In [1]:
pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 71.0 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [78]:
from pathlib import Path
import duckdb
import pandas as pd

BASE_DIR = Path("/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/data/sf0.1")

SNB_SNAPSHOT_DIR = BASE_DIR / "social_network-sf0.1-CsvMergeForeign-StringDateFormatter"
SNB_DYNAMIC_DIR = SNB_SNAPSHOT_DIR / "dynamic"
SNB_STATIC_DIR = SNB_SNAPSHOT_DIR / "static"
SNB_SUBSTITUTION_DIR = BASE_DIR / "substitution_parameters-sf0.1"
SNB_UPDATE_DIR = BASE_DIR / "social_network-sf0.1-numpart-1"

print("Base dir:", BASE_DIR)
print("Snapshot exists:", SNB_SNAPSHOT_DIR.exists())
print("Dynamic exists:", SNB_DYNAMIC_DIR.exists())
print("Static exists:", SNB_STATIC_DIR.exists())
print("Substitution exists:", SNB_SUBSTITUTION_DIR.exists())
print("Update exists:", SNB_UPDATE_DIR.exists())

Base dir: /home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/data/sf0.1
Snapshot exists: True
Dynamic exists: True
Static exists: True
Substitution exists: True
Update exists: True


In [79]:
con = duckdb.connect(database=":memory:")

def count_csv(path_pattern: Path) -> int:
    sql = f"""
    SELECT COUNT(*) AS n
    FROM read_csv(
        '{str(path_pattern)}',
        delim='|',
        header=true,
        all_varchar=true,
        union_by_name=true
    )
    """
    return con.execute(sql).fetchone()[0]

csv_groups = {
    "Person": SNB_DYNAMIC_DIR / "person_*.csv",
    "Forum": SNB_DYNAMIC_DIR / "forum_*.csv",
    "Post": SNB_DYNAMIC_DIR / "post_*.csv",
    "Comment": SNB_DYNAMIC_DIR / "comment_*.csv",
    "Place": SNB_STATIC_DIR / "place_*.csv",
    "Organisation": SNB_STATIC_DIR / "organisation_*.csv",
    "Tag": SNB_STATIC_DIR / "tag_*.csv",
    "TagClass": SNB_STATIC_DIR / "tagclass_*.csv",
    "Person_knows_Person": SNB_DYNAMIC_DIR / "person_knows_person_*.csv",
    "Forum_hasMember_Person": SNB_DYNAMIC_DIR / "forum_hasMember_person_*.csv",
    "Person_likes_Post": SNB_DYNAMIC_DIR / "person_likes_post_*.csv",
    "Person_likes_Comment": SNB_DYNAMIC_DIR / "person_likes_comment_*.csv",
    "Post_hasTag_Tag": SNB_DYNAMIC_DIR / "post_hasTag_tag_*.csv",
    "Comment_hasTag_Tag": SNB_DYNAMIC_DIR / "comment_hasTag_tag_*.csv",
    "Forum_hasTag_Tag": SNB_DYNAMIC_DIR / "forum_hasTag_tag_*.csv",
    "Person_hasInterest_Tag": SNB_DYNAMIC_DIR / "person_hasInterest_tag_*.csv",
    "Person_studyAt_Organisation": SNB_DYNAMIC_DIR / "person_studyAt_organisation_*.csv",
    "Person_workAt_Organisation": SNB_DYNAMIC_DIR / "person_workAt_organisation_*.csv",
    "Person_email_EmailAddress": SNB_DYNAMIC_DIR / "person_email_emailaddress_*.csv",
    "Person_speaks_Language": SNB_DYNAMIC_DIR / "person_speaks_language_*.csv",
}

rows = []

for name, pattern in csv_groups.items():
    try:
        n = count_csv(pattern)
        rows.append({"group": name, "rows": n, "status": "OK"})
    except Exception as e:
        rows.append({
            "group": name,
            "rows": None,
            "status": f"ERROR: {type(e).__name__}: {e}"
        })

counts_df = pd.DataFrame(rows)
counts_df

,group,rows,status
0,Person,171733,OK
1,Forum,184715,OK
2,Post,186819,OK
3,Comment,342346,OK
4,Place,1460,OK
5,Organisation,7955,OK
6,Tag,16080,OK
7,TagClass,71,OK
8,Person_knows_Person,14073,OK
9,Forum_hasMember_Person,123268,OK


In [80]:
# =========================================================
# BLOCK 1A — Inspect LDBC SNB CSV column names
# =========================================================
#
# Goal:
# - Inspect the physical column names of each LDBC SNB CSV group.
# - This is necessary before creating conceptual views.
# - We do this because the framework must know the exact physical fields
#   used for primary keys, foreign keys, dates, and relationship tables.
# =========================================================

from pathlib import Path
import duckdb
import pandas as pd

BASE_DIR = Path("/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/data/sf0.1")

SNB_SNAPSHOT_DIR = BASE_DIR / "social_network-sf0.1-CsvMergeForeign-StringDateFormatter"
SNB_DYNAMIC_DIR = SNB_SNAPSHOT_DIR / "dynamic"
SNB_STATIC_DIR = SNB_SNAPSHOT_DIR / "static"
SNB_SUBSTITUTION_DIR = BASE_DIR / "substitution_parameters-sf0.1"
SNB_UPDATE_DIR = BASE_DIR / "social_network-sf0.1-numpart-1"

con = duckdb.connect(database=":memory:")

def inspect_csv_columns(path_pattern: Path, limit: int = 3) -> pd.DataFrame:
    """
    Read a small sample from a CSV group and return its columns.
    """
    sql = f"""
    SELECT *
    FROM read_csv(
        '{str(path_pattern)}',
        delim='|',
        header=true,
        all_varchar=true,
        union_by_name=true
    )
    LIMIT {limit}
    """
    return con.execute(sql).df()

csv_groups = {
    "Person": SNB_DYNAMIC_DIR / "person_*.csv",
    "Forum": SNB_DYNAMIC_DIR / "forum_*.csv",
    "Post": SNB_DYNAMIC_DIR / "post_*.csv",
    "Comment": SNB_DYNAMIC_DIR / "comment_*.csv",
    "Place": SNB_STATIC_DIR / "place_*.csv",
    "Organisation": SNB_STATIC_DIR / "organisation_*.csv",
    "Tag": SNB_STATIC_DIR / "tag_*.csv",
    "TagClass": SNB_STATIC_DIR / "tagclass_*.csv",

    "Person_knows_Person": SNB_DYNAMIC_DIR / "person_knows_person_*.csv",
    "Forum_hasMember_Person": SNB_DYNAMIC_DIR / "forum_hasMember_person_*.csv",
    "Person_likes_Post": SNB_DYNAMIC_DIR / "person_likes_post_*.csv",
    "Person_likes_Comment": SNB_DYNAMIC_DIR / "person_likes_comment_*.csv",
    "Post_hasTag_Tag": SNB_DYNAMIC_DIR / "post_hasTag_tag_*.csv",
    "Comment_hasTag_Tag": SNB_DYNAMIC_DIR / "comment_hasTag_tag_*.csv",
    "Forum_hasTag_Tag": SNB_DYNAMIC_DIR / "forum_hasTag_tag_*.csv",
    "Person_hasInterest_Tag": SNB_DYNAMIC_DIR / "person_hasInterest_tag_*.csv",
    "Person_studyAt_Organisation": SNB_DYNAMIC_DIR / "person_studyAt_organisation_*.csv",
    "Person_workAt_Organisation": SNB_DYNAMIC_DIR / "person_workAt_organisation_*.csv",
    "Person_email_EmailAddress": SNB_DYNAMIC_DIR / "person_email_emailaddress_*.csv",
    "Person_speaks_Language": SNB_DYNAMIC_DIR / "person_speaks_language_*.csv",
}

column_rows = []

for group_name, pattern in csv_groups.items():
    try:
        sample_df = inspect_csv_columns(pattern)
        column_rows.append({
            "group": group_name,
            "n_columns": len(sample_df.columns),
            "columns": list(sample_df.columns),
            "status": "OK"
        })
    except Exception as e:
        column_rows.append({
            "group": group_name,
            "n_columns": None,
            "columns": None,
            "status": f"ERROR: {type(e).__name__}: {e}"
        })

columns_df = pd.DataFrame(column_rows)
columns_df

,group,n_columns,columns,status
0,Person,19,"[id, firstName, lastName, gender, birthday, creationDate, locationIP, browserUsed, place, Person.id, email, Tag.id, Person.id_1, Comment.id, Post.id, language, Organisation.id, classYear, workFrom]",OK
1,Forum,8,"[id, title, creationDate, moderator, Forum.id, Person.id, joinDate, Tag.id]",OK
2,Post,13,"[id, imageFile, creationDate, locationIP, browserUsed, language, content, length, creator, Forum.id, place, Post.id, Tag.id]",OK
3,Comment,12,"[id, creationDate, locationIP, browserUsed, content, length, creator, place, replyOfPost, replyOfComment, Comment.id, Tag.id]",OK
4,Place,5,"[id, name, url, type, isPartOf]",OK
5,Organisation,5,"[id, type, name, url, place]",OK
6,Tag,4,"[id, name, url, hasType]",OK
7,TagClass,4,"[id, name, url, isSubclassOf]",OK
8,Person_knows_Person,3,"[Person.id, Person.id_1, creationDate]",OK
9,Forum_hasMember_Person,3,"[Forum.id, Person.id, joinDate]",OK


In [81]:
# =========================================================
# BLOCK 1B — Register LDBC SNB raw CSV files as DuckDB views
# =========================================================
#
# Goal:
# - Create one DuckDB raw view per physical CSV group.
# - These raw views are dataset-specific.
# - Later blocks will use conceptual views derived from these raw views.
# =========================================================

from pathlib import Path
import duckdb
import pandas as pd

BASE_DIR = Path("/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/data/sf0.1")

SNB_SNAPSHOT_DIR = BASE_DIR / "social_network-sf0.1-CsvMergeForeign-StringDateFormatter"
SNB_DYNAMIC_DIR = SNB_SNAPSHOT_DIR / "dynamic"
SNB_STATIC_DIR = SNB_SNAPSHOT_DIR / "static"
SNB_SUBSTITUTION_DIR = BASE_DIR / "substitution_parameters-sf0.1"
SNB_UPDATE_DIR = BASE_DIR / "social_network-sf0.1-numpart-1"

con = duckdb.connect(database=":memory:")

def create_csv_view(view_name: str, path_pattern: Path) -> None:
    """
    Create a DuckDB view from one or more LDBC SNB CSV files.
    """
    sql = f"""
    CREATE OR REPLACE VIEW {view_name} AS
    SELECT *
    FROM read_csv(
        '{str(path_pattern)}',
        delim='|',
        header=true,
        all_varchar=true,
        union_by_name=true
    );
    """
    con.execute(sql)

SNB_RAW_VIEW_PATTERNS = {
    # Static entities
    "snb_raw_places": SNB_STATIC_DIR / "place_*.csv",
    "snb_raw_organisations": SNB_STATIC_DIR / "organisation_*.csv",
    "snb_raw_tags": SNB_STATIC_DIR / "tag_*.csv",
    "snb_raw_tagclasses": SNB_STATIC_DIR / "tagclass_*.csv",

    # Dynamic entities
    "snb_raw_persons": SNB_DYNAMIC_DIR / "person_*.csv",
    "snb_raw_forums": SNB_DYNAMIC_DIR / "forum_*.csv",
    "snb_raw_posts": SNB_DYNAMIC_DIR / "post_*.csv",
    "snb_raw_comments": SNB_DYNAMIC_DIR / "comment_*.csv",

    # Dynamic relationships
    "snb_raw_person_knows_person": SNB_DYNAMIC_DIR / "person_knows_person_*.csv",
    "snb_raw_person_hasinterest_tag": SNB_DYNAMIC_DIR / "person_hasInterest_tag_*.csv",
    "snb_raw_person_likes_post": SNB_DYNAMIC_DIR / "person_likes_post_*.csv",
    "snb_raw_person_likes_comment": SNB_DYNAMIC_DIR / "person_likes_comment_*.csv",
    "snb_raw_forum_hasmember_person": SNB_DYNAMIC_DIR / "forum_hasMember_person_*.csv",
    "snb_raw_forum_hastag_tag": SNB_DYNAMIC_DIR / "forum_hasTag_tag_*.csv",
    "snb_raw_post_hastag_tag": SNB_DYNAMIC_DIR / "post_hasTag_tag_*.csv",
    "snb_raw_comment_hastag_tag": SNB_DYNAMIC_DIR / "comment_hasTag_tag_*.csv",
    "snb_raw_person_studyat_organisation": SNB_DYNAMIC_DIR / "person_studyAt_organisation_*.csv",
    "snb_raw_person_workat_organisation": SNB_DYNAMIC_DIR / "person_workAt_organisation_*.csv",
    "snb_raw_person_email_emailaddress": SNB_DYNAMIC_DIR / "person_email_emailaddress_*.csv",
    "snb_raw_person_speaks_language": SNB_DYNAMIC_DIR / "person_speaks_language_*.csv",
}

for view_name, pattern in SNB_RAW_VIEW_PATTERNS.items():
    create_csv_view(view_name, pattern)

# Validate row counts for all raw views.
raw_count_rows = []

for view_name in SNB_RAW_VIEW_PATTERNS:
    try:
        n = con.execute(f"SELECT COUNT(*) FROM {view_name}").fetchone()[0]
        raw_count_rows.append({
            "view_name": view_name,
            "rows": n,
            "status": "OK"
        })
    except Exception as e:
        raw_count_rows.append({
            "view_name": view_name,
            "rows": None,
            "status": f"ERROR: {type(e).__name__}: {e}"
        })

raw_counts_df = pd.DataFrame(raw_count_rows)
raw_counts_df

,view_name,rows,status
0,snb_raw_places,1460,OK
1,snb_raw_organisations,7955,OK
2,snb_raw_tags,16080,OK
3,snb_raw_tagclasses,71,OK
4,snb_raw_persons,171733,OK
5,snb_raw_forums,184715,OK
6,snb_raw_posts,186819,OK
7,snb_raw_comments,342346,OK
8,snb_raw_person_knows_person,14073,OK
9,snb_raw_person_hasinterest_tag,35475,OK


In [82]:
# =========================================================
# BLOCK 1C — Create first LDBC SNB conceptual views
# =========================================================
#
# Goal:
# - Create clean conceptual views from raw CSV views.
# - At this first stage, we keep columns unchanged.
# - Later we can rename or cast columns if needed.
# =========================================================

def create_simple_conceptual_views(con) -> None:
    """
    Create conceptual views for the main LDBC SNB entities and relationships.
    """

    # Main entity views
    con.execute("""
    CREATE OR REPLACE VIEW snb_persons AS
    SELECT * FROM snb_raw_persons
    """)

    con.execute("""
    CREATE OR REPLACE VIEW snb_forums AS
    SELECT * FROM snb_raw_forums
    """)

    con.execute("""
    CREATE OR REPLACE VIEW snb_posts AS
    SELECT * FROM snb_raw_posts
    """)

    con.execute("""
    CREATE OR REPLACE VIEW snb_comments AS
    SELECT * FROM snb_raw_comments
    """)

    con.execute("""
    CREATE OR REPLACE VIEW snb_places AS
    SELECT * FROM snb_raw_places
    """)

    con.execute("""
    CREATE OR REPLACE VIEW snb_organisations AS
    SELECT * FROM snb_raw_organisations
    """)

    con.execute("""
    CREATE OR REPLACE VIEW snb_tags AS
    SELECT * FROM snb_raw_tags
    """)

    con.execute("""
    CREATE OR REPLACE VIEW snb_tagclasses AS
    SELECT * FROM snb_raw_tagclasses
    """)

    # Relationship views
    con.execute("""
    CREATE OR REPLACE VIEW snb_person_knows_person AS
    SELECT * FROM snb_raw_person_knows_person
    """)

    con.execute("""
    CREATE OR REPLACE VIEW snb_person_hasinterest_tag AS
    SELECT * FROM snb_raw_person_hasinterest_tag
    """)

    con.execute("""
    CREATE OR REPLACE VIEW snb_person_likes_post AS
    SELECT * FROM snb_raw_person_likes_post
    """)

    con.execute("""
    CREATE OR REPLACE VIEW snb_person_likes_comment AS
    SELECT * FROM snb_raw_person_likes_comment
    """)

    con.execute("""
    CREATE OR REPLACE VIEW snb_forum_hasmember_person AS
    SELECT * FROM snb_raw_forum_hasmember_person
    """)

    con.execute("""
    CREATE OR REPLACE VIEW snb_forum_hastag_tag AS
    SELECT * FROM snb_raw_forum_hastag_tag
    """)

    con.execute("""
    CREATE OR REPLACE VIEW snb_post_hastag_tag AS
    SELECT * FROM snb_raw_post_hastag_tag
    """)

    con.execute("""
    CREATE OR REPLACE VIEW snb_comment_hastag_tag AS
    SELECT * FROM snb_raw_comment_hastag_tag
    """)

    con.execute("""
    CREATE OR REPLACE VIEW snb_person_studyat_organisation AS
    SELECT * FROM snb_raw_person_studyat_organisation
    """)

    con.execute("""
    CREATE OR REPLACE VIEW snb_person_workat_organisation AS
    SELECT * FROM snb_raw_person_workat_organisation
    """)

    con.execute("""
    CREATE OR REPLACE VIEW snb_person_email_emailaddress AS
    SELECT * FROM snb_raw_person_email_emailaddress
    """)

    con.execute("""
    CREATE OR REPLACE VIEW snb_person_speaks_language AS
    SELECT * FROM snb_raw_person_speaks_language
    """)

create_simple_conceptual_views(con)

conceptual_views = [
    "snb_persons",
    "snb_forums",
    "snb_posts",
    "snb_comments",
    "snb_places",
    "snb_organisations",
    "snb_tags",
    "snb_tagclasses",
    "snb_person_knows_person",
    "snb_forum_hasmember_person",
    "snb_person_likes_post",
    "snb_person_likes_comment",
    "snb_post_hastag_tag",
    "snb_comment_hastag_tag",
    "snb_forum_hastag_tag",
    "snb_person_hasinterest_tag",
    "snb_person_studyat_organisation",
    "snb_person_workat_organisation",
]

conceptual_count_rows = []

for view_name in conceptual_views:
    n = con.execute(f"SELECT COUNT(*) FROM {view_name}").fetchone()[0]
    conceptual_count_rows.append({
        "view_name": view_name,
        "rows": n
    })

conceptual_counts_df = pd.DataFrame(conceptual_count_rows)
conceptual_counts_df

,view_name,rows
0,snb_persons,171733
1,snb_forums,184715
2,snb_posts,186819
3,snb_comments,342346
4,snb_places,1460
5,snb_organisations,7955
6,snb_tags,16080
7,snb_tagclasses,71
8,snb_person_knows_person,14073
9,snb_forum_hasmember_person,123268


In [83]:
# =========================================================
# BLOCK 2A — Print full physical column names
# =========================================================
#
# Goal:
# - Print complete column lists for each LDBC SNB CSV group.
# - This avoids mistakes caused by truncated pandas output.
# - We need exact column names before creating normalized views.
# =========================================================

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

for _, row in columns_df.iterrows():
    print("\n" + "=" * 80)
    print(row["group"])
    print("-" * 80)
    print(row["columns"])


Person
--------------------------------------------------------------------------------
['id', 'firstName', 'lastName', 'gender', 'birthday', 'creationDate', 'locationIP', 'browserUsed', 'place', 'Person.id', 'email', 'Tag.id', 'Person.id_1', 'Comment.id', 'Post.id', 'language', 'Organisation.id', 'classYear', 'workFrom']

Forum
--------------------------------------------------------------------------------
['id', 'title', 'creationDate', 'moderator', 'Forum.id', 'Person.id', 'joinDate', 'Tag.id']

Post
--------------------------------------------------------------------------------
['id', 'imageFile', 'creationDate', 'locationIP', 'browserUsed', 'language', 'content', 'length', 'creator', 'Forum.id', 'place', 'Post.id', 'Tag.id']

Comment
--------------------------------------------------------------------------------
['id', 'creationDate', 'locationIP', 'browserUsed', 'content', 'length', 'creator', 'place', 'replyOfPost', 'replyOfComment', 'Comment.id', 'Tag.id']

Place
----------

In [84]:
# =========================================================
# BLOCK 2B — Create normalized LDBC SNB conceptual views
# =========================================================
#
# Goal:
# - Create normalized conceptual views with clean column names.
# - Avoid physical column names containing dots, such as "Person.id".
# - These views will be used by the framework blocks V0–V24.
# =========================================================

def q(col: str) -> str:
    """
    Quote a DuckDB column name safely.
    Useful for columns such as Person.id or Comment.id.
    """
    return '"' + col.replace('"', '""') + '"'


# ---------------------------------------------------------
# Main entity views
# ---------------------------------------------------------

con.execute("""
CREATE OR REPLACE VIEW snb_persons_norm AS
SELECT
    id AS person_id,
    *
FROM snb_persons
""")

con.execute("""
CREATE OR REPLACE VIEW snb_forums_norm AS
SELECT
    id AS forum_id,
    *
FROM snb_forums
""")

con.execute("""
CREATE OR REPLACE VIEW snb_posts_norm AS
SELECT
    id AS post_id,
    *
FROM snb_posts
""")

con.execute("""
CREATE OR REPLACE VIEW snb_comments_norm AS
SELECT
    id AS comment_id,
    *
FROM snb_comments
""")

con.execute("""
CREATE OR REPLACE VIEW snb_places_norm AS
SELECT
    id AS place_id,
    *
FROM snb_places
""")

con.execute("""
CREATE OR REPLACE VIEW snb_organisations_norm AS
SELECT
    id AS organisation_id,
    *
FROM snb_organisations
""")

con.execute("""
CREATE OR REPLACE VIEW snb_tags_norm AS
SELECT
    id AS tag_id,
    *
FROM snb_tags
""")

con.execute("""
CREATE OR REPLACE VIEW snb_tagclasses_norm AS
SELECT
    id AS tagclass_id,
    *
FROM snb_tagclasses
""")


# ---------------------------------------------------------
# Relationship views
# ---------------------------------------------------------

con.execute(f"""
CREATE OR REPLACE VIEW snb_knows_norm AS
SELECT
    {q("Person.id")} AS person1_id,
    {q("Person.id_1")} AS person2_id,
    creationDate AS creation_date
FROM snb_person_knows_person
""")

con.execute(f"""
CREATE OR REPLACE VIEW snb_forum_members_norm AS
SELECT
    {q("Forum.id")} AS forum_id,
    {q("Person.id")} AS person_id,
    joinDate AS join_date
FROM snb_forum_hasmember_person
""")

con.execute(f"""
CREATE OR REPLACE VIEW snb_person_likes_posts_norm AS
SELECT
    {q("Person.id")} AS person_id,
    {q("Post.id")} AS post_id,
    creationDate AS creation_date
FROM snb_person_likes_post
""")

con.execute(f"""
CREATE OR REPLACE VIEW snb_person_likes_comments_norm AS
SELECT
    {q("Person.id")} AS person_id,
    {q("Comment.id")} AS comment_id,
    creationDate AS creation_date
FROM snb_person_likes_comment
""")

con.execute(f"""
CREATE OR REPLACE VIEW snb_post_tags_norm AS
SELECT
    {q("Post.id")} AS post_id,
    {q("Tag.id")} AS tag_id
FROM snb_post_hastag_tag
""")

con.execute(f"""
CREATE OR REPLACE VIEW snb_comment_tags_norm AS
SELECT
    {q("Comment.id")} AS comment_id,
    {q("Tag.id")} AS tag_id
FROM snb_comment_hastag_tag
""")

con.execute(f"""
CREATE OR REPLACE VIEW snb_forum_tags_norm AS
SELECT
    {q("Forum.id")} AS forum_id,
    {q("Tag.id")} AS tag_id
FROM snb_forum_hastag_tag
""")

con.execute(f"""
CREATE OR REPLACE VIEW snb_person_interests_norm AS
SELECT
    {q("Person.id")} AS person_id,
    {q("Tag.id")} AS tag_id
FROM snb_person_hasinterest_tag
""")

con.execute(f"""
CREATE OR REPLACE VIEW snb_person_studyat_norm AS
SELECT
    {q("Person.id")} AS person_id,
    {q("Organisation.id")} AS organisation_id,
    classYear AS class_year
FROM snb_person_studyat_organisation
""")

con.execute(f"""
CREATE OR REPLACE VIEW snb_person_workat_norm AS
SELECT
    {q("Person.id")} AS person_id,
    {q("Organisation.id")} AS organisation_id,
    workFrom AS work_from
FROM snb_person_workat_organisation
""")

con.execute(f"""
CREATE OR REPLACE VIEW snb_person_emails_norm AS
SELECT
    {q("Person.id")} AS person_id,
    email AS email
FROM snb_person_email_emailaddress
""")

con.execute(f"""
CREATE OR REPLACE VIEW snb_person_languages_norm AS
SELECT
    {q("Person.id")} AS person_id,
    language AS language
FROM snb_person_speaks_language
""")

In [85]:
# =========================================================
# BLOCK 2C — Validate normalized LDBC SNB conceptual views
# =========================================================

normalized_views = [
    "snb_persons_norm",
    "snb_forums_norm",
    "snb_posts_norm",
    "snb_comments_norm",
    "snb_places_norm",
    "snb_organisations_norm",
    "snb_tags_norm",
    "snb_tagclasses_norm",

    "snb_knows_norm",
    "snb_forum_members_norm",
    "snb_person_likes_posts_norm",
    "snb_person_likes_comments_norm",
    "snb_post_tags_norm",
    "snb_comment_tags_norm",
    "snb_forum_tags_norm",
    "snb_person_interests_norm",
    "snb_person_studyat_norm",
    "snb_person_workat_norm",
    "snb_person_emails_norm",
    "snb_person_languages_norm",
]

normalized_count_rows = []

for view_name in normalized_views:
    try:
        n = con.execute(f"SELECT COUNT(*) FROM {view_name}").fetchone()[0]
        sample_cols = con.execute(f"SELECT * FROM {view_name} LIMIT 1").df().columns.tolist()
        normalized_count_rows.append({
            "view_name": view_name,
            "rows": n,
            "columns": sample_cols,
            "status": "OK"
        })
    except Exception as e:
        normalized_count_rows.append({
            "view_name": view_name,
            "rows": None,
            "columns": None,
            "status": f"ERROR: {type(e).__name__}: {e}"
        })

normalized_counts_df = pd.DataFrame(normalized_count_rows)
normalized_counts_df

,view_name,rows,columns,status
0,snb_persons_norm,171733,"[person_id, id, firstName, lastName, gender, birthday, creationDate, locationIP, browserUsed, place, Person.id, email, Tag.id, Person.id_1, Comment.id, Post.id, language, Organisation.id, classYear, workFrom]",OK
1,snb_forums_norm,184715,"[forum_id, id, title, creationDate, moderator, Forum.id, Person.id, joinDate, Tag.id]",OK
2,snb_posts_norm,186819,"[post_id, id, imageFile, creationDate, locationIP, browserUsed, language, content, length, creator, Forum.id, place, Post.id, Tag.id]",OK
3,snb_comments_norm,342346,"[comment_id, id, creationDate, locationIP, browserUsed, content, length, creator, place, replyOfPost, replyOfComment, Comment.id, Tag.id]",OK
4,snb_places_norm,1460,"[place_id, id, name, url, type, isPartOf]",OK
5,snb_organisations_norm,7955,"[organisation_id, id, type, name, url, place]",OK
6,snb_tags_norm,16080,"[tag_id, id, name, url, hasType]",OK
7,snb_tagclasses_norm,71,"[tagclass_id, id, name, url, isSubclassOf]",OK
8,snb_knows_norm,14073,"[person1_id, person2_id, creation_date]",OK
9,snb_forum_members_norm,123268,"[forum_id, person_id, join_date]",OK


In [86]:
# =========================================================
# BLOCK 2D — LDBC SNB entity maps for the framework
# =========================================================
#
# Goal:
# - Define dataset-specific metadata.
# - These maps replace the FIBEN-specific metadata.
# - The core methodology will use these maps generically.
# =========================================================

SNB_ENTITY_VIEW_MAP = {
    "Person": "snb_persons_norm",
    "Forum": "snb_forums_norm",
    "Post": "snb_posts_norm",
    "Comment": "snb_comments_norm",
    "Place": "snb_places_norm",
    "Organisation": "snb_organisations_norm",
    "Tag": "snb_tags_norm",
    "TagClass": "snb_tagclasses_norm",
}

SNB_ENTITY_PK_MAP = {
    "Person": "person_id",
    "Forum": "forum_id",
    "Post": "post_id",
    "Comment": "comment_id",
    "Place": "place_id",
    "Organisation": "organisation_id",
    "Tag": "tag_id",
    "TagClass": "tagclass_id",
}

SNB_COLLECTION_NAME_BY_ENTITY = {
    "Person": "persons",
    "Forum": "forums",
    "Post": "posts",
    "Comment": "comments",
    "Place": "places",
    "Organisation": "organisations",
    "Tag": "tags",
    "TagClass": "tagclasses",
}

SNB_RELATIONSHIP_VIEW_MAP = {
    "person_knows_person": "snb_knows_norm",
    "forum_has_member_person": "snb_forum_members_norm",
    "person_likes_post": "snb_person_likes_posts_norm",
    "person_likes_comment": "snb_person_likes_comments_norm",
    "post_has_tag": "snb_post_tags_norm",
    "comment_has_tag": "snb_comment_tags_norm",
    "forum_has_tag": "snb_forum_tags_norm",
    "person_has_interest_tag": "snb_person_interests_norm",
    "person_study_at_organisation": "snb_person_studyat_norm",
    "person_work_at_organisation": "snb_person_workat_norm",
    "person_has_email": "snb_person_emails_norm",
    "person_speaks_language": "snb_person_languages_norm",
}

snb_entity_metadata_df = pd.DataFrame([
    {
        "entity": entity,
        "view_name": SNB_ENTITY_VIEW_MAP[entity],
        "primary_key": SNB_ENTITY_PK_MAP[entity],
        "collection_name": SNB_COLLECTION_NAME_BY_ENTITY[entity],
    }
    for entity in SNB_ENTITY_VIEW_MAP
])

snb_entity_metadata_df

,entity,view_name,primary_key,collection_name
0,Person,snb_persons_norm,person_id,persons
1,Forum,snb_forums_norm,forum_id,forums
2,Post,snb_posts_norm,post_id,posts
3,Comment,snb_comments_norm,comment_id,comments
4,Place,snb_places_norm,place_id,places
5,Organisation,snb_organisations_norm,organisation_id,organisations
6,Tag,snb_tags_norm,tag_id,tags
7,TagClass,snb_tagclasses_norm,tagclass_id,tagclasses


In [87]:
# =========================================================
# BLOCK 2E — Initial LDBC SNB relationship hints
# =========================================================
#
# Goal:
# - Represent the conceptual relationships of the LDBC SNB schema.
# - These hints will support root selection, traversal depth, Rc, D, Re,
#   semantic profile, sharedness and candidate materialization.
# =========================================================

SNB_RELATIONSHIP_HINTS = {
    # -----------------------------------------------------
    # Person-centered relationships
    # -----------------------------------------------------
    "person_knows_person": {
        "semantic_type": "association",
        "source_entity": "Person",
        "target_entity": "Person",
        "view_name": "snb_knows_norm",
        "source_column": "person1_id",
        "target_column": "person2_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": True,
    },

    "person_has_interest_tag": {
        "semantic_type": "association",
        "source_entity": "Person",
        "target_entity": "Tag",
        "view_name": "snb_person_interests_norm",
        "source_column": "person_id",
        "target_column": "tag_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": False,
    },

    "person_likes_post": {
        "semantic_type": "associative",
        "source_entity": "Person",
        "target_entity": "Post",
        "view_name": "snb_person_likes_posts_norm",
        "source_column": "person_id",
        "target_column": "post_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": False,
    },

    "person_likes_comment": {
        "semantic_type": "associative",
        "source_entity": "Person",
        "target_entity": "Comment",
        "view_name": "snb_person_likes_comments_norm",
        "source_column": "person_id",
        "target_column": "comment_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": False,
    },

    "person_study_at_organisation": {
        "semantic_type": "association",
        "source_entity": "Person",
        "target_entity": "Organisation",
        "view_name": "snb_person_studyat_norm",
        "source_column": "person_id",
        "target_column": "organisation_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": False,
    },

    "person_work_at_organisation": {
        "semantic_type": "association",
        "source_entity": "Person",
        "target_entity": "Organisation",
        "view_name": "snb_person_workat_norm",
        "source_column": "person_id",
        "target_column": "organisation_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": False,
    },

    # -----------------------------------------------------
    # Forum-centered relationships
    # -----------------------------------------------------
    "forum_has_member_person": {
        "semantic_type": "associative",
        "source_entity": "Forum",
        "target_entity": "Person",
        "view_name": "snb_forum_members_norm",
        "source_column": "forum_id",
        "target_column": "person_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": False,
    },

    "forum_has_tag": {
        "semantic_type": "association",
        "source_entity": "Forum",
        "target_entity": "Tag",
        "view_name": "snb_forum_tags_norm",
        "source_column": "forum_id",
        "target_column": "tag_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": False,
    },

    # -----------------------------------------------------
    # Message-centered relationships
    # -----------------------------------------------------
    "post_has_tag": {
        "semantic_type": "association",
        "source_entity": "Post",
        "target_entity": "Tag",
        "view_name": "snb_post_tags_norm",
        "source_column": "post_id",
        "target_column": "tag_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": False,
    },

    "comment_has_tag": {
        "semantic_type": "association",
        "source_entity": "Comment",
        "target_entity": "Tag",
        "view_name": "snb_comment_tags_norm",
        "source_column": "comment_id",
        "target_column": "tag_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": False,
    },

    # -----------------------------------------------------
    # Attribute-like relationships
    # These can be treated as weak/attribute relationships.
    # They are useful for benchmark loading but may not dominate
    # the conceptual activation logic.
    # -----------------------------------------------------
    "person_has_email": {
        "semantic_type": "attribute",
        "source_entity": "Person",
        "target_entity": "EmailAddress",
        "view_name": "snb_person_emails_norm",
        "source_column": "person_id",
        "target_column": "email",
        "relationship_kind": "one_to_many",
        "is_bidirectional": False,
    },

    "person_speaks_language": {
        "semantic_type": "attribute",
        "source_entity": "Person",
        "target_entity": "Language",
        "view_name": "snb_person_languages_norm",
        "source_column": "person_id",
        "target_column": "language",
        "relationship_kind": "one_to_many",
        "is_bidirectional": False,
    },
}

snb_relationship_hints_df = pd.DataFrame([
    {
        "relationship": name,
        "semantic_type": spec["semantic_type"],
        "source_entity": spec["source_entity"],
        "target_entity": spec["target_entity"],
        "view_name": spec["view_name"],
        "source_column": spec["source_column"],
        "target_column": spec["target_column"],
        "relationship_kind": spec["relationship_kind"],
        "is_bidirectional": spec["is_bidirectional"],
    }
    for name, spec in SNB_RELATIONSHIP_HINTS.items()
])

snb_relationship_hints_df

,relationship,semantic_type,source_entity,target_entity,view_name,source_column,target_column,relationship_kind,is_bidirectional
0,person_knows_person,association,Person,Person,snb_knows_norm,person1_id,person2_id,many_to_many,True
1,person_has_interest_tag,association,Person,Tag,snb_person_interests_norm,person_id,tag_id,many_to_many,False
2,person_likes_post,associative,Person,Post,snb_person_likes_posts_norm,person_id,post_id,many_to_many,False
3,person_likes_comment,associative,Person,Comment,snb_person_likes_comments_norm,person_id,comment_id,many_to_many,False
4,person_study_at_organisation,association,Person,Organisation,snb_person_studyat_norm,person_id,organisation_id,many_to_many,False
5,person_work_at_organisation,association,Person,Organisation,snb_person_workat_norm,person_id,organisation_id,many_to_many,False
6,forum_has_member_person,associative,Forum,Person,snb_forum_members_norm,forum_id,person_id,many_to_many,False
7,forum_has_tag,association,Forum,Tag,snb_forum_tags_norm,forum_id,tag_id,many_to_many,False
8,post_has_tag,association,Post,Tag,snb_post_tags_norm,post_id,tag_id,many_to_many,False
9,comment_has_tag,association,Comment,Tag,snb_comment_tags_norm,comment_id,tag_id,many_to_many,False


In [88]:
# =========================================================
# BLOCK 2F — Create clean LDBC SNB entity views
# =========================================================
#
# Goal:
# - Build clean entity views from CsvMergeForeign files.
# - CsvMergeForeign may repeat entity rows because some relationships are merged.
# - Therefore, we use SELECT DISTINCT on entity-level attributes.
# - These clean views will be used by the framework.
# =========================================================

def q(col: str) -> str:
    """
    Quote a DuckDB column name safely.
    Useful for columns such as Person.id or Forum.id.
    """
    return '"' + col.replace('"', '""') + '"'


# ---------------------------------------------------------
# Clean entity views
# ---------------------------------------------------------

con.execute("""
CREATE OR REPLACE VIEW snb_persons_entity AS
SELECT DISTINCT
    id AS person_id,
    firstName AS first_name,
    lastName AS last_name,
    gender,
    birthday,
    creationDate AS creation_date,
    locationIP AS location_ip,
    browserUsed AS browser_used,
    place AS place_id
FROM snb_persons
""")

con.execute("""
CREATE OR REPLACE VIEW snb_forums_entity AS
SELECT DISTINCT
    id AS forum_id,
    title,
    creationDate AS creation_date,
    moderator AS moderator_person_id
FROM snb_forums
""")

con.execute(f"""
CREATE OR REPLACE VIEW snb_posts_entity AS
SELECT DISTINCT
    id AS post_id,
    imageFile AS image_file,
    creationDate AS creation_date,
    locationIP AS location_ip,
    browserUsed AS browser_used,
    language,
    content,
    length,
    creator AS creator_person_id,
    {q("Forum.id")} AS forum_id,
    place AS place_id
FROM snb_posts
""")

con.execute("""
CREATE OR REPLACE VIEW snb_comments_entity AS
SELECT DISTINCT
    id AS comment_id,
    creationDate AS creation_date,
    locationIP AS location_ip,
    browserUsed AS browser_used,
    content,
    length,
    creator AS creator_person_id,
    place AS place_id,
    replyOfPost AS reply_post_id,
    replyOfComment AS reply_comment_id
FROM snb_comments
""")

con.execute("""
CREATE OR REPLACE VIEW snb_places_entity AS
SELECT DISTINCT
    id AS place_id,
    name,
    url,
    type,
    isPartOf AS parent_place_id
FROM snb_places
""")

con.execute("""
CREATE OR REPLACE VIEW snb_organisations_entity AS
SELECT DISTINCT
    id AS organisation_id,
    type,
    name,
    url,
    place AS place_id
FROM snb_organisations
""")

con.execute("""
CREATE OR REPLACE VIEW snb_tags_entity AS
SELECT DISTINCT
    id AS tag_id,
    name,
    url,
    hasType AS tagclass_id
FROM snb_tags
""")

con.execute("""
CREATE OR REPLACE VIEW snb_tagclasses_entity AS
SELECT DISTINCT
    id AS tagclass_id,
    name,
    url,
    isSubclassOf AS parent_tagclass_id
FROM snb_tagclasses
""")


# ---------------------------------------------------------
# Validate clean entity counts
# ---------------------------------------------------------

clean_entity_views = [
    "snb_persons_entity",
    "snb_forums_entity",
    "snb_posts_entity",
    "snb_comments_entity",
    "snb_places_entity",
    "snb_organisations_entity",
    "snb_tags_entity",
    "snb_tagclasses_entity",
]

clean_entity_count_rows = []

for view_name in clean_entity_views:
    n = con.execute(f"SELECT COUNT(*) FROM {view_name}").fetchone()[0]
    cols = con.execute(f"SELECT * FROM {view_name} LIMIT 1").df().columns.tolist()
    clean_entity_count_rows.append({
        "view_name": view_name,
        "rows": n,
        "columns": cols
    })

clean_entity_counts_df = pd.DataFrame(clean_entity_count_rows)
clean_entity_counts_df

,view_name,rows,columns
0,snb_persons_entity,125038,"[person_id, first_name, last_name, gender, birthday, creation_date, location_ip, browser_used, place_id]"
1,snb_forums_entity,13751,"[forum_id, title, creation_date, moderator_person_id]"
2,snb_posts_entity,135702,"[post_id, image_file, creation_date, location_ip, browser_used, language, content, length, creator_person_id, forum_id, place_id]"
3,snb_comments_entity,151044,"[comment_id, creation_date, location_ip, browser_used, content, length, creator_person_id, place_id, reply_post_id, reply_comment_id]"
4,snb_places_entity,1460,"[place_id, name, url, type, parent_place_id]"
5,snb_organisations_entity,7955,"[organisation_id, type, name, url, place_id]"
6,snb_tags_entity,16080,"[tag_id, name, url, tagclass_id]"
7,snb_tagclasses_entity,71,"[tagclass_id, name, url, parent_tagclass_id]"


In [89]:
# =========================================================
# BLOCK 2G — Update LDBC SNB entity maps using clean views
# =========================================================
#
# Goal:
# - Use clean entity views in the framework.
# - These maps are the LDBC SNB replacement for the FIBEN entity maps.
# =========================================================

SNB_ENTITY_VIEW_MAP = {
    "Person": "snb_persons_entity",
    "Forum": "snb_forums_entity",
    "Post": "snb_posts_entity",
    "Comment": "snb_comments_entity",
    "Place": "snb_places_entity",
    "Organisation": "snb_organisations_entity",
    "Tag": "snb_tags_entity",
    "TagClass": "snb_tagclasses_entity",
}

SNB_ENTITY_PK_MAP = {
    "Person": "person_id",
    "Forum": "forum_id",
    "Post": "post_id",
    "Comment": "comment_id",
    "Place": "place_id",
    "Organisation": "organisation_id",
    "Tag": "tag_id",
    "TagClass": "tagclass_id",
}

SNB_COLLECTION_NAME_BY_ENTITY = {
    "Person": "persons",
    "Forum": "forums",
    "Post": "posts",
    "Comment": "comments",
    "Place": "places",
    "Organisation": "organisations",
    "Tag": "tags",
    "TagClass": "tagclasses",
}

snb_entity_metadata_df = pd.DataFrame([
    {
        "entity": entity,
        "view_name": SNB_ENTITY_VIEW_MAP[entity],
        "primary_key": SNB_ENTITY_PK_MAP[entity],
        "collection_name": SNB_COLLECTION_NAME_BY_ENTITY[entity],
    }
    for entity in SNB_ENTITY_VIEW_MAP
])

snb_entity_metadata_df

,entity,view_name,primary_key,collection_name
0,Person,snb_persons_entity,person_id,persons
1,Forum,snb_forums_entity,forum_id,forums
2,Post,snb_posts_entity,post_id,posts
3,Comment,snb_comments_entity,comment_id,comments
4,Place,snb_places_entity,place_id,places
5,Organisation,snb_organisations_entity,organisation_id,organisations
6,Tag,snb_tags_entity,tag_id,tags
7,TagClass,snb_tagclasses_entity,tagclass_id,tagclasses


In [90]:
# =========================================================
# BLOCK 2G — Update LDBC SNB entity maps using clean views
# =========================================================
#
# Goal:
# - Use clean entity views in the framework.
# - These maps are the LDBC SNB replacement for the FIBEN entity maps.
# =========================================================

SNB_ENTITY_VIEW_MAP = {
    "Person": "snb_persons_entity",
    "Forum": "snb_forums_entity",
    "Post": "snb_posts_entity",
    "Comment": "snb_comments_entity",
    "Place": "snb_places_entity",
    "Organisation": "snb_organisations_entity",
    "Tag": "snb_tags_entity",
    "TagClass": "snb_tagclasses_entity",
}

SNB_ENTITY_PK_MAP = {
    "Person": "person_id",
    "Forum": "forum_id",
    "Post": "post_id",
    "Comment": "comment_id",
    "Place": "place_id",
    "Organisation": "organisation_id",
    "Tag": "tag_id",
    "TagClass": "tagclass_id",
}

SNB_COLLECTION_NAME_BY_ENTITY = {
    "Person": "persons",
    "Forum": "forums",
    "Post": "posts",
    "Comment": "comments",
    "Place": "places",
    "Organisation": "organisations",
    "Tag": "tags",
    "TagClass": "tagclasses",
}

snb_entity_metadata_df = pd.DataFrame([
    {
        "entity": entity,
        "view_name": SNB_ENTITY_VIEW_MAP[entity],
        "primary_key": SNB_ENTITY_PK_MAP[entity],
        "collection_name": SNB_COLLECTION_NAME_BY_ENTITY[entity],
    }
    for entity in SNB_ENTITY_VIEW_MAP
])

snb_entity_metadata_df

,entity,view_name,primary_key,collection_name
0,Person,snb_persons_entity,person_id,persons
1,Forum,snb_forums_entity,forum_id,forums
2,Post,snb_posts_entity,post_id,posts
3,Comment,snb_comments_entity,comment_id,comments
4,Place,snb_places_entity,place_id,places
5,Organisation,snb_organisations_entity,organisation_id,organisations
6,Tag,snb_tags_entity,tag_id,tags
7,TagClass,snb_tagclasses_entity,tagclass_id,tagclasses


In [91]:
# =========================================================
# BLOCK 2I — Complete initial LDBC SNB relationship hints
# =========================================================
#
# Goal:
# - Define conceptual relationships used by the framework.
# - Include explicit relationship CSVs and FK-based relationships.
# - Exclude purely attribute-like values from the core schema.
# =========================================================

SNB_RELATIONSHIP_HINTS = {
    # -----------------------------------------------------
    # Person-centered relationships
    # -----------------------------------------------------
    "person_knows_person": {
        "semantic_type": "association",
        "source_entity": "Person",
        "target_entity": "Person",
        "view_name": "snb_knows_norm",
        "source_column": "person1_id",
        "target_column": "person2_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": True,
    },

    "person_is_located_in_place": {
        "semantic_type": "association",
        "source_entity": "Person",
        "target_entity": "Place",
        "view_name": "snb_person_islocatedin_place",
        "source_column": "person_id",
        "target_column": "place_id",
        "relationship_kind": "many_to_one",
        "is_bidirectional": False,
    },

    "person_has_interest_tag": {
        "semantic_type": "association",
        "source_entity": "Person",
        "target_entity": "Tag",
        "view_name": "snb_person_interests_norm",
        "source_column": "person_id",
        "target_column": "tag_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": False,
    },

    "person_likes_post": {
        "semantic_type": "associative",
        "source_entity": "Person",
        "target_entity": "Post",
        "view_name": "snb_person_likes_posts_norm",
        "source_column": "person_id",
        "target_column": "post_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": False,
    },

    "person_likes_comment": {
        "semantic_type": "associative",
        "source_entity": "Person",
        "target_entity": "Comment",
        "view_name": "snb_person_likes_comments_norm",
        "source_column": "person_id",
        "target_column": "comment_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": False,
    },

    "person_study_at_organisation": {
        "semantic_type": "association",
        "source_entity": "Person",
        "target_entity": "Organisation",
        "view_name": "snb_person_studyat_norm",
        "source_column": "person_id",
        "target_column": "organisation_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": False,
    },

    "person_work_at_organisation": {
        "semantic_type": "association",
        "source_entity": "Person",
        "target_entity": "Organisation",
        "view_name": "snb_person_workat_norm",
        "source_column": "person_id",
        "target_column": "organisation_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": False,
    },

    # -----------------------------------------------------
    # Forum-centered relationships
    # -----------------------------------------------------
    "forum_has_moderator_person": {
        "semantic_type": "association",
        "source_entity": "Forum",
        "target_entity": "Person",
        "view_name": "snb_forum_hasmoderator_person",
        "source_column": "forum_id",
        "target_column": "person_id",
        "relationship_kind": "many_to_one",
        "is_bidirectional": False,
    },

    "forum_has_member_person": {
        "semantic_type": "associative",
        "source_entity": "Forum",
        "target_entity": "Person",
        "view_name": "snb_forum_members_norm",
        "source_column": "forum_id",
        "target_column": "person_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": False,
    },

    "forum_has_tag": {
        "semantic_type": "association",
        "source_entity": "Forum",
        "target_entity": "Tag",
        "view_name": "snb_forum_tags_norm",
        "source_column": "forum_id",
        "target_column": "tag_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": False,
    },

    "forum_container_of_post": {
        "semantic_type": "containment",
        "source_entity": "Forum",
        "target_entity": "Post",
        "view_name": "snb_forum_containerof_post",
        "source_column": "forum_id",
        "target_column": "post_id",
        "relationship_kind": "one_to_many",
        "is_bidirectional": False,
    },

    # -----------------------------------------------------
    # Post-centered relationships
    # -----------------------------------------------------
    "post_has_creator_person": {
        "semantic_type": "association",
        "source_entity": "Post",
        "target_entity": "Person",
        "view_name": "snb_post_hascreator_person",
        "source_column": "post_id",
        "target_column": "person_id",
        "relationship_kind": "many_to_one",
        "is_bidirectional": False,
    },

    "post_is_located_in_place": {
        "semantic_type": "association",
        "source_entity": "Post",
        "target_entity": "Place",
        "view_name": "snb_post_islocatedin_place",
        "source_column": "post_id",
        "target_column": "place_id",
        "relationship_kind": "many_to_one",
        "is_bidirectional": False,
    },

    "post_has_tag": {
        "semantic_type": "association",
        "source_entity": "Post",
        "target_entity": "Tag",
        "view_name": "snb_post_tags_norm",
        "source_column": "post_id",
        "target_column": "tag_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": False,
    },

    # -----------------------------------------------------
    # Comment-centered relationships
    # -----------------------------------------------------
    "comment_has_creator_person": {
        "semantic_type": "association",
        "source_entity": "Comment",
        "target_entity": "Person",
        "view_name": "snb_comment_hascreator_person",
        "source_column": "comment_id",
        "target_column": "person_id",
        "relationship_kind": "many_to_one",
        "is_bidirectional": False,
    },

    "comment_is_located_in_place": {
        "semantic_type": "association",
        "source_entity": "Comment",
        "target_entity": "Place",
        "view_name": "snb_comment_islocatedin_place",
        "source_column": "comment_id",
        "target_column": "place_id",
        "relationship_kind": "many_to_one",
        "is_bidirectional": False,
    },

    "comment_has_tag": {
        "semantic_type": "association",
        "source_entity": "Comment",
        "target_entity": "Tag",
        "view_name": "snb_comment_tags_norm",
        "source_column": "comment_id",
        "target_column": "tag_id",
        "relationship_kind": "many_to_many",
        "is_bidirectional": False,
    },

    "comment_reply_of_post": {
        "semantic_type": "containment",
        "source_entity": "Post",
        "target_entity": "Comment",
        "view_name": "snb_comment_replyof_post",
        "source_column": "post_id",
        "target_column": "comment_id",
        "relationship_kind": "one_to_many",
        "is_bidirectional": False,
    },

    "comment_reply_of_comment": {
        "semantic_type": "containment",
        "source_entity": "Comment",
        "target_entity": "Comment",
        "view_name": "snb_comment_replyof_comment",
        "source_column": "parent_comment_id",
        "target_column": "comment_id",
        "relationship_kind": "one_to_many",
        "is_bidirectional": False,
    },

    # -----------------------------------------------------
    # Static hierarchy relationships
    # -----------------------------------------------------
    "organisation_is_located_in_place": {
        "semantic_type": "association",
        "source_entity": "Organisation",
        "target_entity": "Place",
        "view_name": "snb_organisation_islocatedin_place",
        "source_column": "organisation_id",
        "target_column": "place_id",
        "relationship_kind": "many_to_one",
        "is_bidirectional": False,
    },

    "tag_has_type_tagclass": {
        "semantic_type": "association",
        "source_entity": "Tag",
        "target_entity": "TagClass",
        "view_name": "snb_tag_hastype_tagclass",
        "source_column": "tag_id",
        "target_column": "tagclass_id",
        "relationship_kind": "many_to_one",
        "is_bidirectional": False,
    },

    "tagclass_is_subclass_of_tagclass": {
        "semantic_type": "association",
        "source_entity": "TagClass",
        "target_entity": "TagClass",
        "view_name": "snb_tagclass_issubclassof_tagclass",
        "source_column": "tagclass_id",
        "target_column": "parent_tagclass_id",
        "relationship_kind": "many_to_one",
        "is_bidirectional": False,
    },

    "place_is_part_of_place": {
        "semantic_type": "containment",
        "source_entity": "Place",
        "target_entity": "Place",
        "view_name": "snb_place_ispartof_place",
        "source_column": "parent_place_id",
        "target_column": "place_id",
        "relationship_kind": "one_to_many",
        "is_bidirectional": False,
    },
}

snb_relationship_hints_df = pd.DataFrame([
    {
        "relationship": name,
        "semantic_type": spec["semantic_type"],
        "source_entity": spec["source_entity"],
        "target_entity": spec["target_entity"],
        "view_name": spec["view_name"],
        "source_column": spec["source_column"],
        "target_column": spec["target_column"],
        "relationship_kind": spec["relationship_kind"],
        "is_bidirectional": spec["is_bidirectional"],
    }
    for name, spec in SNB_RELATIONSHIP_HINTS.items()
])

snb_relationship_hints_df

,relationship,semantic_type,source_entity,target_entity,view_name,source_column,target_column,relationship_kind,is_bidirectional
0,person_knows_person,association,Person,Person,snb_knows_norm,person1_id,person2_id,many_to_many,True
1,person_is_located_in_place,association,Person,Place,snb_person_islocatedin_place,person_id,place_id,many_to_one,False
2,person_has_interest_tag,association,Person,Tag,snb_person_interests_norm,person_id,tag_id,many_to_many,False
3,person_likes_post,associative,Person,Post,snb_person_likes_posts_norm,person_id,post_id,many_to_many,False
4,person_likes_comment,associative,Person,Comment,snb_person_likes_comments_norm,person_id,comment_id,many_to_many,False
5,person_study_at_organisation,association,Person,Organisation,snb_person_studyat_norm,person_id,organisation_id,many_to_many,False
6,person_work_at_organisation,association,Person,Organisation,snb_person_workat_norm,person_id,organisation_id,many_to_many,False
7,forum_has_moderator_person,association,Forum,Person,snb_forum_hasmoderator_person,forum_id,person_id,many_to_one,False
8,forum_has_member_person,associative,Forum,Person,snb_forum_members_norm,forum_id,person_id,many_to_many,False
9,forum_has_tag,association,Forum,Tag,snb_forum_tags_norm,forum_id,tag_id,many_to_many,False


In [92]:
# =========================================================
# BLOCK 3 — Validate LDBC SNB relationship joins
# =========================================================
#
# Goal:
# - Validate that each relationship view joins correctly to its source
#   and target entity views.
# - This prevents the framework from computing metrics over broken paths.
# =========================================================

def validate_relationship_join(rel_name: str, spec: dict) -> dict:
    view_name = spec["view_name"]
    source_entity = spec["source_entity"]
    target_entity = spec["target_entity"]
    source_column = spec["source_column"]
    target_column = spec["target_column"]

    source_view = SNB_ENTITY_VIEW_MAP[source_entity]
    target_view = SNB_ENTITY_VIEW_MAP[target_entity]

    source_pk = SNB_ENTITY_PK_MAP[source_entity]
    target_pk = SNB_ENTITY_PK_MAP[target_entity]

    sql = f"""
    SELECT
        COUNT(*) AS relationship_rows,

        SUM(CASE WHEN s.{source_pk} IS NOT NULL THEN 1 ELSE 0 END) AS source_matches,
        SUM(CASE WHEN t.{target_pk} IS NOT NULL THEN 1 ELSE 0 END) AS target_matches

    FROM {view_name} r
    LEFT JOIN {source_view} s
      ON r.{source_column} = s.{source_pk}
    LEFT JOIN {target_view} t
      ON r.{target_column} = t.{target_pk}
    """

    row = con.execute(sql).df().iloc[0].to_dict()

    relationship_rows = int(row["relationship_rows"] or 0)
    source_matches = int(row["source_matches"] or 0)
    target_matches = int(row["target_matches"] or 0)

    return {
        "relationship": rel_name,
        "view_name": view_name,
        "source_entity": source_entity,
        "target_entity": target_entity,
        "relationship_rows": relationship_rows,
        "source_matches": source_matches,
        "target_matches": target_matches,
        "source_match_ratio": source_matches / relationship_rows if relationship_rows else None,
        "target_match_ratio": target_matches / relationship_rows if relationship_rows else None,
    }


join_validation_rows = []

for rel_name, spec in SNB_RELATIONSHIP_HINTS.items():
    try:
        join_validation_rows.append(validate_relationship_join(rel_name, spec))
    except Exception as e:
        join_validation_rows.append({
            "relationship": rel_name,
            "view_name": spec.get("view_name"),
            "source_entity": spec.get("source_entity"),
            "target_entity": spec.get("target_entity"),
            "relationship_rows": None,
            "source_matches": None,
            "target_matches": None,
            "source_match_ratio": None,
            "target_match_ratio": None,
            "error": f"{type(e).__name__}: {e}",
        })

join_validation_df = pd.DataFrame(join_validation_rows)

# Put problematic relationships first.
join_validation_df = join_validation_df.sort_values(
    by=["source_match_ratio", "target_match_ratio"],
    ascending=True,
    na_position="first"
)

join_validation_df

,relationship,view_name,source_entity,target_entity,relationship_rows,source_matches,target_matches,source_match_ratio,target_match_ratio,error
1,person_is_located_in_place,snb_person_islocatedin_place,Person,Place,NaN,NaN,NaN,NaN,NaN,"CatalogException: Catalog Error: Table with name snb_person_islocatedin_place does not exist!\nDid you mean ""snb_persons""?\n\nLINE 8: FROM snb_person_islocatedin_place r\n ^"
7,forum_has_moderator_person,snb_forum_hasmoderator_person,Forum,Person,NaN,NaN,NaN,NaN,NaN,"CatalogException: Catalog Error: Table with name snb_forum_hasmoderator_person does not exist!\nDid you mean ""snb_forum_hasmember_person""?\n\nLINE 8: FROM snb_forum_hasmoderator_person r\n ^"
10,forum_container_of_post,snb_forum_containerof_post,Forum,Post,NaN,NaN,NaN,NaN,NaN,"CatalogException: Catalog Error: Table with name snb_forum_containerof_post does not exist!\nDid you mean ""snb_forum_tags_norm""?\n\nLINE 8: FROM snb_forum_containerof_post r\n ^"
11,post_has_creator_person,snb_post_hascreator_person,Post,Person,NaN,NaN,NaN,NaN,NaN,"CatalogException: Catalog Error: Table with name snb_post_hascreator_person does not exist!\nDid you mean ""snb_posts""?\n\nLINE 8: FROM snb_post_hascreator_person r\n ^"
12,post_is_located_in_place,snb_post_islocatedin_place,Post,Place,NaN,NaN,NaN,NaN,NaN,"CatalogException: Catalog Error: Table with name snb_post_islocatedin_place does not exist!\nDid you mean ""snb_posts""?\n\nLINE 8: FROM snb_post_islocatedin_place r\n ^"
14,comment_has_creator_person,snb_comment_hascreator_person,Comment,Person,NaN,NaN,NaN,NaN,NaN,"CatalogException: Catalog Error: Table with name snb_comment_hascreator_person does not exist!\nDid you mean ""snb_comments""?\n\nLINE 8: FROM snb_comment_hascreator_person r\n ^"
15,comment_is_located_in_place,snb_comment_islocatedin_place,Comment,Place,NaN,NaN,NaN,NaN,NaN,"CatalogException: Catalog Error: Table with name snb_comment_islocatedin_place does not exist!\nDid you mean ""snb_comments""?\n\nLINE 8: FROM snb_comment_islocatedin_place r\n ^"
17,comment_reply_of_post,snb_comment_replyof_post,Post,Comment,NaN,NaN,NaN,NaN,NaN,"CatalogException: Catalog Error: Table with name snb_comment_replyof_post does not exist!\nDid you mean ""snb_comments""?\n\nLINE 8: FROM snb_comment_replyof_post r\n ^"
18,comment_reply_of_comment,snb_comment_replyof_comment,Comment,Comment,NaN,NaN,NaN,NaN,NaN,"CatalogException: Catalog Error: Table with name snb_comment_replyof_comment does not exist!\nDid you mean ""snb_comments_norm""?\n\nLINE 8: FROM snb_comment_replyof_comment r\n ^"
19,organisation_is_located_in_place,snb_organisation_islocatedin_place,Organisation,Place,NaN,NaN,NaN,NaN,NaN,"CatalogException: Catalog Error: Table with name snb_organisation_islocatedin_place does not exist!\nDid you mean ""snb_organisations""?\n\nLINE 8: FROM snb_organisation_islocatedin_place r\n ^"


In [93]:
# =========================================================
# BLOCK 3A — Create FK-derived LDBC SNB relationship views
# =========================================================
#
# Goal:
# - Create relationship views from foreign-key columns.
# - These views were referenced in SNB_RELATIONSHIP_HINTS.
# - They must exist before running join validation.
# =========================================================

# Person -> Place
con.execute("""
CREATE OR REPLACE VIEW snb_person_islocatedin_place AS
SELECT DISTINCT
    person_id,
    place_id
FROM snb_persons_entity
WHERE place_id IS NOT NULL
""")

# Forum -> Person moderator
con.execute("""
CREATE OR REPLACE VIEW snb_forum_hasmoderator_person AS
SELECT DISTINCT
    forum_id,
    moderator_person_id AS person_id
FROM snb_forums_entity
WHERE moderator_person_id IS NOT NULL
""")

# Forum -> Post
con.execute("""
CREATE OR REPLACE VIEW snb_forum_containerof_post AS
SELECT DISTINCT
    forum_id,
    post_id
FROM snb_posts_entity
WHERE forum_id IS NOT NULL
""")

# Post -> Person creator
con.execute("""
CREATE OR REPLACE VIEW snb_post_hascreator_person AS
SELECT DISTINCT
    post_id,
    creator_person_id AS person_id
FROM snb_posts_entity
WHERE creator_person_id IS NOT NULL
""")

# Post -> Place
con.execute("""
CREATE OR REPLACE VIEW snb_post_islocatedin_place AS
SELECT DISTINCT
    post_id,
    place_id
FROM snb_posts_entity
WHERE place_id IS NOT NULL
""")

# Comment -> Person creator
con.execute("""
CREATE OR REPLACE VIEW snb_comment_hascreator_person AS
SELECT DISTINCT
    comment_id,
    creator_person_id AS person_id
FROM snb_comments_entity
WHERE creator_person_id IS NOT NULL
""")

# Comment -> Place
con.execute("""
CREATE OR REPLACE VIEW snb_comment_islocatedin_place AS
SELECT DISTINCT
    comment_id,
    place_id
FROM snb_comments_entity
WHERE place_id IS NOT NULL
""")

# Post -> Comment
con.execute("""
CREATE OR REPLACE VIEW snb_comment_replyof_post AS
SELECT DISTINCT
    reply_post_id AS post_id,
    comment_id
FROM snb_comments_entity
WHERE reply_post_id IS NOT NULL
""")

# Comment -> Comment
con.execute("""
CREATE OR REPLACE VIEW snb_comment_replyof_comment AS
SELECT DISTINCT
    reply_comment_id AS parent_comment_id,
    comment_id
FROM snb_comments_entity
WHERE reply_comment_id IS NOT NULL
""")

# Organisation -> Place
con.execute("""
CREATE OR REPLACE VIEW snb_organisation_islocatedin_place AS
SELECT DISTINCT
    organisation_id,
    place_id
FROM snb_organisations_entity
WHERE place_id IS NOT NULL
""")

# Tag -> TagClass
con.execute("""
CREATE OR REPLACE VIEW snb_tag_hastype_tagclass AS
SELECT DISTINCT
    tag_id,
    tagclass_id
FROM snb_tags_entity
WHERE tagclass_id IS NOT NULL
""")

# TagClass -> TagClass
con.execute("""
CREATE OR REPLACE VIEW snb_tagclass_issubclassof_tagclass AS
SELECT DISTINCT
    tagclass_id,
    parent_tagclass_id
FROM snb_tagclasses_entity
WHERE parent_tagclass_id IS NOT NULL
""")

# Place -> Place
con.execute("""
CREATE OR REPLACE VIEW snb_place_ispartof_place AS
SELECT DISTINCT
    parent_place_id,
    place_id
FROM snb_places_entity
WHERE parent_place_id IS NOT NULL
""")

In [94]:
# =========================================================
# BLOCK 3B — Validate FK-derived relationship views exist
# =========================================================

fk_relationship_views = [
    "snb_person_islocatedin_place",
    "snb_forum_hasmoderator_person",
    "snb_forum_containerof_post",
    "snb_post_hascreator_person",
    "snb_post_islocatedin_place",
    "snb_comment_hascreator_person",
    "snb_comment_islocatedin_place",
    "snb_comment_replyof_post",
    "snb_comment_replyof_comment",
    "snb_organisation_islocatedin_place",
    "snb_tag_hastype_tagclass",
    "snb_tagclass_issubclassof_tagclass",
    "snb_place_ispartof_place",
]

fk_view_rows = []

for view_name in fk_relationship_views:
    try:
        n = con.execute(f"SELECT COUNT(*) FROM {view_name}").fetchone()[0]
        cols = con.execute(f"SELECT * FROM {view_name} LIMIT 1").df().columns.tolist()
        fk_view_rows.append({
            "view_name": view_name,
            "rows": n,
            "columns": cols,
            "status": "OK"
        })
    except Exception as e:
        fk_view_rows.append({
            "view_name": view_name,
            "rows": None,
            "columns": None,
            "status": f"ERROR: {type(e).__name__}: {e}"
        })

fk_relationship_views_df = pd.DataFrame(fk_view_rows)
fk_relationship_views_df

,view_name,rows,columns,status
0,snb_person_islocatedin_place,1528,"[person_id, place_id]",OK
1,snb_forum_hasmoderator_person,13750,"[forum_id, person_id]",OK
2,snb_forum_containerof_post,135701,"[forum_id, post_id]",OK
3,snb_post_hascreator_person,135701,"[post_id, person_id]",OK
4,snb_post_islocatedin_place,135701,"[post_id, place_id]",OK
5,snb_comment_hascreator_person,151043,"[comment_id, person_id]",OK
6,snb_comment_islocatedin_place,151043,"[comment_id, place_id]",OK
7,snb_comment_replyof_post,74256,"[post_id, comment_id]",OK
8,snb_comment_replyof_comment,76787,"[parent_comment_id, comment_id]",OK
9,snb_organisation_islocatedin_place,7955,"[organisation_id, place_id]",OK


In [95]:
join_validation_rows = []

for rel_name, spec in SNB_RELATIONSHIP_HINTS.items():
    try:
        join_validation_rows.append(validate_relationship_join(rel_name, spec))
    except Exception as e:
        join_validation_rows.append({
            "relationship": rel_name,
            "view_name": spec.get("view_name"),
            "source_entity": spec.get("source_entity"),
            "target_entity": spec.get("target_entity"),
            "relationship_rows": None,
            "source_matches": None,
            "target_matches": None,
            "source_match_ratio": None,
            "target_match_ratio": None,
            "error": f"{type(e).__name__}: {e}",
        })

join_validation_df = pd.DataFrame(join_validation_rows)

join_validation_df = join_validation_df.sort_values(
    by=["source_match_ratio", "target_match_ratio"],
    ascending=True,
    na_position="first"
)

join_validation_df

,relationship,view_name,source_entity,target_entity,relationship_rows,source_matches,target_matches,source_match_ratio,target_match_ratio
0,person_knows_person,snb_knows_norm,Person,Person,14073,14073,14073,1.0,1.0
1,person_is_located_in_place,snb_person_islocatedin_place,Person,Place,1528,1528,1528,1.0,1.0
2,person_has_interest_tag,snb_person_interests_norm,Person,Tag,35475,35475,35475,1.0,1.0
3,person_likes_post,snb_person_likes_posts_norm,Person,Post,47215,47215,47215,1.0,1.0
4,person_likes_comment,snb_person_likes_comments_norm,Person,Comment,62225,62225,62225,1.0,1.0
5,person_study_at_organisation,snb_person_studyat_norm,Person,Organisation,1209,1209,1209,1.0,1.0
6,person_work_at_organisation,snb_person_workat_norm,Person,Organisation,3313,3313,3313,1.0,1.0
7,forum_has_moderator_person,snb_forum_hasmoderator_person,Forum,Person,13750,13750,13750,1.0,1.0
8,forum_has_member_person,snb_forum_members_norm,Forum,Person,123268,123268,123268,1.0,1.0
9,forum_has_tag,snb_forum_tags_norm,Forum,Tag,47697,47697,47697,1.0,1.0


In [96]:
# =========================================================
# BLOCK 4A — Build LDBC SNB conceptual schema edge table
# =========================================================
#
# Goal:
# - Convert SNB_RELATIONSHIP_HINTS into a clean conceptual edge table.
# - Each row represents one relationship between two conceptual entities.
# - This table is the base for the schema graph used by the methodology.
# =========================================================

import pandas as pd

schema_edge_rows = []

for rel_name, spec in SNB_RELATIONSHIP_HINTS.items():
    schema_edge_rows.append({
        "relationship": rel_name,
        "source_entity": spec["source_entity"],
        "target_entity": spec["target_entity"],
        "semantic_type": spec["semantic_type"],
        "relationship_kind": spec["relationship_kind"],
        "is_bidirectional": spec["is_bidirectional"],
        "view_name": spec["view_name"],
        "source_column": spec["source_column"],
        "target_column": spec["target_column"],
    })

snb_schema_edges_df = pd.DataFrame(schema_edge_rows)

# Sort for easier reading
snb_schema_edges_df = snb_schema_edges_df.sort_values(
    by=["source_entity", "target_entity", "relationship"]
).reset_index(drop=True)

snb_schema_edges_df

,relationship,source_entity,target_entity,semantic_type,relationship_kind,is_bidirectional,view_name,source_column,target_column
0,comment_reply_of_comment,Comment,Comment,containment,one_to_many,False,snb_comment_replyof_comment,parent_comment_id,comment_id
1,comment_has_creator_person,Comment,Person,association,many_to_one,False,snb_comment_hascreator_person,comment_id,person_id
2,comment_is_located_in_place,Comment,Place,association,many_to_one,False,snb_comment_islocatedin_place,comment_id,place_id
3,comment_has_tag,Comment,Tag,association,many_to_many,False,snb_comment_tags_norm,comment_id,tag_id
4,forum_has_member_person,Forum,Person,associative,many_to_many,False,snb_forum_members_norm,forum_id,person_id
5,forum_has_moderator_person,Forum,Person,association,many_to_one,False,snb_forum_hasmoderator_person,forum_id,person_id
6,forum_container_of_post,Forum,Post,containment,one_to_many,False,snb_forum_containerof_post,forum_id,post_id
7,forum_has_tag,Forum,Tag,association,many_to_many,False,snb_forum_tags_norm,forum_id,tag_id
8,organisation_is_located_in_place,Organisation,Place,association,many_to_one,False,snb_organisation_islocatedin_place,organisation_id,place_id
9,person_likes_comment,Person,Comment,associative,many_to_many,False,snb_person_likes_comments_norm,person_id,comment_id


In [97]:
# =========================================================
# BLOCK 4B — Build LDBC SNB conceptual schema graph
# =========================================================
#
# Goal:
# - Build a directed conceptual graph using NetworkX.
# - Nodes are conceptual entities.
# - Edges are conceptual relationships.
# - Bidirectional relationships are added in both directions.
# =========================================================

import networkx as nx

snb_schema_graph = nx.DiGraph()

# Add entity nodes
for entity in SNB_ENTITY_VIEW_MAP.keys():
    snb_schema_graph.add_node(entity)

# Add relationship edges
for _, row in snb_schema_edges_df.iterrows():
    source = row["source_entity"]
    target = row["target_entity"]
    rel_name = row["relationship"]

    snb_schema_graph.add_edge(
        source,
        target,
        relationship=rel_name,
        semantic_type=row["semantic_type"],
        relationship_kind=row["relationship_kind"],
        view_name=row["view_name"],
        source_column=row["source_column"],
        target_column=row["target_column"],
    )

    # For bidirectional conceptual relationships, add reverse edge too.
    if bool(row["is_bidirectional"]):
        snb_schema_graph.add_edge(
            target,
            source,
            relationship=rel_name + "_reverse",
            semantic_type=row["semantic_type"],
            relationship_kind=row["relationship_kind"],
            view_name=row["view_name"],
            source_column=row["target_column"],
            target_column=row["source_column"],
        )

print("Number of entities:", snb_schema_graph.number_of_nodes())
print("Number of directed edges:", snb_schema_graph.number_of_edges())
print("Entities:", list(snb_schema_graph.nodes()))

Number of entities: 8
Number of directed edges: 21
Entities: ['Person', 'Forum', 'Post', 'Comment', 'Place', 'Organisation', 'Tag', 'TagClass']


In [98]:
# =========================================================
# BLOCK 4C — Validate conceptual graph connectivity
# =========================================================
#
# Goal:
# - Check whether conceptual entities are connected.
# - This is important because Rc, D and Re depend on schema paths.
# =========================================================

graph_validation_rows = []

entities = list(SNB_ENTITY_VIEW_MAP.keys())

for source in entities:
    for target in entities:
        if source == target:
            continue

        try:
            shortest_path = nx.shortest_path(snb_schema_graph, source=source, target=target)
            distance = len(shortest_path) - 1
            graph_validation_rows.append({
                "source_entity": source,
                "target_entity": target,
                "reachable": True,
                "distance": distance,
                "path": " -> ".join(shortest_path),
            })
        except nx.NetworkXNoPath:
            graph_validation_rows.append({
                "source_entity": source,
                "target_entity": target,
                "reachable": False,
                "distance": None,
                "path": None,
            })

snb_graph_connectivity_df = pd.DataFrame(graph_validation_rows)

snb_graph_connectivity_summary_df = (
    snb_graph_connectivity_df
    .groupby("source_entity")
    .agg(
        reachable_targets=("reachable", "sum"),
        total_targets=("target_entity", "count"),
        avg_distance=("distance", "mean"),
        max_distance=("distance", "max"),
    )
    .reset_index()
)

snb_graph_connectivity_summary_df

,source_entity,reachable_targets,total_targets,avg_distance,max_distance
0,Comment,6,7,1.500000,2.0
1,Forum,7,7,1.571429,2.0
2,Organisation,1,7,1.000000,1.0
3,Person,6,7,1.166667,2.0
4,Place,0,7,NaN,NaN
5,Post,6,7,1.333333,2.0
6,Tag,1,7,1.000000,1.0
7,TagClass,0,7,NaN,NaN


In [99]:
# =========================================================
# BLOCK 4D — Inspect important LDBC SNB conceptual paths
# =========================================================
#
# Goal:
# - Inspect important paths that will appear in the SNB workload.
# =========================================================

important_pairs = [
    ("Person", "Person"),
    ("Person", "Post"),
    ("Person", "Comment"),
    ("Person", "Forum"),
    ("Forum", "Post"),
    ("Forum", "Comment"),
    ("Post", "Comment"),
    ("Post", "Tag"),
    ("Comment", "Tag"),
    ("Person", "Tag"),
    ("Person", "Organisation"),
    ("Person", "Place"),
    ("Organisation", "Place"),
    ("Tag", "TagClass"),
    ("Place", "Place"),
]

important_path_rows = []

for source, target in important_pairs:
    try:
        path = nx.shortest_path(snb_schema_graph, source=source, target=target)
        important_path_rows.append({
            "source_entity": source,
            "target_entity": target,
            "reachable": True,
            "distance": len(path) - 1,
            "path": " -> ".join(path),
        })
    except nx.NetworkXNoPath:
        important_path_rows.append({
            "source_entity": source,
            "target_entity": target,
            "reachable": False,
            "distance": None,
            "path": None,
        })

important_paths_df = pd.DataFrame(important_path_rows)
important_paths_df

,source_entity,target_entity,reachable,distance,path
0,Person,Person,True,0.0,Person
1,Person,Post,True,1.0,Person -> Post
2,Person,Comment,True,1.0,Person -> Comment
3,Person,Forum,False,NaN,None
4,Forum,Post,True,1.0,Forum -> Post
5,Forum,Comment,True,2.0,Forum -> Person -> Comment
6,Post,Comment,True,1.0,Post -> Comment
7,Post,Tag,True,1.0,Post -> Tag
8,Comment,Tag,True,1.0,Comment -> Tag
9,Person,Tag,True,1.0,Person -> Tag


In [100]:
# =========================================================
# BLOCK 4E — Generate Mermaid ER diagram for LDBC SNB
# =========================================================
#
# Goal:
# - Generate a Mermaid ER diagram from the validated conceptual schema.
# - This can be copied to README, Overleaf notes, or documentation.
# =========================================================

mermaid_lines = []
mermaid_lines.append("erDiagram")
mermaid_lines.append("")

# Entity definitions
entity_attributes = {
    "Person": [
        "bigint person_id PK",
        "string first_name",
        "string last_name",
        "string gender",
        "date birthday",
        "datetime creation_date",
        "string location_ip",
        "string browser_used",
    ],
    "Forum": [
        "bigint forum_id PK",
        "string title",
        "datetime creation_date",
    ],
    "Post": [
        "bigint post_id PK",
        "datetime creation_date",
        "string image_file",
        "string location_ip",
        "string browser_used",
        "string language",
        "string content",
        "int length",
    ],
    "Comment": [
        "bigint comment_id PK",
        "datetime creation_date",
        "string location_ip",
        "string browser_used",
        "string content",
        "int length",
    ],
    "Place": [
        "bigint place_id PK",
        "string name",
        "string url",
        "string type",
    ],
    "Organisation": [
        "bigint organisation_id PK",
        "string type",
        "string name",
        "string url",
    ],
    "Tag": [
        "bigint tag_id PK",
        "string name",
        "string url",
    ],
    "TagClass": [
        "bigint tagclass_id PK",
        "string name",
        "string url",
    ],
}

for entity, attrs in entity_attributes.items():
    mermaid_lines.append(f"    {entity.upper()} {{")
    for attr in attrs:
        mermaid_lines.append(f"        {attr}")
    mermaid_lines.append("    }")
    mermaid_lines.append("")

# Relationship rendering rules
def mermaid_cardinality(kind: str):
    if kind == "one_to_many":
        return "||--o{"
    if kind == "many_to_one":
        return "}o--||"
    if kind == "many_to_many":
        return "}o--o{"
    return "}o--o{"

for _, row in snb_schema_edges_df.iterrows():
    source = row["source_entity"].upper()
    target = row["target_entity"].upper()
    rel = row["relationship"]
    kind = row["relationship_kind"]

    card = mermaid_cardinality(kind)
    mermaid_lines.append(f'    {source} {card} {target} : "{rel}"')

mermaid_diagram = "\n".join(mermaid_lines)

print(mermaid_diagram)

erDiagram

    PERSON {
        bigint person_id PK
        string first_name
        string last_name
        string gender
        date birthday
        datetime creation_date
        string location_ip
        string browser_used
    }

    FORUM {
        bigint forum_id PK
        string title
        datetime creation_date
    }

    POST {
        bigint post_id PK
        datetime creation_date
        string image_file
        string location_ip
        string browser_used
        string language
        string content
        int length
    }

    COMMENT {
        bigint comment_id PK
        datetime creation_date
        string location_ip
        string browser_used
        string content
        int length
    }

    PLACE {
        bigint place_id PK
        string name
        string url
        string type
    }

    ORGANISATION {
        bigint organisation_id PK
        string type
        string name
        string url
    }

    TAG {
        bigint tag_id PK
   

In [101]:
# =========================================================
# BLOCK 4F — Save initial LDBC SNB schema artifacts
# =========================================================
#
# Goal:
# - Save schema validation artifacts for reproducibility.
# - This helps resume work after restarting the kernel.
# =========================================================

from pathlib import Path

ARTIFACTS_DIR = BASE_DIR.parent.parent / "benchmark_artifacts_dir" / "ldbc_snb_schema_validation"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

clean_entity_counts_df.to_csv(ARTIFACTS_DIR / "clean_entity_counts.csv", index=False)
snb_relationship_hints_df.to_csv(ARTIFACTS_DIR / "relationship_hints.csv", index=False)
join_validation_df.to_csv(ARTIFACTS_DIR / "relationship_join_validation.csv", index=False)
snb_schema_edges_df.to_csv(ARTIFACTS_DIR / "schema_edges.csv", index=False)
snb_graph_connectivity_df.to_csv(ARTIFACTS_DIR / "schema_graph_connectivity.csv", index=False)
important_paths_df.to_csv(ARTIFACTS_DIR / "important_schema_paths.csv", index=False)

with open(ARTIFACTS_DIR / "ldbc_snb_mermaid_er_diagram.mmd", "w", encoding="utf-8") as f:
    f.write(mermaid_diagram)

print("Saved schema artifacts to:")
print(ARTIFACTS_DIR)

Saved schema artifacts to:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_schema_validation


In [102]:
# =========================================================
# BLOCK 4G — Build LDBC SNB traversal graph for methodology metrics
# =========================================================
#
# Goal:
# - Build an undirected conceptual graph for distance-based metrics.
# - The directed graph keeps relationship direction.
# - The traversal graph supports conceptual reachability for Rc, D, Re,
#   DeltaR and DeltaRratio.
# =========================================================

import networkx as nx
import pandas as pd

snb_traversal_graph = nx.Graph()

# Add entity nodes.
for entity in SNB_ENTITY_VIEW_MAP.keys():
    snb_traversal_graph.add_node(entity)

# Add undirected edges for conceptual traversal.
for _, row in snb_schema_edges_df.iterrows():
    source = row["source_entity"]
    target = row["target_entity"]

    snb_traversal_graph.add_edge(
        source,
        target,
        relationship=row["relationship"],
        semantic_type=row["semantic_type"],
        relationship_kind=row["relationship_kind"],
        view_name=row["view_name"],
        source_column=row["source_column"],
        target_column=row["target_column"],
    )

print("Traversal graph entities:", snb_traversal_graph.number_of_nodes())
print("Traversal graph edges:", snb_traversal_graph.number_of_edges())
print("Entities:", list(snb_traversal_graph.nodes()))

Traversal graph entities: 8
Traversal graph edges: 19
Entities: ['Person', 'Forum', 'Post', 'Comment', 'Place', 'Organisation', 'Tag', 'TagClass']


In [103]:
# =========================================================
# BLOCK 4H — Validate traversal graph connectivity
# =========================================================
#
# Goal:
# - Recompute reachability using the undirected traversal graph.
# - This is the graph that should be used for conceptual distance.
# =========================================================

traversal_validation_rows = []

entities = list(SNB_ENTITY_VIEW_MAP.keys())

for source in entities:
    for target in entities:
        if source == target:
            continue

        try:
            shortest_path = nx.shortest_path(
                snb_traversal_graph,
                source=source,
                target=target
            )
            distance = len(shortest_path) - 1

            traversal_validation_rows.append({
                "source_entity": source,
                "target_entity": target,
                "reachable": True,
                "distance": distance,
                "path": " -> ".join(shortest_path),
            })

        except nx.NetworkXNoPath:
            traversal_validation_rows.append({
                "source_entity": source,
                "target_entity": target,
                "reachable": False,
                "distance": None,
                "path": None,
            })

snb_traversal_connectivity_df = pd.DataFrame(traversal_validation_rows)

snb_traversal_connectivity_summary_df = (
    snb_traversal_connectivity_df
    .groupby("source_entity")
    .agg(
        reachable_targets=("reachable", "sum"),
        total_targets=("target_entity", "count"),
        avg_distance=("distance", "mean"),
        max_distance=("distance", "max"),
    )
    .reset_index()
)

snb_traversal_connectivity_summary_df

,source_entity,reachable_targets,total_targets,avg_distance,max_distance
0,Comment,7,7,1.428571,2
1,Forum,7,7,1.571429,2
2,Organisation,7,7,1.857143,3
3,Person,7,7,1.142857,2
4,Place,7,7,1.571429,3
5,Post,7,7,1.285714,2
6,Tag,7,7,1.285714,2
7,TagClass,7,7,2.142857,3


In [104]:
# =========================================================
# BLOCK 4I — Inspect important traversal paths
# =========================================================
#
# Goal:
# - Check important conceptual paths using the traversal graph.
# =========================================================

important_pairs = [
    ("Person", "Person"),
    ("Person", "Post"),
    ("Person", "Comment"),
    ("Person", "Forum"),
    ("Forum", "Post"),
    ("Forum", "Comment"),
    ("Post", "Comment"),
    ("Post", "Tag"),
    ("Comment", "Tag"),
    ("Person", "Tag"),
    ("Person", "Organisation"),
    ("Person", "Place"),
    ("Organisation", "Place"),
    ("Tag", "TagClass"),
    ("Place", "Place"),
    ("TagClass", "Post"),
    ("Place", "Person"),
]

important_traversal_path_rows = []

for source, target in important_pairs:
    try:
        path = nx.shortest_path(
            snb_traversal_graph,
            source=source,
            target=target
        )

        important_traversal_path_rows.append({
            "source_entity": source,
            "target_entity": target,
            "reachable": True,
            "distance": len(path) - 1,
            "path": " -> ".join(path),
        })

    except nx.NetworkXNoPath:
        important_traversal_path_rows.append({
            "source_entity": source,
            "target_entity": target,
            "reachable": False,
            "distance": None,
            "path": None,
        })

important_traversal_paths_df = pd.DataFrame(important_traversal_path_rows)
important_traversal_paths_df

,source_entity,target_entity,reachable,distance,path
0,Person,Person,True,0,Person
1,Person,Post,True,1,Person -> Post
2,Person,Comment,True,1,Person -> Comment
3,Person,Forum,True,1,Person -> Forum
4,Forum,Post,True,1,Forum -> Post
5,Forum,Comment,True,2,Forum -> Person -> Comment
6,Post,Comment,True,1,Post -> Comment
7,Post,Tag,True,1,Post -> Tag
8,Comment,Tag,True,1,Comment -> Tag
9,Person,Tag,True,1,Person -> Tag


In [105]:
# =========================================================
# BLOCK 5A — Define official-based LDBC SNB Interactive workload
# =========================================================
#
# Goal:
# - Define the workload using official LDBC SNB Interactive query IDs.
# - Avoid creating an artificial workload disconnected from the benchmark.
# - Start with all official short reads, selected official complex reads,
#   and all official insert operations.
#
# Notes:
# - The official schema uses Message as an abstract supertype.
# - In the CSV files, Message is represented by Post and Comment.
# - Therefore, Message-based queries are mapped to Post and/or Comment.
# =========================================================

SNB_OFFICIAL_WORKLOAD = [
    # -----------------------------------------------------
    # Official Interactive Short Reads
    # -----------------------------------------------------
    {
        "query_name": "IS1_ProfileOfPerson",
        "official_id": "IS1",
        "official_title": "Profile of a person",
        "query_group": "short_read",
        "operation_type": "read",
        "root_entity": "Person",
        "accessed_entities": ["Person", "Place"],
        "relationships_used": [
            "person_is_located_in_place",
        ],
        "description": "Official IS1: retrieve a person's profile and city of residence.",
    },

    {
        "query_name": "IS2_RecentMessagesOfPerson",
        "official_id": "IS2",
        "official_title": "Recent messages of a person",
        "query_group": "short_read",
        "operation_type": "read",
        "root_entity": "Person",
        "accessed_entities": ["Person", "Post", "Comment"],
        "relationships_used": [
            "post_has_creator_person",
            "comment_has_creator_person",
            "comment_reply_of_post",
            "comment_reply_of_comment",
        ],
        "description": "Official IS2: retrieve the last messages created by a person and the original post in the conversation.",
    },

    {
        "query_name": "IS3_FriendsOfPerson",
        "official_id": "IS3",
        "official_title": "Friends of a person",
        "query_group": "short_read",
        "operation_type": "read",
        "root_entity": "Person",
        "accessed_entities": ["Person"],
        "relationships_used": [
            "person_knows_person",
        ],
        "description": "Official IS3: retrieve all friends of a person and the friendship creation date.",
    },

    {
        "query_name": "IS4_ContentOfMessage",
        "official_id": "IS4",
        "official_title": "Content of a message",
        "query_group": "short_read",
        "operation_type": "read",
        "root_entity": "Post",
        "accessed_entities": ["Post", "Comment"],
        "relationships_used": [],
        "description": "Official IS4: retrieve content and creation date of a Message. Message is represented by Post or Comment.",
    },

    {
        "query_name": "IS5_CreatorOfMessage",
        "official_id": "IS5",
        "official_title": "Creator of a message",
        "query_group": "short_read",
        "operation_type": "read",
        "root_entity": "Post",
        "accessed_entities": ["Post", "Comment", "Person"],
        "relationships_used": [
            "post_has_creator_person",
            "comment_has_creator_person",
        ],
        "description": "Official IS5: retrieve the author of a Message. Message is represented by Post or Comment.",
    },

    {
        "query_name": "IS6_ForumOfMessage",
        "official_id": "IS6",
        "official_title": "Forum of a message",
        "query_group": "short_read",
        "operation_type": "read",
        "root_entity": "Post",
        "accessed_entities": ["Post", "Comment", "Forum", "Person"],
        "relationships_used": [
            "forum_container_of_post",
            "comment_reply_of_post",
            "comment_reply_of_comment",
            "forum_has_moderator_person",
        ],
        "description": "Official IS6: retrieve the forum containing a Message and the moderator of that forum.",
    },

    {
        "query_name": "IS7_RepliesOfMessage",
        "official_id": "IS7",
        "official_title": "Replies of a message",
        "query_group": "short_read",
        "operation_type": "read",
        "root_entity": "Post",
        "accessed_entities": ["Post", "Comment", "Person"],
        "relationships_used": [
            "comment_reply_of_post",
            "comment_reply_of_comment",
            "comment_has_creator_person",
            "post_has_creator_person",
            "person_knows_person",
        ],
        "description": "Official IS7: retrieve one-hop replies of a Message and whether reply author knows original author.",
    },

    # -----------------------------------------------------
    # Official Interactive Complex Reads — first stage
    # -----------------------------------------------------
    {
        "query_name": "IC1_TransitiveFriendsWithName",
        "official_id": "IC1",
        "official_title": "Transitive friends with a certain name",
        "query_group": "complex_read",
        "operation_type": "read",
        "root_entity": "Person",
        "accessed_entities": ["Person", "Place", "Organisation"],
        "relationships_used": [
            "person_knows_person",
            "person_is_located_in_place",
            "person_study_at_organisation",
            "person_work_at_organisation",
            "organisation_is_located_in_place",
        ],
        "description": "Official IC1: find people connected by up to 3 knows hops with a given first name, including workplaces and studies.",
    },

    {
        "query_name": "IC2_RecentMessagesByFriends",
        "official_id": "IC2",
        "official_title": "Recent messages by your friends",
        "query_group": "complex_read",
        "operation_type": "read",
        "root_entity": "Person",
        "accessed_entities": ["Person", "Post", "Comment"],
        "relationships_used": [
            "person_knows_person",
            "post_has_creator_person",
            "comment_has_creator_person",
        ],
        "description": "Official IC2: find recent messages created by the friends of a person.",
    },

    {
        "query_name": "IC3_FriendsAndFriendsOfFriendsInCountries",
        "official_id": "IC3",
        "official_title": "Friends and friends of friends that have been to given countries",
        "query_group": "complex_read",
        "operation_type": "read",
        "root_entity": "Person",
        "accessed_entities": ["Person", "Post", "Comment", "Place"],
        "relationships_used": [
            "person_knows_person",
            "person_is_located_in_place",
            "post_has_creator_person",
            "comment_has_creator_person",
            "post_is_located_in_place",
            "comment_is_located_in_place",
            "place_is_part_of_place",
        ],
        "description": "Official IC3: find friends and friends-of-friends who created messages in two countries.",
    },

    {
        "query_name": "IC4_NewTopics",
        "official_id": "IC4",
        "official_title": "New topics",
        "query_group": "complex_read",
        "operation_type": "read",
        "root_entity": "Person",
        "accessed_entities": ["Person", "Post", "Tag"],
        "relationships_used": [
            "person_knows_person",
            "post_has_creator_person",
            "post_has_tag",
        ],
        "description": "Official IC4: find new tags attached to posts created by friends in a time interval.",
    },

    {
        "query_name": "IC5_NewGroups",
        "official_id": "IC5",
        "official_title": "New groups",
        "query_group": "complex_read",
        "operation_type": "read",
        "root_entity": "Person",
        "accessed_entities": ["Person", "Forum", "Post"],
        "relationships_used": [
            "person_knows_person",
            "forum_has_member_person",
            "forum_container_of_post",
            "post_has_creator_person",
        ],
        "description": "Official IC5: find forums joined by friends/friends-of-friends and count their posts.",
    },

    {
        "query_name": "IC6_TagCoOccurrence",
        "official_id": "IC6",
        "official_title": "Tag co-occurrence",
        "query_group": "complex_read",
        "operation_type": "read",
        "root_entity": "Person",
        "accessed_entities": ["Person", "Post", "Tag"],
        "relationships_used": [
            "person_knows_person",
            "post_has_creator_person",
            "post_has_tag",
        ],
        "description": "Official IC6: find tags co-occurring with a given tag on posts by friends/friends-of-friends.",
    },

    {
        "query_name": "IC7_RecentLikers",
        "official_id": "IC7",
        "official_title": "Recent likers",
        "query_group": "complex_read",
        "operation_type": "read",
        "root_entity": "Person",
        "accessed_entities": ["Person", "Post", "Comment"],
        "relationships_used": [
            "post_has_creator_person",
            "comment_has_creator_person",
            "person_likes_post",
            "person_likes_comment",
            "person_knows_person",
        ],
        "description": "Official IC7: find recent likes on a person's messages and whether likers are friends.",
    },

    # -----------------------------------------------------
    # Official Inserts
    # -----------------------------------------------------
    {
        "query_name": "INS1_AddPerson",
        "official_id": "INS1",
        "official_title": "Add person",
        "query_group": "insert",
        "operation_type": "insert",
        "root_entity": "Person",
        "accessed_entities": ["Person", "Place", "Tag", "Organisation"],
        "relationships_used": [
            "person_is_located_in_place",
            "person_has_interest_tag",
            "person_study_at_organisation",
            "person_work_at_organisation",
        ],
        "description": "Official INS1: add a Person connected by location, interests, studyAt and workAt.",
    },

    {
        "query_name": "INS2_AddLikeToPost",
        "official_id": "INS2",
        "official_title": "Add like to post",
        "query_group": "insert",
        "operation_type": "insert",
        "root_entity": "Person",
        "accessed_entities": ["Person", "Post"],
        "relationships_used": [
            "person_likes_post",
        ],
        "description": "Official INS2: add a likes edge from Person to Post.",
    },

    {
        "query_name": "INS3_AddLikeToComment",
        "official_id": "INS3",
        "official_title": "Add like to comment",
        "query_group": "insert",
        "operation_type": "insert",
        "root_entity": "Person",
        "accessed_entities": ["Person", "Comment"],
        "relationships_used": [
            "person_likes_comment",
        ],
        "description": "Official INS3: add a likes edge from Person to Comment.",
    },

    {
        "query_name": "INS4_AddForum",
        "official_id": "INS4",
        "official_title": "Add forum",
        "query_group": "insert",
        "operation_type": "insert",
        "root_entity": "Forum",
        "accessed_entities": ["Forum", "Person", "Tag"],
        "relationships_used": [
            "forum_has_moderator_person",
            "forum_has_tag",
        ],
        "description": "Official INS4: add a Forum connected to a moderator and tags.",
    },

    {
        "query_name": "INS5_AddForumMembership",
        "official_id": "INS5",
        "official_title": "Add forum membership",
        "query_group": "insert",
        "operation_type": "insert",
        "root_entity": "Forum",
        "accessed_entities": ["Forum", "Person"],
        "relationships_used": [
            "forum_has_member_person",
        ],
        "description": "Official INS5: add a hasMember edge between Forum and Person.",
    },

    {
        "query_name": "INS6_AddPost",
        "official_id": "INS6",
        "official_title": "Add post",
        "query_group": "insert",
        "operation_type": "insert",
        "root_entity": "Post",
        "accessed_entities": ["Post", "Person", "Forum", "Place", "Tag"],
        "relationships_used": [
            "post_has_creator_person",
            "forum_container_of_post",
            "post_is_located_in_place",
            "post_has_tag",
        ],
        "description": "Official INS6: add a Post connected by creator, forum, location and tags.",
    },

    {
        "query_name": "INS7_AddComment",
        "official_id": "INS7",
        "official_title": "Add comment",
        "query_group": "insert",
        "operation_type": "insert",
        "root_entity": "Comment",
        "accessed_entities": ["Comment", "Person", "Post", "Place", "Tag"],
        "relationships_used": [
            "comment_has_creator_person",
            "comment_reply_of_post",
            "comment_reply_of_comment",
            "comment_is_located_in_place",
            "comment_has_tag",
        ],
        "description": "Official INS7: add a Comment connected by replyOf, creator, location and tags.",
    },

    {
        "query_name": "INS8_AddFriendship",
        "official_id": "INS8",
        "official_title": "Add friendship",
        "query_group": "insert",
        "operation_type": "insert",
        "root_entity": "Person",
        "accessed_entities": ["Person"],
        "relationships_used": [
            "person_knows_person",
        ],
        "description": "Official INS8: add a knows edge between two Persons.",
    },
]

snb_workload_df = pd.DataFrame(SNB_OFFICIAL_WORKLOAD)
snb_workload_df

,query_name,official_id,official_title,query_group,operation_type,root_entity,accessed_entities,relationships_used,description
0,IS1_ProfileOfPerson,IS1,Profile of a person,short_read,read,Person,"[Person, Place]",[person_is_located_in_place],Official IS1: retrieve a person's profile and city of residence.
1,IS2_RecentMessagesOfPerson,IS2,Recent messages of a person,short_read,read,Person,"[Person, Post, Comment]","[post_has_creator_person, comment_has_creator_person, comment_reply_of_post, comment_reply_of_comment]",Official IS2: retrieve the last messages created by a person and the original post in the conversation.
2,IS3_FriendsOfPerson,IS3,Friends of a person,short_read,read,Person,[Person],[person_knows_person],Official IS3: retrieve all friends of a person and the friendship creation date.
3,IS4_ContentOfMessage,IS4,Content of a message,short_read,read,Post,"[Post, Comment]",[],Official IS4: retrieve content and creation date of a Message. Message is represented by Post or Comment.
4,IS5_CreatorOfMessage,IS5,Creator of a message,short_read,read,Post,"[Post, Comment, Person]","[post_has_creator_person, comment_has_creator_person]",Official IS5: retrieve the author of a Message. Message is represented by Post or Comment.
5,IS6_ForumOfMessage,IS6,Forum of a message,short_read,read,Post,"[Post, Comment, Forum, Person]","[forum_container_of_post, comment_reply_of_post, comment_reply_of_comment, forum_has_moderator_person]",Official IS6: retrieve the forum containing a Message and the moderator of that forum.
6,IS7_RepliesOfMessage,IS7,Replies of a message,short_read,read,Post,"[Post, Comment, Person]","[comment_reply_of_post, comment_reply_of_comment, comment_has_creator_person, post_has_creator_person, person_knows_person]",Official IS7: retrieve one-hop replies of a Message and whether reply author knows original author.
7,IC1_TransitiveFriendsWithName,IC1,Transitive friends with a certain name,complex_read,read,Person,"[Person, Place, Organisation]","[person_knows_person, person_is_located_in_place, person_study_at_organisation, person_work_at_organisation, organisation_is_located_in_place]","Official IC1: find people connected by up to 3 knows hops with a given first name, including workplaces and studies."
8,IC2_RecentMessagesByFriends,IC2,Recent messages by your friends,complex_read,read,Person,"[Person, Post, Comment]","[person_knows_person, post_has_creator_person, comment_has_creator_person]",Official IC2: find recent messages created by the friends of a person.
9,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,Friends and friends of friends that have been to given countries,complex_read,read,Person,"[Person, Post, Comment, Place]","[person_knows_person, person_is_located_in_place, post_has_creator_person, comment_has_creator_person, post_is_located_in_place, comment_is_located_in_place, place_is_part_of_place]",Official IC3: find friends and friends-of-friends who created messages in two countries.


In [106]:
# BLOCK 5B
# Validate workload relationships

defined_relationships = set(SNB_RELATIONSHIP_HINTS.keys())

workload_relationship_validation_rows = []

for query in SNB_OFFICIAL_WORKLOAD:
    query_name = query["query_name"]

    for rel in query["relationships_used"]:
        workload_relationship_validation_rows.append({
            "query_name": query_name,
            "official_id": query["official_id"],
            "relationship": rel,
            "exists_in_schema": rel in defined_relationships,
        })

workload_relationship_validation_df = pd.DataFrame(workload_relationship_validation_rows)
workload_relationship_validation_df

,query_name,official_id,relationship,exists_in_schema
0,IS1_ProfileOfPerson,IS1,person_is_located_in_place,True
1,IS2_RecentMessagesOfPerson,IS2,post_has_creator_person,True
2,IS2_RecentMessagesOfPerson,IS2,comment_has_creator_person,True
3,IS2_RecentMessagesOfPerson,IS2,comment_reply_of_post,True
4,IS2_RecentMessagesOfPerson,IS2,comment_reply_of_comment,True
...,...,...,...,...
61,INS7_AddComment,INS7,comment_reply_of_post,True
62,INS7_AddComment,INS7,comment_reply_of_comment,True
63,INS7_AddComment,INS7,comment_is_located_in_place,True
64,INS7_AddComment,INS7,comment_has_tag,True


In [107]:
# =========================================================
# BLOCK 5C — Infer dominant semantic type per official query
# =========================================================
#
# Goal:
# - Count semantic relationship types used by each official query.
# - Infer the dominant semantic type.
# - Queries without explicit relationships are treated as "lookup".
# =========================================================

from collections import Counter
import pandas as pd

semantic_profile_rows = []

for query in SNB_OFFICIAL_WORKLOAD:
    query_name = query["query_name"]
    official_id = query["official_id"]
    rels = query["relationships_used"]

    semantic_types = [
        SNB_RELATIONSHIP_HINTS[rel]["semantic_type"]
        for rel in rels
        if rel in SNB_RELATIONSHIP_HINTS
    ]

    counter = Counter(semantic_types)

    if len(counter) == 0:
        dominant = "lookup"
    elif len(counter) == 1:
        dominant = list(counter.keys())[0]
    else:
        most_common = counter.most_common()
        top_count = most_common[0][1]
        top_types = [t for t, c in most_common if c == top_count]

        if len(top_types) == 1:
            dominant = top_types[0]
        else:
            dominant = "mixed"

    semantic_profile_rows.append({
        "query_name": query_name,
        "official_id": official_id,
        "association_count": counter.get("association", 0),
        "associative_count": counter.get("associative", 0),
        "containment_count": counter.get("containment", 0),
        "lookup_count": 1 if dominant == "lookup" else 0,
        "dominant_semantic_type": dominant,
    })

snb_query_semantic_profile_df = pd.DataFrame(semantic_profile_rows)
snb_query_semantic_profile_df

,query_name,official_id,association_count,associative_count,containment_count,lookup_count,dominant_semantic_type
0,IS1_ProfileOfPerson,IS1,1,0,0,0,association
1,IS2_RecentMessagesOfPerson,IS2,2,0,2,0,mixed
2,IS3_FriendsOfPerson,IS3,1,0,0,0,association
3,IS4_ContentOfMessage,IS4,0,0,0,1,lookup
4,IS5_CreatorOfMessage,IS5,2,0,0,0,association
5,IS6_ForumOfMessage,IS6,1,0,3,0,containment
6,IS7_RepliesOfMessage,IS7,3,0,2,0,association
7,IC1_TransitiveFriendsWithName,IC1,5,0,0,0,association
8,IC2_RecentMessagesByFriends,IC2,3,0,0,0,association
9,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,6,0,1,0,association


In [108]:
# =========================================================
# BLOCK 5D — Build initial official LDBC SNB workload matrix
# =========================================================
#
# Goal:
# - Merge official workload metadata with semantic profile.
# - This becomes the initial workload matrix for the methodology.
# =========================================================

snb_initial_workload_matrix_df = (
    snb_workload_df
    .merge(snb_query_semantic_profile_df, on=["query_name", "official_id"], how="left")
)

snb_initial_workload_matrix_df

,query_name,official_id,official_title,query_group,operation_type,root_entity,accessed_entities,relationships_used,description,association_count,associative_count,containment_count,lookup_count,dominant_semantic_type
0,IS1_ProfileOfPerson,IS1,Profile of a person,short_read,read,Person,"[Person, Place]",[person_is_located_in_place],Official IS1: retrieve a person's profile and city of residence.,1,0,0,0,association
1,IS2_RecentMessagesOfPerson,IS2,Recent messages of a person,short_read,read,Person,"[Person, Post, Comment]","[post_has_creator_person, comment_has_creator_person, comment_reply_of_post, comment_reply_of_comment]",Official IS2: retrieve the last messages created by a person and the original post in the conversation.,2,0,2,0,mixed
2,IS3_FriendsOfPerson,IS3,Friends of a person,short_read,read,Person,[Person],[person_knows_person],Official IS3: retrieve all friends of a person and the friendship creation date.,1,0,0,0,association
3,IS4_ContentOfMessage,IS4,Content of a message,short_read,read,Post,"[Post, Comment]",[],Official IS4: retrieve content and creation date of a Message. Message is represented by Post or Comment.,0,0,0,1,lookup
4,IS5_CreatorOfMessage,IS5,Creator of a message,short_read,read,Post,"[Post, Comment, Person]","[post_has_creator_person, comment_has_creator_person]",Official IS5: retrieve the author of a Message. Message is represented by Post or Comment.,2,0,0,0,association
5,IS6_ForumOfMessage,IS6,Forum of a message,short_read,read,Post,"[Post, Comment, Forum, Person]","[forum_container_of_post, comment_reply_of_post, comment_reply_of_comment, forum_has_moderator_person]",Official IS6: retrieve the forum containing a Message and the moderator of that forum.,1,0,3,0,containment
6,IS7_RepliesOfMessage,IS7,Replies of a message,short_read,read,Post,"[Post, Comment, Person]","[comment_reply_of_post, comment_reply_of_comment, comment_has_creator_person, post_has_creator_person, person_knows_person]",Official IS7: retrieve one-hop replies of a Message and whether reply author knows original author.,3,0,2,0,association
7,IC1_TransitiveFriendsWithName,IC1,Transitive friends with a certain name,complex_read,read,Person,"[Person, Place, Organisation]","[person_knows_person, person_is_located_in_place, person_study_at_organisation, person_work_at_organisation, organisation_is_located_in_place]","Official IC1: find people connected by up to 3 knows hops with a given first name, including workplaces and studies.",5,0,0,0,association
8,IC2_RecentMessagesByFriends,IC2,Recent messages by your friends,complex_read,read,Person,"[Person, Post, Comment]","[person_knows_person, post_has_creator_person, comment_has_creator_person]",Official IC2: find recent messages created by the friends of a person.,3,0,0,0,association
9,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,Friends and friends of friends that have been to given countries,complex_read,read,Person,"[Person, Post, Comment, Place]","[person_knows_person, person_is_located_in_place, post_has_creator_person, comment_has_creator_person, post_is_located_in_place, comment_is_located_in_place, place_is_part_of_place]",Official IC3: find friends and friends-of-friends who created messages in two countries.,6,0,1,0,association


In [109]:
# =========================================================
# BLOCK 5E — Validate workload root-to-entity traversal paths
# =========================================================
#
# Goal:
# - For each official query, check whether accessed entities can be reached
#   from the root entity in the traversal graph.
# - This prepares the computation of D and Rc.
# =========================================================

import networkx as nx

workload_path_rows = []

for query in SNB_OFFICIAL_WORKLOAD:
    query_name = query["query_name"]
    official_id = query["official_id"]
    root = query["root_entity"]

    for entity in query["accessed_entities"]:
        if entity == root:
            workload_path_rows.append({
                "query_name": query_name,
                "official_id": official_id,
                "root_entity": root,
                "target_entity": entity,
                "reachable": True,
                "distance": 0,
                "path": root,
            })
            continue

        try:
            path = nx.shortest_path(
                snb_traversal_graph,
                source=root,
                target=entity
            )

            workload_path_rows.append({
                "query_name": query_name,
                "official_id": official_id,
                "root_entity": root,
                "target_entity": entity,
                "reachable": True,
                "distance": len(path) - 1,
                "path": " -> ".join(path),
            })

        except nx.NetworkXNoPath:
            workload_path_rows.append({
                "query_name": query_name,
                "official_id": official_id,
                "root_entity": root,
                "target_entity": entity,
                "reachable": False,
                "distance": None,
                "path": None,
            })

snb_workload_paths_df = pd.DataFrame(workload_path_rows)
snb_workload_paths_df

,query_name,official_id,root_entity,target_entity,reachable,distance,path
0,IS1_ProfileOfPerson,IS1,Person,Person,True,0,Person
1,IS1_ProfileOfPerson,IS1,Person,Place,True,1,Person -> Place
2,IS2_RecentMessagesOfPerson,IS2,Person,Person,True,0,Person
3,IS2_RecentMessagesOfPerson,IS2,Person,Post,True,1,Person -> Post
4,IS2_RecentMessagesOfPerson,IS2,Person,Comment,True,1,Person -> Comment
...,...,...,...,...,...,...,...
59,INS7_AddComment,INS7,Comment,Person,True,1,Comment -> Person
60,INS7_AddComment,INS7,Comment,Post,True,1,Comment -> Post
61,INS7_AddComment,INS7,Comment,Place,True,1,Comment -> Place
62,INS7_AddComment,INS7,Comment,Tag,True,1,Comment -> Tag


In [110]:
# =========================================================
# BLOCK 5F — Preliminary D per official query
# =========================================================
#
# Goal:
# - Compute preliminary required depth D.
# - Correct self-relationship queries such as Person-knows-Person.
# =========================================================

SELF_RELATIONSHIPS_WITH_DEPTH_ONE = {
    "person_knows_person",
    "comment_reply_of_comment",
    "place_is_part_of_place",
    "tagclass_is_subclass_of_tagclass",
}

preliminary_depth_rows = []

for query in SNB_OFFICIAL_WORKLOAD:
    query_name = query["query_name"]
    official_id = query["official_id"]
    root = query["root_entity"]
    rels = set(query["relationships_used"])

    query_paths = snb_workload_paths_df[
        snb_workload_paths_df["query_name"] == query_name
    ]

    max_distance = query_paths["distance"].max()
    all_reachable = bool(query_paths["reachable"].all())

    # Correction for self-relationships.
    uses_self_relationship = len(rels.intersection(SELF_RELATIONSHIPS_WITH_DEPTH_ONE)) > 0

    if uses_self_relationship and (max_distance is None or max_distance < 1):
        adjusted_D = 1
    else:
        adjusted_D = max_distance

    preliminary_depth_rows.append({
        "query_name": query_name,
        "official_id": official_id,
        "root_entity": root,
        "preliminary_D_raw": max_distance,
        "uses_self_relationship": uses_self_relationship,
        "preliminary_D": adjusted_D,
        "all_reachable": all_reachable,
    })

snb_preliminary_depth_df = pd.DataFrame(preliminary_depth_rows)
snb_preliminary_depth_df

,query_name,official_id,root_entity,preliminary_D_raw,uses_self_relationship,preliminary_D,all_reachable
0,IS1_ProfileOfPerson,IS1,Person,1,False,1,True
1,IS2_RecentMessagesOfPerson,IS2,Person,1,True,1,True
2,IS3_FriendsOfPerson,IS3,Person,0,True,1,True
3,IS4_ContentOfMessage,IS4,Post,1,False,1,True
4,IS5_CreatorOfMessage,IS5,Post,1,False,1,True
5,IS6_ForumOfMessage,IS6,Post,1,True,1,True
6,IS7_RepliesOfMessage,IS7,Post,1,True,1,True
7,IC1_TransitiveFriendsWithName,IC1,Person,1,True,1,True
8,IC2_RecentMessagesByFriends,IC2,Person,1,True,1,True
9,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,Person,1,True,1,True


In [111]:
# =========================================================
# BLOCK 5H — Official query-path depth overrides
# =========================================================
#
# Goal:
# - Add official query-path depth annotations.
# - The graph-based preliminary_D only captures entity-type distance.
# - Official LDBC queries may repeat the same relationship, especially
#   Person-knows-Person traversals.
# - Therefore, we define a methodology-level D override per query.
# =========================================================

SNB_OFFICIAL_QUERY_PATH_OVERRIDES = {
    # -----------------------------------------------------
    # Short reads
    # -----------------------------------------------------
    "IS1_ProfileOfPerson": {
        "official_D": 1,
        "note": "Person profile plus location place."
    },

    "IS2_RecentMessagesOfPerson": {
        "official_D": 2,
        "note": "Person -> Message plus possible Comment -> original Post traversal."
    },

    "IS3_FriendsOfPerson": {
        "official_D": 1,
        "note": "Person -> knows -> Person."
    },

    "IS4_ContentOfMessage": {
        "official_D": 0,
        "note": "Direct lookup of Message content; represented physically as Post or Comment."
    },

    "IS5_CreatorOfMessage": {
        "official_D": 1,
        "note": "Message -> creator Person."
    },

    "IS6_ForumOfMessage": {
        "official_D": 3,
        "note": "Message -> original Post -> Forum -> Moderator Person. Comments may require reply traversal."
    },

    "IS7_RepliesOfMessage": {
        "official_D": 2,
        "note": "Message -> replies -> reply creator, plus friendship check."
    },

    # -----------------------------------------------------
    # Complex reads, first stage
    # -----------------------------------------------------
    "IC1_TransitiveFriendsWithName": {
        "official_D": 4,
        "note": "Up to 3 knows hops plus profile/organisation/place expansion."
    },

    "IC2_RecentMessagesByFriends": {
        "official_D": 2,
        "note": "Person -> friend -> recent messages."
    },

    "IC3_FriendsAndFriendsOfFriendsInCountries": {
        "official_D": 4,
        "note": "Friends/friends-of-friends plus messages and country/place traversal."
    },

    "IC4_NewTopics": {
        "official_D": 3,
        "note": "Person -> friends -> posts -> tags."
    },

    "IC5_NewGroups": {
        "official_D": 3,
        "note": "Person -> friends/friends-of-friends -> forums -> posts."
    },

    "IC6_TagCoOccurrence": {
        "official_D": 3,
        "note": "Person -> friends/friends-of-friends -> posts -> tags."
    },

    "IC7_RecentLikers": {
        "official_D": 2,
        "note": "Person messages -> likers, plus knows check."
    },

    # -----------------------------------------------------
    # Inserts
    # -----------------------------------------------------
    "INS1_AddPerson": {
        "official_D": 1,
        "note": "Insert Person plus direct relationships to Place, Tag and Organisation."
    },

    "INS2_AddLikeToPost": {
        "official_D": 1,
        "note": "Person -> likes -> Post."
    },

    "INS3_AddLikeToComment": {
        "official_D": 1,
        "note": "Person -> likes -> Comment."
    },

    "INS4_AddForum": {
        "official_D": 1,
        "note": "Forum plus moderator and tags."
    },

    "INS5_AddForumMembership": {
        "official_D": 1,
        "note": "Forum -> hasMember -> Person."
    },

    "INS6_AddPost": {
        "official_D": 1,
        "note": "Post plus creator, forum, place and tags."
    },

    "INS7_AddComment": {
        "official_D": 1,
        "note": "Comment plus creator, reply target, place and tags."
    },

    "INS8_AddFriendship": {
        "official_D": 1,
        "note": "Person -> knows -> Person."
    },
}

depth_override_rows = []

for query in SNB_OFFICIAL_WORKLOAD:
    qname = query["query_name"]
    override = SNB_OFFICIAL_QUERY_PATH_OVERRIDES.get(qname, {})

    depth_override_rows.append({
        "query_name": qname,
        "official_id": query["official_id"],
        "official_D": override.get("official_D"),
        "depth_note": override.get("note"),
    })

snb_query_depth_overrides_df = pd.DataFrame(depth_override_rows)
snb_query_depth_overrides_df

,query_name,official_id,official_D,depth_note
0,IS1_ProfileOfPerson,IS1,1,Person profile plus location place.
1,IS2_RecentMessagesOfPerson,IS2,2,Person -> Message plus possible Comment -> original Post traversal.
2,IS3_FriendsOfPerson,IS3,1,Person -> knows -> Person.
3,IS4_ContentOfMessage,IS4,0,Direct lookup of Message content; represented physically as Post or Comment.
4,IS5_CreatorOfMessage,IS5,1,Message -> creator Person.
5,IS6_ForumOfMessage,IS6,3,Message -> original Post -> Forum -> Moderator Person. Comments may require reply traversal.
6,IS7_RepliesOfMessage,IS7,2,"Message -> replies -> reply creator, plus friendship check."
7,IC1_TransitiveFriendsWithName,IC1,4,Up to 3 knows hops plus profile/organisation/place expansion.
8,IC2_RecentMessagesByFriends,IC2,2,Person -> friend -> recent messages.
9,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,4,Friends/friends-of-friends plus messages and country/place traversal.


In [112]:
# =========================================================
# BLOCK 5I — Build corrected official depth matrix
# =========================================================
#
# Goal:
# - Combine graph-based preliminary depth with official query-path depth.
# - Use official_D as final D when available.
# =========================================================

snb_corrected_depth_df = (
    snb_preliminary_depth_df
    .merge(
        snb_query_depth_overrides_df,
        on=["query_name", "official_id"],
        how="left"
    )
)

snb_corrected_depth_df["D"] = snb_corrected_depth_df["official_D"].combine_first(
    snb_corrected_depth_df["preliminary_D"]
)

snb_corrected_depth_df["depth_source"] = snb_corrected_depth_df["official_D"].apply(
    lambda x: "official_query_path_override" if pd.notna(x) else "graph_preliminary"
)

snb_corrected_depth_df = snb_corrected_depth_df[
    [
        "query_name",
        "official_id",
        "root_entity",
        "preliminary_D_raw",
        "uses_self_relationship",
        "preliminary_D",
        "official_D",
        "D",
        "depth_source",
        "all_reachable",
        "depth_note",
    ]
]

snb_corrected_depth_df

,query_name,official_id,root_entity,preliminary_D_raw,uses_self_relationship,preliminary_D,official_D,D,depth_source,all_reachable,depth_note
0,IS1_ProfileOfPerson,IS1,Person,1,False,1,1,1,official_query_path_override,True,Person profile plus location place.
1,IS2_RecentMessagesOfPerson,IS2,Person,1,True,1,2,2,official_query_path_override,True,Person -> Message plus possible Comment -> original Post traversal.
2,IS3_FriendsOfPerson,IS3,Person,0,True,1,1,1,official_query_path_override,True,Person -> knows -> Person.
3,IS4_ContentOfMessage,IS4,Post,1,False,1,0,0,official_query_path_override,True,Direct lookup of Message content; represented physically as Post or Comment.
4,IS5_CreatorOfMessage,IS5,Post,1,False,1,1,1,official_query_path_override,True,Message -> creator Person.
5,IS6_ForumOfMessage,IS6,Post,1,True,1,3,3,official_query_path_override,True,Message -> original Post -> Forum -> Moderator Person. Comments may require reply traversal.
6,IS7_RepliesOfMessage,IS7,Post,1,True,1,2,2,official_query_path_override,True,"Message -> replies -> reply creator, plus friendship check."
7,IC1_TransitiveFriendsWithName,IC1,Person,1,True,1,4,4,official_query_path_override,True,Up to 3 knows hops plus profile/organisation/place expansion.
8,IC2_RecentMessagesByFriends,IC2,Person,1,True,1,2,2,official_query_path_override,True,Person -> friend -> recent messages.
9,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,Person,1,True,1,4,4,official_query_path_override,True,Friends/friends-of-friends plus messages and country/place traversal.


In [113]:
# =========================================================
# BLOCK 6A — Compute initial Rc for official LDBC SNB workload
# =========================================================
#
# Goal:
# - Compute relationship-count metrics per query.
# - Rc_distinct counts distinct relationship types.
# - Rc_weighted counts repeated relationship traversal when known.
# =========================================================

SNB_RELATIONSHIP_MULTIPLICITY_OVERRIDES = {
    "IC1_TransitiveFriendsWithName": {
        "person_knows_person": 3,
    },

    "IC3_FriendsAndFriendsOfFriendsInCountries": {
        "person_knows_person": 2,
    },

    "IC5_NewGroups": {
        "person_knows_person": 2,
    },

    "IC6_TagCoOccurrence": {
        "person_knows_person": 2,
    },
}

rc_rows = []

for query in SNB_OFFICIAL_WORKLOAD:
    qname = query["query_name"]
    official_id = query["official_id"]
    rels = query["relationships_used"]

    multiplicity_override = SNB_RELATIONSHIP_MULTIPLICITY_OVERRIDES.get(qname, {})

    rc_weighted = 0
    weighted_relationships = []

    for rel in rels:
        m = multiplicity_override.get(rel, 1)
        rc_weighted += m
        weighted_relationships.append(f"{rel} x{m}")

    rc_rows.append({
        "query_name": qname,
        "official_id": official_id,
        "Rc_distinct": len(set(rels)),
        "Rc_weighted": rc_weighted,
        "weighted_relationships": weighted_relationships,
    })

snb_rc_df = pd.DataFrame(rc_rows)
snb_rc_df

,query_name,official_id,Rc_distinct,Rc_weighted,weighted_relationships
0,IS1_ProfileOfPerson,IS1,1,1,[person_is_located_in_place x1]
1,IS2_RecentMessagesOfPerson,IS2,4,4,"[post_has_creator_person x1, comment_has_creator_person x1, comment_reply_of_post x1, comment_reply_of_comment x1]"
2,IS3_FriendsOfPerson,IS3,1,1,[person_knows_person x1]
3,IS4_ContentOfMessage,IS4,0,0,[]
4,IS5_CreatorOfMessage,IS5,2,2,"[post_has_creator_person x1, comment_has_creator_person x1]"
5,IS6_ForumOfMessage,IS6,4,4,"[forum_container_of_post x1, comment_reply_of_post x1, comment_reply_of_comment x1, forum_has_moderator_person x1]"
6,IS7_RepliesOfMessage,IS7,5,5,"[comment_reply_of_post x1, comment_reply_of_comment x1, comment_has_creator_person x1, post_has_creator_person x1, person_knows_person x1]"
7,IC1_TransitiveFriendsWithName,IC1,5,7,"[person_knows_person x3, person_is_located_in_place x1, person_study_at_organisation x1, person_work_at_organisation x1, organisation_is_located_in_place x1]"
8,IC2_RecentMessagesByFriends,IC2,3,3,"[person_knows_person x1, post_has_creator_person x1, comment_has_creator_person x1]"
9,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,7,8,"[person_knows_person x2, person_is_located_in_place x1, post_has_creator_person x1, comment_has_creator_person x1, post_is_located_in_place x1, comment_is_located_in_place x1, place_is_part_of_place x1]"


In [114]:
# =========================================================
# BLOCK 6B — Build official LDBC SNB structural matrix
# =========================================================
#
# Goal:
# - Build the first structural matrix for the official SNB workload.
# - This matrix prepares the later computation of Re, DeltaR,
#   DeltaRratio, volatility and sharedness.
# =========================================================

snb_structural_matrix_df = (
    snb_workload_df
    .merge(snb_query_semantic_profile_df, on=["query_name", "official_id"], how="left")
    .merge(snb_corrected_depth_df[["query_name", "official_id", "D", "depth_source"]], 
           on=["query_name", "official_id"], how="left")
    .merge(snb_rc_df[["query_name", "official_id", "Rc_distinct", "Rc_weighted", "weighted_relationships"]],
           on=["query_name", "official_id"], how="left")
)

snb_structural_matrix_df = snb_structural_matrix_df[
    [
        "query_name",
        "official_id",
        "official_title",
        "query_group",
        "operation_type",
        "root_entity",
        "accessed_entities",
        "relationships_used",
        "Rc_distinct",
        "Rc_weighted",
        "D",
        "depth_source",
        "association_count",
        "associative_count",
        "containment_count",
        "lookup_count",
        "dominant_semantic_type",
        "weighted_relationships",
        "description",
    ]
]

snb_structural_matrix_df

,query_name,official_id,official_title,query_group,operation_type,root_entity,accessed_entities,relationships_used,Rc_distinct,Rc_weighted,D,depth_source,association_count,associative_count,containment_count,lookup_count,dominant_semantic_type,weighted_relationships,description
0,IS1_ProfileOfPerson,IS1,Profile of a person,short_read,read,Person,"[Person, Place]",[person_is_located_in_place],1,1,1,official_query_path_override,1,0,0,0,association,[person_is_located_in_place x1],Official IS1: retrieve a person's profile and city of residence.
1,IS2_RecentMessagesOfPerson,IS2,Recent messages of a person,short_read,read,Person,"[Person, Post, Comment]","[post_has_creator_person, comment_has_creator_person, comment_reply_of_post, comment_reply_of_comment]",4,4,2,official_query_path_override,2,0,2,0,mixed,"[post_has_creator_person x1, comment_has_creator_person x1, comment_reply_of_post x1, comment_reply_of_comment x1]",Official IS2: retrieve the last messages created by a person and the original post in the conversation.
2,IS3_FriendsOfPerson,IS3,Friends of a person,short_read,read,Person,[Person],[person_knows_person],1,1,1,official_query_path_override,1,0,0,0,association,[person_knows_person x1],Official IS3: retrieve all friends of a person and the friendship creation date.
3,IS4_ContentOfMessage,IS4,Content of a message,short_read,read,Post,"[Post, Comment]",[],0,0,0,official_query_path_override,0,0,0,1,lookup,[],Official IS4: retrieve content and creation date of a Message. Message is represented by Post or Comment.
4,IS5_CreatorOfMessage,IS5,Creator of a message,short_read,read,Post,"[Post, Comment, Person]","[post_has_creator_person, comment_has_creator_person]",2,2,1,official_query_path_override,2,0,0,0,association,"[post_has_creator_person x1, comment_has_creator_person x1]",Official IS5: retrieve the author of a Message. Message is represented by Post or Comment.
5,IS6_ForumOfMessage,IS6,Forum of a message,short_read,read,Post,"[Post, Comment, Forum, Person]","[forum_container_of_post, comment_reply_of_post, comment_reply_of_comment, forum_has_moderator_person]",4,4,3,official_query_path_override,1,0,3,0,containment,"[forum_container_of_post x1, comment_reply_of_post x1, comment_reply_of_comment x1, forum_has_moderator_person x1]",Official IS6: retrieve the forum containing a Message and the moderator of that forum.
6,IS7_RepliesOfMessage,IS7,Replies of a message,short_read,read,Post,"[Post, Comment, Person]","[comment_reply_of_post, comment_reply_of_comment, comment_has_creator_person, post_has_creator_person, person_knows_person]",5,5,2,official_query_path_override,3,0,2,0,association,"[comment_reply_of_post x1, comment_reply_of_comment x1, comment_has_creator_person x1, post_has_creator_person x1, person_knows_person x1]",Official IS7: retrieve one-hop replies of a Message and whether reply author knows original author.
7,IC1_TransitiveFriendsWithName,IC1,Transitive friends with a certain name,complex_read,read,Person,"[Person, Place, Organisation]","[person_knows_person, person_is_located_in_place, person_study_at_organisation, person_work_at_organisation, organisation_is_located_in_place]",5,7,4,official_query_path_override,5,0,0,0,association,"[person_knows_person x3, person_is_located_in_place x1, person_study_at_organisation x1, person_work_at_organisation x1, organisation_is_located_in_place x1]","Official IC1: find people connected by up to 3 knows hops with a given first name, including workplaces and studies."
8,IC2_RecentMessagesByFriends,IC2,Recent messages by your friends,complex_read,read,Person,"[Person, Post, Comment]","[person_knows_person, post_has_creator_person, comment_has_creator_person]",3,3,2,official_query_path_override,3,0,0,0,association,"[person_knows_person x1, post_has_creator_person x1, comment_has_creator_person x1]",Official IC2: find recent messages created by the friends of a person.
9,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,Friends and friends of friends that have been to given 

In [115]:
# =========================================================
# BLOCK 6C — Save corrected structural workload artifacts
# =========================================================

STRUCTURAL_ARTIFACTS_DIR = BASE_DIR.parent.parent / "benchmark_artifacts_dir" / "ldbc_snb_structural_matrix_official"
STRUCTURAL_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

snb_query_depth_overrides_df.to_csv(
    STRUCTURAL_ARTIFACTS_DIR / "snb_query_depth_overrides.csv",
    index=False
)

snb_corrected_depth_df.to_csv(
    STRUCTURAL_ARTIFACTS_DIR / "snb_corrected_depth.csv",
    index=False
)

snb_rc_df.to_csv(
    STRUCTURAL_ARTIFACTS_DIR / "snb_rc_metrics.csv",
    index=False
)

snb_structural_matrix_df.to_csv(
    STRUCTURAL_ARTIFACTS_DIR / "snb_structural_matrix_official.csv",
    index=False
)

print("Saved structural artifacts to:")
print(STRUCTURAL_ARTIFACTS_DIR)

Saved structural artifacts to:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_structural_matrix_official


In [116]:
# =========================================================
# BLOCK 7A — Compute Re, DeltaR and DeltaRratio
# =========================================================
#
# Goal:
# - Compute residual explicit traversal (Re).
# - Compute structural reduction (DeltaR).
# - Compute normalized structural reduction (DeltaRratio).
#
# Interpretation:
# - Rc_weighted represents the total relationship cost of the query.
# - D represents the main root-centric traversal depth.
# - Re is the remaining relationship cost not covered by the main path.
# =========================================================

import pandas as pd

snb_reduction_rows = []

for _, row in snb_structural_matrix_df.iterrows():
    qname = row["query_name"]
    official_id = row["official_id"]
    rc = int(row["Rc_weighted"]) if pd.notna(row["Rc_weighted"]) else 0
    d = int(row["D"]) if pd.notna(row["D"]) else 0

    if rc == 0:
        re = 0
        delta_r = 0
        delta_r_ratio = 0.0
    else:
        re = max(rc - d, 0)
        delta_r = rc - re
        delta_r_ratio = delta_r / rc

    snb_reduction_rows.append({
        "query_name": qname,
        "official_id": official_id,
        "Rc_weighted": rc,
        "D": d,
        "Re": re,
        "DeltaR": delta_r,
        "DeltaRratio": delta_r_ratio,
    })

snb_reduction_metrics_df = pd.DataFrame(snb_reduction_rows)

snb_reduction_metrics_df

,query_name,official_id,Rc_weighted,D,Re,DeltaR,DeltaRratio
0,IS1_ProfileOfPerson,IS1,1,1,0,1,1.000000
1,IS2_RecentMessagesOfPerson,IS2,4,2,2,2,0.500000
2,IS3_FriendsOfPerson,IS3,1,1,0,1,1.000000
3,IS4_ContentOfMessage,IS4,0,0,0,0,0.000000
4,IS5_CreatorOfMessage,IS5,2,1,1,1,0.500000
5,IS6_ForumOfMessage,IS6,4,3,1,3,0.750000
6,IS7_RepliesOfMessage,IS7,5,2,3,2,0.400000
7,IC1_TransitiveFriendsWithName,IC1,7,4,3,4,0.571429
8,IC2_RecentMessagesByFriends,IC2,3,2,1,2,0.666667
9,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,8,4,4,4,0.500000


In [117]:
# =========================================================
# BLOCK 7B — Add reduction metrics to structural matrix
# =========================================================
#
# Goal:
# - Extend the official structural matrix with Re, DeltaR and DeltaRratio.
# =========================================================

snb_structural_reduction_matrix_df = (
    snb_structural_matrix_df
    .merge(
        snb_reduction_metrics_df[
            ["query_name", "official_id", "Re", "DeltaR", "DeltaRratio"]
        ],
        on=["query_name", "official_id"],
        how="left"
    )
)

snb_structural_reduction_matrix_df = snb_structural_reduction_matrix_df[
    [
        "query_name",
        "official_id",
        "official_title",
        "query_group",
        "operation_type",
        "root_entity",
        "accessed_entities",
        "relationships_used",
        "Rc_distinct",
        "Rc_weighted",
        "D",
        "Re",
        "DeltaR",
        "DeltaRratio",
        "association_count",
        "associative_count",
        "containment_count",
        "lookup_count",
        "dominant_semantic_type",
        "weighted_relationships",
        "description",
    ]
]

snb_structural_reduction_matrix_df

,query_name,official_id,official_title,query_group,operation_type,root_entity,accessed_entities,relationships_used,Rc_distinct,Rc_weighted,D,Re,DeltaR,DeltaRratio,association_count,associative_count,containment_count,lookup_count,dominant_semantic_type,weighted_relationships,description
0,IS1_ProfileOfPerson,IS1,Profile of a person,short_read,read,Person,"[Person, Place]",[person_is_located_in_place],1,1,1,0,1,1.000000,1,0,0,0,association,[person_is_located_in_place x1],Official IS1: retrieve a person's profile and city of residence.
1,IS2_RecentMessagesOfPerson,IS2,Recent messages of a person,short_read,read,Person,"[Person, Post, Comment]","[post_has_creator_person, comment_has_creator_person, comment_reply_of_post, comment_reply_of_comment]",4,4,2,2,2,0.500000,2,0,2,0,mixed,"[post_has_creator_person x1, comment_has_creator_person x1, comment_reply_of_post x1, comment_reply_of_comment x1]",Official IS2: retrieve the last messages created by a person and the original post in the conversation.
2,IS3_FriendsOfPerson,IS3,Friends of a person,short_read,read,Person,[Person],[person_knows_person],1,1,1,0,1,1.000000,1,0,0,0,association,[person_knows_person x1],Official IS3: retrieve all friends of a person and the friendship creation date.
3,IS4_ContentOfMessage,IS4,Content of a message,short_read,read,Post,"[Post, Comment]",[],0,0,0,0,0,0.000000,0,0,0,1,lookup,[],Official IS4: retrieve content and creation date of a Message. Message is represented by Post or Comment.
4,IS5_CreatorOfMessage,IS5,Creator of a message,short_read,read,Post,"[Post, Comment, Person]","[post_has_creator_person, comment_has_creator_person]",2,2,1,1,1,0.500000,2,0,0,0,association,"[post_has_creator_person x1, comment_has_creator_person x1]",Official IS5: retrieve the author of a Message. Message is represented by Post or Comment.
5,IS6_ForumOfMessage,IS6,Forum of a message,short_read,read,Post,"[Post, Comment, Forum, Person]","[forum_container_of_post, comment_reply_of_post, comment_reply_of_comment, forum_has_moderator_person]",4,4,3,1,3,0.750000,1,0,3,0,containment,"[forum_container_of_post x1, comment_reply_of_post x1, comment_reply_of_comment x1, forum_has_moderator_person x1]",Official IS6: retrieve the forum containing a Message and the moderator of that forum.
6,IS7_RepliesOfMessage,IS7,Replies of a message,short_read,read,Post,"[Post, Comment, Person]","[comment_reply_of_post, comment_reply_of_comment, comment_has_creator_person, post_has_creator_person, person_knows_person]",5,5,2,3,2,0.400000,3,0,2,0,association,"[comment_reply_of_post x1, comment_reply_of_comment x1, comment_has_creator_person x1, post_has_creator_person x1, person_knows_person x1]",Official IS7: retrieve one-hop replies of a Message and whether reply author knows original author.
7,IC1_TransitiveFriendsWithName,IC1,Transitive friends with a certain name,complex_read,read,Person,"[Person, Place, Organisation]","[person_knows_person, person_is_located_in_place, person_study_at_organisation, person_work_at_organisation, organisation_is_located_in_place]",5,7,4,3,4,0.571429,5,0,0,0,association,"[person_knows_person x3, person_is_located_in_place x1, person_study_at_organisation x1, person_work_at_organisation x1, organisation_is_located_in_place x1]","Official IC1: find people connected by up to 3 knows hops with a given first name, including workplaces and studies."
8,IC2_RecentMessagesByFriends,IC2,Recent messages by your friends,complex_read,read,Person,"[Person, Post, Comment]","[person_knows_person, post_has_creator_person, comment_has_creator_person]",3,3,2,1,2,0.666667,3,0,0,0,association,"[person_knows_person x1, post_has_creator_person x1, comment_has_creator_person x1]",Official IC2: find recent messages created by the friends of a person.
9,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,Friends and friends of friends that have been to given countries,complex_read,read,Person,"[Person, Post, Comment, Place]","[person_knows_person, person_is_located_in_place, post_has_creator

In [118]:
# =========================================================
# BLOCK 7C — Rank queries by structural reduction potential
# =========================================================
#
# Goal:
# - Identify queries with high structural reduction potential.
# - Higher DeltaRratio means the root-centric design may reduce more traversal.
# =========================================================

snb_reduction_rank_df = (
    snb_structural_reduction_matrix_df[
        [
            "query_name",
            "official_id",
            "query_group",
            "root_entity",
            "Rc_weighted",
            "D",
            "Re",
            "DeltaR",
            "DeltaRratio",
            "dominant_semantic_type",
        ]
    ]
    .sort_values(
        by=["DeltaRratio", "DeltaR", "Rc_weighted"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

snb_reduction_rank_df

,query_name,official_id,query_group,root_entity,Rc_weighted,D,Re,DeltaR,DeltaRratio,dominant_semantic_type
0,IC4_NewTopics,IC4,complex_read,Person,3,3,0,3,1.000000,association
1,IS1_ProfileOfPerson,IS1,short_read,Person,1,1,0,1,1.000000,association
2,IS3_FriendsOfPerson,IS3,short_read,Person,1,1,0,1,1.000000,association
3,INS2_AddLikeToPost,INS2,insert,Person,1,1,0,1,1.000000,associative
4,INS3_AddLikeToComment,INS3,insert,Person,1,1,0,1,1.000000,associative
5,INS5_AddForumMembership,INS5,insert,Forum,1,1,0,1,1.000000,associative
6,INS8_AddFriendship,INS8,insert,Person,1,1,0,1,1.000000,association
7,IS6_ForumOfMessage,IS6,short_read,Post,4,3,1,3,0.750000,containment
8,IC6_TagCoOccurrence,IC6,complex_read,Person,4,3,1,3,0.750000,association
9,IC2_RecentMessagesByFriends,IC2,complex_read,Person,3,2,1,2,0.666667,association


In [119]:
# =========================================================
# BLOCK 7D — Validate structural reduction metrics
# =========================================================
#
# Goal:
# - Check for impossible values.
# - DeltaR must be between 0 and Rc_weighted.
# - DeltaRratio must be between 0 and 1.
# =========================================================

validation_rows = []

for _, row in snb_reduction_metrics_df.iterrows():
    rc = row["Rc_weighted"]
    d = row["D"]
    re = row["Re"]
    delta = row["DeltaR"]
    ratio = row["DeltaRratio"]

    validation_rows.append({
        "query_name": row["query_name"],
        "official_id": row["official_id"],
        "Rc_weighted": rc,
        "D": d,
        "Re": re,
        "DeltaR": delta,
        "DeltaRratio": ratio,
        "valid_Re": re >= 0,
        "valid_DeltaR": 0 <= delta <= rc if rc > 0 else delta == 0,
        "valid_DeltaRratio": 0 <= ratio <= 1,
    })

snb_reduction_validation_df = pd.DataFrame(validation_rows)

snb_reduction_validation_df

,query_name,official_id,Rc_weighted,D,Re,DeltaR,DeltaRratio,valid_Re,valid_DeltaR,valid_DeltaRratio
0,IS1_ProfileOfPerson,IS1,1,1,0,1,1.000000,True,True,True
1,IS2_RecentMessagesOfPerson,IS2,4,2,2,2,0.500000,True,True,True
2,IS3_FriendsOfPerson,IS3,1,1,0,1,1.000000,True,True,True
3,IS4_ContentOfMessage,IS4,0,0,0,0,0.000000,True,True,True
4,IS5_CreatorOfMessage,IS5,2,1,1,1,0.500000,True,True,True
5,IS6_ForumOfMessage,IS6,4,3,1,3,0.750000,True,True,True
6,IS7_RepliesOfMessage,IS7,5,2,3,2,0.400000,True,True,True
7,IC1_TransitiveFriendsWithName,IC1,7,4,3,4,0.571429,True,True,True
8,IC2_RecentMessagesByFriends,IC2,3,2,1,2,0.666667,True,True,True
9,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,8,4,4,4,0.500000,True,True,True


In [120]:
snb_reduction_validation_df[
    (snb_reduction_validation_df["valid_Re"] == False)
    | (snb_reduction_validation_df["valid_DeltaR"] == False)
    | (snb_reduction_validation_df["valid_DeltaRratio"] == False)
]

,query_name,official_id,Rc_weighted,D,Re,DeltaR,DeltaRratio,valid_Re,valid_DeltaR,valid_DeltaRratio


In [121]:
# =========================================================
# BLOCK 7E — Save structural reduction artifacts
# =========================================================

REDUCTION_ARTIFACTS_DIR = BASE_DIR.parent.parent / "benchmark_artifacts_dir" / "ldbc_snb_structural_reduction_official"
REDUCTION_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

snb_reduction_metrics_df.to_csv(
    REDUCTION_ARTIFACTS_DIR / "snb_reduction_metrics.csv",
    index=False
)

snb_structural_reduction_matrix_df.to_csv(
    REDUCTION_ARTIFACTS_DIR / "snb_structural_reduction_matrix.csv",
    index=False
)

snb_reduction_rank_df.to_csv(
    REDUCTION_ARTIFACTS_DIR / "snb_reduction_rank.csv",
    index=False
)

snb_reduction_validation_df.to_csv(
    REDUCTION_ARTIFACTS_DIR / "snb_reduction_validation.csv",
    index=False
)

print("Saved reduction artifacts to:")
print(REDUCTION_ARTIFACTS_DIR)

Saved reduction artifacts to:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_structural_reduction_official


In [122]:
# =========================================================
# CHECKPOINT 1 — Save LDBC SNB notebook state
# =========================================================
#
# Goal:
# - Save all important DataFrames and metadata dictionaries.
# - This allows resuming the notebook without recomputing all previous blocks.
# =========================================================

from pathlib import Path
import json
import pandas as pd

PROJECT_DIR = BASE_DIR.parent.parent

CHECKPOINT_DIR = PROJECT_DIR / "benchmark_artifacts_dir" / "ldbc_snb_checkpoints"
CHECKPOINT_DATAFRAMES_DIR = CHECKPOINT_DIR / "dataframes"
CHECKPOINT_METADATA_DIR = CHECKPOINT_DIR / "metadata"

CHECKPOINT_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_METADATA_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# DataFrames to save
# ---------------------------------------------------------

CHECKPOINT_DATAFRAMES = [
    "columns_df",
    "raw_counts_df",
    "conceptual_counts_df",
    "clean_entity_counts_df",
    "fk_relationship_views_df",
    "join_validation_df",
    "snb_entity_metadata_df",
    "snb_relationship_hints_df",
    "snb_schema_edges_df",
    "snb_graph_connectivity_df",
    "snb_graph_connectivity_summary_df",
    "important_paths_df",
    "snb_traversal_connectivity_df",
    "snb_traversal_connectivity_summary_df",
    "important_traversal_paths_df",
    "snb_workload_df",
    "workload_relationship_validation_df",
    "snb_query_semantic_profile_df",
    "snb_initial_workload_matrix_df",
    "snb_workload_paths_df",
    "snb_preliminary_depth_df",
    "snb_query_depth_overrides_df",
    "snb_corrected_depth_df",
    "snb_rc_df",
    "snb_structural_matrix_df",
    "snb_reduction_metrics_df",
    "snb_structural_reduction_matrix_df",
    "snb_reduction_rank_df",
    "snb_reduction_validation_df",
]

saved_dataframes = {}

for var_name in CHECKPOINT_DATAFRAMES:
    obj = globals().get(var_name)

    if isinstance(obj, pd.DataFrame):
        out_path = CHECKPOINT_DATAFRAMES_DIR / f"{var_name}.json"

        obj.to_json(
            out_path,
            orient="records",
            indent=2,
            force_ascii=False
        )

        saved_dataframes[var_name] = str(out_path.relative_to(CHECKPOINT_DIR))

print("Saved DataFrames:")
for name in saved_dataframes:
    print(" -", name)

# ---------------------------------------------------------
# Metadata to save
# ---------------------------------------------------------

checkpoint_metadata = {
    "BASE_DIR": str(BASE_DIR),
    "SNB_SNAPSHOT_DIR": str(SNB_SNAPSHOT_DIR),
    "SNB_DYNAMIC_DIR": str(SNB_DYNAMIC_DIR),
    "SNB_STATIC_DIR": str(SNB_STATIC_DIR),
    "SNB_SUBSTITUTION_DIR": str(SNB_SUBSTITUTION_DIR),
    "SNB_UPDATE_DIR": str(SNB_UPDATE_DIR),

    "SNB_ENTITY_VIEW_MAP": SNB_ENTITY_VIEW_MAP,
    "SNB_ENTITY_PK_MAP": SNB_ENTITY_PK_MAP,
    "SNB_COLLECTION_NAME_BY_ENTITY": SNB_COLLECTION_NAME_BY_ENTITY,
    "SNB_RELATIONSHIP_HINTS": SNB_RELATIONSHIP_HINTS,

    "SNB_OFFICIAL_WORKLOAD": SNB_OFFICIAL_WORKLOAD,
    "SNB_OFFICIAL_QUERY_PATH_OVERRIDES": SNB_OFFICIAL_QUERY_PATH_OVERRIDES,
    "SNB_RELATIONSHIP_MULTIPLICITY_OVERRIDES": SNB_RELATIONSHIP_MULTIPLICITY_OVERRIDES,
}

metadata_path = CHECKPOINT_METADATA_DIR / "checkpoint_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(checkpoint_metadata, f, indent=2, ensure_ascii=False)

manifest_path = CHECKPOINT_DIR / "checkpoint_manifest.json"

checkpoint_manifest = {
    "dataframes": saved_dataframes,
    "metadata": str(metadata_path.relative_to(CHECKPOINT_DIR)),
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(checkpoint_manifest, f, indent=2, ensure_ascii=False)

print("\nCheckpoint saved to:")
print(CHECKPOINT_DIR)

Saved DataFrames:
 - columns_df
 - raw_counts_df
 - conceptual_counts_df
 - clean_entity_counts_df
 - fk_relationship_views_df
 - join_validation_df
 - snb_entity_metadata_df
 - snb_relationship_hints_df
 - snb_schema_edges_df
 - snb_graph_connectivity_df
 - snb_graph_connectivity_summary_df
 - important_paths_df
 - snb_traversal_connectivity_df
 - snb_traversal_connectivity_summary_df
 - important_traversal_paths_df
 - snb_workload_df
 - workload_relationship_validation_df
 - snb_query_semantic_profile_df
 - snb_initial_workload_matrix_df
 - snb_workload_paths_df
 - snb_preliminary_depth_df
 - snb_query_depth_overrides_df
 - snb_corrected_depth_df
 - snb_rc_df
 - snb_structural_matrix_df
 - snb_reduction_metrics_df
 - snb_structural_reduction_matrix_df
 - snb_reduction_rank_df
 - snb_reduction_validation_df

Checkpoint saved to:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_checkpoints


In [123]:
# =========================================================
# CHECKPOINT 2 — Persist clean LDBC SNB DuckDB tables
# =========================================================
#
# Goal:
# - Save clean entity views and relationship views into a persistent DuckDB file.
# - After kernel restart, connect to this file instead of rebuilding all views.
# =========================================================

import duckdb
from pathlib import Path

DUCKDB_DIR = PROJECT_DIR / "duckdb"
DUCKDB_DIR.mkdir(parents=True, exist_ok=True)

PERSISTENT_DUCKDB_PATH = DUCKDB_DIR / "ldbc_snb_sf0_1_clean.duckdb"

persistent_con = duckdb.connect(str(PERSISTENT_DUCKDB_PATH))

views_to_persist = set()

# Entity views
for view_name in SNB_ENTITY_VIEW_MAP.values():
    views_to_persist.add(view_name)

# Relationship views
for spec in SNB_RELATIONSHIP_HINTS.values():
    views_to_persist.add(spec["view_name"])

# Optional attribute-like views useful for benchmark loading
extra_views = [
    "snb_person_emails_norm",
    "snb_person_languages_norm",
]

for view_name in extra_views:
    views_to_persist.add(view_name)

persist_rows = []

for view_name in sorted(views_to_persist):
    try:
        df = con.execute(f"SELECT * FROM {view_name}").df()

        persistent_con.register("__tmp_df", df)
        persistent_con.execute(f"""
            CREATE OR REPLACE TABLE {view_name} AS
            SELECT * FROM __tmp_df
        """)
        persistent_con.unregister("__tmp_df")

        n = persistent_con.execute(f"SELECT COUNT(*) FROM {view_name}").fetchone()[0]

        persist_rows.append({
            "table_name": view_name,
            "rows": n,
            "status": "OK"
        })

    except Exception as e:
        persist_rows.append({
            "table_name": view_name,
            "rows": None,
            "status": f"ERROR: {type(e).__name__}: {e}"
        })

persistent_con.close()

persistent_duckdb_df = pd.DataFrame(persist_rows)

# Save DuckDB path into metadata
checkpoint_metadata["PERSISTENT_DUCKDB_PATH"] = str(PERSISTENT_DUCKDB_PATH)

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(checkpoint_metadata, f, indent=2, ensure_ascii=False)

persistent_duckdb_df

,table_name,rows,status
0,snb_comment_hascreator_person,151043,OK
1,snb_comment_islocatedin_place,151043,OK
2,snb_comment_replyof_comment,76787,OK
3,snb_comment_replyof_post,74256,OK
4,snb_comment_tags_norm,191303,OK
5,snb_comments_entity,151044,OK
6,snb_forum_containerof_post,135701,OK
7,snb_forum_hasmoderator_person,13750,OK
8,snb_forum_members_norm,123268,OK
9,snb_forum_tags_norm,47697,OK


In [126]:
# =========================================================
# RESUME BLOCK — Restore LDBC SNB checkpoint
# =========================================================
#
# Goal:
# - Restore saved DataFrames, metadata dictionaries, and DuckDB connection.
# - Use this after kernel restart to continue from the last saved checkpoint.
# =========================================================

from pathlib import Path
import json
import pandas as pd
import duckdb
import networkx as nx

PROJECT_DIR = Path("/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB")

CHECKPOINT_DIR = PROJECT_DIR / "benchmark_artifacts_dir" / "ldbc_snb_checkpoints"
CHECKPOINT_DATAFRAMES_DIR = CHECKPOINT_DIR / "dataframes"
CHECKPOINT_METADATA_DIR = CHECKPOINT_DIR / "metadata"

manifest_path = CHECKPOINT_DIR / "checkpoint_manifest.json"

with open(manifest_path, "r", encoding="utf-8") as f:
    checkpoint_manifest = json.load(f)

metadata_path = CHECKPOINT_DIR / checkpoint_manifest["metadata"]

with open(metadata_path, "r", encoding="utf-8") as f:
    checkpoint_metadata = json.load(f)

# ---------------------------------------------------------
# Restore paths
# ---------------------------------------------------------

BASE_DIR = Path(checkpoint_metadata["BASE_DIR"])
SNB_SNAPSHOT_DIR = Path(checkpoint_metadata["SNB_SNAPSHOT_DIR"])
SNB_DYNAMIC_DIR = Path(checkpoint_metadata["SNB_DYNAMIC_DIR"])
SNB_STATIC_DIR = Path(checkpoint_metadata["SNB_STATIC_DIR"])
SNB_SUBSTITUTION_DIR = Path(checkpoint_metadata["SNB_SUBSTITUTION_DIR"])
SNB_UPDATE_DIR = Path(checkpoint_metadata["SNB_UPDATE_DIR"])

# ---------------------------------------------------------
# Restore metadata dictionaries/lists
# ---------------------------------------------------------

SNB_ENTITY_VIEW_MAP = checkpoint_metadata["SNB_ENTITY_VIEW_MAP"]
SNB_ENTITY_PK_MAP = checkpoint_metadata["SNB_ENTITY_PK_MAP"]
SNB_COLLECTION_NAME_BY_ENTITY = checkpoint_metadata["SNB_COLLECTION_NAME_BY_ENTITY"]
SNB_RELATIONSHIP_HINTS = checkpoint_metadata["SNB_RELATIONSHIP_HINTS"]

SNB_OFFICIAL_WORKLOAD = checkpoint_metadata["SNB_OFFICIAL_WORKLOAD"]
SNB_OFFICIAL_QUERY_PATH_OVERRIDES = checkpoint_metadata["SNB_OFFICIAL_QUERY_PATH_OVERRIDES"]
SNB_RELATIONSHIP_MULTIPLICITY_OVERRIDES = checkpoint_metadata["SNB_RELATIONSHIP_MULTIPLICITY_OVERRIDES"]

# ---------------------------------------------------------
# Restore DataFrames
# ---------------------------------------------------------

for var_name, relative_path in checkpoint_manifest["dataframes"].items():
    df_path = CHECKPOINT_DIR / relative_path
    globals()[var_name] = pd.read_json(df_path, orient="records")

print("Restored DataFrames:")
for name in checkpoint_manifest["dataframes"]:
    print(" -", name)

# ---------------------------------------------------------
# Restore DuckDB connection
# ---------------------------------------------------------

PERSISTENT_DUCKDB_PATH = Path(checkpoint_metadata["PERSISTENT_DUCKDB_PATH"])

con = duckdb.connect(str(PERSISTENT_DUCKDB_PATH))

print("\nConnected to persistent DuckDB:")
print(PERSISTENT_DUCKDB_PATH)

# ---------------------------------------------------------
# Rebuild graphs from schema edges
# ---------------------------------------------------------

snb_schema_graph = nx.DiGraph()

for entity in SNB_ENTITY_VIEW_MAP.keys():
    snb_schema_graph.add_node(entity)

for _, row in snb_schema_edges_df.iterrows():
    source = row["source_entity"]
    target = row["target_entity"]

    snb_schema_graph.add_edge(
        source,
        target,
        relationship=row["relationship"],
        semantic_type=row["semantic_type"],
        relationship_kind=row["relationship_kind"],
        view_name=row["view_name"],
        source_column=row["source_column"],
        target_column=row["target_column"],
    )

    if bool(row["is_bidirectional"]):
        snb_schema_graph.add_edge(
            target,
            source,
            relationship=str(row["relationship"]) + "_reverse",
            semantic_type=row["semantic_type"],
            relationship_kind=row["relationship_kind"],
            view_name=row["view_name"],
            source_column=row["target_column"],
            target_column=row["source_column"],
        )

snb_traversal_graph = nx.Graph()

for entity in SNB_ENTITY_VIEW_MAP.keys():
    snb_traversal_graph.add_node(entity)

for _, row in snb_schema_edges_df.iterrows():
    snb_traversal_graph.add_edge(
        row["source_entity"],
        row["target_entity"],
        relationship=row["relationship"],
        semantic_type=row["semantic_type"],
        relationship_kind=row["relationship_kind"],
        view_name=row["view_name"],
        source_column=row["source_column"],
        target_column=row["target_column"],
    )

print("\nGraphs rebuilt:")
print("Directed graph nodes:", snb_schema_graph.number_of_nodes())
print("Directed graph edges:", snb_schema_graph.number_of_edges())
print("Traversal graph nodes:", snb_traversal_graph.number_of_nodes())
print("Traversal graph edges:", snb_traversal_graph.number_of_edges())

# ---------------------------------------------------------
# Quick sanity check
# ---------------------------------------------------------

print("\nSanity check:")
print(con.execute("SELECT COUNT(*) AS n FROM snb_persons_entity").df())
print(con.execute("SELECT COUNT(*) AS n FROM snb_posts_entity").df())
print(con.execute("SELECT COUNT(*) AS n FROM snb_comments_entity").df())

Restored DataFrames:
 - columns_df
 - raw_counts_df
 - conceptual_counts_df
 - clean_entity_counts_df
 - fk_relationship_views_df
 - join_validation_df
 - snb_entity_metadata_df
 - snb_relationship_hints_df
 - snb_schema_edges_df
 - snb_graph_connectivity_df
 - snb_graph_connectivity_summary_df
 - important_paths_df
 - snb_traversal_connectivity_df
 - snb_traversal_connectivity_summary_df
 - important_traversal_paths_df
 - snb_workload_df
 - workload_relationship_validation_df
 - snb_query_semantic_profile_df
 - snb_initial_workload_matrix_df
 - snb_workload_paths_df
 - snb_preliminary_depth_df
 - snb_query_depth_overrides_df
 - snb_corrected_depth_df
 - snb_rc_df
 - snb_structural_matrix_df
 - snb_reduction_metrics_df
 - snb_structural_reduction_matrix_df
 - snb_reduction_rank_df
 - snb_reduction_validation_df

Connected to persistent DuckDB:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/duckdb/ldbc_snb_sf0_1_clean.duckdb

Graphs rebuilt:
Directed

In [127]:
# =========================================================
# BLOCK 8A — Inspect LDBC SNB update stream files
# =========================================================
#
# Goal:
# - Inspect the physical update stream files.
# - Understand whether they have headers, operation names, and usable fields.
# - This is a diagnostic block before computing update volatility.
# =========================================================

from pathlib import Path
import pandas as pd

print("SNB_UPDATE_DIR:", SNB_UPDATE_DIR)
print("Exists:", SNB_UPDATE_DIR.exists())

update_stream_files = sorted(SNB_UPDATE_DIR.glob("updateStream_*.csv"))

print("\nUpdate stream files:")
for path in update_stream_files:
    print(" -", path.name)

# Show first lines as raw text.
for path in update_stream_files:
    print("\n" + "=" * 100)
    print(path.name)
    print("=" * 100)

    with open(path, "r", encoding="utf-8") as f:
        for i in range(5):
            line = f.readline()
            if not line:
                break
            print(line.rstrip("\n"))

SNB_UPDATE_DIR: /home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/data/sf0.1/social_network-sf0.1-numpart-1
Exists: True

Update stream files:
 - updateStream_0_0_forum.csv
 - updateStream_0_0_person.csv

updateStream_0_0_forum.csv
1347529090363|1331161432355|7|1099511997932|1347529090363|1.12.242.179|Opera|roflol|6|26388279068220|1|-1|1099511997926|
1347529099239|1307796167976|5|1030792157789|17592186044532|1347529099239
1347529219074|1347166209438|2|35184372090183|481036434045|1347529219074
1347529227881|1281275886265|2|6597069766720|1030792332314|1347529227881
1347529249368|1342103263364|3|32985348834174|1030792257093|1347529249368

updateStream_0_0_person.csv
1347615339076|0|1|35184372090047|Witold|Ciesla|female|435456000000|1347615339076|31.183.168.247|Firefox|1282|pl;en|Witold35184372090047@gmail.com|205;1178;1760;1988;9097;9163;9317||979,2009
1347625494732|0|1|35184372089129|Ashin|Michie|female|487555200000|1347625494732|203.215.60.162|Chrome|10

In [128]:
# =========================================================
# BLOCK 8B — Read update stream samples
# =========================================================
#
# Goal:
# - Try to read update stream files as pipe-separated CSVs.
# - Store column names and row counts.
# =========================================================

update_stream_sample_rows = []

for path in update_stream_files:
    try:
        sample_df = pd.read_csv(
            path,
            sep="|",
            dtype=str,
            nrows=5,
            engine="python"
        )

        total_rows = sum(1 for _ in open(path, "r", encoding="utf-8")) - 1

        update_stream_sample_rows.append({
            "file_name": path.name,
            "n_columns": len(sample_df.columns),
            "columns": list(sample_df.columns),
            "sample_rows": sample_df.to_dict(orient="records"),
            "estimated_rows": total_rows,
            "status": "OK_header_true",
        })

    except Exception as e:
        update_stream_sample_rows.append({
            "file_name": path.name,
            "n_columns": None,
            "columns": None,
            "sample_rows": None,
            "estimated_rows": None,
            "status": f"ERROR: {type(e).__name__}: {e}",
        })

update_stream_samples_df = pd.DataFrame(update_stream_sample_rows)
update_stream_samples_df

,file_name,n_columns,columns,sample_rows,estimated_rows,status
0,updateStream_0_0_forum.csv,NaN,None,None,NaN,"ERROR: ParserError: Expected 14 fields in line 6, saw 15"
1,updateStream_0_0_person.csv,17.0,"[1347615339076, 0, 1, 35184372090047, Witold, Ciesla, female, 435456000000, 1347615339076.1, 31.183.168.247, Firefox, 1282, pl;en, Witold35184372090047@gmail.com, 205;1178;1760;1988;9097;9163;9317, Unnamed: 15, 979,2009]","[{'1347615339076': '1347625494732', '0': '0', '1': '1', '35184372090047': '35184372089129', 'Witold': 'Ashin', 'Ciesla': 'Michie', 'female': 'female', '435456000000': '487555200000', '1347615339076.1': '1347625494732', '31.183.168.247': '203.215.60.162', 'Firefox': 'Chrome', '1282': '1010', 'pl;en': 'my;en', 'Witold35184372090047@gmail.com': 'Ashin35184372089129@gmail.com;Ashin35184372089129@gmx.com;Ashin35184372089129@hotmail.com', '205;1178;1760;1988;9097;9163;9317': '205;564;571;577;810;973;978;1177;1668;1988;2081;2096;2107;2799;2818;2840;2845;2879;2923;2926;2963;2965;4986;5162;5342;5546;5648;5650;6368;6645;6990;7507;7522;7572;7594;8792;10107;10322', 'Unnamed: 15': nan, '979,2009': '1009,2010;982,2004;892,2006;109,2004;1133,2010'}, {'1347615339076': '1347681355202', '0': '0', '1': '1', '35184372090047': '35184372090466', 'Witold': 'Rahim', 'Ciesla': 'Zaland', 'female': 'female', '435456000000': '611712000000', '1347615339076.1': '1347681355202', '31.183.168.247': '58.147.130.150', 'Firefox': 'Firefox', '1282': '953', 'pl;en': 'tk;uz;en', 'Witold35184372090047@gmail.com': 'Rahim35184372090466@hotmail.com', '205;1178;1760;1988;9097;9163;9317': '1671;1747;6429;16074', 'Unnamed: 15': '1582,2008', '979,2009': '1,2009;2,2009'}, {'1347615339076': '1347701695628', '0': '0', '1': '1', '35184372090047': '35184372089452', 'Witold': 'Manuel', 'Ciesla': 'Gonzalez', 'female': 'male', '435456000000': '480384000000', '1347615339076.1': '1347701695628', '31.183.168.247': '148.204.205.19', 'Firefox': 'Firefox', '1282': '740', 'pl;en': 'es;en', 'Witold35184372090047@gmail.com': 'Manuel35184372089452@gmail.com;Manuel35184372089452@gmx.com;Manuel35184372089452@hotmail.com', '205;1178;1760;1988;9097;9163;9317': '140;243;468;575;580;776;1021;1452;1616;1763;1764;1987;1995;1997;2000;2034;2043;2051;2067;2087;2092;2781;2782;2783;2790;2793;2890;2950;2982;2996;3001;3101;5116;5313;5490;6395;6704;6964;7517;7545;7960;9170;9174;10197;11537;12742', 'Unnamed: 15': '5039,2004', '979,2009': nan}, {'1347615339076': '1347728341891', '0': '0', '1': '1', '35184372090047': '35184372089227', 'Witold': 'Carlos', 'Ciesla': 'Araujo', 'female': 'female', '435456000000': '340243200000', '1347615339076.1': '1347728341891', '31.183.168.247': '152.92.50.216', 'Firefox': 'Safari', '1282': '582', 'pl;en': 'pt;en', 'Witold35184372090047@gmail.com': 'Carlos35184372089227@gmail.com;Carlos35184372089227@gmx.com;Carlos35184372089227@yahoo.com', '205;1178;1760;1988;9097;9163;9317': '6', 'Unnamed: 15': '1852,1998', '979,2009': nan}, {'1347615339076': '1347808601116', '0': '0', '1': '1', '35184372090047': '35184372090497', 'Witold': 'Eddie', 'Ciesla': 'Cruz', 'female': 'female', '435456000000': '539654400000', '1347615339076.1': '1347808601116', '31.183.168.247': '116.68.182.20', 'Firefox': 'Chrome', '1282': '818', 'pl;en': 'en;tl', 'Witold35184372090047@gmail.com': 'Eddie35184372090497@gmx.com', '205;1178;1760;1988;9097;9163;9317': '294;524;561;565;568;588;1032;1152;1153;1154;1171;1194;1375;1669;1671;1987;1993;1998;2784;2787;2788;2790;2792;2793;2794;3089;5153;5601;6131;7338;7509;7524;9141;9289;9329;10028', 'Unnamed: 15': nan, '979,2009': '963,2008;950,2006'}]",171.0,OK_header_true


In [129]:
# =========================================================
# BLOCK 8B — Parse update streams without headers
# =========================================================
#
# Goal:
# - Parse LDBC SNB update streams line by line.
# - The update stream files do not have headers.
# - The number of fields can vary by insert operation.
# - For update volatility, we mainly need the operation code.
#
# Format assumption:
# - field[0] = update timestamp
# - field[1] = dependency timestamp / auxiliary timestamp
# - field[2] = insert operation code
#   1 = INS1_AddPerson
#   2 = INS2_AddLikeToPost
#   3 = INS3_AddLikeToComment
#   4 = INS4_AddForum
#   5 = INS5_AddForumMembership
#   6 = INS6_AddPost
#   7 = INS7_AddComment
#   8 = INS8_AddFriendship
# =========================================================

import csv
import pandas as pd

UPDATE_OPERATION_CODE_MAP = {
    1: {
        "official_id": "INS1",
        "query_name": "INS1_AddPerson",
        "operation": "Add person",
    },
    2: {
        "official_id": "INS2",
        "query_name": "INS2_AddLikeToPost",
        "operation": "Add like to post",
    },
    3: {
        "official_id": "INS3",
        "query_name": "INS3_AddLikeToComment",
        "operation": "Add like to comment",
    },
    4: {
        "official_id": "INS4",
        "query_name": "INS4_AddForum",
        "operation": "Add forum",
    },
    5: {
        "official_id": "INS5",
        "query_name": "INS5_AddForumMembership",
        "operation": "Add forum membership",
    },
    6: {
        "official_id": "INS6",
        "query_name": "INS6_AddPost",
        "operation": "Add post",
    },
    7: {
        "official_id": "INS7",
        "query_name": "INS7_AddComment",
        "operation": "Add comment",
    },
    8: {
        "official_id": "INS8",
        "query_name": "INS8_AddFriendship",
        "operation": "Add friendship",
    },
}


def parse_update_stream_file(path):
    """
    Parse one LDBC SNB update stream file without assuming a fixed number
    of columns.

    We keep only metadata needed for update volatility:
    - file name
    - line number
    - update timestamp
    - operation code
    - official insert id
    - field count
    """
    parsed_rows = []
    error_rows = []

    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f, delimiter="|")

        for line_no, fields in enumerate(reader, start=1):
            if not fields:
                continue

            # Some lines may end with a trailing "|", producing a final empty field.
            # Keep field count as observed, but do not let this break parsing.
            try:
                update_timestamp = fields[0] if len(fields) > 0 else None
                dependency_timestamp = fields[1] if len(fields) > 1 else None
                operation_code = int(fields[2]) if len(fields) > 2 and fields[2] != "" else None

                op_meta = UPDATE_OPERATION_CODE_MAP.get(operation_code, {})

                parsed_rows.append({
                    "file_name": path.name,
                    "line_no": line_no,
                    "update_timestamp": update_timestamp,
                    "dependency_timestamp": dependency_timestamp,
                    "operation_code": operation_code,
                    "official_id": op_meta.get("official_id"),
                    "query_name": op_meta.get("query_name"),
                    "operation": op_meta.get("operation"),
                    "n_fields": len(fields),
                    "raw_prefix": fields[:8],
                })

            except Exception as e:
                error_rows.append({
                    "file_name": path.name,
                    "line_no": line_no,
                    "error": f"{type(e).__name__}: {e}",
                    "raw_fields": fields[:12],
                })

    return parsed_rows, error_rows


parsed_update_rows = []
parsed_update_errors = []

for path in update_stream_files:
    rows, errors = parse_update_stream_file(path)
    parsed_update_rows.extend(rows)
    parsed_update_errors.extend(errors)

snb_update_stream_operations_df = pd.DataFrame(parsed_update_rows)
snb_update_stream_parse_errors_df = pd.DataFrame(parsed_update_errors)

print("Parsed update operations:", len(snb_update_stream_operations_df))
print("Parse errors:", len(snb_update_stream_parse_errors_df))

snb_update_stream_operations_df.head()

Parsed update operations: 287330
Parse errors: 0


,file_name,line_no,update_timestamp,dependency_timestamp,operation_code,official_id,query_name,operation,n_fields,raw_prefix
0,updateStream_0_0_forum.csv,1,1347529090363,1331161432355,7,INS7,INS7_AddComment,Add comment,14,"[1347529090363, 1331161432355, 7, 1099511997932, 1347529090363, 1.12.242.179, Opera, roflol]"
1,updateStream_0_0_forum.csv,2,1347529099239,1307796167976,5,INS5,INS5_AddForumMembership,Add forum membership,6,"[1347529099239, 1307796167976, 5, 1030792157789, 17592186044532, 1347529099239]"
2,updateStream_0_0_forum.csv,3,1347529219074,1347166209438,2,INS2,INS2_AddLikeToPost,Add like to post,6,"[1347529219074, 1347166209438, 2, 35184372090183, 481036434045, 1347529219074]"
3,updateStream_0_0_forum.csv,4,1347529227881,1281275886265,2,INS2,INS2_AddLikeToPost,Add like to post,6,"[1347529227881, 1281275886265, 2, 6597069766720, 1030792332314, 1347529227881]"
4,updateStream_0_0_forum.csv,5,1347529249368,1342103263364,3,INS3,INS3_AddLikeToComment,Add like to comment,6,"[1347529249368, 1342103263364, 3, 32985348834174, 1030792257093, 1347529249368]"


In [130]:
# =========================================================
# BLOCK 8B.1 — Count observed update operations
# =========================================================

snb_observed_update_operation_counts_df = (
    snb_update_stream_operations_df
    .groupby(["operation_code", "official_id", "query_name", "operation"], dropna=False)
    .agg(
        observed_update_count=("line_no", "count"),
        files=("file_name", lambda x: sorted(set(x))),
        min_fields=("n_fields", "min"),
        max_fields=("n_fields", "max"),
    )
    .reset_index()
    .sort_values("operation_code")
)

snb_observed_update_operation_counts_df

,operation_code,official_id,query_name,operation,observed_update_count,files,min_fields,max_fields
0,1,INS1,INS1_AddPerson,Add person,172,[updateStream_0_0_person.csv],17,17
1,2,INS2,INS2_AddLikeToPost,Add like to post,44313,[updateStream_0_0_forum.csv],6,6
2,3,INS3,INS3_AddLikeToComment,Add like to comment,30395,[updateStream_0_0_forum.csv],6,6
3,4,INS4,INS4_AddForum,Add forum,3059,[updateStream_0_0_forum.csv],8,8
4,5,INS5,INS5_AddForumMembership,Add forum membership,126615,[updateStream_0_0_forum.csv],6,6
5,6,INS6,INS6_AddPost,Add post,32610,[updateStream_0_0_forum.csv],15,15
6,7,INS7,INS7_AddComment,Add comment,46969,[updateStream_0_0_forum.csv],14,14
7,8,INS8,INS8_AddFriendship,Add friendship,3197,[updateStream_0_0_forum.csv],6,6


In [131]:
snb_update_stream_operations_df[
    snb_update_stream_operations_df["official_id"].isna()
]

,file_name,line_no,update_timestamp,dependency_timestamp,operation_code,official_id,query_name,operation,n_fields,raw_prefix


In [132]:
# =========================================================
# BLOCK 8B.2 — Count update operations by file
# =========================================================

snb_observed_update_by_file_df = (
    snb_update_stream_operations_df
    .groupby(["file_name", "operation_code", "official_id", "query_name"], dropna=False)
    .agg(
        observed_update_count=("line_no", "count"),
        min_fields=("n_fields", "min"),
        max_fields=("n_fields", "max"),
    )
    .reset_index()
    .sort_values(["file_name", "operation_code"])
)

snb_observed_update_by_file_df

,file_name,operation_code,official_id,query_name,observed_update_count,min_fields,max_fields
0,updateStream_0_0_forum.csv,2,INS2,INS2_AddLikeToPost,44313,6,6
1,updateStream_0_0_forum.csv,3,INS3,INS3_AddLikeToComment,30395,6,6
2,updateStream_0_0_forum.csv,4,INS4,INS4_AddForum,3059,8,8
3,updateStream_0_0_forum.csv,5,INS5,INS5_AddForumMembership,126615,6,6
4,updateStream_0_0_forum.csv,6,INS6,INS6_AddPost,32610,15,15
5,updateStream_0_0_forum.csv,7,INS7,INS7_AddComment,46969,14,14
6,updateStream_0_0_forum.csv,8,INS8,INS8_AddFriendship,3197,6,6
7,updateStream_0_0_person.csv,1,INS1,INS1_AddPerson,172,17,17


In [133]:
# =========================================================
# BLOCK 8C — Build observed official insert impact map
# =========================================================
#
# Goal:
# - Estimate update volatility using the actual update stream counts.
# - Each INS operation contributes according to its observed frequency.
# =========================================================

# Map official_id/query_name to observed update count.
observed_insert_count_map = dict(
    zip(
        snb_observed_update_operation_counts_df["query_name"],
        snb_observed_update_operation_counts_df["observed_update_count"]
    )
)

official_insert_queries = [
    q for q in SNB_OFFICIAL_WORKLOAD
    if q["query_group"] == "insert"
]

insert_entity_impact_rows = []
insert_relationship_impact_rows = []

for query in official_insert_queries:
    qname = query["query_name"]
    official_id = query["official_id"]

    observed_count = int(observed_insert_count_map.get(qname, 0))

    # Entity impacts
    for entity in sorted(set(query["accessed_entities"])):
        insert_entity_impact_rows.append({
            "query_name": qname,
            "official_id": official_id,
            "entity": entity,
            "observed_update_count": observed_count,
            "impact_count": observed_count,
        })

    # Relationship impacts
    for rel in sorted(set(query["relationships_used"])):
        insert_relationship_impact_rows.append({
            "query_name": qname,
            "official_id": official_id,
            "relationship": rel,
            "observed_update_count": observed_count,
            "impact_count": observed_count,
        })

snb_insert_entity_impact_df = pd.DataFrame(insert_entity_impact_rows)
snb_insert_relationship_impact_df = pd.DataFrame(insert_relationship_impact_rows)

snb_insert_entity_impact_df

,query_name,official_id,entity,observed_update_count,impact_count
0,INS1_AddPerson,INS1,Organisation,172,172
1,INS1_AddPerson,INS1,Person,172,172
2,INS1_AddPerson,INS1,Place,172,172
3,INS1_AddPerson,INS1,Tag,172,172
4,INS2_AddLikeToPost,INS2,Person,44313,44313
5,INS2_AddLikeToPost,INS2,Post,44313,44313
6,INS3_AddLikeToComment,INS3,Comment,30395,30395
7,INS3_AddLikeToComment,INS3,Person,30395,30395
8,INS4_AddForum,INS4,Forum,3059,3059
9,INS4_AddForum,INS4,Person,3059,3059


In [134]:
snb_insert_relationship_impact_df

,query_name,official_id,relationship,observed_update_count,impact_count
0,INS1_AddPerson,INS1,person_has_interest_tag,172,172
1,INS1_AddPerson,INS1,person_is_located_in_place,172,172
2,INS1_AddPerson,INS1,person_study_at_organisation,172,172
3,INS1_AddPerson,INS1,person_work_at_organisation,172,172
4,INS2_AddLikeToPost,INS2,person_likes_post,44313,44313
5,INS3_AddLikeToComment,INS3,person_likes_comment,30395,30395
6,INS4_AddForum,INS4,forum_has_moderator_person,3059,3059
7,INS4_AddForum,INS4,forum_has_tag,3059,3059
8,INS5_AddForumMembership,INS5,forum_has_member_person,126615,126615
9,INS6_AddPost,INS6,forum_container_of_post,32610,32610


In [135]:
# =========================================================
# BLOCK 8D — Compute entity update volatility
# =========================================================
#
# Goal:
# - Count how often each entity is affected by official insert operations.
# - Normalize by the maximum entity update frequency.
# =========================================================

snb_entity_update_volatility_df = (
    snb_insert_entity_impact_df
    .groupby("entity")
    .agg(
        update_impact_count=("impact_count", "sum"),
        affected_by_inserts=("official_id", lambda x: sorted(set(x))),
    )
    .reset_index()
)

max_entity_update_count = snb_entity_update_volatility_df["update_impact_count"].max()

snb_entity_update_volatility_df["entity_update_volatility"] = (
    snb_entity_update_volatility_df["update_impact_count"] / max_entity_update_count
    if max_entity_update_count > 0
    else 0.0
)

snb_entity_update_volatility_df = snb_entity_update_volatility_df.sort_values(
    by=["entity_update_volatility", "update_impact_count"],
    ascending=False
).reset_index(drop=True)

snb_entity_update_volatility_df

,entity,update_impact_count,affected_by_inserts,entity_update_volatility
0,Person,287330,"[INS1, INS2, INS3, INS4, INS5, INS6, INS7, INS8]",1.000000
1,Forum,162284,"[INS4, INS5, INS6]",0.564800
2,Post,123892,"[INS2, INS6, INS7]",0.431184
3,Tag,82810,"[INS1, INS4, INS6, INS7]",0.288205
4,Place,79751,"[INS1, INS6, INS7]",0.277559
5,Comment,77364,"[INS3, INS7]",0.269251
6,Organisation,172,[INS1],0.000599


In [136]:
# =========================================================
# BLOCK 8E — Compute relationship update volatility
# =========================================================
#
# Goal:
# - Count how often each relationship is affected by official insert operations.
# - Normalize by the maximum relationship update frequency.
# =========================================================

snb_relationship_update_volatility_df = (
    snb_insert_relationship_impact_df
    .groupby("relationship")
    .agg(
        update_impact_count=("impact_count", "sum"),
        affected_by_inserts=("official_id", lambda x: sorted(set(x))),
    )
    .reset_index()
)

max_relationship_update_count = snb_relationship_update_volatility_df["update_impact_count"].max()

snb_relationship_update_volatility_df["relationship_update_volatility"] = (
    snb_relationship_update_volatility_df["update_impact_count"] / max_relationship_update_count
    if max_relationship_update_count > 0
    else 0.0
)

snb_relationship_update_volatility_df = snb_relationship_update_volatility_df.sort_values(
    by=["relationship_update_volatility", "update_impact_count"],
    ascending=False
).reset_index(drop=True)

snb_relationship_update_volatility_df

,relationship,update_impact_count,affected_by_inserts,relationship_update_volatility
0,forum_has_member_person,126615,[INS5],1.000000
1,comment_has_creator_person,46969,[INS7],0.370959
2,comment_has_tag,46969,[INS7],0.370959
3,comment_is_located_in_place,46969,[INS7],0.370959
4,comment_reply_of_comment,46969,[INS7],0.370959
5,comment_reply_of_post,46969,[INS7],0.370959
6,person_likes_post,44313,[INS2],0.349982
7,forum_container_of_post,32610,[INS6],0.257552
8,post_has_creator_person,32610,[INS6],0.257552
9,post_has_tag,32610,[INS6],0.257552


In [137]:
# =========================================================
# BLOCK 8F — Compute query-level update volatility
# =========================================================
#
# Goal:
# - Estimate update volatility per official query.
# - Use the maximum and mean volatility of touched entities and relationships.
# =========================================================

entity_vol_map = dict(
    zip(
        snb_entity_update_volatility_df["entity"],
        snb_entity_update_volatility_df["entity_update_volatility"]
    )
)

relationship_vol_map = dict(
    zip(
        snb_relationship_update_volatility_df["relationship"],
        snb_relationship_update_volatility_df["relationship_update_volatility"]
    )
)

query_update_volatility_rows = []

for query in SNB_OFFICIAL_WORKLOAD:
    qname = query["query_name"]
    official_id = query["official_id"]

    accessed_entities = list(set(query["accessed_entities"]))
    relationships_used = list(set(query["relationships_used"]))

    entity_vols = [
        entity_vol_map.get(entity, 0.0)
        for entity in accessed_entities
    ]

    relationship_vols = [
        relationship_vol_map.get(rel, 0.0)
        for rel in relationships_used
    ]

    all_vols = entity_vols + relationship_vols

    if all_vols:
        update_volatility_mean = sum(all_vols) / len(all_vols)
        update_volatility_max = max(all_vols)
    else:
        update_volatility_mean = 0.0
        update_volatility_max = 0.0

    query_update_volatility_rows.append({
        "query_name": qname,
        "official_id": official_id,
        "query_group": query["query_group"],
        "operation_type": query["operation_type"],
        "root_entity": query["root_entity"],
        "n_touched_entities": len(accessed_entities),
        "n_touched_relationships": len(relationships_used),
        "entity_volatility_mean": (
            sum(entity_vols) / len(entity_vols)
            if entity_vols else 0.0
        ),
        "entity_volatility_max": (
            max(entity_vols)
            if entity_vols else 0.0
        ),
        "relationship_volatility_mean": (
            sum(relationship_vols) / len(relationship_vols)
            if relationship_vols else 0.0
        ),
        "relationship_volatility_max": (
            max(relationship_vols)
            if relationship_vols else 0.0
        ),
        "update_volatility_mean": update_volatility_mean,
        "update_volatility_max": update_volatility_max,
    })

snb_query_update_volatility_df = pd.DataFrame(query_update_volatility_rows)

snb_query_update_volatility_df = snb_query_update_volatility_df.sort_values(
    by=["update_volatility_max", "update_volatility_mean"],
    ascending=False
).reset_index(drop=True)

snb_query_update_volatility_df

,query_name,official_id,query_group,operation_type,root_entity,n_touched_entities,n_touched_relationships,entity_volatility_mean,entity_volatility_max,relationship_volatility_mean,relationship_volatility_max,update_volatility_mean,update_volatility_max
0,INS5_AddForumMembership,INS5,insert,insert,Forum,2,1,0.782400,1.000000,1.000000,1.000000,0.854933,1.000000
1,INS2_AddLikeToPost,INS2,insert,insert,Person,2,1,0.715592,1.000000,0.349982,0.349982,0.593722,1.000000
2,IS3_FriendsOfPerson,IS3,short_read,read,Person,1,1,1.000000,1.000000,0.025250,0.025250,0.512625,1.000000
3,INS8_AddFriendship,INS8,insert,insert,Person,1,1,1.000000,1.000000,0.025250,0.025250,0.512625,1.000000
4,IC5_NewGroups,IC5,complex_read,read,Person,3,4,0.665328,1.000000,0.385089,1.000000,0.505191,1.000000
5,INS3_AddLikeToComment,INS3,insert,insert,Person,2,1,0.634626,1.000000,0.240058,0.240058,0.503103,1.000000
6,IS5_CreatorOfMessage,IS5,short_read,read,Post,3,2,0.566812,1.000000,0.314256,0.370959,0.465789,1.000000
7,IS2_RecentMessagesOfPerson,IS2,short_read,read,Person,3,4,0.566812,1.000000,0.342608,0.370959,0.438695,1.000000
8,IS1_ProfileOfPerson,IS1,short_read,read,Person,2,1,0.638779,1.000000,0.001358,0.001358,0.426306,1.000000
9,INS7_AddComment,INS7,insert,insert,Comment,5,5,0.453240,1.000000,0.370959,0.370959,0.412100,1.000000


In [138]:
# =========================================================
# BLOCK 8G — Add update volatility to structural reduction matrix
# =========================================================
#
# Goal:
# - Extend the current matrix with update volatility variables.
# =========================================================

snb_matrix_with_volatility_df = (
    snb_structural_reduction_matrix_df
    .merge(
        snb_query_update_volatility_df[
            [
                "query_name",
                "official_id",
                "entity_volatility_mean",
                "entity_volatility_max",
                "relationship_volatility_mean",
                "relationship_volatility_max",
                "update_volatility_mean",
                "update_volatility_max",
            ]
        ],
        on=["query_name", "official_id"],
        how="left"
    )
)

snb_matrix_with_volatility_df

,query_name,official_id,official_title,query_group,operation_type,root_entity,accessed_entities,relationships_used,Rc_distinct,Rc_weighted,D,Re,DeltaR,DeltaRratio,association_count,associative_count,containment_count,lookup_count,dominant_semantic_type,weighted_relationships,description,entity_volatility_mean,entity_volatility_max,relationship_volatility_mean,relationship_volatility_max,update_volatility_mean,update_volatility_max
0,IS1_ProfileOfPerson,IS1,Profile of a person,short_read,read,Person,"[Person, Place]",[person_is_located_in_place],1,1,1,0,1,1.000000,1,0,0,0,association,[person_is_located_in_place x1],Official IS1: retrieve a person's profile and city of residence.,0.638779,1.000000,0.001358,0.001358,0.426306,1.000000
1,IS2_RecentMessagesOfPerson,IS2,Recent messages of a person,short_read,read,Person,"[Person, Post, Comment]","[post_has_creator_person, comment_has_creator_person, comment_reply_of_post, comment_reply_of_comment]",4,4,2,2,2,0.500000,2,0,2,0,mixed,"[post_has_creator_person x1, comment_has_creator_person x1, comment_reply_of_post x1, comment_reply_of_comment x1]",Official IS2: retrieve the last messages created by a person and the original post in the conversation.,0.566812,1.000000,0.342608,0.370959,0.438695,1.000000
2,IS3_FriendsOfPerson,IS3,Friends of a person,short_read,read,Person,[Person],[person_knows_person],1,1,1,0,1,1.000000,1,0,0,0,association,[person_knows_person x1],Official IS3: retrieve all friends of a person and the friendship creation date.,1.000000,1.000000,0.025250,0.025250,0.512625,1.000000
3,IS4_ContentOfMessage,IS4,Content of a message,short_read,read,Post,"[Post, Comment]",[],0,0,0,0,0,0.000000,0,0,0,1,lookup,[],Official IS4: retrieve content and creation date of a Message. Message is represented by Post or Comment.,0.350218,0.431184,0.000000,0.000000,0.350218,0.431184
4,IS5_CreatorOfMessage,IS5,Creator of a message,short_read,read,Post,"[Post, Comment, Person]","[post_has_creator_person, comment_has_creator_person]",2,2,1,1,1,0.500000,2,0,0,0,association,"[post_has_creator_person x1, comment_has_creator_person x1]",Official IS5: retrieve the author of a Message. Message is represented by Post or Comment.,0.566812,1.000000,0.314256,0.370959,0.465789,1.000000
5,IS6_ForumOfMessage,IS6,Forum of a message,short_read,read,Post,"[Post, Comment, Forum, Person]","[forum_container_of_post, comment_reply_of_post, comment_reply_of_comment, forum_has_moderator_person]",4,4,3,1,3,0.750000,1,0,3,0,containment,"[forum_container_of_post x1, comment_reply_of_post x1, comment_reply_of_comment x1, forum_has_moderator_person x1]",Official IS6: retrieve the forum containing a Message and the moderator of that forum.,0.566309,1.000000,0.255908,0.370959,0.411108,1.000000
6,IS7_RepliesOfMessage,IS7,Replies of a message,short_read,read,Post,"[Post, Comment, Person]","[comment_reply_of_post, comment_reply_of_comment, comment_has_creator_person, post_has_creator_person, person_knows_person]",5,5,2,3,2,0.400000,3,0,2,0,association,"[comment_reply_of_post x1, comment_reply_of_comment x1, comment_has_creator_person x1, post_has_creator_person x1, person_knows_person x1]",Official IS7: retrieve one-hop replies of a Message and whether reply author knows original author.,0.566812,1.000000,0.279136,0.370959,0.387014,1.000000
7,IC1_TransitiveFriendsWithName,IC1,Transitive friends with a certain name,complex_read,read,Person,"[Person, Place, Organisation]","[person_knows_person, person_is_located_in_place, person_study_at_organisation, person_work_at_organisation, organisation_is_located_in_place]",5,7,4,3,4,0.571429,5,0,0,0,association,"[person_knows_person x3, person_is_located_in_place x1, person_study_at_organisation x1, person_work_at_organisation x1, organisation_is_located_in_place x1]","Official IC1: find people connected by up to 3 knows hops with a given first name, including workplaces and studies.",0.426053,1.000000,0.005865,0.025250,0.163435,1.000000
8,IC2_RecentMessagesByFriends,IC2,Recent messages

In [139]:
# =========================================================
# BLOCK 8H — Rank official queries by update volatility
# =========================================================

snb_volatility_rank_df = (
    snb_matrix_with_volatility_df[
        [
            "query_name",
            "official_id",
            "query_group",
            "operation_type",
            "root_entity",
            "Rc_weighted",
            "D",
            "Re",
            "DeltaRratio",
            "dominant_semantic_type",
            "update_volatility_mean",
            "update_volatility_max",
        ]
    ]
    .sort_values(
        by=["update_volatility_max", "update_volatility_mean", "Rc_weighted"],
        ascending=False
    )
    .reset_index(drop=True)
)

snb_volatility_rank_df

,query_name,official_id,query_group,operation_type,root_entity,Rc_weighted,D,Re,DeltaRratio,dominant_semantic_type,update_volatility_mean,update_volatility_max
0,INS5_AddForumMembership,INS5,insert,insert,Forum,1,1,0,1.000000,associative,0.854933,1.000000
1,INS2_AddLikeToPost,INS2,insert,insert,Person,1,1,0,1.000000,associative,0.593722,1.000000
2,IS3_FriendsOfPerson,IS3,short_read,read,Person,1,1,0,1.000000,association,0.512625,1.000000
3,INS8_AddFriendship,INS8,insert,insert,Person,1,1,0,1.000000,association,0.512625,1.000000
4,IC5_NewGroups,IC5,complex_read,read,Person,5,3,2,0.600000,association,0.505191,1.000000
5,INS3_AddLikeToComment,INS3,insert,insert,Person,1,1,0,1.000000,associative,0.503103,1.000000
6,IS5_CreatorOfMessage,IS5,short_read,read,Post,2,1,1,0.500000,association,0.465789,1.000000
7,IS2_RecentMessagesOfPerson,IS2,short_read,read,Person,4,2,2,0.500000,mixed,0.438695,1.000000
8,IS1_ProfileOfPerson,IS1,short_read,read,Person,1,1,0,1.000000,association,0.426306,1.000000
9,INS7_AddComment,INS7,insert,insert,Comment,5,1,4,0.200000,association,0.412100,1.000000


In [141]:
# =========================================================
# BLOCK 8I — Save update volatility artifacts
# =========================================================
#
# Goal:
# - Save all artifacts produced by Block 8.
# - This version is corrected for LDBC SNB update streams without headers.
# - It saves parsed update operations, observed update counts,
#   entity volatility, relationship volatility, and query-level volatility.
# =========================================================

from pathlib import Path
import pandas as pd

VOLATILITY_ARTIFACTS_DIR = (
    BASE_DIR.parent.parent
    / "benchmark_artifacts_dir"
    / "ldbc_snb_update_volatility_official"
)

VOLATILITY_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 1. Save parsed update stream artifacts
# ---------------------------------------------------------

snb_update_stream_operations_df.to_csv(
    VOLATILITY_ARTIFACTS_DIR / "update_stream_operations.csv",
    index=False
)

snb_update_stream_parse_errors_df.to_csv(
    VOLATILITY_ARTIFACTS_DIR / "update_stream_parse_errors.csv",
    index=False
)

snb_observed_update_operation_counts_df.to_csv(
    VOLATILITY_ARTIFACTS_DIR / "observed_update_operation_counts.csv",
    index=False
)

snb_observed_update_by_file_df.to_csv(
    VOLATILITY_ARTIFACTS_DIR / "observed_update_by_file.csv",
    index=False
)

# ---------------------------------------------------------
# 2. Save insert impact artifacts
# ---------------------------------------------------------

snb_insert_entity_impact_df.to_csv(
    VOLATILITY_ARTIFACTS_DIR / "insert_entity_impact.csv",
    index=False
)

snb_insert_relationship_impact_df.to_csv(
    VOLATILITY_ARTIFACTS_DIR / "insert_relationship_impact.csv",
    index=False
)

# ---------------------------------------------------------
# 3. Save volatility artifacts
# ---------------------------------------------------------

snb_entity_update_volatility_df.to_csv(
    VOLATILITY_ARTIFACTS_DIR / "entity_update_volatility.csv",
    index=False
)

snb_relationship_update_volatility_df.to_csv(
    VOLATILITY_ARTIFACTS_DIR / "relationship_update_volatility.csv",
    index=False
)

snb_query_update_volatility_df.to_csv(
    VOLATILITY_ARTIFACTS_DIR / "query_update_volatility.csv",
    index=False
)

# ---------------------------------------------------------
# 4. Save matrix with volatility
# ---------------------------------------------------------

snb_matrix_with_volatility_df.to_csv(
    VOLATILITY_ARTIFACTS_DIR / "matrix_with_update_volatility.csv",
    index=False
)

snb_volatility_rank_df.to_csv(
    VOLATILITY_ARTIFACTS_DIR / "volatility_rank.csv",
    index=False
)

print("Saved update volatility artifacts to:")
print(VOLATILITY_ARTIFACTS_DIR)

print("\nSaved files:")
for path in sorted(VOLATILITY_ARTIFACTS_DIR.glob("*.csv")):
    print(" -", path.name)

Saved update volatility artifacts to:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_update_volatility_official

Saved files:
 - entity_update_volatility.csv
 - insert_entity_impact.csv
 - insert_relationship_impact.csv
 - matrix_with_update_volatility.csv
 - observed_update_by_file.csv
 - observed_update_operation_counts.csv
 - query_update_volatility.csv
 - relationship_update_volatility.csv
 - update_stream_operations.csv
 - update_stream_parse_errors.csv
 - volatility_rank.csv


In [142]:
# =========================================================
# BLOCK 8J — Save checkpoint after update volatility
# =========================================================
#
# Goal:
# - Save the Block 8 state into the checkpoint folder.
# - If the kernel or connection drops, the notebook can resume after Block 8.
# =========================================================

from pathlib import Path
import json
import pandas as pd

# ---------------------------------------------------------
# 1. Recreate checkpoint paths if they are not in memory
# ---------------------------------------------------------

PROJECT_DIR = BASE_DIR.parent.parent

CHECKPOINT_DIR = PROJECT_DIR / "benchmark_artifacts_dir" / "ldbc_snb_checkpoints"
CHECKPOINT_DATAFRAMES_DIR = CHECKPOINT_DIR / "dataframes"
CHECKPOINT_METADATA_DIR = CHECKPOINT_DIR / "metadata"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_METADATA_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = CHECKPOINT_DIR / "checkpoint_manifest.json"
metadata_path = CHECKPOINT_METADATA_DIR / "checkpoint_metadata.json"

# ---------------------------------------------------------
# 2. Load existing manifest if available
# ---------------------------------------------------------

if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        checkpoint_manifest = json.load(f)
else:
    checkpoint_manifest = {
        "dataframes": {},
        "metadata": str(metadata_path.relative_to(CHECKPOINT_DIR)),
    }

# ---------------------------------------------------------
# 3. Load existing metadata if available
# ---------------------------------------------------------

if metadata_path.exists():
    with open(metadata_path, "r", encoding="utf-8") as f:
        checkpoint_metadata = json.load(f)
else:
    checkpoint_metadata = {}

# ---------------------------------------------------------
# 4. Update metadata paths and core objects
# ---------------------------------------------------------

checkpoint_metadata.update({
    "BASE_DIR": str(BASE_DIR),
    "SNB_SNAPSHOT_DIR": str(SNB_SNAPSHOT_DIR),
    "SNB_DYNAMIC_DIR": str(SNB_DYNAMIC_DIR),
    "SNB_STATIC_DIR": str(SNB_STATIC_DIR),
    "SNB_SUBSTITUTION_DIR": str(SNB_SUBSTITUTION_DIR),
    "SNB_UPDATE_DIR": str(SNB_UPDATE_DIR),

    "SNB_ENTITY_VIEW_MAP": SNB_ENTITY_VIEW_MAP,
    "SNB_ENTITY_PK_MAP": SNB_ENTITY_PK_MAP,
    "SNB_COLLECTION_NAME_BY_ENTITY": SNB_COLLECTION_NAME_BY_ENTITY,
    "SNB_RELATIONSHIP_HINTS": SNB_RELATIONSHIP_HINTS,

    "SNB_OFFICIAL_WORKLOAD": SNB_OFFICIAL_WORKLOAD,
    "SNB_OFFICIAL_QUERY_PATH_OVERRIDES": SNB_OFFICIAL_QUERY_PATH_OVERRIDES,
    "SNB_RELATIONSHIP_MULTIPLICITY_OVERRIDES": SNB_RELATIONSHIP_MULTIPLICITY_OVERRIDES,
    "UPDATE_OPERATION_CODE_MAP": UPDATE_OPERATION_CODE_MAP,
})

# Keep persistent DuckDB path if it already exists in memory.
if "PERSISTENT_DUCKDB_PATH" in globals():
    checkpoint_metadata["PERSISTENT_DUCKDB_PATH"] = str(PERSISTENT_DUCKDB_PATH)

# ---------------------------------------------------------
# 5. Save Block 8 DataFrames
# ---------------------------------------------------------

BLOCK_8_CHECKPOINT_DATAFRAMES = [
    # Update stream parsing
    "snb_update_stream_operations_df",
    "snb_update_stream_parse_errors_df",
    "snb_observed_update_operation_counts_df",
    "snb_observed_update_by_file_df",

    # Insert impact
    "snb_insert_entity_impact_df",
    "snb_insert_relationship_impact_df",

    # Volatility
    "snb_entity_update_volatility_df",
    "snb_relationship_update_volatility_df",
    "snb_query_update_volatility_df",

    # Matrix with volatility
    "snb_matrix_with_volatility_df",
    "snb_volatility_rank_df",
]

saved_block_8_dataframes = []

for var_name in BLOCK_8_CHECKPOINT_DATAFRAMES:
    obj = globals().get(var_name)

    if isinstance(obj, pd.DataFrame):
        out_path = CHECKPOINT_DATAFRAMES_DIR / f"{var_name}.json"

        obj.to_json(
            out_path,
            orient="records",
            indent=2,
            force_ascii=False
        )

        checkpoint_manifest["dataframes"][var_name] = str(
            out_path.relative_to(CHECKPOINT_DIR)
        )

        saved_block_8_dataframes.append(var_name)

# ---------------------------------------------------------
# 6. Save updated metadata and manifest
# ---------------------------------------------------------

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(checkpoint_metadata, f, indent=2, ensure_ascii=False)

checkpoint_manifest["metadata"] = str(metadata_path.relative_to(CHECKPOINT_DIR))

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(checkpoint_manifest, f, indent=2, ensure_ascii=False)

print("Block 8 checkpoint saved to:")
print(CHECKPOINT_DIR)

print("\nSaved Block 8 DataFrames:")
for name in saved_block_8_dataframes:
    print(" -", name)

print("\nManifest:")
print(manifest_path)

Block 8 checkpoint saved to:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_checkpoints

Saved Block 8 DataFrames:
 - snb_update_stream_operations_df
 - snb_update_stream_parse_errors_df
 - snb_observed_update_operation_counts_df
 - snb_observed_update_by_file_df
 - snb_insert_entity_impact_df
 - snb_insert_relationship_impact_df
 - snb_entity_update_volatility_df
 - snb_relationship_update_volatility_df
 - snb_query_update_volatility_df
 - snb_matrix_with_volatility_df
 - snb_volatility_rank_df

Manifest:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_checkpoints/checkpoint_manifest.json


In [ ]:
# =========================================================
# RESUME AFTER BLOCK 8 — Restore checkpoint
# =========================================================
#
# Goal:
# - Restore notebook state after Block 8.
# - Continue directly to Block 9.
# =========================================================

from pathlib import Path
import json
import pandas as pd
import duckdb
import networkx as nx

PROJECT_DIR = Path("/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB")

CHECKPOINT_DIR = PROJECT_DIR / "benchmark_artifacts_dir" / "ldbc_snb_checkpoints"
manifest_path = CHECKPOINT_DIR / "checkpoint_manifest.json"

with open(manifest_path, "r", encoding="utf-8") as f:
    checkpoint_manifest = json.load(f)

metadata_path = CHECKPOINT_DIR / checkpoint_manifest["metadata"]

with open(metadata_path, "r", encoding="utf-8") as f:
    checkpoint_metadata = json.load(f)

# ---------------------------------------------------------
# Restore paths
# ---------------------------------------------------------

BASE_DIR = Path(checkpoint_metadata["BASE_DIR"])
SNB_SNAPSHOT_DIR = Path(checkpoint_metadata["SNB_SNAPSHOT_DIR"])
SNB_DYNAMIC_DIR = Path(checkpoint_metadata["SNB_DYNAMIC_DIR"])
SNB_STATIC_DIR = Path(checkpoint_metadata["SNB_STATIC_DIR"])
SNB_SUBSTITUTION_DIR = Path(checkpoint_metadata["SNB_SUBSTITUTION_DIR"])
SNB_UPDATE_DIR = Path(checkpoint_metadata["SNB_UPDATE_DIR"])

# ---------------------------------------------------------
# Restore dictionaries and lists
# ---------------------------------------------------------

SNB_ENTITY_VIEW_MAP = checkpoint_metadata["SNB_ENTITY_VIEW_MAP"]
SNB_ENTITY_PK_MAP = checkpoint_metadata["SNB_ENTITY_PK_MAP"]
SNB_COLLECTION_NAME_BY_ENTITY = checkpoint_metadata["SNB_COLLECTION_NAME_BY_ENTITY"]
SNB_RELATIONSHIP_HINTS = checkpoint_metadata["SNB_RELATIONSHIP_HINTS"]

SNB_OFFICIAL_WORKLOAD = checkpoint_metadata["SNB_OFFICIAL_WORKLOAD"]
SNB_OFFICIAL_QUERY_PATH_OVERRIDES = checkpoint_metadata["SNB_OFFICIAL_QUERY_PATH_OVERRIDES"]
SNB_RELATIONSHIP_MULTIPLICITY_OVERRIDES = checkpoint_metadata["SNB_RELATIONSHIP_MULTIPLICITY_OVERRIDES"]
UPDATE_OPERATION_CODE_MAP = checkpoint_metadata["UPDATE_OPERATION_CODE_MAP"]

# ---------------------------------------------------------
# Restore all checkpointed DataFrames
# ---------------------------------------------------------

for var_name, relative_path in checkpoint_manifest["dataframes"].items():
    df_path = CHECKPOINT_DIR / relative_path
    globals()[var_name] = pd.read_json(df_path, orient="records")

print("Restored DataFrames:")
for name in checkpoint_manifest["dataframes"]:
    print(" -", name)

# ---------------------------------------------------------
# Restore DuckDB connection
# ---------------------------------------------------------

if "PERSISTENT_DUCKDB_PATH" in checkpoint_metadata:
    PERSISTENT_DUCKDB_PATH = Path(checkpoint_metadata["PERSISTENT_DUCKDB_PATH"])
    con = duckdb.connect(str(PERSISTENT_DUCKDB_PATH))

    print("\nConnected to persistent DuckDB:")
    print(PERSISTENT_DUCKDB_PATH)
else:
    print("\nWARNING: PERSISTENT_DUCKDB_PATH not found in metadata.")
    print("You may need to recreate DuckDB views or rerun the persistent DuckDB checkpoint.")

# ---------------------------------------------------------
# Rebuild graphs if schema edges are available
# ---------------------------------------------------------

if "snb_schema_edges_df" in globals():
    snb_schema_graph = nx.DiGraph()

    for entity in SNB_ENTITY_VIEW_MAP.keys():
        snb_schema_graph.add_node(entity)

    for _, row in snb_schema_edges_df.iterrows():
        source = row["source_entity"]
        target = row["target_entity"]

        snb_schema_graph.add_edge(
            source,
            target,
            relationship=row["relationship"],
            semantic_type=row["semantic_type"],
            relationship_kind=row["relationship_kind"],
            view_name=row["view_name"],
            source_column=row["source_column"],
            target_column=row["target_column"],
        )

        if bool(row["is_bidirectional"]):
            snb_schema_graph.add_edge(
                target,
                source,
                relationship=str(row["relationship"]) + "_reverse",
                semantic_type=row["semantic_type"],
                relationship_kind=row["relationship_kind"],
                view_name=row["view_name"],
                source_column=row["target_column"],
                target_column=row["source_column"],
            )

    snb_traversal_graph = nx.Graph()

    for entity in SNB_ENTITY_VIEW_MAP.keys():
        snb_traversal_graph.add_node(entity)

    for _, row in snb_schema_edges_df.iterrows():
        snb_traversal_graph.add_edge(
            row["source_entity"],
            row["target_entity"],
            relationship=row["relationship"],
            semantic_type=row["semantic_type"],
            relationship_kind=row["relationship_kind"],
            view_name=row["view_name"],
            source_column=row["source_column"],
            target_column=row["target_column"],
        )

    print("\nGraphs rebuilt:")
    print("Directed graph nodes:", snb_schema_graph.number_of_nodes())
    print("Directed graph edges:", snb_schema_graph.number_of_edges())
    print("Traversal graph nodes:", snb_traversal_graph.number_of_nodes())
    print("Traversal graph edges:", snb_traversal_graph.number_of_edges())

# ---------------------------------------------------------
# Quick sanity check after Block 8
# ---------------------------------------------------------

print("\nBlock 8 objects restored:")
for name in [
    "snb_update_stream_operations_df",
    "snb_observed_update_operation_counts_df",
    "snb_entity_update_volatility_df",
    "snb_relationship_update_volatility_df",
    "snb_query_update_volatility_df",
    "snb_matrix_with_volatility_df",
    "snb_volatility_rank_df",
]:
    print(name, "=", "OK" if name in globals() else "MISSING")

In [144]:
# =========================================================
# BLOCK 9A — Compute relationship-level observed sharedness
# =========================================================
#
# Goal:
# - Measure how much target entities are reused across source entities.
# - Also measure source fanout.
#
# Definitions:
# - target_reuse_mean:
#     average number of distinct source records pointing to the same target.
#
# - target_reuse_max:
#     maximum number of distinct source records pointing to one target.
#
# - source_fanout_mean:
#     average number of distinct targets per source.
#
# - source_fanout_max:
#     maximum number of distinct targets for one source.
#
# Interpretation:
# - High target reuse means high observed sharedness.
# - High source fanout means one root may connect to many children.
# =========================================================

import pandas as pd
import numpy as np

def q(col: str) -> str:
    """
    Quote a DuckDB identifier safely.
    """
    return '"' + str(col).replace('"', '""') + '"'


relationship_sharedness_rows = []

for rel_name, spec in SNB_RELATIONSHIP_HINTS.items():
    view_name = spec["view_name"]
    source_entity = spec["source_entity"]
    target_entity = spec["target_entity"]
    source_col = spec["source_column"]
    target_col = spec["target_column"]

    try:
        sql = f"""
        WITH base AS (
            SELECT
                CAST({q(source_col)} AS VARCHAR) AS source_id,
                CAST({q(target_col)} AS VARCHAR) AS target_id
            FROM {view_name}
            WHERE {q(source_col)} IS NOT NULL
              AND {q(target_col)} IS NOT NULL
        ),

        source_fanout AS (
            SELECT
                source_id,
                COUNT(DISTINCT target_id) AS n_targets
            FROM base
            GROUP BY source_id
        ),

        target_reuse AS (
            SELECT
                target_id,
                COUNT(DISTINCT source_id) AS n_sources
            FROM base
            GROUP BY target_id
        )

        SELECT
            (SELECT COUNT(*) FROM base) AS relationship_rows,
            (SELECT COUNT(DISTINCT source_id) FROM base) AS distinct_sources,
            (SELECT COUNT(DISTINCT target_id) FROM base) AS distinct_targets,

            (SELECT AVG(n_targets) FROM source_fanout) AS source_fanout_mean,
            (SELECT MAX(n_targets) FROM source_fanout) AS source_fanout_max,

            (SELECT AVG(n_sources) FROM target_reuse) AS target_reuse_mean,
            (SELECT MAX(n_sources) FROM target_reuse) AS target_reuse_max
        """

        stats = con.execute(sql).df().iloc[0].to_dict()

        relationship_sharedness_rows.append({
            "relationship": rel_name,
            "source_entity": source_entity,
            "target_entity": target_entity,
            "semantic_type": spec["semantic_type"],
            "relationship_kind": spec["relationship_kind"],
            "view_name": view_name,
            "relationship_rows": int(stats["relationship_rows"] or 0),
            "distinct_sources": int(stats["distinct_sources"] or 0),
            "distinct_targets": int(stats["distinct_targets"] or 0),
            "source_fanout_mean": float(stats["source_fanout_mean"] or 0.0),
            "source_fanout_max": float(stats["source_fanout_max"] or 0.0),
            "target_reuse_mean": float(stats["target_reuse_mean"] or 0.0),
            "target_reuse_max": float(stats["target_reuse_max"] or 0.0),
            "status": "OK",
        })

    except Exception as e:
        relationship_sharedness_rows.append({
            "relationship": rel_name,
            "source_entity": source_entity,
            "target_entity": target_entity,
            "semantic_type": spec.get("semantic_type"),
            "relationship_kind": spec.get("relationship_kind"),
            "view_name": view_name,
            "relationship_rows": None,
            "distinct_sources": None,
            "distinct_targets": None,
            "source_fanout_mean": None,
            "source_fanout_max": None,
            "target_reuse_mean": None,
            "target_reuse_max": None,
            "status": f"ERROR: {type(e).__name__}: {e}",
        })

snb_relationship_sharedness_raw_df = pd.DataFrame(relationship_sharedness_rows)

snb_relationship_sharedness_raw_df

,relationship,source_entity,target_entity,semantic_type,relationship_kind,view_name,relationship_rows,distinct_sources,distinct_targets,source_fanout_mean,source_fanout_max,target_reuse_mean,target_reuse_max,status
0,person_knows_person,Person,Person,association,many_to_many,snb_knows_norm,14073,1199,1205,11.737281,243.0,11.678838,331.0,OK
1,person_is_located_in_place,Person,Place,association,many_to_one,snb_person_islocatedin_place,1528,1528,903,1.000000,1.0,1.692137,6.0,OK
2,person_has_interest_tag,Person,Tag,association,many_to_many,snb_person_interests_norm,35475,1528,4182,23.216623,80.0,8.482783,173.0,OK
3,person_likes_post,Person,Post,associative,many_to_many,snb_person_likes_posts_norm,47215,1385,8419,34.090253,637.0,5.608148,338.0,OK
4,person_likes_comment,Person,Comment,associative,many_to_many,snb_person_likes_comments_norm,62225,1423,4701,43.728039,471.0,13.236545,269.0,OK
5,person_study_at_organisation,Person,Organisation,association,many_to_many,snb_person_studyat_norm,1209,1209,524,1.000000,1.0,2.307252,22.0,OK
6,person_work_at_organisation,Person,Organisation,association,many_to_many,snb_person_workat_norm,3313,1207,1030,2.744822,5.0,3.216505,32.0,OK
7,forum_has_moderator_person,Forum,Person,association,many_to_one,snb_forum_hasmoderator_person,13750,13750,1528,1.000000,1.0,8.998691,37.0,OK
8,forum_has_member_person,Forum,Person,associative,many_to_many,snb_forum_members_norm,123268,11077,1480,11.128284,340.0,83.289189,1493.0,OK
9,forum_has_tag,Forum,Tag,association,many_to_many,snb_forum_tags_norm,47697,13750,4182,3.468873,80.0,11.405308,366.0,OK


In [145]:
# =========================================================
# BLOCK 9B — Normalize relationship observed sharedness
# =========================================================
#
# Goal:
# - Normalize target reuse and source fanout.
# - target_reuse is used as observed sharedness.
# =========================================================

snb_relationship_sharedness_df = snb_relationship_sharedness_raw_df.copy()

ok_mask = snb_relationship_sharedness_df["status"] == "OK"

max_target_reuse_mean = snb_relationship_sharedness_df.loc[
    ok_mask, "target_reuse_mean"
].max()

max_target_reuse_max = snb_relationship_sharedness_df.loc[
    ok_mask, "target_reuse_max"
].max()

max_source_fanout_mean = snb_relationship_sharedness_df.loc[
    ok_mask, "source_fanout_mean"
].max()

max_source_fanout_max = snb_relationship_sharedness_df.loc[
    ok_mask, "source_fanout_max"
].max()

snb_relationship_sharedness_df["relationship_sharedness_mean_norm"] = (
    snb_relationship_sharedness_df["target_reuse_mean"] / max_target_reuse_mean
    if max_target_reuse_mean and max_target_reuse_mean > 0
    else 0.0
)

snb_relationship_sharedness_df["relationship_sharedness_max_norm"] = (
    snb_relationship_sharedness_df["target_reuse_max"] / max_target_reuse_max
    if max_target_reuse_max and max_target_reuse_max > 0
    else 0.0
)

snb_relationship_sharedness_df["source_fanout_mean_norm"] = (
    snb_relationship_sharedness_df["source_fanout_mean"] / max_source_fanout_mean
    if max_source_fanout_mean and max_source_fanout_mean > 0
    else 0.0
)

snb_relationship_sharedness_df["source_fanout_max_norm"] = (
    snb_relationship_sharedness_df["source_fanout_max"] / max_source_fanout_max
    if max_source_fanout_max and max_source_fanout_max > 0
    else 0.0
)

snb_relationship_sharedness_df = snb_relationship_sharedness_df.sort_values(
    by=["relationship_sharedness_max_norm", "relationship_sharedness_mean_norm"],
    ascending=False
).reset_index(drop=True)

snb_relationship_sharedness_df

,relationship,source_entity,target_entity,semantic_type,relationship_kind,view_name,relationship_rows,distinct_sources,distinct_targets,source_fanout_mean,source_fanout_max,target_reuse_mean,target_reuse_max,status,relationship_sharedness_mean_norm,relationship_sharedness_max_norm,source_fanout_mean_norm,source_fanout_max_norm
0,post_is_located_in_place,Post,Place,association,many_to_one,snb_post_islocatedin_place,135701,135701,111,1.000000,1.0,1222.531532,20470.0,OK,0.898426,1.000000,0.022869,0.001570
1,comment_is_located_in_place,Comment,Place,association,many_to_one,snb_comment_islocatedin_place,151043,151043,111,1.000000,1.0,1360.747748,20390.0,OK,1.000000,0.996092,0.022869,0.001570
2,tag_has_type_tagclass,Tag,TagClass,association,many_to_one,snb_tag_hastype_tagclass,16080,16080,62,1.000000,1.0,259.354839,5061.0,OK,0.190597,0.247240,0.022869,0.001570
3,comment_has_creator_person,Comment,Person,association,many_to_one,snb_comment_hascreator_person,151043,151043,1400,1.000000,1.0,107.887857,2179.0,OK,0.079286,0.106448,0.022869,0.001570
4,forum_has_member_person,Forum,Person,associative,many_to_many,snb_forum_members_norm,123268,11077,1480,11.128284,340.0,83.289189,1493.0,OK,0.061208,0.072936,0.254489,0.533752
5,comment_has_tag,Comment,Tag,association,many_to_many,snb_comment_tags_norm,191303,50339,10802,3.800294,19.0,17.709961,1064.0,OK,0.013015,0.051979,0.086907,0.029827
6,post_has_tag,Post,Tag,association,many_to_many,snb_post_tags_norm,51118,15445,7343,3.309680,14.0,6.961460,828.0,OK,0.005116,0.040449,0.075688,0.021978
7,post_has_creator_person,Post,Person,association,many_to_one,snb_post_hascreator_person,135701,135701,1347,1.000000,1.0,100.743133,473.0,OK,0.074035,0.023107,0.022869,0.001570
8,forum_has_tag,Forum,Tag,association,many_to_many,snb_forum_tags_norm,47697,13750,4182,3.468873,80.0,11.405308,366.0,OK,0.008382,0.017880,0.079328,0.125589
9,person_likes_post,Person,Post,associative,many_to_many,snb_person_likes_posts_norm,47215,1385,8419,34.090253,637.0,5.608148,338.0,OK,0.004121,0.016512,0.779597,1.000000


In [146]:
# =========================================================
# BLOCK 9C — Compute entity-level observed sharedness
# =========================================================
#
# Goal:
# - Aggregate observed sharedness by entity.
# - Main signal:
#   entity appears as target of relationships reused by many sources.
# =========================================================

entity_sharedness_rows = []

entities = sorted(SNB_ENTITY_VIEW_MAP.keys())

for entity in entities:
    as_target = snb_relationship_sharedness_df[
        snb_relationship_sharedness_df["target_entity"] == entity
    ]

    as_source = snb_relationship_sharedness_df[
        snb_relationship_sharedness_df["source_entity"] == entity
    ]

    if len(as_target) > 0:
        target_sharedness_mean = as_target["relationship_sharedness_mean_norm"].mean()
        target_sharedness_max = as_target["relationship_sharedness_max_norm"].max()
        target_reuse_mean_raw = as_target["target_reuse_mean"].mean()
        target_reuse_max_raw = as_target["target_reuse_max"].max()
    else:
        target_sharedness_mean = 0.0
        target_sharedness_max = 0.0
        target_reuse_mean_raw = 0.0
        target_reuse_max_raw = 0.0

    if len(as_source) > 0:
        source_fanout_mean = as_source["source_fanout_mean_norm"].mean()
        source_fanout_max = as_source["source_fanout_max_norm"].max()
        source_fanout_mean_raw = as_source["source_fanout_mean"].mean()
        source_fanout_max_raw = as_source["source_fanout_max"].max()
    else:
        source_fanout_mean = 0.0
        source_fanout_max = 0.0
        source_fanout_mean_raw = 0.0
        source_fanout_max_raw = 0.0

    entity_sharedness_rows.append({
        "entity": entity,
        "n_relationships_as_target": len(as_target),
        "n_relationships_as_source": len(as_source),

        "target_reuse_mean_raw": target_reuse_mean_raw,
        "target_reuse_max_raw": target_reuse_max_raw,

        "source_fanout_mean_raw": source_fanout_mean_raw,
        "source_fanout_max_raw": source_fanout_max_raw,

        "entity_sharedness_mean": target_sharedness_mean,
        "entity_sharedness_max": target_sharedness_max,

        "entity_source_fanout_mean": source_fanout_mean,
        "entity_source_fanout_max": source_fanout_max,
    })

snb_entity_sharedness_df = pd.DataFrame(entity_sharedness_rows)

snb_entity_sharedness_df = snb_entity_sharedness_df.sort_values(
    by=["entity_sharedness_max", "entity_sharedness_mean"],
    ascending=False
).reset_index(drop=True)

snb_entity_sharedness_df

,entity,n_relationships_as_target,n_relationships_as_source,target_reuse_mean_raw,target_reuse_max_raw,source_fanout_mean_raw,source_fanout_max_raw,entity_sharedness_mean,entity_sharedness_max,entity_source_fanout_mean,entity_source_fanout_max
0,Place,5,1,518.516810,20470.0,12.427350,199.0,0.381053,1.000000,0.284196,0.312402
1,TagClass,2,1,131.519525,5061.0,1.000000,1.0,0.096652,0.247240,0.022869,0.001570
2,Person,5,7,62.519542,2179.0,16.788145,637.0,0.045945,0.106448,0.383922,1.000000
3,Tag,4,1,11.139878,1064.0,1.000000,1.0,0.008187,0.051979,0.022869,0.001570
4,Post,2,4,3.304074,338.0,2.611237,19.0,0.002428,0.016512,0.059715,0.029827
5,Comment,3,4,5.078848,269.0,2.040289,19.0,0.003732,0.013141,0.046659,0.029827
6,Organisation,2,1,2.761878,32.0,1.000000,1.0,0.002030,0.001563,0.022869,0.001570
7,Forum,0,4,0.000000,0.0,6.588141,340.0,0.000000,0.000000,0.150662,0.533752


In [147]:
# =========================================================
# BLOCK 9D — Compute query-level observed sharedness
# =========================================================
#
# Goal:
# - Estimate observed sharedness per official query.
# - Uses:
#   1. sharedness of accessed entities
#   2. sharedness of relationships used by the query
# =========================================================

entity_sharedness_mean_map = dict(
    zip(
        snb_entity_sharedness_df["entity"],
        snb_entity_sharedness_df["entity_sharedness_mean"]
    )
)

entity_sharedness_max_map = dict(
    zip(
        snb_entity_sharedness_df["entity"],
        snb_entity_sharedness_df["entity_sharedness_max"]
    )
)

relationship_sharedness_mean_map = dict(
    zip(
        snb_relationship_sharedness_df["relationship"],
        snb_relationship_sharedness_df["relationship_sharedness_mean_norm"]
    )
)

relationship_sharedness_max_map = dict(
    zip(
        snb_relationship_sharedness_df["relationship"],
        snb_relationship_sharedness_df["relationship_sharedness_max_norm"]
    )
)

query_sharedness_rows = []

for query in SNB_OFFICIAL_WORKLOAD:
    qname = query["query_name"]
    official_id = query["official_id"]

    accessed_entities = sorted(set(query["accessed_entities"]))
    relationships_used = sorted(set(query["relationships_used"]))

    entity_mean_values = [
        entity_sharedness_mean_map.get(entity, 0.0)
        for entity in accessed_entities
    ]

    entity_max_values = [
        entity_sharedness_max_map.get(entity, 0.0)
        for entity in accessed_entities
    ]

    relationship_mean_values = [
        relationship_sharedness_mean_map.get(rel, 0.0)
        for rel in relationships_used
    ]

    relationship_max_values = [
        relationship_sharedness_max_map.get(rel, 0.0)
        for rel in relationships_used
    ]

    all_mean_values = entity_mean_values + relationship_mean_values
    all_max_values = entity_max_values + relationship_max_values

    query_sharedness_rows.append({
        "query_name": qname,
        "official_id": official_id,
        "query_group": query["query_group"],
        "operation_type": query["operation_type"],
        "root_entity": query["root_entity"],

        "n_accessed_entities": len(accessed_entities),
        "n_relationships_used": len(relationships_used),

        "entity_sharedness_mean": (
            sum(entity_mean_values) / len(entity_mean_values)
            if entity_mean_values else 0.0
        ),
        "entity_sharedness_max": (
            max(entity_max_values)
            if entity_max_values else 0.0
        ),

        "relationship_sharedness_mean": (
            sum(relationship_mean_values) / len(relationship_mean_values)
            if relationship_mean_values else 0.0
        ),
        "relationship_sharedness_max": (
            max(relationship_max_values)
            if relationship_max_values else 0.0
        ),

        "observed_sharedness_mean": (
            sum(all_mean_values) / len(all_mean_values)
            if all_mean_values else 0.0
        ),
        "observed_sharedness_max": (
            max(all_max_values)
            if all_max_values else 0.0
        ),
    })

snb_query_sharedness_df = pd.DataFrame(query_sharedness_rows)

snb_query_sharedness_df = snb_query_sharedness_df.sort_values(
    by=["observed_sharedness_max", "observed_sharedness_mean"],
    ascending=False
).reset_index(drop=True)

snb_query_sharedness_df

,query_name,official_id,query_group,operation_type,root_entity,n_accessed_entities,n_relationships_used,entity_sharedness_mean,entity_sharedness_max,relationship_sharedness_mean,relationship_sharedness_max,observed_sharedness_mean,observed_sharedness_max
0,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,complex_read,read,Person,4,7,0.108290,1.000000,0.294615,1.000000,0.226861,1.000000
1,INS6_AddPost,INS6,insert,insert,Post,5,4,0.087523,1.000000,0.244578,1.000000,0.157325,1.000000
2,INS7_AddComment,INS7,insert,insert,Comment,5,5,0.088269,1.000000,0.218754,0.996092,0.153512,1.000000
3,IS1_ProfileOfPerson,IS1,short_read,read,Person,2,1,0.213499,1.000000,0.001244,0.000293,0.142747,1.000000
4,INS1_AddPerson,INS1,insert,insert,Person,4,4,0.109304,1.000000,0.002884,0.008451,0.056094,1.000000
5,IC1_TransitiveFriendsWithName,IC1,complex_read,read,Person,3,5,0.143009,1.000000,0.003749,0.016170,0.055972,1.000000
6,IS5_CreatorOfMessage,IS5,short_read,read,Post,3,2,0.017369,0.106448,0.076660,0.106448,0.041085,0.106448
7,INS5_AddForumMembership,INS5,insert,insert,Forum,2,1,0.022972,0.106448,0.061208,0.072936,0.035718,0.106448
8,IC2_RecentMessagesByFriends,IC2,complex_read,read,Person,3,3,0.017369,0.106448,0.053968,0.106448,0.035668,0.106448
9,IS2_RecentMessagesOfPerson,IS2,short_read,read,Person,3,4,0.017369,0.106448,0.038698,0.106448,0.029557,0.106448


In [148]:
# =========================================================
# BLOCK 9E — Add observed sharedness to final matrix
# =========================================================
#
# Goal:
# - Extend the matrix with observed sharedness.
# - This matrix will feed the activation logic later.
# =========================================================

snb_matrix_with_sharedness_df = (
    snb_matrix_with_volatility_df
    .merge(
        snb_query_sharedness_df[
            [
                "query_name",
                "official_id",
                "entity_sharedness_mean",
                "entity_sharedness_max",
                "relationship_sharedness_mean",
                "relationship_sharedness_max",
                "observed_sharedness_mean",
                "observed_sharedness_max",
            ]
        ],
        on=["query_name", "official_id"],
        how="left"
    )
)

snb_matrix_with_sharedness_df

,query_name,official_id,official_title,query_group,operation_type,root_entity,accessed_entities,relationships_used,Rc_distinct,Rc_weighted,D,Re,DeltaR,DeltaRratio,association_count,associative_count,containment_count,lookup_count,dominant_semantic_type,weighted_relationships,description,entity_volatility_mean,entity_volatility_max,relationship_volatility_mean,relationship_volatility_max,update_volatility_mean,update_volatility_max,entity_sharedness_mean,entity_sharedness_max,relationship_sharedness_mean,relationship_sharedness_max,observed_sharedness_mean,observed_sharedness_max
0,IS1_ProfileOfPerson,IS1,Profile of a person,short_read,read,Person,"[Person, Place]",[person_is_located_in_place],1,1,1,0,1,1.000000,1,0,0,0,association,[person_is_located_in_place x1],Official IS1: retrieve a person's profile and city of residence.,0.638779,1.000000,0.001358,0.001358,0.426306,1.000000,0.213499,1.000000,0.001244,0.000293,0.142747,1.000000
1,IS2_RecentMessagesOfPerson,IS2,Recent messages of a person,short_read,read,Person,"[Person, Post, Comment]","[post_has_creator_person, comment_has_creator_person, comment_reply_of_post, comment_reply_of_comment]",4,4,2,2,2,0.500000,2,0,2,0,mixed,"[post_has_creator_person x1, comment_has_creator_person x1, comment_reply_of_post x1, comment_reply_of_comment x1]",Official IS2: retrieve the last messages created by a person and the original post in the conversation.,0.566812,1.000000,0.342608,0.370959,0.438695,1.000000,0.017369,0.106448,0.038698,0.106448,0.029557,0.106448
2,IS3_FriendsOfPerson,IS3,Friends of a person,short_read,read,Person,[Person],[person_knows_person],1,1,1,0,1,1.000000,1,0,0,0,association,[person_knows_person x1],Official IS3: retrieve all friends of a person and the friendship creation date.,1.000000,1.000000,0.025250,0.025250,0.512625,1.000000,0.045945,0.106448,0.008583,0.016170,0.027264,0.106448
3,IS4_ContentOfMessage,IS4,Content of a message,short_read,read,Post,"[Post, Comment]",[],0,0,0,0,0,0.000000,0,0,0,1,lookup,[],Official IS4: retrieve content and creation date of a Message. Message is represented by Post or Comment.,0.350218,0.431184,0.000000,0.000000,0.350218,0.431184,0.003080,0.016512,0.000000,0.000000,0.003080,0.016512
4,IS5_CreatorOfMessage,IS5,Creator of a message,short_read,read,Post,"[Post, Comment, Person]","[post_has_creator_person, comment_has_creator_person]",2,2,1,1,1,0.500000,2,0,0,0,association,"[post_has_creator_person x1, comment_has_creator_person x1]",Official IS5: retrieve the author of a Message. Message is represented by Post or Comment.,0.566812,1.000000,0.314256,0.370959,0.465789,1.000000,0.017369,0.106448,0.076660,0.106448,0.041085,0.106448
5,IS6_ForumOfMessage,IS6,Forum of a message,short_read,read,Post,"[Post, Comment, Forum, Person]","[forum_container_of_post, comment_reply_of_post, comment_reply_of_comment, forum_has_moderator_person]",4,4,3,1,3,0.750000,1,0,3,0,containment,"[forum_container_of_post x1, comment_reply_of_post x1, comment_reply_of_comment x1, forum_has_moderator_person x1]",Official IS6: retrieve the forum containing a Message and the moderator of that forum.,0.566309,1.000000,0.255908,0.370959,0.411108,1.000000,0.013026,0.106448,0.002204,0.001808,0.007615,0.106448
6,IS7_RepliesOfMessage,IS7,Replies of a message,short_read,read,Post,"[Post, Comment, Person]","[comment_reply_of_post, comment_reply_of_comment, comment_has_creator_person, post_has_creator_person, person_knows_person]",5,5,2,3,2,0.400000,3,0,2,0,association,"[comment_reply_of_post x1, comment_reply_of_comment x1, comment_has_creator_person x1, post_has_creator_person x1, person_knows_person x1]",Official IS7: retrieve one-hop replies of a Message and whether reply author knows original author.,0.566812,1.000000,0.279136,0.370959,0.387014,1.000000,0.017369,0.106448,0.032675,0.106448,0.026935,0.106448
7,IC1_TransitiveFriendsWithName,IC1,Transitive friends with a certain name,complex_read,read,Person,"[Person, Place, Organisation]","[person_knows_person, person_is_loca

In [149]:
# =========================================================
# BLOCK 9F — Rank queries by observed sharedness
# =========================================================

snb_sharedness_rank_df = (
    snb_matrix_with_sharedness_df[
        [
            "query_name",
            "official_id",
            "query_group",
            "operation_type",
            "root_entity",
            "Rc_weighted",
            "D",
            "Re",
            "DeltaRratio",
            "dominant_semantic_type",
            "update_volatility_mean",
            "update_volatility_max",
            "observed_sharedness_mean",
            "observed_sharedness_max",
        ]
    ]
    .sort_values(
        by=["observed_sharedness_max", "observed_sharedness_mean", "Rc_weighted"],
        ascending=False
    )
    .reset_index(drop=True)
)

snb_sharedness_rank_df

,query_name,official_id,query_group,operation_type,root_entity,Rc_weighted,D,Re,DeltaRratio,dominant_semantic_type,update_volatility_mean,update_volatility_max,observed_sharedness_mean,observed_sharedness_max
0,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,complex_read,read,Person,8,4,4,0.500000,association,0.296511,1.000000,0.226861,1.000000
1,INS6_AddPost,INS6,insert,insert,Post,4,1,3,0.250000,association,0.399106,1.000000,0.157325,1.000000
2,INS7_AddComment,INS7,insert,insert,Comment,5,1,4,0.200000,association,0.412100,1.000000,0.153512,1.000000
3,IS1_ProfileOfPerson,IS1,short_read,read,Person,1,1,0,1.000000,association,0.426306,1.000000,0.142747,1.000000
4,INS1_AddPerson,INS1,insert,insert,Person,4,1,3,0.250000,association,0.196475,1.000000,0.056094,1.000000
5,IC1_TransitiveFriendsWithName,IC1,complex_read,read,Person,7,4,3,0.571429,association,0.163435,1.000000,0.055972,1.000000
6,IS5_CreatorOfMessage,IS5,short_read,read,Post,2,1,1,0.500000,association,0.465789,1.000000,0.041085,0.106448
7,INS5_AddForumMembership,INS5,insert,insert,Forum,1,1,0,1.000000,associative,0.854933,1.000000,0.035718,0.106448
8,IC2_RecentMessagesByFriends,IC2,complex_read,read,Person,3,2,1,0.666667,association,0.392366,1.000000,0.035668,0.106448
9,IS2_RecentMessagesOfPerson,IS2,short_read,read,Person,4,2,2,0.500000,mixed,0.438695,1.000000,0.029557,0.106448


In [150]:
# =========================================================
# BLOCK 9G — Validate observed sharedness metrics
# =========================================================
#
# Goal:
# - Check that normalized values are between 0 and 1.
# =========================================================

sharedness_validation_rows = []

for _, row in snb_query_sharedness_df.iterrows():
    sharedness_validation_rows.append({
        "query_name": row["query_name"],
        "official_id": row["official_id"],
        "observed_sharedness_mean": row["observed_sharedness_mean"],
        "observed_sharedness_max": row["observed_sharedness_max"],
        "valid_mean": 0 <= row["observed_sharedness_mean"] <= 1,
        "valid_max": 0 <= row["observed_sharedness_max"] <= 1,
    })

snb_sharedness_validation_df = pd.DataFrame(sharedness_validation_rows)

snb_sharedness_validation_df

,query_name,official_id,observed_sharedness_mean,observed_sharedness_max,valid_mean,valid_max
0,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,0.226861,1.000000,True,True
1,INS6_AddPost,INS6,0.157325,1.000000,True,True
2,INS7_AddComment,INS7,0.153512,1.000000,True,True
3,IS1_ProfileOfPerson,IS1,0.142747,1.000000,True,True
4,INS1_AddPerson,INS1,0.056094,1.000000,True,True
5,IC1_TransitiveFriendsWithName,IC1,0.055972,1.000000,True,True
6,IS5_CreatorOfMessage,IS5,0.041085,0.106448,True,True
7,INS5_AddForumMembership,INS5,0.035718,0.106448,True,True
8,IC2_RecentMessagesByFriends,IC2,0.035668,0.106448,True,True
9,IS2_RecentMessagesOfPerson,IS2,0.029557,0.106448,True,True


In [152]:
snb_sharedness_validation_df[
    (snb_sharedness_validation_df["valid_mean"] == False)
    | (snb_sharedness_validation_df["valid_max"] == False)
]

,query_name,official_id,observed_sharedness_mean,observed_sharedness_max,valid_mean,valid_max


In [153]:
# =========================================================
# BLOCK 9H — Save observed sharedness artifacts
# =========================================================

SHAREDNESS_ARTIFACTS_DIR = (
    BASE_DIR.parent.parent
    / "benchmark_artifacts_dir"
    / "ldbc_snb_observed_sharedness_official"
)

SHAREDNESS_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

snb_relationship_sharedness_raw_df.to_csv(
    SHAREDNESS_ARTIFACTS_DIR / "relationship_sharedness_raw.csv",
    index=False
)

snb_relationship_sharedness_df.to_csv(
    SHAREDNESS_ARTIFACTS_DIR / "relationship_sharedness_normalized.csv",
    index=False
)

snb_entity_sharedness_df.to_csv(
    SHAREDNESS_ARTIFACTS_DIR / "entity_sharedness.csv",
    index=False
)

snb_query_sharedness_df.to_csv(
    SHAREDNESS_ARTIFACTS_DIR / "query_sharedness.csv",
    index=False
)

snb_matrix_with_sharedness_df.to_csv(
    SHAREDNESS_ARTIFACTS_DIR / "matrix_with_sharedness.csv",
    index=False
)

snb_sharedness_rank_df.to_csv(
    SHAREDNESS_ARTIFACTS_DIR / "sharedness_rank.csv",
    index=False
)

snb_sharedness_validation_df.to_csv(
    SHAREDNESS_ARTIFACTS_DIR / "sharedness_validation.csv",
    index=False
)

print("Saved observed sharedness artifacts to:")
print(SHAREDNESS_ARTIFACTS_DIR)

print("\nSaved files:")
for path in sorted(SHAREDNESS_ARTIFACTS_DIR.glob("*.csv")):
    print(" -", path.name)

Saved observed sharedness artifacts to:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_observed_sharedness_official

Saved files:
 - entity_sharedness.csv
 - matrix_with_sharedness.csv
 - query_sharedness.csv
 - relationship_sharedness_normalized.csv
 - relationship_sharedness_raw.csv
 - sharedness_rank.csv
 - sharedness_validation.csv


In [154]:
# =========================================================
# BLOCK 9I — Save checkpoint after observed sharedness
# =========================================================
#
# Goal:
# - Save Block 9 DataFrames into the checkpoint.
# - If the connection drops, continue after Block 9.
# =========================================================

from pathlib import Path
import json
import pandas as pd

PROJECT_DIR = BASE_DIR.parent.parent

CHECKPOINT_DIR = PROJECT_DIR / "benchmark_artifacts_dir" / "ldbc_snb_checkpoints"
CHECKPOINT_DATAFRAMES_DIR = CHECKPOINT_DIR / "dataframes"
CHECKPOINT_METADATA_DIR = CHECKPOINT_DIR / "metadata"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_METADATA_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = CHECKPOINT_DIR / "checkpoint_manifest.json"
metadata_path = CHECKPOINT_METADATA_DIR / "checkpoint_metadata.json"

if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        checkpoint_manifest = json.load(f)
else:
    checkpoint_manifest = {
        "dataframes": {},
        "metadata": str(metadata_path.relative_to(CHECKPOINT_DIR)),
    }

if metadata_path.exists():
    with open(metadata_path, "r", encoding="utf-8") as f:
        checkpoint_metadata = json.load(f)
else:
    checkpoint_metadata = {}

BLOCK_9_CHECKPOINT_DATAFRAMES = [
    "snb_relationship_sharedness_raw_df",
    "snb_relationship_sharedness_df",
    "snb_entity_sharedness_df",
    "snb_query_sharedness_df",
    "snb_matrix_with_sharedness_df",
    "snb_sharedness_rank_df",
    "snb_sharedness_validation_df",
]

saved_block_9_dataframes = []

for var_name in BLOCK_9_CHECKPOINT_DATAFRAMES:
    obj = globals().get(var_name)

    if isinstance(obj, pd.DataFrame):
        out_path = CHECKPOINT_DATAFRAMES_DIR / f"{var_name}.json"

        obj.to_json(
            out_path,
            orient="records",
            indent=2,
            force_ascii=False
        )

        checkpoint_manifest["dataframes"][var_name] = str(
            out_path.relative_to(CHECKPOINT_DIR)
        )

        saved_block_9_dataframes.append(var_name)

# Keep metadata updated.
checkpoint_metadata.update({
    "BASE_DIR": str(BASE_DIR),
    "SNB_SNAPSHOT_DIR": str(SNB_SNAPSHOT_DIR),
    "SNB_DYNAMIC_DIR": str(SNB_DYNAMIC_DIR),
    "SNB_STATIC_DIR": str(SNB_STATIC_DIR),
    "SNB_SUBSTITUTION_DIR": str(SNB_SUBSTITUTION_DIR),
    "SNB_UPDATE_DIR": str(SNB_UPDATE_DIR),

    "SNB_ENTITY_VIEW_MAP": SNB_ENTITY_VIEW_MAP,
    "SNB_ENTITY_PK_MAP": SNB_ENTITY_PK_MAP,
    "SNB_COLLECTION_NAME_BY_ENTITY": SNB_COLLECTION_NAME_BY_ENTITY,
    "SNB_RELATIONSHIP_HINTS": SNB_RELATIONSHIP_HINTS,

    "SNB_OFFICIAL_WORKLOAD": SNB_OFFICIAL_WORKLOAD,
    "SNB_OFFICIAL_QUERY_PATH_OVERRIDES": SNB_OFFICIAL_QUERY_PATH_OVERRIDES,
    "SNB_RELATIONSHIP_MULTIPLICITY_OVERRIDES": SNB_RELATIONSHIP_MULTIPLICITY_OVERRIDES,
})

if "UPDATE_OPERATION_CODE_MAP" in globals():
    checkpoint_metadata["UPDATE_OPERATION_CODE_MAP"] = UPDATE_OPERATION_CODE_MAP

if "PERSISTENT_DUCKDB_PATH" in globals():
    checkpoint_metadata["PERSISTENT_DUCKDB_PATH"] = str(PERSISTENT_DUCKDB_PATH)

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(checkpoint_metadata, f, indent=2, ensure_ascii=False)

checkpoint_manifest["metadata"] = str(metadata_path.relative_to(CHECKPOINT_DIR))

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(checkpoint_manifest, f, indent=2, ensure_ascii=False)

print("Block 9 checkpoint saved to:")
print(CHECKPOINT_DIR)

print("\nSaved Block 9 DataFrames:")
for name in saved_block_9_dataframes:
    print(" -", name)

Block 9 checkpoint saved to:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_checkpoints

Saved Block 9 DataFrames:
 - snb_relationship_sharedness_raw_df
 - snb_relationship_sharedness_df
 - snb_entity_sharedness_df
 - snb_query_sharedness_df
 - snb_matrix_with_sharedness_df
 - snb_sharedness_rank_df
 - snb_sharedness_validation_df


In [155]:
# =========================================================
# LOAD BLOCK 9 ARTIFACTS — Restore observed sharedness outputs
# =========================================================

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB")

SHAREDNESS_ARTIFACTS_DIR = (
    PROJECT_DIR
    / "benchmark_artifacts_dir"
    / "ldbc_snb_observed_sharedness_official"
)

snb_relationship_sharedness_raw_df = pd.read_csv(
    SHAREDNESS_ARTIFACTS_DIR / "relationship_sharedness_raw.csv"
)

snb_relationship_sharedness_df = pd.read_csv(
    SHAREDNESS_ARTIFACTS_DIR / "relationship_sharedness_normalized.csv"
)

snb_entity_sharedness_df = pd.read_csv(
    SHAREDNESS_ARTIFACTS_DIR / "entity_sharedness.csv"
)

snb_query_sharedness_df = pd.read_csv(
    SHAREDNESS_ARTIFACTS_DIR / "query_sharedness.csv"
)

snb_matrix_with_sharedness_df = pd.read_csv(
    SHAREDNESS_ARTIFACTS_DIR / "matrix_with_sharedness.csv"
)

snb_sharedness_rank_df = pd.read_csv(
    SHAREDNESS_ARTIFACTS_DIR / "sharedness_rank.csv"
)

snb_sharedness_validation_df = pd.read_csv(
    SHAREDNESS_ARTIFACTS_DIR / "sharedness_validation.csv"
)

print("Loaded Block 9 artifacts successfully.")

print("\nRows:")
print("relationship_sharedness:", len(snb_relationship_sharedness_df))
print("entity_sharedness:", len(snb_entity_sharedness_df))
print("query_sharedness:", len(snb_query_sharedness_df))
print("matrix_with_sharedness:", len(snb_matrix_with_sharedness_df))

Loaded Block 9 artifacts successfully.

Rows:
relationship_sharedness: 23
entity_sharedness: 8
query_sharedness: 22
matrix_with_sharedness: 22


In [156]:
# =========================================================
# BLOCK 10A — Build final analytical matrix
# =========================================================
#
# Goal:
# - Build the final analytical matrix for the official LDBC SNB workload.
# - This matrix is equivalent to the final document variable matrix
#   used before the activation logic.
# =========================================================

import pandas as pd

snb_final_analytical_matrix_df = snb_matrix_with_sharedness_df.copy()

# ---------------------------------------------------------
# Ensure key numeric variables are numeric
# ---------------------------------------------------------

numeric_columns = [
    "Rc_distinct",
    "Rc_weighted",
    "D",
    "Re",
    "DeltaR",
    "DeltaRratio",
    "association_count",
    "associative_count",
    "containment_count",
    "lookup_count",
    "entity_volatility_mean",
    "entity_volatility_max",
    "relationship_volatility_mean",
    "relationship_volatility_max",
    "update_volatility_mean",
    "update_volatility_max",
    "entity_sharedness_mean",
    "entity_sharedness_max",
    "relationship_sharedness_mean",
    "relationship_sharedness_max",
    "observed_sharedness_mean",
    "observed_sharedness_max",
]

for col in numeric_columns:
    if col in snb_final_analytical_matrix_df.columns:
        snb_final_analytical_matrix_df[col] = pd.to_numeric(
            snb_final_analytical_matrix_df[col],
            errors="coerce"
        ).fillna(0)

# ---------------------------------------------------------
# Add compact methodological labels
# ---------------------------------------------------------

def classify_depth(d):
    if d == 0:
        return "lookup"
    if d == 1:
        return "shallow"
    if d == 2:
        return "medium"
    return "deep"


def classify_reduction_ratio(x):
    if x >= 0.75:
        return "high"
    if x >= 0.40:
        return "medium"
    if x > 0:
        return "low"
    return "none"


def classify_volatility(x):
    if x >= 0.75:
        return "high"
    if x >= 0.35:
        return "medium"
    if x > 0:
        return "low"
    return "none"


def classify_sharedness(x):
    if x >= 0.75:
        return "high"
    if x >= 0.25:
        return "medium"
    if x > 0:
        return "low"
    return "none"


snb_final_analytical_matrix_df["depth_class"] = snb_final_analytical_matrix_df["D"].apply(classify_depth)

snb_final_analytical_matrix_df["structural_reduction_class"] = (
    snb_final_analytical_matrix_df["DeltaRratio"].apply(classify_reduction_ratio)
)

snb_final_analytical_matrix_df["update_volatility_class"] = (
    snb_final_analytical_matrix_df["update_volatility_max"].apply(classify_volatility)
)

snb_final_analytical_matrix_df["observed_sharedness_class"] = (
    snb_final_analytical_matrix_df["observed_sharedness_max"].apply(classify_sharedness)
)

# ---------------------------------------------------------
# Reorder columns for readability
# ---------------------------------------------------------

preferred_columns = [
    "query_name",
    "official_id",
    "official_title",
    "query_group",
    "operation_type",
    "root_entity",

    "accessed_entities",
    "relationships_used",
    "weighted_relationships",

    "Rc_distinct",
    "Rc_weighted",
    "D",
    "Re",
    "DeltaR",
    "DeltaRratio",

    "depth_class",
    "structural_reduction_class",

    "association_count",
    "associative_count",
    "containment_count",
    "lookup_count",
    "dominant_semantic_type",

    "entity_volatility_mean",
    "entity_volatility_max",
    "relationship_volatility_mean",
    "relationship_volatility_max",
    "update_volatility_mean",
    "update_volatility_max",
    "update_volatility_class",

    "entity_sharedness_mean",
    "entity_sharedness_max",
    "relationship_sharedness_mean",
    "relationship_sharedness_max",
    "observed_sharedness_mean",
    "observed_sharedness_max",
    "observed_sharedness_class",

    "description",
]

existing_columns = [
    col for col in preferred_columns
    if col in snb_final_analytical_matrix_df.columns
]

remaining_columns = [
    col for col in snb_final_analytical_matrix_df.columns
    if col not in existing_columns
]

snb_final_analytical_matrix_df = snb_final_analytical_matrix_df[
    existing_columns + remaining_columns
]

snb_final_analytical_matrix_df

,query_name,official_id,official_title,query_group,operation_type,root_entity,accessed_entities,relationships_used,weighted_relationships,Rc_distinct,Rc_weighted,D,Re,DeltaR,DeltaRratio,depth_class,structural_reduction_class,association_count,associative_count,containment_count,lookup_count,dominant_semantic_type,entity_volatility_mean,entity_volatility_max,relationship_volatility_mean,relationship_volatility_max,update_volatility_mean,update_volatility_max,update_volatility_class,entity_sharedness_mean,entity_sharedness_max,relationship_sharedness_mean,relationship_sharedness_max,observed_sharedness_mean,observed_sharedness_max,observed_sharedness_class,description
0,IS1_ProfileOfPerson,IS1,Profile of a person,short_read,read,Person,"['Person', 'Place']",['person_is_located_in_place'],['person_is_located_in_place x1'],1,1,1,0,1,1.000000,shallow,high,1,0,0,0,association,0.638779,1.000000,0.001358,0.001358,0.426306,1.000000,high,0.213499,1.000000,0.001244,0.000293,0.142747,1.000000,high,Official IS1: retrieve a person's profile and city of residence.
1,IS2_RecentMessagesOfPerson,IS2,Recent messages of a person,short_read,read,Person,"['Person', 'Post', 'Comment']","['post_has_creator_person', 'comment_has_creator_person', 'comment_reply_of_post', 'comment_reply_of_comment']","['post_has_creator_person x1', 'comment_has_creator_person x1', 'comment_reply_of_post x1', 'comment_reply_of_comment x1']",4,4,2,2,2,0.500000,medium,medium,2,0,2,0,mixed,0.566812,1.000000,0.342608,0.370959,0.438695,1.000000,high,0.017369,0.106448,0.038698,0.106448,0.029557,0.106448,low,Official IS2: retrieve the last messages created by a person and the original post in the conversation.
2,IS3_FriendsOfPerson,IS3,Friends of a person,short_read,read,Person,['Person'],['person_knows_person'],['person_knows_person x1'],1,1,1,0,1,1.000000,shallow,high,1,0,0,0,association,1.000000,1.000000,0.025250,0.025250,0.512625,1.000000,high,0.045945,0.106448,0.008583,0.016170,0.027264,0.106448,low,Official IS3: retrieve all friends of a person and the friendship creation date.
3,IS4_ContentOfMessage,IS4,Content of a message,short_read,read,Post,"['Post', 'Comment']",[],[],0,0,0,0,0,0.000000,lookup,none,0,0,0,1,lookup,0.350218,0.431184,0.000000,0.000000,0.350218,0.431184,medium,0.003080,0.016512,0.000000,0.000000,0.003080,0.016512,low,Official IS4: retrieve content and creation date of a Message. Message is represented by Post or Comment.
4,IS5_CreatorOfMessage,IS5,Creator of a message,short_read,read,Post,"['Post', 'Comment', 'Person']","['post_has_creator_person', 'comment_has_creator_person']","['post_has_creator_person x1', 'comment_has_creator_person x1']",2,2,1,1,1,0.500000,shallow,medium,2,0,0,0,association,0.566812,1.000000,0.314256,0.370959,0.465789,1.000000,high,0.017369,0.106448,0.076660,0.106448,0.041085,0.106448,low,Official IS5: retrieve the author of a Message. Message is represented by Post or Comment.
5,IS6_ForumOfMessage,IS6,Forum of a message,short_read,read,Post,"['Post', 'Comment', 'Forum', 'Person']","['forum_container_of_post', 'comment_reply_of_post', 'comment_reply_of_comment', 'forum_has_moderator_person']","['forum_container_of_post x1', 'comment_reply_of_post x1', 'comment_reply_of_comment x1', 'forum_has_moderator_person x1']",4,4,3,1,3,0.750000,deep,high,1,0,3,0,containment,0.566309,1.000000,0.255908,0.370959,0.411108,1.000000,high,0.013026,0.106448,0.002204,0.001808,0.007615,0.106448,low,Official IS6: retrieve the forum containing a Message and the moderator of that forum.
6,IS7_RepliesOfMessage,IS7,Replies of a message,short_read,read,Post,"['Post', 'Comment', 'Person']","['comment_reply_of_post', 'comment_reply_of_comment', 'comment_has_creator_person', 'post_has_creator_person', 'person_knows_person']","['comment_reply_of_post x1', 'comment_reply_of_comment x1', 'comment_has_creator_person x1', 'post_has_creator_person x1', 'person_knows_person x1']",5,5,2,3,2,0.400000,medium,medium,3,0,2,0,association,0.566812,1.000000,0.279136,0.370959,

In [157]:
# =========================================================
# BLOCK 10B — Validate final analytical matrix
# =========================================================
#
# Goal:
# - Check missing values and invalid ranges.
# - Confirm that the final matrix is ready for activation.
# =========================================================

validation_rows = []

required_columns = [
    "query_name",
    "official_id",
    "query_group",
    "operation_type",
    "root_entity",
    "Rc_weighted",
    "D",
    "Re",
    "DeltaR",
    "DeltaRratio",
    "dominant_semantic_type",
    "update_volatility_max",
    "observed_sharedness_max",
]

for col in required_columns:
    validation_rows.append({
        "check": f"missing_values_in_{col}",
        "value": int(snb_final_analytical_matrix_df[col].isna().sum())
        if col in snb_final_analytical_matrix_df.columns
        else None,
        "status": "OK"
        if col in snb_final_analytical_matrix_df.columns
        and snb_final_analytical_matrix_df[col].isna().sum() == 0
        else "ERROR",
    })

# Range checks
range_checks = [
    ("DeltaRratio", 0, 1),
    ("update_volatility_mean", 0, 1),
    ("update_volatility_max", 0, 1),
    ("observed_sharedness_mean", 0, 1),
    ("observed_sharedness_max", 0, 1),
]

for col, lo, hi in range_checks:
    if col in snb_final_analytical_matrix_df.columns:
        invalid_count = int(
            (
                (snb_final_analytical_matrix_df[col] < lo)
                | (snb_final_analytical_matrix_df[col] > hi)
            ).sum()
        )

        validation_rows.append({
            "check": f"range_{col}_{lo}_{hi}",
            "value": invalid_count,
            "status": "OK" if invalid_count == 0 else "ERROR",
        })

snb_final_matrix_validation_df = pd.DataFrame(validation_rows)
snb_final_matrix_validation_df

,check,value,status
0,missing_values_in_query_name,0,OK
1,missing_values_in_official_id,0,OK
2,missing_values_in_query_group,0,OK
3,missing_values_in_operation_type,0,OK
4,missing_values_in_root_entity,0,OK
5,missing_values_in_Rc_weighted,0,OK
6,missing_values_in_D,0,OK
7,missing_values_in_Re,0,OK
8,missing_values_in_DeltaR,0,OK
9,missing_values_in_DeltaRratio,0,OK


In [158]:
snb_final_matrix_validation_df[
    snb_final_matrix_validation_df["status"] != "OK"
]

,check,value,status


In [159]:
# =========================================================
# BLOCK 10C — Summarize final matrix by query group
# =========================================================
#
# Goal:
# - Summarize the analytical behavior of short reads, complex reads,
#   and inserts.
# =========================================================

snb_final_matrix_group_summary_df = (
    snb_final_analytical_matrix_df
    .groupby("query_group")
    .agg(
        n_queries=("query_name", "count"),
        avg_Rc_weighted=("Rc_weighted", "mean"),
        avg_D=("D", "mean"),
        avg_Re=("Re", "mean"),
        avg_DeltaRratio=("DeltaRratio", "mean"),
        avg_update_volatility=("update_volatility_mean", "mean"),
        max_update_volatility=("update_volatility_max", "max"),
        avg_observed_sharedness=("observed_sharedness_mean", "mean"),
        max_observed_sharedness=("observed_sharedness_max", "max"),
    )
    .reset_index()
)

snb_final_matrix_group_summary_df

,query_group,n_queries,avg_Rc_weighted,avg_D,avg_Re,avg_DeltaRratio,avg_update_volatility,max_update_volatility,avg_observed_sharedness,max_observed_sharedness
0,complex_read,7,5.000000,3.000000,2.000,0.641156,0.354112,1.0,0.060377,1.0
1,insert,8,2.375000,1.000000,1.375,0.650000,0.481541,1.0,0.060130,1.0
2,short_read,7,2.428571,1.428571,1.000,0.592857,0.427394,1.0,0.039755,1.0


In [160]:
# =========================================================
# BLOCK 10D — Summarize final matrix by semantic type
# =========================================================

snb_final_matrix_semantic_summary_df = (
    snb_final_analytical_matrix_df
    .groupby("dominant_semantic_type")
    .agg(
        n_queries=("query_name", "count"),
        avg_Rc_weighted=("Rc_weighted", "mean"),
        avg_D=("D", "mean"),
        avg_Re=("Re", "mean"),
        avg_DeltaRratio=("DeltaRratio", "mean"),
        avg_update_volatility=("update_volatility_mean", "mean"),
        max_update_volatility=("update_volatility_max", "max"),
        avg_observed_sharedness=("observed_sharedness_mean", "mean"),
        max_observed_sharedness=("observed_sharedness_max", "max"),
    )
    .reset_index()
    .sort_values("n_queries", ascending=False)
)

snb_final_matrix_semantic_summary_df

,dominant_semantic_type,n_queries,avg_Rc_weighted,avg_D,avg_Re,avg_DeltaRratio,avg_update_volatility,max_update_volatility,avg_observed_sharedness,max_observed_sharedness
0,association,16,3.75,1.9375,1.8125,0.599256,0.385693,1.000000,0.066793,1.000000
1,associative,3,1.00,1.0000,0.0000,1.000000,0.650586,1.000000,0.024339,0.106448
2,containment,1,4.00,3.0000,1.0000,0.750000,0.411108,1.000000,0.007615,0.106448
3,lookup,1,0.00,0.0000,0.0000,0.000000,0.350218,0.431184,0.003080,0.016512
4,mixed,1,4.00,2.0000,2.0000,0.500000,0.438695,1.000000,0.029557,0.106448


In [161]:
# =========================================================
# BLOCK 10E — Save final analytical matrix artifacts
# =========================================================

from pathlib import Path

FINAL_MATRIX_ARTIFACTS_DIR = (
    BASE_DIR.parent.parent
    / "benchmark_artifacts_dir"
    / "ldbc_snb_final_analytical_matrix_official"
)

FINAL_MATRIX_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

snb_final_analytical_matrix_df.to_csv(
    FINAL_MATRIX_ARTIFACTS_DIR / "snb_final_analytical_matrix.csv",
    index=False
)

snb_final_matrix_validation_df.to_csv(
    FINAL_MATRIX_ARTIFACTS_DIR / "snb_final_matrix_validation.csv",
    index=False
)

snb_final_matrix_group_summary_df.to_csv(
    FINAL_MATRIX_ARTIFACTS_DIR / "snb_final_matrix_group_summary.csv",
    index=False
)

snb_final_matrix_semantic_summary_df.to_csv(
    FINAL_MATRIX_ARTIFACTS_DIR / "snb_final_matrix_semantic_summary.csv",
    index=False
)

print("Saved final analytical matrix artifacts to:")
print(FINAL_MATRIX_ARTIFACTS_DIR)

print("\nSaved files:")
for path in sorted(FINAL_MATRIX_ARTIFACTS_DIR.glob("*.csv")):
    print(" -", path.name)

Saved final analytical matrix artifacts to:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_final_analytical_matrix_official

Saved files:
 - snb_final_analytical_matrix.csv
 - snb_final_matrix_group_summary.csv
 - snb_final_matrix_semantic_summary.csv
 - snb_final_matrix_validation.csv


In [162]:
# =========================================================
# BLOCK 10F — Save checkpoint after final analytical matrix
# =========================================================

from pathlib import Path
import json
import pandas as pd

PROJECT_DIR = BASE_DIR.parent.parent

CHECKPOINT_DIR = PROJECT_DIR / "benchmark_artifacts_dir" / "ldbc_snb_checkpoints"
CHECKPOINT_DATAFRAMES_DIR = CHECKPOINT_DIR / "dataframes"
CHECKPOINT_METADATA_DIR = CHECKPOINT_DIR / "metadata"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_METADATA_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = CHECKPOINT_DIR / "checkpoint_manifest.json"
metadata_path = CHECKPOINT_METADATA_DIR / "checkpoint_metadata.json"

if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        checkpoint_manifest = json.load(f)
else:
    checkpoint_manifest = {
        "dataframes": {},
        "metadata": str(metadata_path.relative_to(CHECKPOINT_DIR)),
    }

if metadata_path.exists():
    with open(metadata_path, "r", encoding="utf-8") as f:
        checkpoint_metadata = json.load(f)
else:
    checkpoint_metadata = {}

BLOCK_10_CHECKPOINT_DATAFRAMES = [
    "snb_final_analytical_matrix_df",
    "snb_final_matrix_validation_df",
    "snb_final_matrix_group_summary_df",
    "snb_final_matrix_semantic_summary_df",
]

saved_block_10_dataframes = []

for var_name in BLOCK_10_CHECKPOINT_DATAFRAMES:
    obj = globals().get(var_name)

    if isinstance(obj, pd.DataFrame):
        out_path = CHECKPOINT_DATAFRAMES_DIR / f"{var_name}.json"

        obj.to_json(
            out_path,
            orient="records",
            indent=2,
            force_ascii=False
        )

        checkpoint_manifest["dataframes"][var_name] = str(
            out_path.relative_to(CHECKPOINT_DIR)
        )

        saved_block_10_dataframes.append(var_name)

# Preserve metadata
if "BASE_DIR" in globals():
    checkpoint_metadata["BASE_DIR"] = str(BASE_DIR)

if "SNB_ENTITY_VIEW_MAP" in globals():
    checkpoint_metadata["SNB_ENTITY_VIEW_MAP"] = SNB_ENTITY_VIEW_MAP

if "SNB_ENTITY_PK_MAP" in globals():
    checkpoint_metadata["SNB_ENTITY_PK_MAP"] = SNB_ENTITY_PK_MAP

if "SNB_COLLECTION_NAME_BY_ENTITY" in globals():
    checkpoint_metadata["SNB_COLLECTION_NAME_BY_ENTITY"] = SNB_COLLECTION_NAME_BY_ENTITY

if "SNB_RELATIONSHIP_HINTS" in globals():
    checkpoint_metadata["SNB_RELATIONSHIP_HINTS"] = SNB_RELATIONSHIP_HINTS

if "SNB_OFFICIAL_WORKLOAD" in globals():
    checkpoint_metadata["SNB_OFFICIAL_WORKLOAD"] = SNB_OFFICIAL_WORKLOAD

if "PERSISTENT_DUCKDB_PATH" in globals():
    checkpoint_metadata["PERSISTENT_DUCKDB_PATH"] = str(PERSISTENT_DUCKDB_PATH)

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(checkpoint_metadata, f, indent=2, ensure_ascii=False)

checkpoint_manifest["metadata"] = str(metadata_path.relative_to(CHECKPOINT_DIR))

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(checkpoint_manifest, f, indent=2, ensure_ascii=False)

print("Block 10 checkpoint saved to:")
print(CHECKPOINT_DIR)

print("\nSaved Block 10 DataFrames:")
for name in saved_block_10_dataframes:
    print(" -", name)

Block 10 checkpoint saved to:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_checkpoints

Saved Block 10 DataFrames:
 - snb_final_analytical_matrix_df
 - snb_final_matrix_validation_df
 - snb_final_matrix_group_summary_df
 - snb_final_matrix_semantic_summary_df


In [169]:
# =========================================================
# RESUME AFTER BLOCK 10 — Restore final analytical matrix
# =========================================================
#
# Goal:
# - Restore the notebook state after Block 10.
# - Continue directly to BLOCK 11 — Activation Logic G0–G9.
# =========================================================

from pathlib import Path
import json
import pandas as pd
import duckdb
import networkx as nx
import ast

# ---------------------------------------------------------
# 1. Project paths
# ---------------------------------------------------------

PROJECT_DIR = Path(
    "/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB"
)

CHECKPOINT_DIR = PROJECT_DIR / "benchmark_artifacts_dir" / "ldbc_snb_checkpoints"
FINAL_MATRIX_ARTIFACTS_DIR = (
    PROJECT_DIR
    / "benchmark_artifacts_dir"
    / "ldbc_snb_final_analytical_matrix_official"
)

manifest_path = CHECKPOINT_DIR / "checkpoint_manifest.json"

print("Project dir:", PROJECT_DIR)
print("Checkpoint exists:", manifest_path.exists())
print("Final matrix dir exists:", FINAL_MATRIX_ARTIFACTS_DIR.exists())

Project dir: /home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB
Checkpoint exists: True
Final matrix dir exists: True


In [170]:
# =========================================================
# RESUME 10A — Load checkpoint manifest, metadata and DataFrames
# =========================================================

if not manifest_path.exists():
    raise FileNotFoundError(
        f"Checkpoint manifest not found: {manifest_path}\n"
        "If Block 10F was not executed, use the fallback block below."
    )

with open(manifest_path, "r", encoding="utf-8") as f:
    checkpoint_manifest = json.load(f)

metadata_path = CHECKPOINT_DIR / checkpoint_manifest["metadata"]

with open(metadata_path, "r", encoding="utf-8") as f:
    checkpoint_metadata = json.load(f)

# ---------------------------------------------------------
# Restore paths
# ---------------------------------------------------------

BASE_DIR = Path(checkpoint_metadata["BASE_DIR"])

SNB_SNAPSHOT_DIR = Path(checkpoint_metadata.get("SNB_SNAPSHOT_DIR", ""))
SNB_DYNAMIC_DIR = Path(checkpoint_metadata.get("SNB_DYNAMIC_DIR", ""))
SNB_STATIC_DIR = Path(checkpoint_metadata.get("SNB_STATIC_DIR", ""))
SNB_SUBSTITUTION_DIR = Path(checkpoint_metadata.get("SNB_SUBSTITUTION_DIR", ""))
SNB_UPDATE_DIR = Path(checkpoint_metadata.get("SNB_UPDATE_DIR", ""))

# ---------------------------------------------------------
# Restore dictionaries/lists
# ---------------------------------------------------------

SNB_ENTITY_VIEW_MAP = checkpoint_metadata.get("SNB_ENTITY_VIEW_MAP", {})
SNB_ENTITY_PK_MAP = checkpoint_metadata.get("SNB_ENTITY_PK_MAP", {})
SNB_COLLECTION_NAME_BY_ENTITY = checkpoint_metadata.get("SNB_COLLECTION_NAME_BY_ENTITY", {})
SNB_RELATIONSHIP_HINTS = checkpoint_metadata.get("SNB_RELATIONSHIP_HINTS", {})
SNB_OFFICIAL_WORKLOAD = checkpoint_metadata.get("SNB_OFFICIAL_WORKLOAD", [])

SNB_OFFICIAL_QUERY_PATH_OVERRIDES = checkpoint_metadata.get(
    "SNB_OFFICIAL_QUERY_PATH_OVERRIDES", {}
)

SNB_RELATIONSHIP_MULTIPLICITY_OVERRIDES = checkpoint_metadata.get(
    "SNB_RELATIONSHIP_MULTIPLICITY_OVERRIDES", {}
)

UPDATE_OPERATION_CODE_MAP = checkpoint_metadata.get("UPDATE_OPERATION_CODE_MAP", {})

# ---------------------------------------------------------
# Restore all DataFrames saved in checkpoint
# ---------------------------------------------------------

for var_name, relative_path in checkpoint_manifest["dataframes"].items():
    df_path = CHECKPOINT_DIR / relative_path

    if df_path.exists():
        globals()[var_name] = pd.read_json(df_path, orient="records")

print("Restored DataFrames from checkpoint:")
for name in checkpoint_manifest["dataframes"]:
    if name in globals():
        print(" -", name)

Restored DataFrames from checkpoint:
 - columns_df
 - raw_counts_df
 - conceptual_counts_df
 - clean_entity_counts_df
 - fk_relationship_views_df
 - join_validation_df
 - snb_entity_metadata_df
 - snb_relationship_hints_df
 - snb_schema_edges_df
 - snb_graph_connectivity_df
 - snb_graph_connectivity_summary_df
 - important_paths_df
 - snb_traversal_connectivity_df
 - snb_traversal_connectivity_summary_df
 - important_traversal_paths_df
 - snb_workload_df
 - workload_relationship_validation_df
 - snb_query_semantic_profile_df
 - snb_initial_workload_matrix_df
 - snb_workload_paths_df
 - snb_preliminary_depth_df
 - snb_query_depth_overrides_df
 - snb_corrected_depth_df
 - snb_rc_df
 - snb_structural_matrix_df
 - snb_reduction_metrics_df
 - snb_structural_reduction_matrix_df
 - snb_reduction_rank_df
 - snb_reduction_validation_df
 - snb_update_stream_operations_df
 - snb_update_stream_parse_errors_df
 - snb_observed_update_operation_counts_df
 - snb_observed_update_by_file_df
 - snb_inser

In [171]:
# =========================================================
# RESUME 10B — Fallback load Block 10 artifacts from CSV
# =========================================================
#
# Use this if Block 10E saved the CSV files but Block 10F checkpoint
# was not completed.

def parse_possible_list_cell(value):
    """
    Convert list-like strings back to Python lists when possible.
    Keeps normal strings unchanged.
    """
    if pd.isna(value):
        return value

    if isinstance(value, list):
        return value

    if isinstance(value, str):
        value_strip = value.strip()

        if value_strip.startswith("[") and value_strip.endswith("]"):
            try:
                return ast.literal_eval(value_strip)
            except Exception:
                return value

    return value


final_matrix_path = FINAL_MATRIX_ARTIFACTS_DIR / "snb_final_analytical_matrix.csv"
validation_path = FINAL_MATRIX_ARTIFACTS_DIR / "snb_final_matrix_validation.csv"
group_summary_path = FINAL_MATRIX_ARTIFACTS_DIR / "snb_final_matrix_group_summary.csv"
semantic_summary_path = FINAL_MATRIX_ARTIFACTS_DIR / "snb_final_matrix_semantic_summary.csv"

if final_matrix_path.exists():
    snb_final_analytical_matrix_df = pd.read_csv(final_matrix_path)

    # Restore list-like columns if they were saved as strings.
    for col in ["accessed_entities", "relationships_used", "weighted_relationships"]:
        if col in snb_final_analytical_matrix_df.columns:
            snb_final_analytical_matrix_df[col] = (
                snb_final_analytical_matrix_df[col].apply(parse_possible_list_cell)
            )

    print("Loaded:", final_matrix_path)
else:
    raise FileNotFoundError(f"Final matrix CSV not found: {final_matrix_path}")

if validation_path.exists():
    snb_final_matrix_validation_df = pd.read_csv(validation_path)
    print("Loaded:", validation_path)

if group_summary_path.exists():
    snb_final_matrix_group_summary_df = pd.read_csv(group_summary_path)
    print("Loaded:", group_summary_path)

if semantic_summary_path.exists():
    snb_final_matrix_semantic_summary_df = pd.read_csv(semantic_summary_path)
    print("Loaded:", semantic_summary_path)

Loaded: /home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_final_analytical_matrix_official/snb_final_analytical_matrix.csv
Loaded: /home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_final_analytical_matrix_official/snb_final_matrix_validation.csv
Loaded: /home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_final_analytical_matrix_official/snb_final_matrix_group_summary.csv
Loaded: /home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_final_analytical_matrix_official/snb_final_matrix_semantic_summary.csv


In [172]:
# =========================================================
# RESUME 10C — Restore persistent DuckDB connection
# =========================================================

if "PERSISTENT_DUCKDB_PATH" in checkpoint_metadata:
    PERSISTENT_DUCKDB_PATH = Path(checkpoint_metadata["PERSISTENT_DUCKDB_PATH"])

    if PERSISTENT_DUCKDB_PATH.exists():
        con = duckdb.connect(str(PERSISTENT_DUCKDB_PATH))
        print("Connected to persistent DuckDB:")
        print(PERSISTENT_DUCKDB_PATH)
    else:
        print("WARNING: Persistent DuckDB file not found:")
        print(PERSISTENT_DUCKDB_PATH)
else:
    print("WARNING: PERSISTENT_DUCKDB_PATH not found in checkpoint metadata.")
    print("This is OK if Block 11 only uses the final analytical matrix.")

Connected to persistent DuckDB:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/duckdb/ldbc_snb_sf0_1_clean.duckdb


In [173]:
# =========================================================
# RESUME 10D — Rebuild schema graphs if schema edges exist
# =========================================================

if "snb_schema_edges_df" in globals():
    snb_schema_graph = nx.DiGraph()

    for entity in SNB_ENTITY_VIEW_MAP.keys():
        snb_schema_graph.add_node(entity)

    for _, row in snb_schema_edges_df.iterrows():
        source = row["source_entity"]
        target = row["target_entity"]

        snb_schema_graph.add_edge(
            source,
            target,
            relationship=row["relationship"],
            semantic_type=row["semantic_type"],
            relationship_kind=row["relationship_kind"],
            view_name=row["view_name"],
            source_column=row["source_column"],
            target_column=row["target_column"],
        )

        if bool(row.get("is_bidirectional", False)):
            snb_schema_graph.add_edge(
                target,
                source,
                relationship=str(row["relationship"]) + "_reverse",
                semantic_type=row["semantic_type"],
                relationship_kind=row["relationship_kind"],
                view_name=row["view_name"],
                source_column=row["target_column"],
                target_column=row["source_column"],
            )

    snb_traversal_graph = nx.Graph()

    for entity in SNB_ENTITY_VIEW_MAP.keys():
        snb_traversal_graph.add_node(entity)

    for _, row in snb_schema_edges_df.iterrows():
        snb_traversal_graph.add_edge(
            row["source_entity"],
            row["target_entity"],
            relationship=row["relationship"],
            semantic_type=row["semantic_type"],
            relationship_kind=row["relationship_kind"],
            view_name=row["view_name"],
            source_column=row["source_column"],
            target_column=row["target_column"],
        )

    print("Graphs rebuilt.")
    print("Directed graph nodes:", snb_schema_graph.number_of_nodes())
    print("Directed graph edges:", snb_schema_graph.number_of_edges())
    print("Traversal graph nodes:", snb_traversal_graph.number_of_nodes())
    print("Traversal graph edges:", snb_traversal_graph.number_of_edges())
else:
    print("snb_schema_edges_df not found. Graph rebuild skipped.")
    print("This is OK if Block 11 only uses snb_final_analytical_matrix_df.")

Graphs rebuilt.
Directed graph nodes: 8
Directed graph edges: 21
Traversal graph nodes: 8
Traversal graph edges: 19


In [174]:
# =========================================================
# RESUME 10E — Sanity check
# =========================================================

required_after_block_10 = [
    "snb_final_analytical_matrix_df",
    "snb_final_matrix_validation_df",
    "snb_final_matrix_group_summary_df",
    "snb_final_matrix_semantic_summary_df",
]

print("Objects needed after Block 10:")

for name in required_after_block_10:
    print(name, "=", "OK" if name in globals() else "MISSING")

print("\nFinal analytical matrix shape:")
print(snb_final_analytical_matrix_df.shape)

print("\nValidation problems:")
if "snb_final_matrix_validation_df" in globals():
    display(
        snb_final_matrix_validation_df[
            snb_final_matrix_validation_df["status"] != "OK"
        ]
    )

print("\nPreview:")
display(
    snb_final_analytical_matrix_df[
        [
            "query_name",
            "official_id",
            "query_group",
            "root_entity",
            "Rc_weighted",
            "D",
            "Re",
            "DeltaRratio",
            "dominant_semantic_type",
            "update_volatility_max",
            "observed_sharedness_max",
        ]
    ].head(10)
)

Objects needed after Block 10:
snb_final_analytical_matrix_df = OK
snb_final_matrix_validation_df = OK
snb_final_matrix_group_summary_df = OK
snb_final_matrix_semantic_summary_df = OK

Final analytical matrix shape:
(22, 37)

Validation problems:


,check,value,status



Preview:


,query_name,official_id,query_group,root_entity,Rc_weighted,D,Re,DeltaRratio,dominant_semantic_type,update_volatility_max,observed_sharedness_max
0,IS1_ProfileOfPerson,IS1,short_read,Person,1,1,0,1.000000,association,1.000000,1.000000
1,IS2_RecentMessagesOfPerson,IS2,short_read,Person,4,2,2,0.500000,mixed,1.000000,0.106448
2,IS3_FriendsOfPerson,IS3,short_read,Person,1,1,0,1.000000,association,1.000000,0.106448
3,IS4_ContentOfMessage,IS4,short_read,Post,0,0,0,0.000000,lookup,0.431184,0.016512
4,IS5_CreatorOfMessage,IS5,short_read,Post,2,1,1,0.500000,association,1.000000,0.106448
5,IS6_ForumOfMessage,IS6,short_read,Post,4,3,1,0.750000,containment,1.000000,0.106448
6,IS7_RepliesOfMessage,IS7,short_read,Post,5,2,3,0.400000,association,1.000000,0.106448
7,IC1_TransitiveFriendsWithName,IC1,complex_read,Person,7,4,3,0.571429,association,1.000000,1.000000
8,IC2_RecentMessagesByFriends,IC2,complex_read,Person,3,2,1,0.666667,association,1.000000,0.106448
9,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,complex_read,Person,8,4,4,0.500000,association,1.000000,1.000000


In [175]:
# =========================================================
# BLOCK 11A — Define G0–G9 generic MongoDB configuration classes
# =========================================================
#
# Goal:
# - Define the generic configuration classes used by the activation logic.
# - These are dataset-independent classes.
# - The LDBC SNB workload will activate a subset of these classes per query.
# =========================================================

import pandas as pd
import json
from pathlib import Path

GENERIC_CONFIGURATION_CLASSES = {
    "G0": {
        "family": "association",
        "role": "baseline",
        "label": "Root document with referenced associations",
        "description": "Baseline root-centric document. Associated entities are kept mostly as references.",
    },
    "G1": {
        "family": "lookup",
        "role": "baseline",
        "label": "Single-entity lookup document",
        "description": "Minimal document for direct lookup queries with no structural traversal.",
    },
    "G2": {
        "family": "association",
        "role": "embedding_candidate",
        "label": "Root document with embedded low-risk associations",
        "description": "Embed associated data when reduction is high and sharedness/volatility are not high.",
    },
    "G3": {
        "family": "association",
        "role": "reference_or_summary_candidate",
        "label": "Root document with references or denormalized summaries",
        "description": "Use references or summaries when association paths involve high sharedness, high volatility, or deeper traversals.",
    },
    "G4": {
        "family": "associative",
        "role": "baseline",
        "label": "Associative relationship as explicit document",
        "description": "Represent associative relationships such as likes, membership, or friendship as first-class documents.",
    },
    "G5": {
        "family": "associative",
        "role": "source_embedded_edges",
        "label": "Source-root document with embedded associative edges",
        "description": "Embed associative edges near the source when fanout/sharedness risk is manageable.",
    },
    "G6": {
        "family": "associative",
        "role": "referenced_or_reverse_associative",
        "label": "Referenced or reverse-indexed associative pattern",
        "description": "Use references, reverse lookups, or edge collections when associative relationships are volatile or highly shared.",
    },
    "G7": {
        "family": "containment",
        "role": "baseline",
        "label": "Containment baseline",
        "description": "Baseline containment-oriented root document.",
    },
    "G8": {
        "family": "containment",
        "role": "deep_embedding_candidate",
        "label": "Deep containment embedding",
        "description": "Embed contained children when containment is deep and sharedness/volatility are low.",
    },
    "G9": {
        "family": "containment",
        "role": "hybrid_containment_candidate",
        "label": "Hybrid containment with references or summaries",
        "description": "Use hybrid containment when fanout, sharedness, or volatility makes full embedding risky.",
    },
}

generic_classes_df = pd.DataFrame([
    {
        "g_class": g,
        "family": spec["family"],
        "role": spec["role"],
        "label": spec["label"],
        "description": spec["description"],
    }
    for g, spec in GENERIC_CONFIGURATION_CLASSES.items()
])

generic_classes_df

,g_class,family,role,label,description
0,G0,association,baseline,Root document with referenced associations,Baseline root-centric document. Associated entities are kept mostly as references.
1,G1,lookup,baseline,Single-entity lookup document,Minimal document for direct lookup queries with no structural traversal.
2,G2,association,embedding_candidate,Root document with embedded low-risk associations,Embed associated data when reduction is high and sharedness/volatility are not high.
3,G3,association,reference_or_summary_candidate,Root document with references or denormalized summaries,"Use references or summaries when association paths involve high sharedness, high volatility, or deeper traversals."
4,G4,associative,baseline,Associative relationship as explicit document,"Represent associative relationships such as likes, membership, or friendship as first-class documents."
5,G5,associative,source_embedded_edges,Source-root document with embedded associative edges,Embed associative edges near the source when fanout/sharedness risk is manageable.
6,G6,associative,referenced_or_reverse_associative,Referenced or reverse-indexed associative pattern,"Use references, reverse lookups, or edge collections when associative relationships are volatile or highly shared."
7,G7,containment,baseline,Containment baseline,Baseline containment-oriented root document.
8,G8,containment,deep_embedding_candidate,Deep containment embedding,Embed contained children when containment is deep and sharedness/volatility are low.
9,G9,containment,hybrid_containment_candidate,Hybrid containment with references or summaries,"Use hybrid containment when fanout, sharedness, or volatility makes full embedding risky."


In [176]:
# =========================================================
# BLOCK 11B — Define activation thresholds
# =========================================================
#
# Goal:
# - Define explicit thresholds used by the activation logic.
# - These thresholds can be tuned later, but keeping them explicit makes
#   the methodology reproducible.
# =========================================================

ACTIVATION_THRESHOLDS = {
    "high_delta_ratio": 0.75,
    "medium_delta_ratio": 0.40,

    "high_volatility": 0.75,
    "medium_volatility": 0.35,

    "high_sharedness": 0.75,
    "medium_sharedness": 0.25,

    "deep_depth": 3,
    "medium_depth": 2,

    "high_residual_traversal": 3,
}

activation_thresholds_df = pd.DataFrame([
    {"threshold": k, "value": v}
    for k, v in ACTIVATION_THRESHOLDS.items()
])

activation_thresholds_df

,threshold,value
0,high_delta_ratio,0.75
1,medium_delta_ratio,0.40
2,high_volatility,0.75
3,medium_volatility,0.35
4,high_sharedness,0.75
5,medium_sharedness,0.25
6,deep_depth,3.00
7,medium_depth,2.00
8,high_residual_traversal,3.00


In [177]:
# =========================================================
# BLOCK 11C — Activation helper functions
# =========================================================
#
# Goal:
# - Implement reusable helper functions for G0–G9 activation.
# =========================================================

def as_float(value, default=0.0):
    try:
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


def as_int(value, default=0):
    try:
        if pd.isna(value):
            return default
        return int(value)
    except Exception:
        return default


def activate_g_class(activated, g_class, reason, strength="candidate"):
    """
    Add an activated class with reason and strength.
    """
    activated.append({
        "g_class": g_class,
        "activation_strength": strength,
        "activation_reason": reason,
    })


def compute_activation_for_query(row):
    """
    Compute activated G classes for one query row.

    The logic follows the generic family selector:
    - lookup       -> G1
    - association  -> G0, G2, G3
    - associative  -> G4, G5, G6
    - containment  -> G7, G8, G9
    - mixed        -> combine relevant families
    """

    qname = row["query_name"]
    semantic = row["dominant_semantic_type"]

    rc = as_int(row.get("Rc_weighted", 0))
    d = as_int(row.get("D", 0))
    re = as_int(row.get("Re", 0))
    delta_ratio = as_float(row.get("DeltaRratio", 0.0))

    volatility_max = as_float(row.get("update_volatility_max", 0.0))
    volatility_mean = as_float(row.get("update_volatility_mean", 0.0))

    sharedness_max = as_float(row.get("observed_sharedness_max", 0.0))
    sharedness_mean = as_float(row.get("observed_sharedness_mean", 0.0))

    assoc_count = as_int(row.get("association_count", 0))
    associative_count = as_int(row.get("associative_count", 0))
    containment_count = as_int(row.get("containment_count", 0))
    lookup_count = as_int(row.get("lookup_count", 0))

    high_delta = delta_ratio >= ACTIVATION_THRESHOLDS["high_delta_ratio"]
    medium_delta = delta_ratio >= ACTIVATION_THRESHOLDS["medium_delta_ratio"]

    high_volatility = volatility_max >= ACTIVATION_THRESHOLDS["high_volatility"]
    medium_volatility = volatility_max >= ACTIVATION_THRESHOLDS["medium_volatility"]

    high_sharedness = sharedness_max >= ACTIVATION_THRESHOLDS["high_sharedness"]
    medium_sharedness = sharedness_max >= ACTIVATION_THRESHOLDS["medium_sharedness"]

    deep = d >= ACTIVATION_THRESHOLDS["deep_depth"]
    medium_depth = d >= ACTIVATION_THRESHOLDS["medium_depth"]

    high_re = re >= ACTIVATION_THRESHOLDS["high_residual_traversal"]

    activated = []

    # -----------------------------------------------------
    # Lookup family
    # -----------------------------------------------------
    if semantic == "lookup" or rc == 0:
        activate_g_class(
            activated,
            "G1",
            "Lookup query with no explicit relationship traversal.",
            "primary"
        )

        # Keep G0 as a weak control only if the query has a root entity.
        activate_g_class(
            activated,
            "G0",
            "Control baseline for root document comparison.",
            "control"
        )

        return activated

    # -----------------------------------------------------
    # Association family
    # -----------------------------------------------------
    if semantic == "association" or assoc_count > 0 or semantic == "mixed":
        activate_g_class(
            activated,
            "G0",
            "Association-family baseline root document.",
            "primary" if semantic == "association" else "secondary"
        )

        if medium_delta and not high_sharedness and not high_volatility and not deep:
            activate_g_class(
                activated,
                "G2",
                "Association path has structural reduction potential and no high sharedness/volatility risk.",
                "primary" if semantic == "association" else "secondary"
            )

        if high_sharedness or high_volatility or deep or high_re:
            activate_g_class(
                activated,
                "G3",
                "Association path has high sharedness, high volatility, deep traversal, or residual traversal.",
                "primary" if semantic == "association" else "secondary"
            )

        # If G2 was not activated and G3 was not activated, keep one association variant.
        already = {x["g_class"] for x in activated}
        if "G2" not in already and "G3" not in already:
            activate_g_class(
                activated,
                "G3",
                "Default association variant when embedding conditions are not clearly favorable.",
                "secondary"
            )

    # -----------------------------------------------------
    # Associative family
    # -----------------------------------------------------
    if semantic == "associative" or associative_count > 0 or semantic == "mixed":
        activate_g_class(
            activated,
            "G4",
            "Associative-family baseline using explicit relationship documents.",
            "primary" if semantic == "associative" else "secondary"
        )

        if high_delta and not high_sharedness and not high_volatility:
            activate_g_class(
                activated,
                "G5",
                "Associative relation is direct and may benefit from source-side embedded edges.",
                "primary" if semantic == "associative" else "secondary"
            )

        if high_sharedness or high_volatility or medium_volatility:
            activate_g_class(
                activated,
                "G6",
                "Associative relation is volatile/shared and should be represented with references or reverse indexes.",
                "primary" if semantic == "associative" else "secondary"
            )

        already = {x["g_class"] for x in activated}
        if "G5" not in already and "G6" not in already:
            activate_g_class(
                activated,
                "G6",
                "Default associative variant when embedding is not clearly safe.",
                "secondary"
            )

    # -----------------------------------------------------
    # Containment family
    # -----------------------------------------------------
    if semantic == "containment" or containment_count > 0 or semantic == "mixed":
        activate_g_class(
            activated,
            "G7",
            "Containment-family baseline.",
            "primary" if semantic == "containment" else "secondary"
        )

        if medium_delta and not high_sharedness and not high_volatility:
            activate_g_class(
                activated,
                "G8",
                "Containment path has structural reduction potential and embedding risk is acceptable.",
                "primary" if semantic == "containment" else "secondary"
            )

        if high_sharedness or high_volatility or high_re:
            activate_g_class(
                activated,
                "G9",
                "Containment path has high sharedness, high volatility, or high residual traversal.",
                "primary" if semantic == "containment" else "secondary"
            )

        already = {x["g_class"] for x in activated}
        if "G8" not in already and "G9" not in already:
            activate_g_class(
                activated,
                "G9",
                "Default hybrid containment variant when full embedding is not clearly safe.",
                "secondary"
            )

    # -----------------------------------------------------
    # Deduplicate while preserving the strongest role
    # -----------------------------------------------------
    priority = {
        "primary": 3,
        "secondary": 2,
        "candidate": 1,
        "control": 0,
    }

    dedup = {}

    for item in activated:
        g = item["g_class"]

        if g not in dedup:
            dedup[g] = item
        else:
            old_strength = dedup[g]["activation_strength"]
            new_strength = item["activation_strength"]

            if priority.get(new_strength, 0) > priority.get(old_strength, 0):
                dedup[g] = item
            else:
                dedup[g]["activation_reason"] += " | " + item["activation_reason"]

    return list(dedup.values())

In [178]:
# =========================================================
# BLOCK 11D — Apply G0–G9 activation logic
# =========================================================
#
# Goal:
# - Apply the activation logic to every official LDBC SNB query.
# - Produce one row per activated query/class pair.
# =========================================================

activation_rows = []

for _, row in snb_final_analytical_matrix_df.iterrows():
    activated = compute_activation_for_query(row)

    for item in activated:
        activation_rows.append({
            "query_name": row["query_name"],
            "official_id": row["official_id"],
            "official_title": row.get("official_title"),
            "query_group": row["query_group"],
            "operation_type": row["operation_type"],
            "root_entity": row["root_entity"],

            "dominant_semantic_type": row["dominant_semantic_type"],
            "Rc_weighted": row["Rc_weighted"],
            "D": row["D"],
            "Re": row["Re"],
            "DeltaRratio": row["DeltaRratio"],
            "update_volatility_max": row["update_volatility_max"],
            "observed_sharedness_max": row["observed_sharedness_max"],

            "g_class": item["g_class"],
            "activation_strength": item["activation_strength"],
            "activation_reason": item["activation_reason"],

            "g_family": GENERIC_CONFIGURATION_CLASSES[item["g_class"]]["family"],
            "g_role": GENERIC_CONFIGURATION_CLASSES[item["g_class"]]["role"],
            "g_label": GENERIC_CONFIGURATION_CLASSES[item["g_class"]]["label"],
        })

snb_activation_matrix_df = pd.DataFrame(activation_rows)

snb_activation_matrix_df = snb_activation_matrix_df.sort_values(
    by=["query_group", "official_id", "activation_strength", "g_class"],
    ascending=[True, True, False, True]
).reset_index(drop=True)

snb_activation_matrix_df

,query_name,official_id,official_title,query_group,operation_type,root_entity,dominant_semantic_type,Rc_weighted,D,Re,DeltaRratio,update_volatility_max,observed_sharedness_max,g_class,activation_strength,activation_reason,g_family,g_role,g_label
0,IC1_TransitiveFriendsWithName,IC1,Transitive friends with a certain name,complex_read,read,Person,association,7,4,3,0.571429,1.0,1.000000,G0,primary,Association-family baseline root document.,association,baseline,Root document with referenced associations
1,IC1_TransitiveFriendsWithName,IC1,Transitive friends with a certain name,complex_read,read,Person,association,7,4,3,0.571429,1.0,1.000000,G3,primary,"Association path has high sharedness, high volatility, deep traversal, or residual traversal.",association,reference_or_summary_candidate,Root document with references or denormalized summaries
2,IC2_RecentMessagesByFriends,IC2,Recent messages by your friends,complex_read,read,Person,association,3,2,1,0.666667,1.0,0.106448,G0,primary,Association-family baseline root document.,association,baseline,Root document with referenced associations
3,IC2_RecentMessagesByFriends,IC2,Recent messages by your friends,complex_read,read,Person,association,3,2,1,0.666667,1.0,0.106448,G3,primary,"Association path has high sharedness, high volatility, deep traversal, or residual traversal.",association,reference_or_summary_candidate,Root document with references or denormalized summaries
4,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,Friends and friends of friends that have been to given countries,complex_read,read,Person,association,8,4,4,0.500000,1.0,1.000000,G7,secondary,Containment-family baseline.,containment,baseline,Containment baseline
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,IS6_ForumOfMessage,IS6,Forum of a message,short_read,read,Post,containment,4,3,1,0.750000,1.0,0.106448,G9,primary,"Containment path has high sharedness, high volatility, or high residual traversal.",containment,hybrid_containment_candidate,Hybrid containment with references or summaries
60,IS7_RepliesOfMessage,IS7,Replies of a message,short_read,read,Post,association,5,2,3,0.400000,1.0,0.106448,G7,secondary,Containment-family baseline.,containment,baseline,Containment baseline
61,IS7_RepliesOfMessage,IS7,Replies of a message,short_read,read,Post,association,5,2,3,0.400000,1.0,0.106448,G9,secondary,"Containment path has high sharedness, high volatility, or high residual traversal.",containment,hybrid_containment_candidate,Hybrid containment with references or summaries
62,IS7_RepliesOfMessage,IS7,Replies of a message,short_read,read,Post,association,5,2,3,0.400000,1.0,0.106448,G0,primary,Association-family baseline root document.,association,baseline,Root document with referenced associations


In [179]:
# =========================================================
# BLOCK 11E — Build compact activation summary per query
# =========================================================
#
# Goal:
# - Produce one row per query with activated classes grouped as lists.
# =========================================================

def sorted_unique_list(values):
    return sorted(set(str(v) for v in values if pd.notna(v)))


snb_activation_summary_df = (
    snb_activation_matrix_df
    .groupby(
        [
            "query_name",
            "official_id",
            "official_title",
            "query_group",
            "operation_type",
            "root_entity",
            "dominant_semantic_type",
        ],
        dropna=False
    )
    .agg(
        activated_g_classes=("g_class", sorted_unique_list),
        primary_g_classes=(
            "g_class",
            lambda x: sorted_unique_list(
                snb_activation_matrix_df.loc[x.index]
                .query("activation_strength == 'primary'")["g_class"]
            )
        ),
        secondary_g_classes=(
            "g_class",
            lambda x: sorted_unique_list(
                snb_activation_matrix_df.loc[x.index]
                .query("activation_strength == 'secondary'")["g_class"]
            )
        ),
        control_g_classes=(
            "g_class",
            lambda x: sorted_unique_list(
                snb_activation_matrix_df.loc[x.index]
                .query("activation_strength == 'control'")["g_class"]
            )
        ),
        n_activated_classes=("g_class", "count"),
    )
    .reset_index()
)

snb_activation_summary_df

,query_name,official_id,official_title,query_group,operation_type,root_entity,dominant_semantic_type,activated_g_classes,primary_g_classes,secondary_g_classes,control_g_classes,n_activated_classes
0,IC1_TransitiveFriendsWithName,IC1,Transitive friends with a certain name,complex_read,read,Person,association,"[G0, G3]","[G0, G3]",[],[],2
1,IC2_RecentMessagesByFriends,IC2,Recent messages by your friends,complex_read,read,Person,association,"[G0, G3]","[G0, G3]",[],[],2
2,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,Friends and friends of friends that have been to given countries,complex_read,read,Person,association,"[G0, G3, G7, G9]","[G0, G3]","[G7, G9]",[],4
3,IC4_NewTopics,IC4,New topics,complex_read,read,Person,association,"[G0, G3]","[G0, G3]",[],[],2
4,IC5_NewGroups,IC5,New groups,complex_read,read,Person,association,"[G0, G3, G4, G6, G7, G9]","[G0, G3]","[G4, G6, G7, G9]",[],6
5,IC6_TagCoOccurrence,IC6,Tag co-occurrence,complex_read,read,Person,association,"[G0, G3]","[G0, G3]",[],[],2
6,IC7_RecentLikers,IC7,Recent likers,complex_read,read,Person,association,"[G0, G3, G4, G6]","[G0, G3]","[G4, G6]",[],4
7,INS1_AddPerson,INS1,Add person,insert,insert,Person,association,"[G0, G3]","[G0, G3]",[],[],2
8,INS2_AddLikeToPost,INS2,Add like to post,insert,insert,Person,associative,"[G4, G6]","[G4, G6]",[],[],2
9,INS3_AddLikeToComment,INS3,Add like to comment,insert,insert,Person,associative,"[G4, G6]","[G4, G6]",[],[],2


In [180]:
# =========================================================
# BLOCK 11F — Activation counts by G class
# =========================================================

snb_activation_counts_by_g_df = (
    snb_activation_matrix_df
    .groupby(["g_class", "g_family", "g_role", "g_label"])
    .agg(
        n_activations=("query_name", "count"),
        n_primary=("activation_strength", lambda x: int((x == "primary").sum())),
        n_secondary=("activation_strength", lambda x: int((x == "secondary").sum())),
        n_control=("activation_strength", lambda x: int((x == "control").sum())),
        queries=("query_name", lambda x: sorted(set(x))),
    )
    .reset_index()
    .sort_values(["g_class"])
)

snb_activation_counts_by_g_df

,g_class,g_family,g_role,g_label,n_activations,n_primary,n_secondary,n_control,queries
0,G0,association,baseline,Root document with referenced associations,19,16,2,1,"[IC1_TransitiveFriendsWithName, IC2_RecentMessagesByFriends, IC3_FriendsAndFriendsOfFriendsInCountries, IC4_NewTopics, IC5_NewGroups, IC6_TagCoOccurrence, IC7_RecentLikers, INS1_AddPerson, INS4_AddForum, INS6_AddPost, INS7_AddComment, INS8_AddFriendship, IS1_ProfileOfPerson, IS2_RecentMessagesOfPerson, IS3_FriendsOfPerson, IS4_ContentOfMessage, IS5_CreatorOfMessage, IS6_ForumOfMessage, IS7_RepliesOfMessage]"
1,G1,lookup,baseline,Single-entity lookup document,1,1,0,0,[IS4_ContentOfMessage]
2,G3,association,reference_or_summary_candidate,Root document with references or denormalized summaries,18,16,2,0,"[IC1_TransitiveFriendsWithName, IC2_RecentMessagesByFriends, IC3_FriendsAndFriendsOfFriendsInCountries, IC4_NewTopics, IC5_NewGroups, IC6_TagCoOccurrence, IC7_RecentLikers, INS1_AddPerson, INS4_AddForum, INS6_AddPost, INS7_AddComment, INS8_AddFriendship, IS1_ProfileOfPerson, IS2_RecentMessagesOfPerson, IS3_FriendsOfPerson, IS5_CreatorOfMessage, IS6_ForumOfMessage, IS7_RepliesOfMessage]"
3,G4,associative,baseline,Associative relationship as explicit document,6,3,3,0,"[IC5_NewGroups, IC7_RecentLikers, INS2_AddLikeToPost, INS3_AddLikeToComment, INS5_AddForumMembership, IS2_RecentMessagesOfPerson]"
4,G6,associative,referenced_or_reverse_associative,Referenced or reverse-indexed associative pattern,6,3,3,0,"[IC5_NewGroups, IC7_RecentLikers, INS2_AddLikeToPost, INS3_AddLikeToComment, INS5_AddForumMembership, IS2_RecentMessagesOfPerson]"
5,G7,containment,baseline,Containment baseline,7,1,6,0,"[IC3_FriendsAndFriendsOfFriendsInCountries, IC5_NewGroups, INS6_AddPost, INS7_AddComment, IS2_RecentMessagesOfPerson, IS6_ForumOfMessage, IS7_RepliesOfMessage]"
6,G9,containment,hybrid_containment_candidate,Hybrid containment with references or summaries,7,1,6,0,"[IC3_FriendsAndFriendsOfFriendsInCountries, IC5_NewGroups, INS6_AddPost, INS7_AddComment, IS2_RecentMessagesOfPerson, IS6_ForumOfMessage, IS7_RepliesOfMessage]"


In [181]:
# =========================================================
# BLOCK 11G — Activation counts by query group and class
# =========================================================

snb_activation_counts_by_group_df = (
    snb_activation_matrix_df
    .groupby(["query_group", "g_class"])
    .agg(
        n_activations=("query_name", "count"),
        queries=("query_name", lambda x: sorted(set(x))),
    )
    .reset_index()
    .sort_values(["query_group", "g_class"])
)

snb_activation_counts_by_group_df

,query_group,g_class,n_activations,queries
0,complex_read,G0,7,"[IC1_TransitiveFriendsWithName, IC2_RecentMessagesByFriends, IC3_FriendsAndFriendsOfFriendsInCountries, IC4_NewTopics, IC5_NewGroups, IC6_TagCoOccurrence, IC7_RecentLikers]"
1,complex_read,G3,7,"[IC1_TransitiveFriendsWithName, IC2_RecentMessagesByFriends, IC3_FriendsAndFriendsOfFriendsInCountries, IC4_NewTopics, IC5_NewGroups, IC6_TagCoOccurrence, IC7_RecentLikers]"
2,complex_read,G4,2,"[IC5_NewGroups, IC7_RecentLikers]"
3,complex_read,G6,2,"[IC5_NewGroups, IC7_RecentLikers]"
4,complex_read,G7,2,"[IC3_FriendsAndFriendsOfFriendsInCountries, IC5_NewGroups]"
5,complex_read,G9,2,"[IC3_FriendsAndFriendsOfFriendsInCountries, IC5_NewGroups]"
6,insert,G0,5,"[INS1_AddPerson, INS4_AddForum, INS6_AddPost, INS7_AddComment, INS8_AddFriendship]"
7,insert,G3,5,"[INS1_AddPerson, INS4_AddForum, INS6_AddPost, INS7_AddComment, INS8_AddFriendship]"
8,insert,G4,3,"[INS2_AddLikeToPost, INS3_AddLikeToComment, INS5_AddForumMembership]"
9,insert,G6,3,"[INS2_AddLikeToPost, INS3_AddLikeToComment, INS5_AddForumMembership]"


In [182]:
# =========================================================
# BLOCK 11H — Validate activation matrix
# =========================================================
#
# Goal:
# - Ensure every query has at least one activated class.
# - Ensure every activated class is in G0–G9.
# =========================================================

all_queries = set(snb_final_analytical_matrix_df["query_name"])
activated_queries = set(snb_activation_matrix_df["query_name"])

missing_activation_queries = sorted(all_queries - activated_queries)

invalid_g_classes = sorted(
    set(snb_activation_matrix_df["g_class"])
    - set(GENERIC_CONFIGURATION_CLASSES.keys())
)

activation_validation_rows = [
    {
        "check": "queries_without_activation",
        "value": len(missing_activation_queries),
        "details": missing_activation_queries,
        "status": "OK" if len(missing_activation_queries) == 0 else "ERROR",
    },
    {
        "check": "invalid_g_classes",
        "value": len(invalid_g_classes),
        "details": invalid_g_classes,
        "status": "OK" if len(invalid_g_classes) == 0 else "ERROR",
    },
]

snb_activation_validation_df = pd.DataFrame(activation_validation_rows)
snb_activation_validation_df

,check,value,details,status
0,queries_without_activation,0,[],OK
1,invalid_g_classes,0,[],OK


In [183]:
snb_activation_validation_df[
    snb_activation_validation_df["status"] != "OK"
]

,check,value,details,status


In [184]:
# =========================================================
# BLOCK 11I — Save activation artifacts
# =========================================================

ACTIVATION_ARTIFACTS_DIR = (
    BASE_DIR.parent.parent
    / "benchmark_artifacts_dir"
    / "ldbc_snb_activation_official"
)

ACTIVATION_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

generic_classes_df.to_csv(
    ACTIVATION_ARTIFACTS_DIR / "generic_configuration_classes_g0_g9.csv",
    index=False
)

activation_thresholds_df.to_csv(
    ACTIVATION_ARTIFACTS_DIR / "activation_thresholds.csv",
    index=False
)

snb_activation_matrix_df.to_csv(
    ACTIVATION_ARTIFACTS_DIR / "snb_activation_matrix.csv",
    index=False
)

snb_activation_summary_df.to_csv(
    ACTIVATION_ARTIFACTS_DIR / "snb_activation_summary.csv",
    index=False
)

snb_activation_counts_by_g_df.to_csv(
    ACTIVATION_ARTIFACTS_DIR / "activation_counts_by_g_class.csv",
    index=False
)

snb_activation_counts_by_group_df.to_csv(
    ACTIVATION_ARTIFACTS_DIR / "activation_counts_by_query_group.csv",
    index=False
)

snb_activation_validation_df.to_csv(
    ACTIVATION_ARTIFACTS_DIR / "activation_validation.csv",
    index=False
)

print("Saved activation artifacts to:")
print(ACTIVATION_ARTIFACTS_DIR)

print("\nSaved files:")
for path in sorted(ACTIVATION_ARTIFACTS_DIR.glob("*.csv")):
    print(" -", path.name)

Saved activation artifacts to:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_activation_official

Saved files:
 - activation_counts_by_g_class.csv
 - activation_counts_by_query_group.csv
 - activation_thresholds.csv
 - activation_validation.csv
 - generic_configuration_classes_g0_g9.csv
 - snb_activation_matrix.csv
 - snb_activation_summary.csv


In [185]:
# =========================================================
# BLOCK 11J — Save checkpoint after activation logic
# =========================================================

from pathlib import Path
import json
import pandas as pd

PROJECT_DIR = BASE_DIR.parent.parent

CHECKPOINT_DIR = PROJECT_DIR / "benchmark_artifacts_dir" / "ldbc_snb_checkpoints"
CHECKPOINT_DATAFRAMES_DIR = CHECKPOINT_DIR / "dataframes"
CHECKPOINT_METADATA_DIR = CHECKPOINT_DIR / "metadata"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_METADATA_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = CHECKPOINT_DIR / "checkpoint_manifest.json"
metadata_path = CHECKPOINT_METADATA_DIR / "checkpoint_metadata.json"

if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        checkpoint_manifest = json.load(f)
else:
    checkpoint_manifest = {
        "dataframes": {},
        "metadata": str(metadata_path.relative_to(CHECKPOINT_DIR)),
    }

if metadata_path.exists():
    with open(metadata_path, "r", encoding="utf-8") as f:
        checkpoint_metadata = json.load(f)
else:
    checkpoint_metadata = {}

BLOCK_11_CHECKPOINT_DATAFRAMES = [
    "generic_classes_df",
    "activation_thresholds_df",
    "snb_activation_matrix_df",
    "snb_activation_summary_df",
    "snb_activation_counts_by_g_df",
    "snb_activation_counts_by_group_df",
    "snb_activation_validation_df",
]

saved_block_11_dataframes = []

for var_name in BLOCK_11_CHECKPOINT_DATAFRAMES:
    obj = globals().get(var_name)

    if isinstance(obj, pd.DataFrame):
        out_path = CHECKPOINT_DATAFRAMES_DIR / f"{var_name}.json"

        obj.to_json(
            out_path,
            orient="records",
            indent=2,
            force_ascii=False
        )

        checkpoint_manifest["dataframes"][var_name] = str(
            out_path.relative_to(CHECKPOINT_DIR)
        )

        saved_block_11_dataframes.append(var_name)

checkpoint_metadata.update({
    "BASE_DIR": str(BASE_DIR),
    "GENERIC_CONFIGURATION_CLASSES": GENERIC_CONFIGURATION_CLASSES,
    "ACTIVATION_THRESHOLDS": ACTIVATION_THRESHOLDS,
})

if "PERSISTENT_DUCKDB_PATH" in globals():
    checkpoint_metadata["PERSISTENT_DUCKDB_PATH"] = str(PERSISTENT_DUCKDB_PATH)

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(checkpoint_metadata, f, indent=2, ensure_ascii=False)

checkpoint_manifest["metadata"] = str(metadata_path.relative_to(CHECKPOINT_DIR))

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(checkpoint_manifest, f, indent=2, ensure_ascii=False)

print("Block 11 checkpoint saved to:")
print(CHECKPOINT_DIR)

print("\nSaved Block 11 DataFrames:")
for name in saved_block_11_dataframes:
    print(" -", name)

Block 11 checkpoint saved to:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_checkpoints

Saved Block 11 DataFrames:
 - generic_classes_df
 - activation_thresholds_df
 - snb_activation_matrix_df
 - snb_activation_summary_df
 - snb_activation_counts_by_g_df
 - snb_activation_counts_by_group_df
 - snb_activation_validation_df


In [199]:
# =========================================================
# RESUME AFTER BLOCK 11 — Restore activation state
# =========================================================
#
# Goal:
# - Restore notebook state after Block 11.
# - Continue directly to BLOCK 12 — Generate MongoDB candidate configurations.
# =========================================================

from pathlib import Path
import json
import ast
import pandas as pd
import duckdb
import networkx as nx

# ---------------------------------------------------------
# 1. Project paths
# ---------------------------------------------------------

PROJECT_DIR = Path(
    "/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB"
)

CHECKPOINT_DIR = PROJECT_DIR / "benchmark_artifacts_dir" / "ldbc_snb_checkpoints"

FINAL_MATRIX_ARTIFACTS_DIR = (
    PROJECT_DIR
    / "benchmark_artifacts_dir"
    / "ldbc_snb_final_analytical_matrix_official"
)

ACTIVATION_ARTIFACTS_DIR = (
    PROJECT_DIR
    / "benchmark_artifacts_dir"
    / "ldbc_snb_activation_official"
)

manifest_path = CHECKPOINT_DIR / "checkpoint_manifest.json"

print("Project dir:", PROJECT_DIR)
print("Checkpoint manifest exists:", manifest_path.exists())
print("Final matrix dir exists:", FINAL_MATRIX_ARTIFACTS_DIR.exists())
print("Activation dir exists:", ACTIVATION_ARTIFACTS_DIR.exists())

Project dir: /home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB
Checkpoint manifest exists: True
Final matrix dir exists: True
Activation dir exists: True


In [200]:
# =========================================================
# RESUME 11A — Load checkpoint manifest, metadata and DataFrames
# =========================================================

if not manifest_path.exists():
    raise FileNotFoundError(
        f"Checkpoint manifest not found: {manifest_path}\n"
        "If Block 11J was not executed, use the CSV fallback blocks below."
    )

with open(manifest_path, "r", encoding="utf-8") as f:
    checkpoint_manifest = json.load(f)

metadata_path = CHECKPOINT_DIR / checkpoint_manifest["metadata"]

with open(metadata_path, "r", encoding="utf-8") as f:
    checkpoint_metadata = json.load(f)

# ---------------------------------------------------------
# Restore paths
# ---------------------------------------------------------

BASE_DIR = Path(checkpoint_metadata.get("BASE_DIR", ""))

SNB_SNAPSHOT_DIR = Path(checkpoint_metadata.get("SNB_SNAPSHOT_DIR", ""))
SNB_DYNAMIC_DIR = Path(checkpoint_metadata.get("SNB_DYNAMIC_DIR", ""))
SNB_STATIC_DIR = Path(checkpoint_metadata.get("SNB_STATIC_DIR", ""))
SNB_SUBSTITUTION_DIR = Path(checkpoint_metadata.get("SNB_SUBSTITUTION_DIR", ""))
SNB_UPDATE_DIR = Path(checkpoint_metadata.get("SNB_UPDATE_DIR", ""))

# ---------------------------------------------------------
# Restore dictionaries / lists
# ---------------------------------------------------------

SNB_ENTITY_VIEW_MAP = checkpoint_metadata.get("SNB_ENTITY_VIEW_MAP", {})
SNB_ENTITY_PK_MAP = checkpoint_metadata.get("SNB_ENTITY_PK_MAP", {})
SNB_COLLECTION_NAME_BY_ENTITY = checkpoint_metadata.get("SNB_COLLECTION_NAME_BY_ENTITY", {})
SNB_RELATIONSHIP_HINTS = checkpoint_metadata.get("SNB_RELATIONSHIP_HINTS", {})

SNB_OFFICIAL_WORKLOAD = checkpoint_metadata.get("SNB_OFFICIAL_WORKLOAD", [])
SNB_OFFICIAL_QUERY_PATH_OVERRIDES = checkpoint_metadata.get(
    "SNB_OFFICIAL_QUERY_PATH_OVERRIDES", {}
)
SNB_RELATIONSHIP_MULTIPLICITY_OVERRIDES = checkpoint_metadata.get(
    "SNB_RELATIONSHIP_MULTIPLICITY_OVERRIDES", {}
)

GENERIC_CONFIGURATION_CLASSES = checkpoint_metadata.get(
    "GENERIC_CONFIGURATION_CLASSES", {}
)

ACTIVATION_THRESHOLDS = checkpoint_metadata.get(
    "ACTIVATION_THRESHOLDS", {}
)

UPDATE_OPERATION_CODE_MAP = checkpoint_metadata.get(
    "UPDATE_OPERATION_CODE_MAP", {}
)

# ---------------------------------------------------------
# Restore all checkpointed DataFrames
# ---------------------------------------------------------

for var_name, relative_path in checkpoint_manifest.get("dataframes", {}).items():
    df_path = CHECKPOINT_DIR / relative_path

    if df_path.exists():
        globals()[var_name] = pd.read_json(df_path, orient="records")

print("Restored DataFrames from checkpoint:")
for name in checkpoint_manifest.get("dataframes", {}):
    if name in globals():
        print(" -", name)

Restored DataFrames from checkpoint:
 - columns_df
 - raw_counts_df
 - conceptual_counts_df
 - clean_entity_counts_df
 - fk_relationship_views_df
 - join_validation_df
 - snb_entity_metadata_df
 - snb_relationship_hints_df
 - snb_schema_edges_df
 - snb_graph_connectivity_df
 - snb_graph_connectivity_summary_df
 - important_paths_df
 - snb_traversal_connectivity_df
 - snb_traversal_connectivity_summary_df
 - important_traversal_paths_df
 - snb_workload_df
 - workload_relationship_validation_df
 - snb_query_semantic_profile_df
 - snb_initial_workload_matrix_df
 - snb_workload_paths_df
 - snb_preliminary_depth_df
 - snb_query_depth_overrides_df
 - snb_corrected_depth_df
 - snb_rc_df
 - snb_structural_matrix_df
 - snb_reduction_metrics_df
 - snb_structural_reduction_matrix_df
 - snb_reduction_rank_df
 - snb_reduction_validation_df
 - snb_update_stream_operations_df
 - snb_update_stream_parse_errors_df
 - snb_observed_update_operation_counts_df
 - snb_observed_update_by_file_df
 - snb_inser

In [201]:
# =========================================================
# RESUME 11B — Fallback load Block 10 and Block 11 CSV artifacts
# =========================================================

def parse_possible_list_cell(value):
    """
    Convert list-like strings back to Python lists when possible.
    Keeps normal strings unchanged.
    """
    if pd.isna(value):
        return value

    if isinstance(value, list):
        return value

    if isinstance(value, str):
        value_strip = value.strip()

        if value_strip.startswith("[") and value_strip.endswith("]"):
            try:
                return ast.literal_eval(value_strip)
            except Exception:
                return value

    return value


# ---------------------------------------------------------
# Load final analytical matrix if missing
# ---------------------------------------------------------

if "snb_final_analytical_matrix_df" not in globals():
    final_matrix_path = FINAL_MATRIX_ARTIFACTS_DIR / "snb_final_analytical_matrix.csv"

    if final_matrix_path.exists():
        snb_final_analytical_matrix_df = pd.read_csv(final_matrix_path)

        for col in ["accessed_entities", "relationships_used", "weighted_relationships"]:
            if col in snb_final_analytical_matrix_df.columns:
                snb_final_analytical_matrix_df[col] = (
                    snb_final_analytical_matrix_df[col].apply(parse_possible_list_cell)
                )

        print("Loaded final matrix:", final_matrix_path)
    else:
        raise FileNotFoundError(f"Final matrix not found: {final_matrix_path}")


if "snb_final_matrix_validation_df" not in globals():
    path = FINAL_MATRIX_ARTIFACTS_DIR / "snb_final_matrix_validation.csv"
    if path.exists():
        snb_final_matrix_validation_df = pd.read_csv(path)
        print("Loaded:", path)


if "snb_final_matrix_group_summary_df" not in globals():
    path = FINAL_MATRIX_ARTIFACTS_DIR / "snb_final_matrix_group_summary.csv"
    if path.exists():
        snb_final_matrix_group_summary_df = pd.read_csv(path)
        print("Loaded:", path)


if "snb_final_matrix_semantic_summary_df" not in globals():
    path = FINAL_MATRIX_ARTIFACTS_DIR / "snb_final_matrix_semantic_summary.csv"
    if path.exists():
        snb_final_matrix_semantic_summary_df = pd.read_csv(path)
        print("Loaded:", path)


# ---------------------------------------------------------
# Load activation artifacts if missing
# ---------------------------------------------------------

if "generic_classes_df" not in globals():
    path = ACTIVATION_ARTIFACTS_DIR / "generic_configuration_classes_g0_g9.csv"
    if path.exists():
        generic_classes_df = pd.read_csv(path)
        print("Loaded:", path)

if "activation_thresholds_df" not in globals():
    path = ACTIVATION_ARTIFACTS_DIR / "activation_thresholds.csv"
    if path.exists():
        activation_thresholds_df = pd.read_csv(path)
        print("Loaded:", path)

if "snb_activation_matrix_df" not in globals():
    path = ACTIVATION_ARTIFACTS_DIR / "snb_activation_matrix.csv"
    if path.exists():
        snb_activation_matrix_df = pd.read_csv(path)
        print("Loaded:", path)
    else:
        raise FileNotFoundError(f"Activation matrix not found: {path}")

if "snb_activation_summary_df" not in globals():
    path = ACTIVATION_ARTIFACTS_DIR / "snb_activation_summary.csv"
    if path.exists():
        snb_activation_summary_df = pd.read_csv(path)

        for col in [
            "activated_g_classes",
            "primary_g_classes",
            "secondary_g_classes",
            "control_g_classes",
        ]:
            if col in snb_activation_summary_df.columns:
                snb_activation_summary_df[col] = (
                    snb_activation_summary_df[col].apply(parse_possible_list_cell)
                )

        print("Loaded:", path)

if "snb_activation_counts_by_g_df" not in globals():
    path = ACTIVATION_ARTIFACTS_DIR / "activation_counts_by_g_class.csv"
    if path.exists():
        snb_activation_counts_by_g_df = pd.read_csv(path)

        if "queries" in snb_activation_counts_by_g_df.columns:
            snb_activation_counts_by_g_df["queries"] = (
                snb_activation_counts_by_g_df["queries"].apply(parse_possible_list_cell)
            )

        print("Loaded:", path)

if "snb_activation_counts_by_group_df" not in globals():
    path = ACTIVATION_ARTIFACTS_DIR / "activation_counts_by_query_group.csv"
    if path.exists():
        snb_activation_counts_by_group_df = pd.read_csv(path)

        if "queries" in snb_activation_counts_by_group_df.columns:
            snb_activation_counts_by_group_df["queries"] = (
                snb_activation_counts_by_group_df["queries"].apply(parse_possible_list_cell)
            )

        print("Loaded:", path)

if "snb_activation_validation_df" not in globals():
    path = ACTIVATION_ARTIFACTS_DIR / "activation_validation.csv"
    if path.exists():
        snb_activation_validation_df = pd.read_csv(path)

        if "details" in snb_activation_validation_df.columns:
            snb_activation_validation_df["details"] = (
                snb_activation_validation_df["details"].apply(parse_possible_list_cell)
            )

        print("Loaded:", path)

In [202]:
# =========================================================
# RESUME 11C — Rebuild generic configuration dictionaries if missing
# =========================================================

if not GENERIC_CONFIGURATION_CLASSES and "generic_classes_df" in globals():
    GENERIC_CONFIGURATION_CLASSES = {}

    for _, row in generic_classes_df.iterrows():
        GENERIC_CONFIGURATION_CLASSES[row["g_class"]] = {
            "family": row["family"],
            "role": row["role"],
            "label": row["label"],
            "description": row["description"],
        }

    print("Rebuilt GENERIC_CONFIGURATION_CLASSES from generic_classes_df.")


if not ACTIVATION_THRESHOLDS and "activation_thresholds_df" in globals():
    ACTIVATION_THRESHOLDS = dict(
        zip(
            activation_thresholds_df["threshold"],
            activation_thresholds_df["value"]
        )
    )

    print("Rebuilt ACTIVATION_THRESHOLDS from activation_thresholds_df.")

print("Number of generic classes:", len(GENERIC_CONFIGURATION_CLASSES))
print("Number of activation thresholds:", len(ACTIVATION_THRESHOLDS))

Number of generic classes: 10
Number of activation thresholds: 9


In [196]:
# =========================================================
# RESUME 11D — Restore persistent DuckDB connection
# =========================================================

if "PERSISTENT_DUCKDB_PATH" in checkpoint_metadata:
    PERSISTENT_DUCKDB_PATH = Path(checkpoint_metadata["PERSISTENT_DUCKDB_PATH"])

    if PERSISTENT_DUCKDB_PATH.exists():
        con = duckdb.connect(str(PERSISTENT_DUCKDB_PATH))
        print("Connected to persistent DuckDB:")
        print(PERSISTENT_DUCKDB_PATH)
    else:
        print("WARNING: Persistent DuckDB file not found:")
        print(PERSISTENT_DUCKDB_PATH)
else:
    print("WARNING: PERSISTENT_DUCKDB_PATH not found in metadata.")
    print("This may be OK if Block 12 only generates JSON specs.")

Connected to persistent DuckDB:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/duckdb/ldbc_snb_sf0_1_clean.duckdb


In [203]:
# =========================================================
# RESUME 11E — Rebuild schema graphs if schema edges exist
# =========================================================

if "snb_schema_edges_df" in globals():
    snb_schema_graph = nx.DiGraph()

    for entity in SNB_ENTITY_VIEW_MAP.keys():
        snb_schema_graph.add_node(entity)

    for _, row in snb_schema_edges_df.iterrows():
        source = row["source_entity"]
        target = row["target_entity"]

        snb_schema_graph.add_edge(
            source,
            target,
            relationship=row["relationship"],
            semantic_type=row["semantic_type"],
            relationship_kind=row["relationship_kind"],
            view_name=row["view_name"],
            source_column=row["source_column"],
            target_column=row["target_column"],
        )

        if bool(row.get("is_bidirectional", False)):
            snb_schema_graph.add_edge(
                target,
                source,
                relationship=str(row["relationship"]) + "_reverse",
                semantic_type=row["semantic_type"],
                relationship_kind=row["relationship_kind"],
                view_name=row["view_name"],
                source_column=row["target_column"],
                target_column=row["source_column"],
            )

    snb_traversal_graph = nx.Graph()

    for entity in SNB_ENTITY_VIEW_MAP.keys():
        snb_traversal_graph.add_node(entity)

    for _, row in snb_schema_edges_df.iterrows():
        snb_traversal_graph.add_edge(
            row["source_entity"],
            row["target_entity"],
            relationship=row["relationship"],
            semantic_type=row["semantic_type"],
            relationship_kind=row["relationship_kind"],
            view_name=row["view_name"],
            source_column=row["source_column"],
            target_column=row["target_column"],
        )

    print("Graphs rebuilt.")
    print("Directed graph nodes:", snb_schema_graph.number_of_nodes())
    print("Directed graph edges:", snb_schema_graph.number_of_edges())
    print("Traversal graph nodes:", snb_traversal_graph.number_of_nodes())
    print("Traversal graph edges:", snb_traversal_graph.number_of_edges())
else:
    print("snb_schema_edges_df not found. Graph rebuild skipped.")

Graphs rebuilt.
Directed graph nodes: 8
Directed graph edges: 21
Traversal graph nodes: 8
Traversal graph edges: 19


In [207]:
# =========================================================
# RESUME FROM BLOCK 11 — Restore state before continuing Block 12
# =========================================================
#
# Use this after kernel restart.
# It restores everything saved after Block 11:
# - final analytical matrix
# - activation matrix
# - activation summary
# - G0-G9 classes
# - metadata dictionaries
#
# After running this block, run:
#   BLOCK 12A
#   BLOCK 12B
#   BLOCK 12C
#   BLOCK 12D
# =========================================================

from pathlib import Path
import json
import ast
import pandas as pd
import duckdb
import networkx as nx

# ---------------------------------------------------------
# 1. Project and checkpoint paths
# ---------------------------------------------------------

PROJECT_DIR = Path(
    "/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB"
)

CHECKPOINT_DIR = PROJECT_DIR / "benchmark_artifacts_dir" / "ldbc_snb_checkpoints"
manifest_path = CHECKPOINT_DIR / "checkpoint_manifest.json"

FINAL_MATRIX_ARTIFACTS_DIR = (
    PROJECT_DIR
    / "benchmark_artifacts_dir"
    / "ldbc_snb_final_analytical_matrix_official"
)

ACTIVATION_ARTIFACTS_DIR = (
    PROJECT_DIR
    / "benchmark_artifacts_dir"
    / "ldbc_snb_activation_official"
)

print("Project dir:", PROJECT_DIR)
print("Checkpoint manifest exists:", manifest_path.exists())

if not manifest_path.exists():
    raise FileNotFoundError(f"Checkpoint manifest not found: {manifest_path}")

# ---------------------------------------------------------
# 2. Load checkpoint manifest and metadata
# ---------------------------------------------------------

with open(manifest_path, "r", encoding="utf-8") as f:
    checkpoint_manifest = json.load(f)

metadata_path = CHECKPOINT_DIR / checkpoint_manifest["metadata"]

with open(metadata_path, "r", encoding="utf-8") as f:
    checkpoint_metadata = json.load(f)

# ---------------------------------------------------------
# 3. Restore paths
# ---------------------------------------------------------

BASE_DIR = Path(checkpoint_metadata.get("BASE_DIR", ""))

SNB_SNAPSHOT_DIR = Path(checkpoint_metadata.get("SNB_SNAPSHOT_DIR", ""))
SNB_DYNAMIC_DIR = Path(checkpoint_metadata.get("SNB_DYNAMIC_DIR", ""))
SNB_STATIC_DIR = Path(checkpoint_metadata.get("SNB_STATIC_DIR", ""))
SNB_SUBSTITUTION_DIR = Path(checkpoint_metadata.get("SNB_SUBSTITUTION_DIR", ""))
SNB_UPDATE_DIR = Path(checkpoint_metadata.get("SNB_UPDATE_DIR", ""))

# ---------------------------------------------------------
# 4. Restore metadata dictionaries/lists
# ---------------------------------------------------------

SNB_ENTITY_VIEW_MAP = checkpoint_metadata.get("SNB_ENTITY_VIEW_MAP", {})
SNB_ENTITY_PK_MAP = checkpoint_metadata.get("SNB_ENTITY_PK_MAP", {})
SNB_COLLECTION_NAME_BY_ENTITY = checkpoint_metadata.get("SNB_COLLECTION_NAME_BY_ENTITY", {})
SNB_RELATIONSHIP_HINTS = checkpoint_metadata.get("SNB_RELATIONSHIP_HINTS", {})

SNB_OFFICIAL_WORKLOAD = checkpoint_metadata.get("SNB_OFFICIAL_WORKLOAD", [])
SNB_OFFICIAL_QUERY_PATH_OVERRIDES = checkpoint_metadata.get(
    "SNB_OFFICIAL_QUERY_PATH_OVERRIDES", {}
)
SNB_RELATIONSHIP_MULTIPLICITY_OVERRIDES = checkpoint_metadata.get(
    "SNB_RELATIONSHIP_MULTIPLICITY_OVERRIDES", {}
)

GENERIC_CONFIGURATION_CLASSES = checkpoint_metadata.get(
    "GENERIC_CONFIGURATION_CLASSES", {}
)

ACTIVATION_THRESHOLDS = checkpoint_metadata.get(
    "ACTIVATION_THRESHOLDS", {}
)

UPDATE_OPERATION_CODE_MAP = checkpoint_metadata.get(
    "UPDATE_OPERATION_CODE_MAP", {}
)

# ---------------------------------------------------------
# 5. Restore all DataFrames from checkpoint
# ---------------------------------------------------------

for var_name, relative_path in checkpoint_manifest.get("dataframes", {}).items():
    df_path = CHECKPOINT_DIR / relative_path

    if df_path.exists():
        globals()[var_name] = pd.read_json(df_path, orient="records")

print("\nRestored DataFrames from checkpoint:")
for name in checkpoint_manifest.get("dataframes", {}):
    if name in globals():
        print(" -", name)

# ---------------------------------------------------------
# 6. Helper to recover list-like columns from CSV fallback
# ---------------------------------------------------------

def parse_possible_list_cell(value):
    if pd.isna(value):
        return value

    if isinstance(value, list):
        return value

    if isinstance(value, str):
        value_strip = value.strip()
        if value_strip.startswith("[") and value_strip.endswith("]"):
            try:
                return ast.literal_eval(value_strip)
            except Exception:
                return value

    return value

# ---------------------------------------------------------
# 7. Fallback: load final matrix if missing
# ---------------------------------------------------------

if "snb_final_analytical_matrix_df" not in globals():
    final_matrix_path = FINAL_MATRIX_ARTIFACTS_DIR / "snb_final_analytical_matrix.csv"

    if not final_matrix_path.exists():
        raise FileNotFoundError(f"Final matrix not found: {final_matrix_path}")

    snb_final_analytical_matrix_df = pd.read_csv(final_matrix_path)

    for col in ["accessed_entities", "relationships_used", "weighted_relationships"]:
        if col in snb_final_analytical_matrix_df.columns:
            snb_final_analytical_matrix_df[col] = (
                snb_final_analytical_matrix_df[col].apply(parse_possible_list_cell)
            )

    print("\nLoaded final matrix from CSV fallback.")

# ---------------------------------------------------------
# 8. Fallback: load activation artifacts if missing
# ---------------------------------------------------------

activation_files = {
    "generic_classes_df": "generic_configuration_classes_g0_g9.csv",
    "activation_thresholds_df": "activation_thresholds.csv",
    "snb_activation_matrix_df": "snb_activation_matrix.csv",
    "snb_activation_summary_df": "snb_activation_summary.csv",
    "snb_activation_counts_by_g_df": "activation_counts_by_g_class.csv",
    "snb_activation_counts_by_group_df": "activation_counts_by_query_group.csv",
    "snb_activation_validation_df": "activation_validation.csv",
}

for var_name, file_name in activation_files.items():
    if var_name not in globals():
        path = ACTIVATION_ARTIFACTS_DIR / file_name

        if path.exists():
            globals()[var_name] = pd.read_csv(path)
            print(f"Loaded {var_name} from CSV fallback.")

# Recover list-like columns in activation summary.
if "snb_activation_summary_df" in globals():
    for col in [
        "activated_g_classes",
        "primary_g_classes",
        "secondary_g_classes",
        "control_g_classes",
    ]:
        if col in snb_activation_summary_df.columns:
            snb_activation_summary_df[col] = (
                snb_activation_summary_df[col].apply(parse_possible_list_cell)
            )

# Recover list-like columns in activation counts.
if "snb_activation_counts_by_g_df" in globals():
    if "queries" in snb_activation_counts_by_g_df.columns:
        snb_activation_counts_by_g_df["queries"] = (
            snb_activation_counts_by_g_df["queries"].apply(parse_possible_list_cell)
        )

if "snb_activation_counts_by_group_df" in globals():
    if "queries" in snb_activation_counts_by_group_df.columns:
        snb_activation_counts_by_group_df["queries"] = (
            snb_activation_counts_by_group_df["queries"].apply(parse_possible_list_cell)
        )

# ---------------------------------------------------------
# 9. Rebuild generic class dictionaries if needed
# ---------------------------------------------------------

if not GENERIC_CONFIGURATION_CLASSES and "generic_classes_df" in globals():
    GENERIC_CONFIGURATION_CLASSES = {}

    for _, row in generic_classes_df.iterrows():
        GENERIC_CONFIGURATION_CLASSES[row["g_class"]] = {
            "family": row["family"],
            "role": row["role"],
            "label": row["label"],
            "description": row["description"],
        }

    print("\nRebuilt GENERIC_CONFIGURATION_CLASSES.")

if not ACTIVATION_THRESHOLDS and "activation_thresholds_df" in globals():
    ACTIVATION_THRESHOLDS = dict(
        zip(
            activation_thresholds_df["threshold"],
            activation_thresholds_df["value"]
        )
    )

    print("Rebuilt ACTIVATION_THRESHOLDS.")

# ---------------------------------------------------------
# 10. Restore DuckDB connection if persistent file exists
# ---------------------------------------------------------

if "PERSISTENT_DUCKDB_PATH" in checkpoint_metadata:
    PERSISTENT_DUCKDB_PATH = Path(checkpoint_metadata["PERSISTENT_DUCKDB_PATH"])

    if PERSISTENT_DUCKDB_PATH.exists():
        con = duckdb.connect(str(PERSISTENT_DUCKDB_PATH))
        print("\nConnected to persistent DuckDB:")
        print(PERSISTENT_DUCKDB_PATH)
    else:
        print("\nWARNING: Persistent DuckDB file not found:")
        print(PERSISTENT_DUCKDB_PATH)
else:
    print("\nNo persistent DuckDB path in metadata. This is OK for candidate generation.")

# ---------------------------------------------------------
# 11. Sanity check before continuing Block 12
# ---------------------------------------------------------

required_before_block_12 = [
    "snb_final_analytical_matrix_df",
    "snb_activation_matrix_df",
    "snb_activation_summary_df",
    "GENERIC_CONFIGURATION_CLASSES",
    "SNB_COLLECTION_NAME_BY_ENTITY",
    "SNB_ENTITY_PK_MAP",
    "SNB_RELATIONSHIP_HINTS",
]

print("\nObjects required before Block 12:")

for name in required_before_block_12:
    print(name, "=", "OK" if name in globals() else "MISSING")

print("\nActivation validation problems:")
if "snb_activation_validation_df" in globals():
    display(
        snb_activation_validation_df[
            snb_activation_validation_df["status"] != "OK"
        ]
    )

print("\nFinal matrix shape:")
print(snb_final_analytical_matrix_df.shape)

print("\nActivation matrix shape:")
print(snb_activation_matrix_df.shape)

Project dir: /home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB
Checkpoint manifest exists: True

Restored DataFrames from checkpoint:
 - columns_df
 - raw_counts_df
 - conceptual_counts_df
 - clean_entity_counts_df
 - fk_relationship_views_df
 - join_validation_df
 - snb_entity_metadata_df
 - snb_relationship_hints_df
 - snb_schema_edges_df
 - snb_graph_connectivity_df
 - snb_graph_connectivity_summary_df
 - important_paths_df
 - snb_traversal_connectivity_df
 - snb_traversal_connectivity_summary_df
 - important_traversal_paths_df
 - snb_workload_df
 - workload_relationship_validation_df
 - snb_query_semantic_profile_df
 - snb_initial_workload_matrix_df
 - snb_workload_paths_df
 - snb_preliminary_depth_df
 - snb_query_depth_overrides_df
 - snb_corrected_depth_df
 - snb_rc_df
 - snb_structural_matrix_df
 - snb_reduction_metrics_df
 - snb_structural_reduction_matrix_df
 - snb_reduction_rank_df
 - snb_reduction_validation_df
 - snb_update_stream_operation

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [208]:
# =========================================================
# FIX RESUME — Safe list parser after checkpoint restore
# =========================================================
#
# Problem:
# - Some checkpointed columns are already Python lists/arrays.
# - pd.isna(list) causes ValueError.
#
# Goal:
# - Safely normalize list-like columns.
# - Continue restoration before running Block 12A–12D.
# =========================================================

import ast
import pandas as pd
import numpy as np
import duckdb
from pathlib import Path

def parse_possible_list_cell_safe(value):
    """
    Convert list-like strings back to Python lists when possible.
    Safely handles lists, tuples, numpy arrays, NaN, and normal strings.
    """

    # Already list-like
    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if isinstance(value, np.ndarray):
        return value.tolist()

    # Missing scalar
    try:
        if pd.isna(value):
            return value
    except Exception:
        pass

    # Stringified list
    if isinstance(value, str):
        value_strip = value.strip()

        if value_strip.startswith("[") and value_strip.endswith("]"):
            try:
                parsed = ast.literal_eval(value_strip)
                if isinstance(parsed, list):
                    return parsed
                if isinstance(parsed, tuple):
                    return list(parsed)
                return parsed
            except Exception:
                return value

        if value_strip == "":
            return []

    return value


# ---------------------------------------------------------
# Recover list-like columns in activation summary
# ---------------------------------------------------------

if "snb_activation_summary_df" in globals():
    for col in [
        "activated_g_classes",
        "primary_g_classes",
        "secondary_g_classes",
        "control_g_classes",
    ]:
        if col in snb_activation_summary_df.columns:
            snb_activation_summary_df[col] = (
                snb_activation_summary_df[col].apply(parse_possible_list_cell_safe)
            )

# ---------------------------------------------------------
# Recover list-like columns in activation counts
# ---------------------------------------------------------

if "snb_activation_counts_by_g_df" in globals():
    if "queries" in snb_activation_counts_by_g_df.columns:
        snb_activation_counts_by_g_df["queries"] = (
            snb_activation_counts_by_g_df["queries"].apply(parse_possible_list_cell_safe)
        )

if "snb_activation_counts_by_group_df" in globals():
    if "queries" in snb_activation_counts_by_group_df.columns:
        snb_activation_counts_by_group_df["queries"] = (
            snb_activation_counts_by_group_df["queries"].apply(parse_possible_list_cell_safe)
        )

# ---------------------------------------------------------
# Recover list-like columns in final analytical matrix
# ---------------------------------------------------------

if "snb_final_analytical_matrix_df" in globals():
    for col in ["accessed_entities", "relationships_used", "weighted_relationships"]:
        if col in snb_final_analytical_matrix_df.columns:
            snb_final_analytical_matrix_df[col] = (
                snb_final_analytical_matrix_df[col].apply(parse_possible_list_cell_safe)
            )

print("List-like columns safely normalized.")

List-like columns safely normalized.


In [209]:
# =========================================================
# FIX RESUME — Rebuild dictionaries if needed
# =========================================================

# Rebuild GENERIC_CONFIGURATION_CLASSES from generic_classes_df
if "GENERIC_CONFIGURATION_CLASSES" not in globals() or not GENERIC_CONFIGURATION_CLASSES:
    GENERIC_CONFIGURATION_CLASSES = {}

    if "generic_classes_df" in globals():
        for _, row in generic_classes_df.iterrows():
            GENERIC_CONFIGURATION_CLASSES[row["g_class"]] = {
                "family": row["family"],
                "role": row["role"],
                "label": row["label"],
                "description": row["description"],
            }

        print("Rebuilt GENERIC_CONFIGURATION_CLASSES.")

# Rebuild ACTIVATION_THRESHOLDS from activation_thresholds_df
if "ACTIVATION_THRESHOLDS" not in globals() or not ACTIVATION_THRESHOLDS:
    if "activation_thresholds_df" in globals():
        ACTIVATION_THRESHOLDS = dict(
            zip(
                activation_thresholds_df["threshold"],
                activation_thresholds_df["value"]
            )
        )

        print("Rebuilt ACTIVATION_THRESHOLDS.")

print("Number of generic classes:", len(GENERIC_CONFIGURATION_CLASSES))
print("Number of activation thresholds:", len(ACTIVATION_THRESHOLDS))

Number of generic classes: 10
Number of activation thresholds: 9


In [210]:
# =========================================================
# FIX RESUME — Restore persistent DuckDB connection if available
# =========================================================

if "checkpoint_metadata" in globals() and "PERSISTENT_DUCKDB_PATH" in checkpoint_metadata:
    PERSISTENT_DUCKDB_PATH = Path(checkpoint_metadata["PERSISTENT_DUCKDB_PATH"])

    if PERSISTENT_DUCKDB_PATH.exists():
        con = duckdb.connect(str(PERSISTENT_DUCKDB_PATH))
        print("Connected to persistent DuckDB:")
        print(PERSISTENT_DUCKDB_PATH)
    else:
        print("WARNING: Persistent DuckDB file not found:")
        print(PERSISTENT_DUCKDB_PATH)
else:
    print("No persistent DuckDB path found. This is OK for Block 12 candidate generation.")

Connected to persistent DuckDB:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/duckdb/ldbc_snb_sf0_1_clean.duckdb


In [211]:
# =========================================================
# FIX RESUME — Sanity check before Block 12
# =========================================================

required_before_block_12 = [
    "snb_final_analytical_matrix_df",
    "snb_activation_matrix_df",
    "snb_activation_summary_df",
    "GENERIC_CONFIGURATION_CLASSES",
    "SNB_COLLECTION_NAME_BY_ENTITY",
    "SNB_ENTITY_PK_MAP",
    "SNB_RELATIONSHIP_HINTS",
]

print("Objects required before Block 12:")

for name in required_before_block_12:
    print(name, "=", "OK" if name in globals() else "MISSING")

print("\nFinal matrix shape:")
print(snb_final_analytical_matrix_df.shape)

print("\nActivation matrix shape:")
print(snb_activation_matrix_df.shape)

print("\nActivation validation problems:")
if "snb_activation_validation_df" in globals():
    display(
        snb_activation_validation_df[
            snb_activation_validation_df["status"] != "OK"
        ]
    )

print("\nPreview activation summary:")
display(
    snb_activation_summary_df[
        [
            "query_name",
            "official_id",
            "query_group",
            "root_entity",
            "dominant_semantic_type",
            "activated_g_classes",
            "primary_g_classes",
            "secondary_g_classes",
            "control_g_classes",
        ]
    ].head(10)
)

Objects required before Block 12:
snb_final_analytical_matrix_df = OK
snb_activation_matrix_df = OK
snb_activation_summary_df = OK
GENERIC_CONFIGURATION_CLASSES = OK
SNB_COLLECTION_NAME_BY_ENTITY = OK
SNB_ENTITY_PK_MAP = OK
SNB_RELATIONSHIP_HINTS = OK

Final matrix shape:
(22, 37)

Activation matrix shape:
(64, 19)

Activation validation problems:


,check,value,details,status



Preview activation summary:


,query_name,official_id,query_group,root_entity,dominant_semantic_type,activated_g_classes,primary_g_classes,secondary_g_classes,control_g_classes
0,IC1_TransitiveFriendsWithName,IC1,complex_read,Person,association,"[G0, G3]","[G0, G3]",[],[]
1,IC2_RecentMessagesByFriends,IC2,complex_read,Person,association,"[G0, G3]","[G0, G3]",[],[]
2,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,complex_read,Person,association,"[G0, G3, G7, G9]","[G0, G3]","[G7, G9]",[]
3,IC4_NewTopics,IC4,complex_read,Person,association,"[G0, G3]","[G0, G3]",[],[]
4,IC5_NewGroups,IC5,complex_read,Person,association,"[G0, G3, G4, G6, G7, G9]","[G0, G3]","[G4, G6, G7, G9]",[]
5,IC6_TagCoOccurrence,IC6,complex_read,Person,association,"[G0, G3]","[G0, G3]",[],[]
6,IC7_RecentLikers,IC7,complex_read,Person,association,"[G0, G3, G4, G6]","[G0, G3]","[G4, G6]",[]
7,INS1_AddPerson,INS1,insert,Person,association,"[G0, G3]","[G0, G3]",[],[]
8,INS2_AddLikeToPost,INS2,insert,Person,associative,"[G4, G6]","[G4, G6]",[],[]
9,INS3_AddLikeToComment,INS3,insert,Person,associative,"[G4, G6]","[G4, G6]",[],[]


In [212]:
# =========================================================
# BLOCK 12A — Define MongoDB pattern templates for G0–G9
# =========================================================
#
# Goal:
# - Define how each generic G class maps to a MongoDB modeling pattern.
# - These templates are still generic.
# - Later blocks will instantiate them using each query root, entities,
#   relationships, and risk variables.
# =========================================================

import pandas as pd
import json
from pathlib import Path
from datetime import datetime

MONGODB_PATTERN_TEMPLATES = {
    "G0": {
        "mongodb_pattern": "root_with_references",
        "document_strategy": "reference",
        "description": "Root collection stores root entity fields and references to related entities.",
        "embedding_level": "none_or_minimal",
        "reference_level": "high",
        "summary_level": "optional",
        "risk_profile": "safe_baseline",
    },

    "G1": {
        "mongodb_pattern": "single_entity_lookup",
        "document_strategy": "single_collection",
        "description": "Single entity collection optimized for direct lookup by primary key.",
        "embedding_level": "none",
        "reference_level": "none",
        "summary_level": "none",
        "risk_profile": "lookup_baseline",
    },

    "G2": {
        "mongodb_pattern": "root_with_embedded_low_risk_associations",
        "document_strategy": "embed",
        "description": "Embed low-risk associated data inside the root document.",
        "embedding_level": "medium",
        "reference_level": "low",
        "summary_level": "optional",
        "risk_profile": "embedding_when_safe",
    },

    "G3": {
        "mongodb_pattern": "root_with_references_or_summaries",
        "document_strategy": "reference_summary",
        "description": "Use references and/or denormalized summaries for associated entities.",
        "embedding_level": "low",
        "reference_level": "high",
        "summary_level": "medium",
        "risk_profile": "shared_or_volatile_safe",
    },

    "G4": {
        "mongodb_pattern": "explicit_edge_collection",
        "document_strategy": "edge_document",
        "description": "Represent associative relationships as explicit edge/relationship collections.",
        "embedding_level": "none",
        "reference_level": "high",
        "summary_level": "none",
        "risk_profile": "associative_baseline",
    },

    "G5": {
        "mongodb_pattern": "source_embedded_edges",
        "document_strategy": "embed_edges",
        "description": "Embed associative edges near the source root when fanout is manageable.",
        "embedding_level": "medium",
        "reference_level": "medium",
        "summary_level": "optional",
        "risk_profile": "direct_associative_embedding",
    },

    "G6": {
        "mongodb_pattern": "referenced_or_reverse_indexed_edges",
        "document_strategy": "edge_reference_reverse_index",
        "description": "Use references, edge collections, and optional reverse indexes for volatile/shared associative relationships.",
        "embedding_level": "low",
        "reference_level": "high",
        "summary_level": "optional",
        "risk_profile": "volatile_associative_safe",
    },

    "G7": {
        "mongodb_pattern": "containment_baseline",
        "document_strategy": "containment_reference",
        "description": "Baseline containment layout with parent-child references.",
        "embedding_level": "low",
        "reference_level": "medium",
        "summary_level": "optional",
        "risk_profile": "containment_baseline",
    },

    "G8": {
        "mongodb_pattern": "deep_containment_embedding",
        "document_strategy": "embed_contained_children",
        "description": "Embed contained child entities inside the root document when risk is low.",
        "embedding_level": "high",
        "reference_level": "low",
        "summary_level": "optional",
        "risk_profile": "deep_embedding_when_safe",
    },

    "G9": {
        "mongodb_pattern": "hybrid_containment",
        "document_strategy": "hybrid_embed_reference_summary",
        "description": "Use partial embedding, references, and summaries for containment paths with fanout/sharedness/volatility risk.",
        "embedding_level": "medium_partial",
        "reference_level": "high",
        "summary_level": "medium",
        "risk_profile": "hybrid_containment_safe",
    },
}

mongodb_pattern_templates_df = pd.DataFrame([
    {
        "g_class": g,
        **spec
    }
    for g, spec in MONGODB_PATTERN_TEMPLATES.items()
])

mongodb_pattern_templates_df

,g_class,mongodb_pattern,document_strategy,description,embedding_level,reference_level,summary_level,risk_profile
0,G0,root_with_references,reference,Root collection stores root entity fields and references to related entities.,none_or_minimal,high,optional,safe_baseline
1,G1,single_entity_lookup,single_collection,Single entity collection optimized for direct lookup by primary key.,none,none,none,lookup_baseline
2,G2,root_with_embedded_low_risk_associations,embed,Embed low-risk associated data inside the root document.,medium,low,optional,embedding_when_safe
3,G3,root_with_references_or_summaries,reference_summary,Use references and/or denormalized summaries for associated entities.,low,high,medium,shared_or_volatile_safe
4,G4,explicit_edge_collection,edge_document,Represent associative relationships as explicit edge/relationship collections.,none,high,none,associative_baseline
5,G5,source_embedded_edges,embed_edges,Embed associative edges near the source root when fanout is manageable.,medium,medium,optional,direct_associative_embedding
6,G6,referenced_or_reverse_indexed_edges,edge_reference_reverse_index,"Use references, edge collections, and optional reverse indexes for volatile/shared associative relationships.",low,high,optional,volatile_associative_safe
7,G7,containment_baseline,containment_reference,Baseline containment layout with parent-child references.,low,medium,optional,containment_baseline
8,G8,deep_containment_embedding,embed_contained_children,Embed contained child entities inside the root document when risk is low.,high,low,optional,deep_embedding_when_safe
9,G9,hybrid_containment,hybrid_embed_reference_summary,"Use partial embedding, references, and summaries for containment paths with fanout/sharedness/volatility risk.",medium_partial,high,medium,hybrid_containment_safe


In [213]:
# =========================================================
# BLOCK 12A — Define MongoDB pattern templates for G0–G9
# =========================================================
#
# Goal:
# - Define how each generic G class maps to a MongoDB modeling pattern.
# - These templates are still generic.
# - Later blocks will instantiate them using each query root, entities,
#   relationships, and risk variables.
# =========================================================

import pandas as pd
import json
from pathlib import Path
from datetime import datetime

MONGODB_PATTERN_TEMPLATES = {
    "G0": {
        "mongodb_pattern": "root_with_references",
        "document_strategy": "reference",
        "description": "Root collection stores root entity fields and references to related entities.",
        "embedding_level": "none_or_minimal",
        "reference_level": "high",
        "summary_level": "optional",
        "risk_profile": "safe_baseline",
    },

    "G1": {
        "mongodb_pattern": "single_entity_lookup",
        "document_strategy": "single_collection",
        "description": "Single entity collection optimized for direct lookup by primary key.",
        "embedding_level": "none",
        "reference_level": "none",
        "summary_level": "none",
        "risk_profile": "lookup_baseline",
    },

    "G2": {
        "mongodb_pattern": "root_with_embedded_low_risk_associations",
        "document_strategy": "embed",
        "description": "Embed low-risk associated data inside the root document.",
        "embedding_level": "medium",
        "reference_level": "low",
        "summary_level": "optional",
        "risk_profile": "embedding_when_safe",
    },

    "G3": {
        "mongodb_pattern": "root_with_references_or_summaries",
        "document_strategy": "reference_summary",
        "description": "Use references and/or denormalized summaries for associated entities.",
        "embedding_level": "low",
        "reference_level": "high",
        "summary_level": "medium",
        "risk_profile": "shared_or_volatile_safe",
    },

    "G4": {
        "mongodb_pattern": "explicit_edge_collection",
        "document_strategy": "edge_document",
        "description": "Represent associative relationships as explicit edge/relationship collections.",
        "embedding_level": "none",
        "reference_level": "high",
        "summary_level": "none",
        "risk_profile": "associative_baseline",
    },

    "G5": {
        "mongodb_pattern": "source_embedded_edges",
        "document_strategy": "embed_edges",
        "description": "Embed associative edges near the source root when fanout is manageable.",
        "embedding_level": "medium",
        "reference_level": "medium",
        "summary_level": "optional",
        "risk_profile": "direct_associative_embedding",
    },

    "G6": {
        "mongodb_pattern": "referenced_or_reverse_indexed_edges",
        "document_strategy": "edge_reference_reverse_index",
        "description": "Use references, edge collections, and optional reverse indexes for volatile/shared associative relationships.",
        "embedding_level": "low",
        "reference_level": "high",
        "summary_level": "optional",
        "risk_profile": "volatile_associative_safe",
    },

    "G7": {
        "mongodb_pattern": "containment_baseline",
        "document_strategy": "containment_reference",
        "description": "Baseline containment layout with parent-child references.",
        "embedding_level": "low",
        "reference_level": "medium",
        "summary_level": "optional",
        "risk_profile": "containment_baseline",
    },

    "G8": {
        "mongodb_pattern": "deep_containment_embedding",
        "document_strategy": "embed_contained_children",
        "description": "Embed contained child entities inside the root document when risk is low.",
        "embedding_level": "high",
        "reference_level": "low",
        "summary_level": "optional",
        "risk_profile": "deep_embedding_when_safe",
    },

    "G9": {
        "mongodb_pattern": "hybrid_containment",
        "document_strategy": "hybrid_embed_reference_summary",
        "description": "Use partial embedding, references, and summaries for containment paths with fanout/sharedness/volatility risk.",
        "embedding_level": "medium_partial",
        "reference_level": "high",
        "summary_level": "medium",
        "risk_profile": "hybrid_containment_safe",
    },
}

mongodb_pattern_templates_df = pd.DataFrame([
    {
        "g_class": g,
        **spec
    }
    for g, spec in MONGODB_PATTERN_TEMPLATES.items()
])

mongodb_pattern_templates_df

,g_class,mongodb_pattern,document_strategy,description,embedding_level,reference_level,summary_level,risk_profile
0,G0,root_with_references,reference,Root collection stores root entity fields and references to related entities.,none_or_minimal,high,optional,safe_baseline
1,G1,single_entity_lookup,single_collection,Single entity collection optimized for direct lookup by primary key.,none,none,none,lookup_baseline
2,G2,root_with_embedded_low_risk_associations,embed,Embed low-risk associated data inside the root document.,medium,low,optional,embedding_when_safe
3,G3,root_with_references_or_summaries,reference_summary,Use references and/or denormalized summaries for associated entities.,low,high,medium,shared_or_volatile_safe
4,G4,explicit_edge_collection,edge_document,Represent associative relationships as explicit edge/relationship collections.,none,high,none,associative_baseline
5,G5,source_embedded_edges,embed_edges,Embed associative edges near the source root when fanout is manageable.,medium,medium,optional,direct_associative_embedding
6,G6,referenced_or_reverse_indexed_edges,edge_reference_reverse_index,"Use references, edge collections, and optional reverse indexes for volatile/shared associative relationships.",low,high,optional,volatile_associative_safe
7,G7,containment_baseline,containment_reference,Baseline containment layout with parent-child references.,low,medium,optional,containment_baseline
8,G8,deep_containment_embedding,embed_contained_children,Embed contained child entities inside the root document when risk is low.,high,low,optional,deep_embedding_when_safe
9,G9,hybrid_containment,hybrid_embed_reference_summary,"Use partial embedding, references, and summaries for containment paths with fanout/sharedness/volatility risk.",medium_partial,high,medium,hybrid_containment_safe


In [215]:
# =========================================================
# BLOCK 12B — Helper functions for MongoDB candidate generation
# =========================================================
#
# Goal:
# - Create reusable functions to instantiate MongoDB candidate specs.
# =========================================================

import hashlib
import json
import pandas as pd


def stable_candidate_id(query_name: str, official_id: str, g_class: str, root_entity: str) -> str:
    """
    Create a stable candidate id for reproducibility.
    """
    raw = f"{official_id}|{query_name}|{g_class}|{root_entity}"
    digest = hashlib.md5(raw.encode("utf-8")).hexdigest()[:8]
    return f"ldbc_snb_{official_id.lower()}_{g_class.lower()}_{digest}"


def ensure_list(value):
    """
    Ensure a value is a Python list.
    Handles real lists and stringified lists.
    """
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    if isinstance(value, str):
        value = value.strip()

        if value.startswith("[") and value.endswith("]"):
            try:
                import ast
                parsed = ast.literal_eval(value)
                if isinstance(parsed, list):
                    return parsed
            except Exception:
                pass

        if value == "":
            return []

        return [value]

    return [value]


def get_collection_name(entity: str) -> str:
    """
    Return MongoDB collection name for an entity.
    """
    return SNB_COLLECTION_NAME_BY_ENTITY.get(entity, entity.lower() + "s")


def get_entity_pk(entity: str) -> str:
    """
    Return primary key field for an entity.
    """
    return SNB_ENTITY_PK_MAP.get(entity, "id")


def build_relationship_specs(relationships_used):
    """
    Convert relationship names into detailed relationship specs.
    """
    specs = []

    for rel in ensure_list(relationships_used):
        if rel not in SNB_RELATIONSHIP_HINTS:
            specs.append({
                "relationship": rel,
                "status": "missing_in_relationship_hints",
            })
            continue

        hint = SNB_RELATIONSHIP_HINTS[rel]

        specs.append({
            "relationship": rel,
            "semantic_type": hint["semantic_type"],
            "source_entity": hint["source_entity"],
            "target_entity": hint["target_entity"],
            "relationship_kind": hint["relationship_kind"],
            "view_name": hint["view_name"],
            "source_column": hint["source_column"],
            "target_column": hint["target_column"],
            "is_bidirectional": hint["is_bidirectional"],
        })

    return specs


def choose_materialization_strategy(g_class: str, row: pd.Series) -> dict:
    """
    Convert a G class and query metrics into concrete materialization decisions.
    """
    template = MONGODB_PATTERN_TEMPLATES[g_class]

    root_entity = row["root_entity"]
    accessed_entities = ensure_list(row["accessed_entities"])
    relationships_used = ensure_list(row["relationships_used"])

    root_collection = get_collection_name(root_entity)

    update_volatility_max = float(row.get("update_volatility_max", 0.0))
    observed_sharedness_max = float(row.get("observed_sharedness_max", 0.0))
    delta_ratio = float(row.get("DeltaRratio", 0.0))
    d = int(row.get("D", 0))
    re = int(row.get("Re", 0))

    high_volatility = update_volatility_max >= 0.75
    high_sharedness = observed_sharedness_max >= 0.75
    deep = d >= 3
    high_re = re >= 3

    embedded_entities = []
    referenced_entities = []
    summarized_entities = []
    edge_collections = []
    reverse_indexes = []

    non_root_entities = [
        e for e in accessed_entities
        if e != root_entity
    ]

    # -----------------------------------------------------
    # Pattern-specific materialization decisions
    # -----------------------------------------------------

    if g_class == "G1":
        embedded_entities = []
        referenced_entities = []
        summarized_entities = []
        edge_collections = []

    elif g_class == "G0":
        referenced_entities = non_root_entities
        edge_collections = relationships_used

    elif g_class == "G2":
        # Low-risk association embedding.
        embedded_entities = non_root_entities
        referenced_entities = []
        edge_collections = []

    elif g_class == "G3":
        # Safer association: references + summaries.
        referenced_entities = non_root_entities
        summarized_entities = non_root_entities if delta_ratio >= 0.4 else []
        edge_collections = relationships_used if high_volatility or high_sharedness or high_re else []

    elif g_class == "G4":
        # Explicit associative edge collections.
        referenced_entities = non_root_entities
        edge_collections = relationships_used

    elif g_class == "G5":
        # Embed edge summaries near source/root.
        embedded_entities = []
        summarized_entities = non_root_entities
        edge_collections = relationships_used

    elif g_class == "G6":
        # Referenced/reverse indexed associative model.
        referenced_entities = non_root_entities
        edge_collections = relationships_used
        reverse_indexes = relationships_used

    elif g_class == "G7":
        # Containment baseline: parent-child references.
        referenced_entities = non_root_entities
        edge_collections = relationships_used

    elif g_class == "G8":
        # Deep embedding when safe.
        embedded_entities = non_root_entities
        referenced_entities = []
        edge_collections = []

    elif g_class == "G9":
        # Hybrid containment.
        if high_sharedness or high_volatility:
            referenced_entities = non_root_entities
            summarized_entities = non_root_entities
            edge_collections = relationships_used
        else:
            embedded_entities = non_root_entities
            summarized_entities = non_root_entities
            edge_collections = relationships_used if deep else []

    return {
        "mongodb_pattern": template["mongodb_pattern"],
        "document_strategy": template["document_strategy"],
        "root_collection": root_collection,
        "root_entity": root_entity,
        "root_pk": get_entity_pk(root_entity),

        "accessed_entities": accessed_entities,
        "relationships_used": relationships_used,
        "relationship_specs": build_relationship_specs(relationships_used),

        "embedded_entities": sorted(set(embedded_entities)),
        "referenced_entities": sorted(set(referenced_entities)),
        "summarized_entities": sorted(set(summarized_entities)),
        "edge_collections": sorted(set(edge_collections)),
        "reverse_indexes": sorted(set(reverse_indexes)),

        "risk_flags": {
            "high_volatility": high_volatility,
            "high_sharedness": high_sharedness,
            "deep_query": deep,
            "high_residual_traversal": high_re,
        },
    }

In [216]:
# =========================================================
# BLOCK 12C — Generate MongoDB candidate specifications
# =========================================================
#
# Goal:
# - Generate one MongoDB candidate specification per activated query/G class.
# - Each candidate spec is reproducible and benchmark-ready.
# =========================================================

candidate_specs = {}
candidate_rows = []

# We need query metrics from final matrix indexed by query_name + official_id.
final_matrix_indexed = snb_final_analytical_matrix_df.set_index(
    ["query_name", "official_id"]
)

for _, act_row in snb_activation_matrix_df.iterrows():
    qname = act_row["query_name"]
    official_id = act_row["official_id"]
    g_class = act_row["g_class"]

    if (qname, official_id) not in final_matrix_indexed.index:
        continue

    qrow = final_matrix_indexed.loc[(qname, official_id)]

    if isinstance(qrow, pd.DataFrame):
        qrow = qrow.iloc[0]

    root_entity = qrow["root_entity"]

    candidate_id = stable_candidate_id(
        query_name=qname,
        official_id=official_id,
        g_class=g_class,
        root_entity=root_entity,
    )

    materialization = choose_materialization_strategy(g_class, qrow)

    candidate_spec = {
        "candidate_id": candidate_id,
        "dataset": "ldbc_snb",
        "scale_label": "sf0.1",
        "workload": "LDBC SNB Interactive v1 official-based subset",
        "query_name": qname,
        "official_id": official_id,
        "official_title": qrow.get("official_title"),
        "query_group": qrow.get("query_group"),
        "operation_type": qrow.get("operation_type"),

        "g_class": g_class,
        "g_family": GENERIC_CONFIGURATION_CLASSES[g_class]["family"],
        "g_role": GENERIC_CONFIGURATION_CLASSES[g_class]["role"],
        "g_label": GENERIC_CONFIGURATION_CLASSES[g_class]["label"],

        "activation_strength": act_row["activation_strength"],
        "activation_reason": act_row["activation_reason"],

        "metrics": {
            "Rc_distinct": int(qrow.get("Rc_distinct", 0)),
            "Rc_weighted": int(qrow.get("Rc_weighted", 0)),
            "D": int(qrow.get("D", 0)),
            "Re": int(qrow.get("Re", 0)),
            "DeltaR": int(qrow.get("DeltaR", 0)),
            "DeltaRratio": float(qrow.get("DeltaRratio", 0.0)),

            "dominant_semantic_type": qrow.get("dominant_semantic_type"),
            "association_count": int(qrow.get("association_count", 0)),
            "associative_count": int(qrow.get("associative_count", 0)),
            "containment_count": int(qrow.get("containment_count", 0)),
            "lookup_count": int(qrow.get("lookup_count", 0)),

            "update_volatility_mean": float(qrow.get("update_volatility_mean", 0.0)),
            "update_volatility_max": float(qrow.get("update_volatility_max", 0.0)),
            "observed_sharedness_mean": float(qrow.get("observed_sharedness_mean", 0.0)),
            "observed_sharedness_max": float(qrow.get("observed_sharedness_max", 0.0)),
        },

        "materialization": materialization,
    }

    candidate_specs[candidate_id] = candidate_spec

    candidate_rows.append({
        "candidate_id": candidate_id,
        "query_name": qname,
        "official_id": official_id,
        "query_group": qrow.get("query_group"),
        "operation_type": qrow.get("operation_type"),
        "root_entity": root_entity,
        "root_collection": materialization["root_collection"],
        "g_class": g_class,
        "g_family": candidate_spec["g_family"],
        "g_role": candidate_spec["g_role"],
        "activation_strength": act_row["activation_strength"],
        "mongodb_pattern": materialization["mongodb_pattern"],
        "document_strategy": materialization["document_strategy"],
        "embedded_entities": materialization["embedded_entities"],
        "referenced_entities": materialization["referenced_entities"],
        "summarized_entities": materialization["summarized_entities"],
        "edge_collections": materialization["edge_collections"],
        "reverse_indexes": materialization["reverse_indexes"],
        "DeltaRratio": candidate_spec["metrics"]["DeltaRratio"],
        "update_volatility_max": candidate_spec["metrics"]["update_volatility_max"],
        "observed_sharedness_max": candidate_spec["metrics"]["observed_sharedness_max"],
    })

snb_mongodb_candidate_specs_by_candidate_id = candidate_specs
snb_mongodb_candidate_specs_df = pd.DataFrame(candidate_rows)

snb_mongodb_candidate_specs_df

,candidate_id,query_name,official_id,query_group,operation_type,root_entity,root_collection,g_class,g_family,g_role,activation_strength,mongodb_pattern,document_strategy,embedded_entities,referenced_entities,summarized_entities,edge_collections,reverse_indexes,DeltaRratio,update_volatility_max,observed_sharedness_max
0,ldbc_snb_ic1_g0_6aac8974,IC1_TransitiveFriendsWithName,IC1,complex_read,read,Person,persons,G0,association,baseline,primary,root_with_references,reference,[],"[Organisation, Place]",[],"[organisation_is_located_in_place, person_is_located_in_place, person_knows_person, person_study_at_organisation, person_work_at_organisation]",[],0.571429,1.0,1.000000
1,ldbc_snb_ic1_g3_89a97e97,IC1_TransitiveFriendsWithName,IC1,complex_read,read,Person,persons,G3,association,reference_or_summary_candidate,primary,root_with_references_or_summaries,reference_summary,[],"[Organisation, Place]","[Organisation, Place]","[organisation_is_located_in_place, person_is_located_in_place, person_knows_person, person_study_at_organisation, person_work_at_organisation]",[],0.571429,1.0,1.000000
2,ldbc_snb_ic2_g0_bcc20fff,IC2_RecentMessagesByFriends,IC2,complex_read,read,Person,persons,G0,association,baseline,primary,root_with_references,reference,[],"[Comment, Post]",[],"[comment_has_creator_person, person_knows_person, post_has_creator_person]",[],0.666667,1.0,0.106448
3,ldbc_snb_ic2_g3_8f05f032,IC2_RecentMessagesByFriends,IC2,complex_read,read,Person,persons,G3,association,reference_or_summary_candidate,primary,root_with_references_or_summaries,reference_summary,[],"[Comment, Post]","[Comment, Post]","[comment_has_creator_person, person_knows_person, post_has_creator_person]",[],0.666667,1.0,0.106448
4,ldbc_snb_ic3_g7_a1750f34,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,complex_read,read,Person,persons,G7,containment,baseline,secondary,containment_baseline,containment_reference,[],"[Comment, Place, Post]",[],"[comment_has_creator_person, comment_is_located_in_place, person_is_located_in_place, person_knows_person, place_is_part_of_place, post_has_creator_person, post_is_located_in_place]",[],0.500000,1.0,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,ldbc_snb_is6_g9_c7584187,IS6_ForumOfMessage,IS6,short_read,read,Post,posts,G9,containment,hybrid_containment_candidate,primary,hybrid_containment,hybrid_embed_reference_summary,[],"[Comment, Forum, Person]","[Comment, Forum, Person]","[comment_reply_of_comment, comment_reply_of_post, forum_container_of_post, forum_has_moderator_person]",[],0.750000,1.0,0.106448
60,ldbc_snb_is7_g7_4e6a7bf8,IS7_RepliesOfMessage,IS7,short_read,read,Post,posts,G7,containment,baseline,secondary,containment_baseline,containment_reference,[],"[Comment, Person]",[],"[comment_has_creator_person, comment_reply_of_comment, comment_reply_of_post, person_knows_person, post_has_creator_person]",[],0.400000,1.0,0.106448
61,ldbc_snb_is7_g9_392f7469,IS7_RepliesOfMessage,IS7,short_read,read,Post,posts,G9,containment,hybrid_containment_candidate,secondary,hybrid_containment,hybrid_embed_reference_summary,[],"[Comment, Person]","[Comment, Person]","[comment_has_creator_person, comment_reply_of_comment, comment_reply_of_post, person_knows_person, post_has_creator_person]",[],0.400000,1.0,0.106448
62,ldbc_snb_is7_g0_7dff7761,IS7_RepliesOfMessage,IS7,short_read,read,Post,posts,G0,association,baseline,primary,root_with_references,reference,[],"[Comment, Person]",[],"[comment_has_creator_person, comment_reply_of_comment, comment_reply_of_post, person_knows_person, post_has_creator_person]",[],0.400000,1.0,0.106448


In [217]:
# =========================================================
# BLOCK 12C — Generate MongoDB candidate specifications
# =========================================================
#
# Goal:
# - Generate one MongoDB candidate specification per activated query/G class.
# - Each candidate spec is reproducible and benchmark-ready.
# =========================================================

candidate_specs = {}
candidate_rows = []

# We need query metrics from final matrix indexed by query_name + official_id.
final_matrix_indexed = snb_final_analytical_matrix_df.set_index(
    ["query_name", "official_id"]
)

for _, act_row in snb_activation_matrix_df.iterrows():
    qname = act_row["query_name"]
    official_id = act_row["official_id"]
    g_class = act_row["g_class"]

    if (qname, official_id) not in final_matrix_indexed.index:
        continue

    qrow = final_matrix_indexed.loc[(qname, official_id)]

    if isinstance(qrow, pd.DataFrame):
        qrow = qrow.iloc[0]

    root_entity = qrow["root_entity"]

    candidate_id = stable_candidate_id(
        query_name=qname,
        official_id=official_id,
        g_class=g_class,
        root_entity=root_entity,
    )

    materialization = choose_materialization_strategy(g_class, qrow)

    candidate_spec = {
        "candidate_id": candidate_id,
        "dataset": "ldbc_snb",
        "scale_label": "sf0.1",
        "workload": "LDBC SNB Interactive v1 official-based subset",
        "query_name": qname,
        "official_id": official_id,
        "official_title": qrow.get("official_title"),
        "query_group": qrow.get("query_group"),
        "operation_type": qrow.get("operation_type"),

        "g_class": g_class,
        "g_family": GENERIC_CONFIGURATION_CLASSES[g_class]["family"],
        "g_role": GENERIC_CONFIGURATION_CLASSES[g_class]["role"],
        "g_label": GENERIC_CONFIGURATION_CLASSES[g_class]["label"],

        "activation_strength": act_row["activation_strength"],
        "activation_reason": act_row["activation_reason"],

        "metrics": {
            "Rc_distinct": int(qrow.get("Rc_distinct", 0)),
            "Rc_weighted": int(qrow.get("Rc_weighted", 0)),
            "D": int(qrow.get("D", 0)),
            "Re": int(qrow.get("Re", 0)),
            "DeltaR": int(qrow.get("DeltaR", 0)),
            "DeltaRratio": float(qrow.get("DeltaRratio", 0.0)),

            "dominant_semantic_type": qrow.get("dominant_semantic_type"),
            "association_count": int(qrow.get("association_count", 0)),
            "associative_count": int(qrow.get("associative_count", 0)),
            "containment_count": int(qrow.get("containment_count", 0)),
            "lookup_count": int(qrow.get("lookup_count", 0)),

            "update_volatility_mean": float(qrow.get("update_volatility_mean", 0.0)),
            "update_volatility_max": float(qrow.get("update_volatility_max", 0.0)),
            "observed_sharedness_mean": float(qrow.get("observed_sharedness_mean", 0.0)),
            "observed_sharedness_max": float(qrow.get("observed_sharedness_max", 0.0)),
        },

        "materialization": materialization,
    }

    candidate_specs[candidate_id] = candidate_spec

    candidate_rows.append({
        "candidate_id": candidate_id,
        "query_name": qname,
        "official_id": official_id,
        "query_group": qrow.get("query_group"),
        "operation_type": qrow.get("operation_type"),
        "root_entity": root_entity,
        "root_collection": materialization["root_collection"],
        "g_class": g_class,
        "g_family": candidate_spec["g_family"],
        "g_role": candidate_spec["g_role"],
        "activation_strength": act_row["activation_strength"],
        "mongodb_pattern": materialization["mongodb_pattern"],
        "document_strategy": materialization["document_strategy"],
        "embedded_entities": materialization["embedded_entities"],
        "referenced_entities": materialization["referenced_entities"],
        "summarized_entities": materialization["summarized_entities"],
        "edge_collections": materialization["edge_collections"],
        "reverse_indexes": materialization["reverse_indexes"],
        "DeltaRratio": candidate_spec["metrics"]["DeltaRratio"],
        "update_volatility_max": candidate_spec["metrics"]["update_volatility_max"],
        "observed_sharedness_max": candidate_spec["metrics"]["observed_sharedness_max"],
    })

snb_mongodb_candidate_specs_by_candidate_id = candidate_specs
snb_mongodb_candidate_specs_df = pd.DataFrame(candidate_rows)

snb_mongodb_candidate_specs_df

,candidate_id,query_name,official_id,query_group,operation_type,root_entity,root_collection,g_class,g_family,g_role,activation_strength,mongodb_pattern,document_strategy,embedded_entities,referenced_entities,summarized_entities,edge_collections,reverse_indexes,DeltaRratio,update_volatility_max,observed_sharedness_max
0,ldbc_snb_ic1_g0_6aac8974,IC1_TransitiveFriendsWithName,IC1,complex_read,read,Person,persons,G0,association,baseline,primary,root_with_references,reference,[],"[Organisation, Place]",[],"[organisation_is_located_in_place, person_is_located_in_place, person_knows_person, person_study_at_organisation, person_work_at_organisation]",[],0.571429,1.0,1.000000
1,ldbc_snb_ic1_g3_89a97e97,IC1_TransitiveFriendsWithName,IC1,complex_read,read,Person,persons,G3,association,reference_or_summary_candidate,primary,root_with_references_or_summaries,reference_summary,[],"[Organisation, Place]","[Organisation, Place]","[organisation_is_located_in_place, person_is_located_in_place, person_knows_person, person_study_at_organisation, person_work_at_organisation]",[],0.571429,1.0,1.000000
2,ldbc_snb_ic2_g0_bcc20fff,IC2_RecentMessagesByFriends,IC2,complex_read,read,Person,persons,G0,association,baseline,primary,root_with_references,reference,[],"[Comment, Post]",[],"[comment_has_creator_person, person_knows_person, post_has_creator_person]",[],0.666667,1.0,0.106448
3,ldbc_snb_ic2_g3_8f05f032,IC2_RecentMessagesByFriends,IC2,complex_read,read,Person,persons,G3,association,reference_or_summary_candidate,primary,root_with_references_or_summaries,reference_summary,[],"[Comment, Post]","[Comment, Post]","[comment_has_creator_person, person_knows_person, post_has_creator_person]",[],0.666667,1.0,0.106448
4,ldbc_snb_ic3_g7_a1750f34,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,complex_read,read,Person,persons,G7,containment,baseline,secondary,containment_baseline,containment_reference,[],"[Comment, Place, Post]",[],"[comment_has_creator_person, comment_is_located_in_place, person_is_located_in_place, person_knows_person, place_is_part_of_place, post_has_creator_person, post_is_located_in_place]",[],0.500000,1.0,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,ldbc_snb_is6_g9_c7584187,IS6_ForumOfMessage,IS6,short_read,read,Post,posts,G9,containment,hybrid_containment_candidate,primary,hybrid_containment,hybrid_embed_reference_summary,[],"[Comment, Forum, Person]","[Comment, Forum, Person]","[comment_reply_of_comment, comment_reply_of_post, forum_container_of_post, forum_has_moderator_person]",[],0.750000,1.0,0.106448
60,ldbc_snb_is7_g7_4e6a7bf8,IS7_RepliesOfMessage,IS7,short_read,read,Post,posts,G7,containment,baseline,secondary,containment_baseline,containment_reference,[],"[Comment, Person]",[],"[comment_has_creator_person, comment_reply_of_comment, comment_reply_of_post, person_knows_person, post_has_creator_person]",[],0.400000,1.0,0.106448
61,ldbc_snb_is7_g9_392f7469,IS7_RepliesOfMessage,IS7,short_read,read,Post,posts,G9,containment,hybrid_containment_candidate,secondary,hybrid_containment,hybrid_embed_reference_summary,[],"[Comment, Person]","[Comment, Person]","[comment_has_creator_person, comment_reply_of_comment, comment_reply_of_post, person_knows_person, post_has_creator_person]",[],0.400000,1.0,0.106448
62,ldbc_snb_is7_g0_7dff7761,IS7_RepliesOfMessage,IS7,short_read,read,Post,posts,G0,association,baseline,primary,root_with_references,reference,[],"[Comment, Person]",[],"[comment_has_creator_person, comment_reply_of_comment, comment_reply_of_post, person_knows_person, post_has_creator_person]",[],0.400000,1.0,0.106448


In [218]:
# =========================================================
# BLOCK 12D — Validate generated MongoDB candidate specs
# =========================================================
#
# Goal:
# - Check that every activation generated one candidate.
# - Check that candidate IDs are unique.
# - Check that all candidate G classes are valid.
# =========================================================

candidate_validation_rows = []

n_activations = len(snb_activation_matrix_df)
n_candidates = len(snb_mongodb_candidate_specs_df)
n_unique_candidate_ids = snb_mongodb_candidate_specs_df["candidate_id"].nunique()

candidate_validation_rows.append({
    "check": "candidate_count_matches_activation_count",
    "value": f"{n_candidates} / {n_activations}",
    "status": "OK" if n_candidates == n_activations else "ERROR",
})

candidate_validation_rows.append({
    "check": "candidate_ids_are_unique",
    "value": f"{n_unique_candidate_ids} / {n_candidates}",
    "status": "OK" if n_unique_candidate_ids == n_candidates else "ERROR",
})

invalid_candidate_g_classes = sorted(
    set(snb_mongodb_candidate_specs_df["g_class"])
    - set(GENERIC_CONFIGURATION_CLASSES.keys())
)

candidate_validation_rows.append({
    "check": "candidate_g_classes_valid",
    "value": len(invalid_candidate_g_classes),
    "details": invalid_candidate_g_classes,
    "status": "OK" if len(invalid_candidate_g_classes) == 0 else "ERROR",
})

missing_roots = snb_mongodb_candidate_specs_df[
    snb_mongodb_candidate_specs_df["root_entity"].isna()
]

candidate_validation_rows.append({
    "check": "candidate_root_entities_present",
    "value": len(missing_roots),
    "status": "OK" if len(missing_roots) == 0 else "ERROR",
})

snb_mongodb_candidate_validation_df = pd.DataFrame(candidate_validation_rows)
snb_mongodb_candidate_validation_df

,check,value,status,details
0,candidate_count_matches_activation_count,64 / 64,OK,NaN
1,candidate_ids_are_unique,64 / 64,OK,NaN
2,candidate_g_classes_valid,0,OK,[]
3,candidate_root_entities_present,0,OK,NaN


In [219]:
snb_mongodb_candidate_validation_df[
    snb_mongodb_candidate_validation_df["status"] != "OK"
]

,check,value,status,details


In [220]:
# =========================================================
# BLOCK 12E — Candidate summary by G class and pattern
# =========================================================

snb_candidate_summary_by_g_df = (
    snb_mongodb_candidate_specs_df
    .groupby(["g_class", "g_family", "mongodb_pattern", "document_strategy"])
    .agg(
        n_candidates=("candidate_id", "count"),
        n_primary=("activation_strength", lambda x: int((x == "primary").sum())),
        n_secondary=("activation_strength", lambda x: int((x == "secondary").sum())),
        n_control=("activation_strength", lambda x: int((x == "control").sum())),
        queries=("query_name", lambda x: sorted(set(x))),
    )
    .reset_index()
    .sort_values(["g_class"])
)

snb_candidate_summary_by_g_df

,g_class,g_family,mongodb_pattern,document_strategy,n_candidates,n_primary,n_secondary,n_control,queries
0,G0,association,root_with_references,reference,19,16,2,1,"[IC1_TransitiveFriendsWithName, IC2_RecentMessagesByFriends, IC3_FriendsAndFriendsOfFriendsInCountries, IC4_NewTopics, IC5_NewGroups, IC6_TagCoOccurrence, IC7_RecentLikers, INS1_AddPerson, INS4_AddForum, INS6_AddPost, INS7_AddComment, INS8_AddFriendship, IS1_ProfileOfPerson, IS2_RecentMessagesOfPerson, IS3_FriendsOfPerson, IS4_ContentOfMessage, IS5_CreatorOfMessage, IS6_ForumOfMessage, IS7_RepliesOfMessage]"
1,G1,lookup,single_entity_lookup,single_collection,1,1,0,0,[IS4_ContentOfMessage]
2,G3,association,root_with_references_or_summaries,reference_summary,18,16,2,0,"[IC1_TransitiveFriendsWithName, IC2_RecentMessagesByFriends, IC3_FriendsAndFriendsOfFriendsInCountries, IC4_NewTopics, IC5_NewGroups, IC6_TagCoOccurrence, IC7_RecentLikers, INS1_AddPerson, INS4_AddForum, INS6_AddPost, INS7_AddComment, INS8_AddFriendship, IS1_ProfileOfPerson, IS2_RecentMessagesOfPerson, IS3_FriendsOfPerson, IS5_CreatorOfMessage, IS6_ForumOfMessage, IS7_RepliesOfMessage]"
3,G4,associative,explicit_edge_collection,edge_document,6,3,3,0,"[IC5_NewGroups, IC7_RecentLikers, INS2_AddLikeToPost, INS3_AddLikeToComment, INS5_AddForumMembership, IS2_RecentMessagesOfPerson]"
4,G6,associative,referenced_or_reverse_indexed_edges,edge_reference_reverse_index,6,3,3,0,"[IC5_NewGroups, IC7_RecentLikers, INS2_AddLikeToPost, INS3_AddLikeToComment, INS5_AddForumMembership, IS2_RecentMessagesOfPerson]"
5,G7,containment,containment_baseline,containment_reference,7,1,6,0,"[IC3_FriendsAndFriendsOfFriendsInCountries, IC5_NewGroups, INS6_AddPost, INS7_AddComment, IS2_RecentMessagesOfPerson, IS6_ForumOfMessage, IS7_RepliesOfMessage]"
6,G9,containment,hybrid_containment,hybrid_embed_reference_summary,7,1,6,0,"[IC3_FriendsAndFriendsOfFriendsInCountries, IC5_NewGroups, INS6_AddPost, INS7_AddComment, IS2_RecentMessagesOfPerson, IS6_ForumOfMessage, IS7_RepliesOfMessage]"


In [221]:
# =========================================================
# BLOCK 12E — Candidate summary by G class and pattern
# =========================================================

snb_candidate_summary_by_g_df = (
    snb_mongodb_candidate_specs_df
    .groupby(["g_class", "g_family", "mongodb_pattern", "document_strategy"])
    .agg(
        n_candidates=("candidate_id", "count"),
        n_primary=("activation_strength", lambda x: int((x == "primary").sum())),
        n_secondary=("activation_strength", lambda x: int((x == "secondary").sum())),
        n_control=("activation_strength", lambda x: int((x == "control").sum())),
        queries=("query_name", lambda x: sorted(set(x))),
    )
    .reset_index()
    .sort_values(["g_class"])
)

snb_candidate_summary_by_g_df

,g_class,g_family,mongodb_pattern,document_strategy,n_candidates,n_primary,n_secondary,n_control,queries
0,G0,association,root_with_references,reference,19,16,2,1,"[IC1_TransitiveFriendsWithName, IC2_RecentMessagesByFriends, IC3_FriendsAndFriendsOfFriendsInCountries, IC4_NewTopics, IC5_NewGroups, IC6_TagCoOccurrence, IC7_RecentLikers, INS1_AddPerson, INS4_AddForum, INS6_AddPost, INS7_AddComment, INS8_AddFriendship, IS1_ProfileOfPerson, IS2_RecentMessagesOfPerson, IS3_FriendsOfPerson, IS4_ContentOfMessage, IS5_CreatorOfMessage, IS6_ForumOfMessage, IS7_RepliesOfMessage]"
1,G1,lookup,single_entity_lookup,single_collection,1,1,0,0,[IS4_ContentOfMessage]
2,G3,association,root_with_references_or_summaries,reference_summary,18,16,2,0,"[IC1_TransitiveFriendsWithName, IC2_RecentMessagesByFriends, IC3_FriendsAndFriendsOfFriendsInCountries, IC4_NewTopics, IC5_NewGroups, IC6_TagCoOccurrence, IC7_RecentLikers, INS1_AddPerson, INS4_AddForum, INS6_AddPost, INS7_AddComment, INS8_AddFriendship, IS1_ProfileOfPerson, IS2_RecentMessagesOfPerson, IS3_FriendsOfPerson, IS5_CreatorOfMessage, IS6_ForumOfMessage, IS7_RepliesOfMessage]"
3,G4,associative,explicit_edge_collection,edge_document,6,3,3,0,"[IC5_NewGroups, IC7_RecentLikers, INS2_AddLikeToPost, INS3_AddLikeToComment, INS5_AddForumMembership, IS2_RecentMessagesOfPerson]"
4,G6,associative,referenced_or_reverse_indexed_edges,edge_reference_reverse_index,6,3,3,0,"[IC5_NewGroups, IC7_RecentLikers, INS2_AddLikeToPost, INS3_AddLikeToComment, INS5_AddForumMembership, IS2_RecentMessagesOfPerson]"
5,G7,containment,containment_baseline,containment_reference,7,1,6,0,"[IC3_FriendsAndFriendsOfFriendsInCountries, IC5_NewGroups, INS6_AddPost, INS7_AddComment, IS2_RecentMessagesOfPerson, IS6_ForumOfMessage, IS7_RepliesOfMessage]"
6,G9,containment,hybrid_containment,hybrid_embed_reference_summary,7,1,6,0,"[IC3_FriendsAndFriendsOfFriendsInCountries, IC5_NewGroups, INS6_AddPost, INS7_AddComment, IS2_RecentMessagesOfPerson, IS6_ForumOfMessage, IS7_RepliesOfMessage]"


In [222]:
# =========================================================
# BLOCK 12F — Candidate summary by query
# =========================================================

snb_candidate_summary_by_query_df = (
    snb_mongodb_candidate_specs_df
    .groupby(["query_name", "official_id", "query_group", "root_entity"])
    .agg(
        candidate_ids=("candidate_id", lambda x: sorted(set(x))),
        g_classes=("g_class", lambda x: sorted(set(x))),
        mongodb_patterns=("mongodb_pattern", lambda x: sorted(set(x))),
        n_candidates=("candidate_id", "count"),
    )
    .reset_index()
    .sort_values(["query_group", "official_id"])
)

snb_candidate_summary_by_query_df

,query_name,official_id,query_group,root_entity,candidate_ids,g_classes,mongodb_patterns,n_candidates
0,IC1_TransitiveFriendsWithName,IC1,complex_read,Person,"[ldbc_snb_ic1_g0_6aac8974, ldbc_snb_ic1_g3_89a97e97]","[G0, G3]","[root_with_references, root_with_references_or_summaries]",2
1,IC2_RecentMessagesByFriends,IC2,complex_read,Person,"[ldbc_snb_ic2_g0_bcc20fff, ldbc_snb_ic2_g3_8f05f032]","[G0, G3]","[root_with_references, root_with_references_or_summaries]",2
2,IC3_FriendsAndFriendsOfFriendsInCountries,IC3,complex_read,Person,"[ldbc_snb_ic3_g0_f7817cbe, ldbc_snb_ic3_g3_f4732bc1, ldbc_snb_ic3_g7_a1750f34, ldbc_snb_ic3_g9_75aa81cd]","[G0, G3, G7, G9]","[containment_baseline, hybrid_containment, root_with_references, root_with_references_or_summaries]",4
3,IC4_NewTopics,IC4,complex_read,Person,"[ldbc_snb_ic4_g0_54d8940c, ldbc_snb_ic4_g3_00df9f3e]","[G0, G3]","[root_with_references, root_with_references_or_summaries]",2
4,IC5_NewGroups,IC5,complex_read,Person,"[ldbc_snb_ic5_g0_f44bd6f8, ldbc_snb_ic5_g3_45734260, ldbc_snb_ic5_g4_69144c92, ldbc_snb_ic5_g6_733601c1, ldbc_snb_ic5_g7_a2002d4c, ldbc_snb_ic5_g9_7a7a63d2]","[G0, G3, G4, G6, G7, G9]","[containment_baseline, explicit_edge_collection, hybrid_containment, referenced_or_reverse_indexed_edges, root_with_references, root_with_references_or_summaries]",6
5,IC6_TagCoOccurrence,IC6,complex_read,Person,"[ldbc_snb_ic6_g0_f6a698ed, ldbc_snb_ic6_g3_9a00a0b5]","[G0, G3]","[root_with_references, root_with_references_or_summaries]",2
6,IC7_RecentLikers,IC7,complex_read,Person,"[ldbc_snb_ic7_g0_39cfc8f6, ldbc_snb_ic7_g3_d6d26f02, ldbc_snb_ic7_g4_5de4a081, ldbc_snb_ic7_g6_084f8a43]","[G0, G3, G4, G6]","[explicit_edge_collection, referenced_or_reverse_indexed_edges, root_with_references, root_with_references_or_summaries]",4
7,INS1_AddPerson,INS1,insert,Person,"[ldbc_snb_ins1_g0_ffbfd07e, ldbc_snb_ins1_g3_10a467d5]","[G0, G3]","[root_with_references, root_with_references_or_summaries]",2
8,INS2_AddLikeToPost,INS2,insert,Person,"[ldbc_snb_ins2_g4_9fb1f96c, ldbc_snb_ins2_g6_f226f1c0]","[G4, G6]","[explicit_edge_collection, referenced_or_reverse_indexed_edges]",2
9,INS3_AddLikeToComment,INS3,insert,Person,"[ldbc_snb_ins3_g4_9ec9f8f6, ldbc_snb_ins3_g6_c1c01729]","[G4, G6]","[explicit_edge_collection, referenced_or_reverse_indexed_edges]",2


In [223]:
# =========================================================
# BLOCK 12G — Save MongoDB candidate artifacts
# =========================================================

MONGODB_CANDIDATES_DIR = (
    BASE_DIR.parent.parent
    / "benchmark_artifacts_dir"
    / "ldbc_snb_mongo_configurations"
)

MONGODB_CANDIDATES_DIR.mkdir(parents=True, exist_ok=True)

# Save candidate specs as JSON
candidate_specs_json_path = (
    MONGODB_CANDIDATES_DIR
    / "mongodb_candidate_specs_by_candidate_id.json"
)

with open(candidate_specs_json_path, "w", encoding="utf-8") as f:
    json.dump(
        snb_mongodb_candidate_specs_by_candidate_id,
        f,
        indent=2,
        ensure_ascii=False
    )

# Save tabular candidate overview
snb_mongodb_candidate_specs_df.to_csv(
    MONGODB_CANDIDATES_DIR / "mongodb_candidate_specs_overview.csv",
    index=False
)

snb_mongodb_candidate_validation_df.to_csv(
    MONGODB_CANDIDATES_DIR / "mongodb_candidate_validation.csv",
    index=False
)

snb_candidate_summary_by_g_df.to_csv(
    MONGODB_CANDIDATES_DIR / "mongodb_candidate_summary_by_g_class.csv",
    index=False
)

snb_candidate_summary_by_query_df.to_csv(
    MONGODB_CANDIDATES_DIR / "mongodb_candidate_summary_by_query.csv",
    index=False
)

# Save pattern templates too
mongodb_pattern_templates_df.to_csv(
    MONGODB_CANDIDATES_DIR / "mongodb_pattern_templates_g0_g9.csv",
    index=False
)

print("Saved MongoDB candidate artifacts to:")
print(MONGODB_CANDIDATES_DIR)

print("\nSaved files:")
for path in sorted(MONGODB_CANDIDATES_DIR.glob("*")):
    print(" -", path.name)

Saved MongoDB candidate artifacts to:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_mongo_configurations

Saved files:
 - mongodb_candidate_specs_by_candidate_id.json
 - mongodb_candidate_specs_overview.csv
 - mongodb_candidate_summary_by_g_class.csv
 - mongodb_candidate_summary_by_query.csv
 - mongodb_candidate_validation.csv
 - mongodb_pattern_templates_g0_g9.csv


In [224]:
# =========================================================
# BLOCK 12H — Create MongoDB candidate manifest
# =========================================================
#
# Goal:
# - Create a manifest that the benchmark runner can read.
# =========================================================

candidate_manifest = {
    "dataset": "ldbc_snb",
    "scale_label": "sf0.1",
    "workload": "LDBC SNB Interactive v1 official-based subset",
    "created_at": datetime.utcnow().isoformat() + "Z",

    "n_queries": int(snb_final_analytical_matrix_df["query_name"].nunique()),
    "n_activations": int(len(snb_activation_matrix_df)),
    "n_candidates": int(len(snb_mongodb_candidate_specs_df)),

    "files": {
        "candidate_specs": "mongodb_candidate_specs_by_candidate_id.json",
        "candidate_specs_overview": "mongodb_candidate_specs_overview.csv",
        "candidate_validation": "mongodb_candidate_validation.csv",
        "candidate_summary_by_g_class": "mongodb_candidate_summary_by_g_class.csv",
        "candidate_summary_by_query": "mongodb_candidate_summary_by_query.csv",
        "pattern_templates": "mongodb_pattern_templates_g0_g9.csv",
    },

    "notes": [
        "Candidates are generated from the official-based LDBC SNB Interactive workload subset.",
        "These specs are not benchmark results; they define MongoDB materialization candidates.",
        "Benchmark execution will be handled by a separate runner script.",
    ],
}

benchmark_manifest_path = MONGODB_CANDIDATES_DIR / "benchmark_manifest.json"

with open(benchmark_manifest_path, "w", encoding="utf-8") as f:
    json.dump(candidate_manifest, f, indent=2, ensure_ascii=False)

candidate_manifest

{'dataset': 'ldbc_snb',
 'scale_label': 'sf0.1',
 'workload': 'LDBC SNB Interactive v1 official-based subset',
 'created_at': '2026-05-08T18:01:11.375015Z',
 'n_queries': 22,
 'n_activations': 64,
 'n_candidates': 64,
 'files': {'candidate_specs': 'mongodb_candidate_specs_by_candidate_id.json',
  'candidate_specs_overview': 'mongodb_candidate_specs_overview.csv',
  'candidate_validation': 'mongodb_candidate_validation.csv',
  'candidate_summary_by_g_class': 'mongodb_candidate_summary_by_g_class.csv',
  'candidate_summary_by_query': 'mongodb_candidate_summary_by_query.csv',
  'pattern_templates': 'mongodb_pattern_templates_g0_g9.csv'},
 'notes': ['Candidates are generated from the official-based LDBC SNB Interactive workload subset.',
  'These specs are not benchmark results; they define MongoDB materialization candidates.',
  'Benchmark execution will be handled by a separate runner script.']}

In [225]:
# =========================================================
# BLOCK 12I — Save checkpoint after MongoDB candidate generation
# =========================================================

from pathlib import Path
import json
import pandas as pd

PROJECT_DIR = BASE_DIR.parent.parent

CHECKPOINT_DIR = PROJECT_DIR / "benchmark_artifacts_dir" / "ldbc_snb_checkpoints"
CHECKPOINT_DATAFRAMES_DIR = CHECKPOINT_DIR / "dataframes"
CHECKPOINT_METADATA_DIR = CHECKPOINT_DIR / "metadata"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_METADATA_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = CHECKPOINT_DIR / "checkpoint_manifest.json"
metadata_path = CHECKPOINT_METADATA_DIR / "checkpoint_metadata.json"

if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        checkpoint_manifest = json.load(f)
else:
    checkpoint_manifest = {
        "dataframes": {},
        "metadata": str(metadata_path.relative_to(CHECKPOINT_DIR)),
    }

if metadata_path.exists():
    with open(metadata_path, "r", encoding="utf-8") as f:
        checkpoint_metadata = json.load(f)
else:
    checkpoint_metadata = {}

BLOCK_12_CHECKPOINT_DATAFRAMES = [
    "mongodb_pattern_templates_df",
    "snb_mongodb_candidate_specs_df",
    "snb_mongodb_candidate_validation_df",
    "snb_candidate_summary_by_g_df",
    "snb_candidate_summary_by_query_df",
]

saved_block_12_dataframes = []

for var_name in BLOCK_12_CHECKPOINT_DATAFRAMES:
    obj = globals().get(var_name)

    if isinstance(obj, pd.DataFrame):
        out_path = CHECKPOINT_DATAFRAMES_DIR / f"{var_name}.json"

        obj.to_json(
            out_path,
            orient="records",
            indent=2,
            force_ascii=False
        )

        checkpoint_manifest["dataframes"][var_name] = str(
            out_path.relative_to(CHECKPOINT_DIR)
        )

        saved_block_12_dataframes.append(var_name)

checkpoint_metadata.update({
    "BASE_DIR": str(BASE_DIR),
    "MONGODB_CANDIDATES_DIR": str(MONGODB_CANDIDATES_DIR),
    "MONGODB_PATTERN_TEMPLATES": MONGODB_PATTERN_TEMPLATES,
    "candidate_manifest": candidate_manifest,
})

if "PERSISTENT_DUCKDB_PATH" in globals():
    checkpoint_metadata["PERSISTENT_DUCKDB_PATH"] = str(PERSISTENT_DUCKDB_PATH)

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(checkpoint_metadata, f, indent=2, ensure_ascii=False)

checkpoint_manifest["metadata"] = str(metadata_path.relative_to(CHECKPOINT_DIR))

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(checkpoint_manifest, f, indent=2, ensure_ascii=False)

print("Block 12 checkpoint saved to:")
print(CHECKPOINT_DIR)

print("\nSaved Block 12 DataFrames:")
for name in saved_block_12_dataframes:
    print(" -", name)

Block 12 checkpoint saved to:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_checkpoints

Saved Block 12 DataFrames:
 - mongodb_pattern_templates_df
 - snb_mongodb_candidate_specs_df
 - snb_mongodb_candidate_validation_df
 - snb_candidate_summary_by_g_df
 - snb_candidate_summary_by_query_df


In [226]:
# =========================================================
# CHECK — Verify Block 12 checkpoint
# =========================================================

from pathlib import Path
import json

PROJECT_DIR = Path(
    "/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB"
)

CHECKPOINT_DIR = PROJECT_DIR / "benchmark_artifacts_dir" / "ldbc_snb_checkpoints"
manifest_path = CHECKPOINT_DIR / "checkpoint_manifest.json"

print("Checkpoint manifest exists:", manifest_path.exists())

if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        checkpoint_manifest = json.load(f)

    expected_block_12 = [
        "mongodb_pattern_templates_df",
        "snb_mongodb_candidate_specs_df",
        "snb_mongodb_candidate_validation_df",
        "snb_candidate_summary_by_g_df",
        "snb_candidate_summary_by_query_df",
    ]

    print("\nBlock 12 DataFrames in checkpoint:")
    for name in expected_block_12:
        exists = name in checkpoint_manifest.get("dataframes", {})
        print(name, "=", "OK" if exists else "MISSING")

Checkpoint manifest exists: True

Block 12 DataFrames in checkpoint:
mongodb_pattern_templates_df = OK
snb_mongodb_candidate_specs_df = OK
snb_mongodb_candidate_validation_df = OK
snb_candidate_summary_by_g_df = OK
snb_candidate_summary_by_query_df = OK


In [227]:
# =========================================================
# BLOCK 13A — Validate inputs for benchmark execution plan
# =========================================================
#
# Goal:
# - Confirm that Block 12 outputs exist.
# - Prepare candidate specs for benchmark plan selection.
# =========================================================

import pandas as pd
import json
from pathlib import Path

required_before_block_13 = [
    "snb_mongodb_candidate_specs_df",
    "snb_mongodb_candidate_specs_by_candidate_id",
    "snb_mongodb_candidate_validation_df",
    "snb_final_analytical_matrix_df",
    "snb_activation_matrix_df",
]

print("Required objects before Block 13:")

for name in required_before_block_13:
    print(name, "=", "OK" if name in globals() else "MISSING")

print("\nCandidate validation problems:")

display(
    snb_mongodb_candidate_validation_df[
        snb_mongodb_candidate_validation_df["status"] != "OK"
    ]
)

print("\nNumber of candidates:", len(snb_mongodb_candidate_specs_df))
print("Number of unique candidates:", snb_mongodb_candidate_specs_df["candidate_id"].nunique())

Required objects before Block 13:
snb_mongodb_candidate_specs_df = OK
snb_mongodb_candidate_specs_by_candidate_id = OK
snb_mongodb_candidate_validation_df = OK
snb_final_analytical_matrix_df = OK
snb_activation_matrix_df = OK

Candidate validation problems:


,check,value,status,details



Number of candidates: 64
Number of unique candidates: 64


In [228]:
# =========================================================
# BLOCK 13B — Define benchmark group assignment rules
# =========================================================
#
# Goal:
# - Assign each candidate to one benchmark group.
#
# Groups:
# - primary:
#     Main candidate selected by activation logic for the query.
#
# - secondary_affected:
#     Alternative candidate activated because the query also touches
#     another semantic family or pattern.
#
# - control:
#     Baseline candidate used for comparison.
# =========================================================

def assign_benchmark_group(row):
    """
    Assign benchmark group from activation strength and G class.
    """

    activation_strength = row.get("activation_strength")
    g_class = row.get("g_class")

    if activation_strength == "control":
        return "control"

    if activation_strength == "primary":
        return "primary"

    if activation_strength == "secondary":
        return "secondary_affected"

    # Fallback: use baseline G classes as controls when activation is unclear.
    if g_class in {"G0", "G1", "G4", "G7"}:
        return "control"

    return "secondary_affected"


def benchmark_group_priority(group):
    """
    Lower number means higher priority in execution ordering.
    """
    priority = {
        "primary": 1,
        "secondary_affected": 2,
        "control": 3,
    }
    return priority.get(group, 99)


def run_priority_by_query_group(query_group):
    """
    Execution priority by workload family.
    """
    priority = {
        "short_read": 1,
        "complex_read": 2,
        "insert": 3,
    }
    return priority.get(query_group, 99)

In [229]:
# =========================================================
# BLOCK 13C — Build full benchmark execution plan
# =========================================================
#
# Goal:
# - Build one execution-plan row per candidate.
# - This plan will be used by the benchmark runner.
# =========================================================

benchmark_plan_rows = []

for _, row in snb_mongodb_candidate_specs_df.iterrows():
    candidate_id = row["candidate_id"]
    spec = snb_mongodb_candidate_specs_by_candidate_id[candidate_id]

    benchmark_group = assign_benchmark_group(row)

    materialization = spec["materialization"]
    metrics = spec["metrics"]

    benchmark_plan_rows.append({
        "candidate_id": candidate_id,

        "dataset": spec["dataset"],
        "scale_label": spec["scale_label"],
        "workload": spec["workload"],

        "query_name": spec["query_name"],
        "official_id": spec["official_id"],
        "official_title": spec["official_title"],
        "query_group": spec["query_group"],
        "operation_type": spec["operation_type"],

        "benchmark_group": benchmark_group,
        "benchmark_group_priority": benchmark_group_priority(benchmark_group),
        "query_group_priority": run_priority_by_query_group(spec["query_group"]),

        "activation_strength": spec["activation_strength"],
        "activation_reason": spec["activation_reason"],

        "root_entity": materialization["root_entity"],
        "root_collection": materialization["root_collection"],
        "root_pk": materialization["root_pk"],

        "g_class": spec["g_class"],
        "g_family": spec["g_family"],
        "g_role": spec["g_role"],
        "g_label": spec["g_label"],

        "mongodb_pattern": materialization["mongodb_pattern"],
        "document_strategy": materialization["document_strategy"],

        "embedded_entities": materialization["embedded_entities"],
        "referenced_entities": materialization["referenced_entities"],
        "summarized_entities": materialization["summarized_entities"],
        "edge_collections": materialization["edge_collections"],
        "reverse_indexes": materialization["reverse_indexes"],

        "Rc_weighted": metrics["Rc_weighted"],
        "D": metrics["D"],
        "Re": metrics["Re"],
        "DeltaRratio": metrics["DeltaRratio"],
        "dominant_semantic_type": metrics["dominant_semantic_type"],
        "update_volatility_max": metrics["update_volatility_max"],
        "observed_sharedness_max": metrics["observed_sharedness_max"],

        "include_in_full_benchmark": True,
    })

snb_benchmark_execution_plan_df = pd.DataFrame(benchmark_plan_rows)

snb_benchmark_execution_plan_df = snb_benchmark_execution_plan_df.sort_values(
    by=[
        "query_group_priority",
        "official_id",
        "benchmark_group_priority",
        "g_class",
    ]
).reset_index(drop=True)

snb_benchmark_execution_plan_df

,candidate_id,dataset,scale_label,workload,query_name,official_id,official_title,query_group,operation_type,benchmark_group,benchmark_group_priority,query_group_priority,activation_strength,activation_reason,root_entity,root_collection,root_pk,g_class,g_family,g_role,g_label,mongodb_pattern,document_strategy,embedded_entities,referenced_entities,summarized_entities,edge_collections,reverse_indexes,Rc_weighted,D,Re,DeltaRratio,dominant_semantic_type,update_volatility_max,observed_sharedness_max,include_in_full_benchmark
0,ldbc_snb_is1_g0_edd90ee3,ldbc_snb,sf0.1,LDBC SNB Interactive v1 official-based subset,IS1_ProfileOfPerson,IS1,Profile of a person,short_read,read,primary,1,1,primary,Association-family baseline root document.,Person,persons,person_id,G0,association,baseline,Root document with referenced associations,root_with_references,reference,[],[Place],[],[person_is_located_in_place],[],1,1,0,1.0,association,1.0,1.000000,True
1,ldbc_snb_is1_g3_6c817501,ldbc_snb,sf0.1,LDBC SNB Interactive v1 official-based subset,IS1_ProfileOfPerson,IS1,Profile of a person,short_read,read,primary,1,1,primary,"Association path has high sharedness, high volatility, deep traversal, or residual traversal.",Person,persons,person_id,G3,association,reference_or_summary_candidate,Root document with references or denormalized summaries,root_with_references_or_summaries,reference_summary,[],[Place],[Place],[person_is_located_in_place],[],1,1,0,1.0,association,1.0,1.000000,True
2,ldbc_snb_is2_g0_1d094f1c,ldbc_snb,sf0.1,LDBC SNB Interactive v1 official-based subset,IS2_RecentMessagesOfPerson,IS2,Recent messages of a person,short_read,read,secondary_affected,2,1,secondary,Association-family baseline root document.,Person,persons,person_id,G0,association,baseline,Root document with referenced associations,root_with_references,reference,[],"[Comment, Post]",[],"[comment_has_creator_person, comment_reply_of_comment, comment_reply_of_post, post_has_creator_person]",[],4,2,2,0.5,mixed,1.0,0.106448,True
3,ldbc_snb_is2_g3_b993236d,ldbc_snb,sf0.1,LDBC SNB Interactive v1 official-based subset,IS2_RecentMessagesOfPerson,IS2,Recent messages of a person,short_read,read,secondary_affected,2,1,secondary,"Association path has high sharedness, high volatility, deep traversal, or residual traversal.",Person,persons,person_id,G3,association,reference_or_summary_candidate,Root document with references or denormalized summaries,root_with_references_or_summaries,reference_summary,[],"[Comment, Post]","[Comment, Post]","[comment_has_creator_person, comment_reply_of_comment, comment_reply_of_post, post_has_creator_person]",[],4,2,2,0.5,mixed,1.0,0.106448,True
4,ldbc_snb_is2_g4_9622e695,ldbc_snb,sf0.1,LDBC SNB Interactive v1 official-based subset,IS2_RecentMessagesOfPerson,IS2,Recent messages of a person,short_read,read,secondary_affected,2,1,secondary,Associative-family baseline using explicit relationship documents.,Person,persons,person_id,G4,associative,baseline,Associative relationship as explicit document,explicit_edge_collection,edge_document,[],"[Comment, Post]",[],"[comment_has_creator_person, comment_reply_of_comment, comment_reply_of_post, post_has_creator_person]",[],4,2,2,0.5,mixed,1.0,0.106448,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,ldbc_snb_ins7_g3_e7300dd8,ldbc_snb,sf0.1,LDBC SNB Interactive v1 official-based subset,INS7_AddComment,INS7,Add comment,insert,insert,primary,1,3,primary,"Association path has high sharedness, high volatility, deep traversal, or residual traversal.",Comment,comments,comment_id,G3,association,reference_or_summary_candidate,Root document with references or denormalized summaries,root_with_references_or_summaries,reference_summary,[],"[Person, Place, Post, Tag]",[],"[comment_has_creator_person, comment_has_tag, comment_is_located_in_place, comment_reply_of_comment, comment_reply_of_post]",[],5,1,4,0.2,association,1.0,1.000

In [230]:
# =========================================================
# BLOCK 13D — Build smoke benchmark execution plan
# =========================================================
#
# Goal:
# - Create a smaller benchmark plan for first execution.
# - This avoids running all 64 candidates before validating the runner.
# =========================================================

SMOKE_OFFICIAL_IDS = [
    # Short reads
    "IS1",
    "IS3",
    "IS4",
    "IS6",

    # Complex reads
    "IC2",
    "IC5",

    # Inserts
    "INS2",
    "INS6",
]

smoke_df = snb_benchmark_execution_plan_df[
    snb_benchmark_execution_plan_df["official_id"].isin(SMOKE_OFFICIAL_IDS)
].copy()

# Keep all candidates for selected smoke queries.
snb_smoke_benchmark_execution_plan_df = smoke_df.sort_values(
    by=[
        "query_group_priority",
        "official_id",
        "benchmark_group_priority",
        "g_class",
    ]
).reset_index(drop=True)

snb_smoke_benchmark_execution_plan_df

,candidate_id,dataset,scale_label,workload,query_name,official_id,official_title,query_group,operation_type,benchmark_group,benchmark_group_priority,query_group_priority,activation_strength,activation_reason,root_entity,root_collection,root_pk,g_class,g_family,g_role,g_label,mongodb_pattern,document_strategy,embedded_entities,referenced_entities,summarized_entities,edge_collections,reverse_indexes,Rc_weighted,D,Re,DeltaRratio,dominant_semantic_type,update_volatility_max,observed_sharedness_max,include_in_full_benchmark
0,ldbc_snb_is1_g0_edd90ee3,ldbc_snb,sf0.1,LDBC SNB Interactive v1 official-based subset,IS1_ProfileOfPerson,IS1,Profile of a person,short_read,read,primary,1,1,primary,Association-family baseline root document.,Person,persons,person_id,G0,association,baseline,Root document with referenced associations,root_with_references,reference,[],[Place],[],[person_is_located_in_place],[],1,1,0,1.000000,association,1.000000,1.000000,True
1,ldbc_snb_is1_g3_6c817501,ldbc_snb,sf0.1,LDBC SNB Interactive v1 official-based subset,IS1_ProfileOfPerson,IS1,Profile of a person,short_read,read,primary,1,1,primary,"Association path has high sharedness, high volatility, deep traversal, or residual traversal.",Person,persons,person_id,G3,association,reference_or_summary_candidate,Root document with references or denormalized summaries,root_with_references_or_summaries,reference_summary,[],[Place],[Place],[person_is_located_in_place],[],1,1,0,1.000000,association,1.000000,1.000000,True
2,ldbc_snb_is3_g0_fc24df68,ldbc_snb,sf0.1,LDBC SNB Interactive v1 official-based subset,IS3_FriendsOfPerson,IS3,Friends of a person,short_read,read,primary,1,1,primary,Association-family baseline root document.,Person,persons,person_id,G0,association,baseline,Root document with referenced associations,root_with_references,reference,[],[],[],[person_knows_person],[],1,1,0,1.000000,association,1.000000,0.106448,True
3,ldbc_snb_is3_g3_ebc00ba6,ldbc_snb,sf0.1,LDBC SNB Interactive v1 official-based subset,IS3_FriendsOfPerson,IS3,Friends of a person,short_read,read,primary,1,1,primary,"Association path has high sharedness, high volatility, deep traversal, or residual traversal.",Person,persons,person_id,G3,association,reference_or_summary_candidate,Root document with references or denormalized summaries,root_with_references_or_summaries,reference_summary,[],[],[],[person_knows_person],[],1,1,0,1.000000,association,1.000000,0.106448,True
4,ldbc_snb_is4_g1_c923c633,ldbc_snb,sf0.1,LDBC SNB Interactive v1 official-based subset,IS4_ContentOfMessage,IS4,Content of a message,short_read,read,primary,1,1,primary,Lookup query with no explicit relationship traversal.,Post,posts,post_id,G1,lookup,baseline,Single-entity lookup document,single_entity_lookup,single_collection,[],[],[],[],[],0,0,0,0.000000,lookup,0.431184,0.016512,True
5,ldbc_snb_is4_g0_103e0c60,ldbc_snb,sf0.1,LDBC SNB Interactive v1 official-based subset,IS4_ContentOfMessage,IS4,Content of a message,short_read,read,control,3,1,control,Control baseline for root document comparison.,Post,posts,post_id,G0,association,baseline,Root document with referenced associations,root_with_references,reference,[],[Comment],[],[],[],0,0,0,0.000000,lookup,0.431184,0.016512,True
6,ldbc_snb_is6_g7_0aecbe5c,ldbc_snb,sf0.1,LDBC SNB Interactive v1 official-based subset,IS6_ForumOfMessage,IS6,Forum of a message,short_read,read,primary,1,1,primary,Containment-family baseline.,Post,posts,post_id,G7,containment,baseline,Containment baseline,containment_baseline,containment_reference,[],"[Comment, Forum, Person]",[],"[comment_reply_of_comment, comment_reply_of_post, forum_container_of_post, forum_has_moderator_person]",[],4,3,1,0.750000,containment,1.000000,0.106448,True
7,ldbc_snb_is6_g9_c7584187,ldbc_snb,sf0.1,LDBC SNB Interactive v1 official-based subset,IS6_ForumOfMessage,IS6,Forum of a message,short_read,read,primary,1,1,primary,"Containment path has high sharedness, high volatility, or high residual traversal.",Post,posts,post_id,G

In [231]:
# =========================================================
# BLOCK 13E — Summarize benchmark execution plans
# =========================================================

snb_benchmark_plan_summary_by_group_df = (
    snb_benchmark_execution_plan_df
    .groupby(["benchmark_group", "query_group"])
    .agg(
        n_candidates=("candidate_id", "count"),
        n_queries=("query_name", "nunique"),
        g_classes=("g_class", lambda x: sorted(set(x))),
    )
    .reset_index()
    .sort_values(["benchmark_group", "query_group"])
)

snb_benchmark_plan_summary_by_g_df = (
    snb_benchmark_execution_plan_df
    .groupby(["g_class", "benchmark_group", "mongodb_pattern"])
    .agg(
        n_candidates=("candidate_id", "count"),
        queries=("query_name", lambda x: sorted(set(x))),
    )
    .reset_index()
    .sort_values(["g_class", "benchmark_group"])
)

snb_smoke_plan_summary_df = (
    snb_smoke_benchmark_execution_plan_df
    .groupby(["benchmark_group", "query_group"])
    .agg(
        n_candidates=("candidate_id", "count"),
        n_queries=("query_name", "nunique"),
        g_classes=("g_class", lambda x: sorted(set(x))),
    )
    .reset_index()
    .sort_values(["benchmark_group", "query_group"])
)

print("Full plan candidates:", len(snb_benchmark_execution_plan_df))
print("Smoke plan candidates:", len(snb_smoke_benchmark_execution_plan_df))

display(snb_benchmark_plan_summary_by_group_df)
display(snb_benchmark_plan_summary_by_g_df)
display(snb_smoke_plan_summary_df)

Full plan candidates: 64
Smoke plan candidates: 24


,benchmark_group,query_group,n_candidates,n_queries,g_classes
0,control,short_read,1,1,[G0]
1,primary,complex_read,14,7,"[G0, G3]"
2,primary,insert,16,8,"[G0, G3, G4, G6]"
3,primary,short_read,11,6,"[G0, G1, G3, G7, G9]"
4,secondary_affected,complex_read,8,3,"[G4, G6, G7, G9]"
5,secondary_affected,insert,4,2,"[G7, G9]"
6,secondary_affected,short_read,10,3,"[G0, G3, G4, G6, G7, G9]"


,g_class,benchmark_group,mongodb_pattern,n_candidates,queries
0,G0,control,root_with_references,1,[IS4_ContentOfMessage]
1,G0,primary,root_with_references,16,"[IC1_TransitiveFriendsWithName, IC2_RecentMessagesByFriends, IC3_FriendsAndFriendsOfFriendsInCountries, IC4_NewTopics, IC5_NewGroups, IC6_TagCoOccurrence, IC7_RecentLikers, INS1_AddPerson, INS4_AddForum, INS6_AddPost, INS7_AddComment, INS8_AddFriendship, IS1_ProfileOfPerson, IS3_FriendsOfPerson, IS5_CreatorOfMessage, IS7_RepliesOfMessage]"
2,G0,secondary_affected,root_with_references,2,"[IS2_RecentMessagesOfPerson, IS6_ForumOfMessage]"
3,G1,primary,single_entity_lookup,1,[IS4_ContentOfMessage]
4,G3,primary,root_with_references_or_summaries,16,"[IC1_TransitiveFriendsWithName, IC2_RecentMessagesByFriends, IC3_FriendsAndFriendsOfFriendsInCountries, IC4_NewTopics, IC5_NewGroups, IC6_TagCoOccurrence, IC7_RecentLikers, INS1_AddPerson, INS4_AddForum, INS6_AddPost, INS7_AddComment, INS8_AddFriendship, IS1_ProfileOfPerson, IS3_FriendsOfPerson, IS5_CreatorOfMessage, IS7_RepliesOfMessage]"
5,G3,secondary_affected,root_with_references_or_summaries,2,"[IS2_RecentMessagesOfPerson, IS6_ForumOfMessage]"
6,G4,primary,explicit_edge_collection,3,"[INS2_AddLikeToPost, INS3_AddLikeToComment, INS5_AddForumMembership]"
7,G4,secondary_affected,explicit_edge_collection,3,"[IC5_NewGroups, IC7_RecentLikers, IS2_RecentMessagesOfPerson]"
8,G6,primary,referenced_or_reverse_indexed_edges,3,"[INS2_AddLikeToPost, INS3_AddLikeToComment, INS5_AddForumMembership]"
9,G6,secondary_affected,referenced_or_reverse_indexed_edges,3,"[IC5_NewGroups, IC7_RecentLikers, IS2_RecentMessagesOfPerson]"


,benchmark_group,query_group,n_candidates,n_queries,g_classes
0,control,short_read,1,1,[G0]
1,primary,complex_read,4,2,"[G0, G3]"
2,primary,insert,4,2,"[G0, G3, G4, G6]"
3,primary,short_read,7,4,"[G0, G1, G3, G7, G9]"
4,secondary_affected,complex_read,4,1,"[G4, G6, G7, G9]"
5,secondary_affected,insert,2,1,"[G7, G9]"
6,secondary_affected,short_read,2,1,"[G0, G3]"


In [232]:
# =========================================================
# BLOCK 13F — Validate benchmark execution plan
# =========================================================
#
# Goal:
# - Ensure every candidate appears once.
# - Ensure benchmark groups are valid.
# - Ensure smoke plan is non-empty and covers all benchmark groups.
# =========================================================

valid_benchmark_groups = {
    "primary",
    "secondary_affected",
    "control",
}

plan_validation_rows = []

# Full plan count
n_candidates = len(snb_mongodb_candidate_specs_df)
n_plan_rows = len(snb_benchmark_execution_plan_df)
n_unique_plan_candidates = snb_benchmark_execution_plan_df["candidate_id"].nunique()

plan_validation_rows.append({
    "check": "full_plan_count_matches_candidates",
    "value": f"{n_plan_rows} / {n_candidates}",
    "status": "OK" if n_plan_rows == n_candidates else "ERROR",
})

plan_validation_rows.append({
    "check": "full_plan_candidate_ids_unique",
    "value": f"{n_unique_plan_candidates} / {n_plan_rows}",
    "status": "OK" if n_unique_plan_candidates == n_plan_rows else "ERROR",
})

# Groups valid
invalid_groups = sorted(
    set(snb_benchmark_execution_plan_df["benchmark_group"])
    - valid_benchmark_groups
)

plan_validation_rows.append({
    "check": "valid_benchmark_groups",
    "value": len(invalid_groups),
    "details": invalid_groups,
    "status": "OK" if len(invalid_groups) == 0 else "ERROR",
})

# Smoke plan non-empty
plan_validation_rows.append({
    "check": "smoke_plan_non_empty",
    "value": len(snb_smoke_benchmark_execution_plan_df),
    "status": "OK" if len(snb_smoke_benchmark_execution_plan_df) > 0 else "ERROR",
})

# Smoke group coverage
smoke_groups = set(snb_smoke_benchmark_execution_plan_df["benchmark_group"])
missing_smoke_groups = sorted(valid_benchmark_groups - smoke_groups)

plan_validation_rows.append({
    "check": "smoke_plan_covers_all_benchmark_groups",
    "value": len(missing_smoke_groups),
    "details": missing_smoke_groups,
    "status": "OK" if len(missing_smoke_groups) == 0 else "WARNING",
})

snb_benchmark_plan_validation_df = pd.DataFrame(plan_validation_rows)

snb_benchmark_plan_validation_df

,check,value,status,details
0,full_plan_count_matches_candidates,64 / 64,OK,NaN
1,full_plan_candidate_ids_unique,64 / 64,OK,NaN
2,valid_benchmark_groups,0,OK,[]
3,smoke_plan_non_empty,24,OK,NaN
4,smoke_plan_covers_all_benchmark_groups,0,OK,[]


In [233]:
snb_benchmark_plan_validation_df[
    snb_benchmark_plan_validation_df["status"] == "ERROR"
]

,check,value,status,details


In [234]:
# =========================================================
# BLOCK 13G — Save benchmark execution plan artifacts
# =========================================================

BENCHMARK_PLAN_DIR = (
    BASE_DIR.parent.parent
    / "benchmark_artifacts_dir"
    / "ldbc_snb_mongo_configurations"
)

BENCHMARK_PLAN_DIR.mkdir(parents=True, exist_ok=True)

# Full plan
snb_benchmark_execution_plan_df.to_csv(
    BENCHMARK_PLAN_DIR / "benchmark_execution_plan.csv",
    index=False
)

# Smoke plan
snb_smoke_benchmark_execution_plan_df.to_csv(
    BENCHMARK_PLAN_DIR / "benchmark_execution_plan_smoke.csv",
    index=False
)

# Summaries
snb_benchmark_plan_summary_by_group_df.to_csv(
    BENCHMARK_PLAN_DIR / "benchmark_plan_summary_by_group.csv",
    index=False
)

snb_benchmark_plan_summary_by_g_df.to_csv(
    BENCHMARK_PLAN_DIR / "benchmark_plan_summary_by_g_class.csv",
    index=False
)

snb_smoke_plan_summary_df.to_csv(
    BENCHMARK_PLAN_DIR / "benchmark_plan_smoke_summary.csv",
    index=False
)

snb_benchmark_plan_validation_df.to_csv(
    BENCHMARK_PLAN_DIR / "benchmark_plan_validation.csv",
    index=False
)

print("Saved benchmark execution plan artifacts to:")
print(BENCHMARK_PLAN_DIR)

print("\nSaved files:")
for path in sorted(BENCHMARK_PLAN_DIR.glob("*")):
    print(" -", path.name)

Saved benchmark execution plan artifacts to:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_mongo_configurations

Saved files:
 - benchmark_execution_plan.csv
 - benchmark_execution_plan_smoke.csv
 - benchmark_manifest.json
 - benchmark_plan_smoke_summary.csv
 - benchmark_plan_summary_by_g_class.csv
 - benchmark_plan_summary_by_group.csv
 - benchmark_plan_validation.csv
 - mongodb_candidate_specs_by_candidate_id.json
 - mongodb_candidate_specs_overview.csv
 - mongodb_candidate_summary_by_g_class.csv
 - mongodb_candidate_summary_by_query.csv
 - mongodb_candidate_validation.csv
 - mongodb_pattern_templates_g0_g9.csv


In [235]:
# =========================================================
# BLOCK 13H — Update benchmark manifest with execution plan
# =========================================================

from datetime import datetime

manifest_path = BENCHMARK_PLAN_DIR / "benchmark_manifest.json"

if manifest_path.exists():
    with open(manifest_path, "r", encoding="utf-8") as f:
        benchmark_manifest = json.load(f)
else:
    benchmark_manifest = {
        "dataset": "ldbc_snb",
        "scale_label": "sf0.1",
        "workload": "LDBC SNB Interactive v1 official-based subset",
    }

benchmark_manifest.update({
    "updated_at": datetime.utcnow().isoformat() + "Z",
    "n_benchmark_plan_rows": int(len(snb_benchmark_execution_plan_df)),
    "n_smoke_plan_rows": int(len(snb_smoke_benchmark_execution_plan_df)),
    "benchmark_groups": sorted(
        snb_benchmark_execution_plan_df["benchmark_group"].unique().tolist()
    ),
    "files": {
        **benchmark_manifest.get("files", {}),
        "benchmark_execution_plan": "benchmark_execution_plan.csv",
        "benchmark_execution_plan_smoke": "benchmark_execution_plan_smoke.csv",
        "benchmark_plan_summary_by_group": "benchmark_plan_summary_by_group.csv",
        "benchmark_plan_summary_by_g_class": "benchmark_plan_summary_by_g_class.csv",
        "benchmark_plan_smoke_summary": "benchmark_plan_smoke_summary.csv",
        "benchmark_plan_validation": "benchmark_plan_validation.csv",
    },
})

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(benchmark_manifest, f, indent=2, ensure_ascii=False)

benchmark_manifest

{'dataset': 'ldbc_snb',
 'scale_label': 'sf0.1',
 'workload': 'LDBC SNB Interactive v1 official-based subset',
 'created_at': '2026-05-08T18:01:11.375015Z',
 'n_queries': 22,
 'n_activations': 64,
 'n_candidates': 64,
 'files': {'candidate_specs': 'mongodb_candidate_specs_by_candidate_id.json',
  'candidate_specs_overview': 'mongodb_candidate_specs_overview.csv',
  'candidate_validation': 'mongodb_candidate_validation.csv',
  'candidate_summary_by_g_class': 'mongodb_candidate_summary_by_g_class.csv',
  'candidate_summary_by_query': 'mongodb_candidate_summary_by_query.csv',
  'pattern_templates': 'mongodb_pattern_templates_g0_g9.csv',
  'benchmark_execution_plan': 'benchmark_execution_plan.csv',
  'benchmark_execution_plan_smoke': 'benchmark_execution_plan_smoke.csv',
  'benchmark_plan_summary_by_group': 'benchmark_plan_summary_by_group.csv',
  'benchmark_plan_summary_by_g_class': 'benchmark_plan_summary_by_g_class.csv',
  'benchmark_plan_smoke_summary': 'benchmark_plan_smoke_summary.cs

In [236]:
# =========================================================
# BLOCK 13I — Save checkpoint after benchmark plan selection
# =========================================================

from pathlib import Path
import json
import pandas as pd

PROJECT_DIR = BASE_DIR.parent.parent

CHECKPOINT_DIR = PROJECT_DIR / "benchmark_artifacts_dir" / "ldbc_snb_checkpoints"
CHECKPOINT_DATAFRAMES_DIR = CHECKPOINT_DIR / "dataframes"
CHECKPOINT_METADATA_DIR = CHECKPOINT_DIR / "metadata"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_METADATA_DIR.mkdir(parents=True, exist_ok=True)

manifest_path_checkpoint = CHECKPOINT_DIR / "checkpoint_manifest.json"
metadata_path = CHECKPOINT_METADATA_DIR / "checkpoint_metadata.json"

if manifest_path_checkpoint.exists():
    with open(manifest_path_checkpoint, "r", encoding="utf-8") as f:
        checkpoint_manifest = json.load(f)
else:
    checkpoint_manifest = {
        "dataframes": {},
        "metadata": str(metadata_path.relative_to(CHECKPOINT_DIR)),
    }

if metadata_path.exists():
    with open(metadata_path, "r", encoding="utf-8") as f:
        checkpoint_metadata = json.load(f)
else:
    checkpoint_metadata = {}

BLOCK_13_CHECKPOINT_DATAFRAMES = [
    "snb_benchmark_execution_plan_df",
    "snb_smoke_benchmark_execution_plan_df",
    "snb_benchmark_plan_summary_by_group_df",
    "snb_benchmark_plan_summary_by_g_df",
    "snb_smoke_plan_summary_df",
    "snb_benchmark_plan_validation_df",
]

saved_block_13_dataframes = []

for var_name in BLOCK_13_CHECKPOINT_DATAFRAMES:
    obj = globals().get(var_name)

    if isinstance(obj, pd.DataFrame):
        out_path = CHECKPOINT_DATAFRAMES_DIR / f"{var_name}.json"

        obj.to_json(
            out_path,
            orient="records",
            indent=2,
            force_ascii=False
        )

        checkpoint_manifest["dataframes"][var_name] = str(
            out_path.relative_to(CHECKPOINT_DIR)
        )

        saved_block_13_dataframes.append(var_name)

checkpoint_metadata.update({
    "BASE_DIR": str(BASE_DIR),
    "BENCHMARK_PLAN_DIR": str(BENCHMARK_PLAN_DIR),
    "benchmark_manifest": benchmark_manifest,
})

if "PERSISTENT_DUCKDB_PATH" in globals():
    checkpoint_metadata["PERSISTENT_DUCKDB_PATH"] = str(PERSISTENT_DUCKDB_PATH)

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(checkpoint_metadata, f, indent=2, ensure_ascii=False)

checkpoint_manifest["metadata"] = str(metadata_path.relative_to(CHECKPOINT_DIR))

with open(manifest_path_checkpoint, "w", encoding="utf-8") as f:
    json.dump(checkpoint_manifest, f, indent=2, ensure_ascii=False)

print("Block 13 checkpoint saved to:")
print(CHECKPOINT_DIR)

print("\nSaved Block 13 DataFrames:")
for name in saved_block_13_dataframes:
    print(" -", name)

Block 13 checkpoint saved to:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_checkpoints

Saved Block 13 DataFrames:
 - snb_benchmark_execution_plan_df
 - snb_smoke_benchmark_execution_plan_df
 - snb_benchmark_plan_summary_by_group_df
 - snb_benchmark_plan_summary_by_g_df
 - snb_smoke_plan_summary_df
 - snb_benchmark_plan_validation_df


In [237]:
# =========================================================
# BLOCK 14A — Define final benchmark bundle paths
# =========================================================
#
# Goal:
# - Define source artifact folders.
# - Define final export bundle folder.
# - This bundle can be copied to the server where MongoDB benchmark runs.
# =========================================================

from pathlib import Path
import shutil
import json
import pandas as pd
from datetime import datetime

PROJECT_DIR = Path(
    "/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB"
)

ARTIFACTS_ROOT = PROJECT_DIR / "benchmark_artifacts_dir"

MONGO_CONFIG_DIR = ARTIFACTS_ROOT / "ldbc_snb_mongo_configurations"
FINAL_MATRIX_DIR = ARTIFACTS_ROOT / "ldbc_snb_final_analytical_matrix_official"
ACTIVATION_DIR = ARTIFACTS_ROOT / "ldbc_snb_activation_official"
VOLATILITY_DIR = ARTIFACTS_ROOT / "ldbc_snb_update_volatility_official"
SHAREDNESS_DIR = ARTIFACTS_ROOT / "ldbc_snb_observed_sharedness_official"
STRUCTURAL_DIR = ARTIFACTS_ROOT / "ldbc_snb_structural_reduction_official"

EXPORT_ROOT = PROJECT_DIR / "exported_benchmark_bundles"
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

BUNDLE_NAME = "ldbc_snb_sf0_1_mongo_benchmark_bundle"
BUNDLE_DIR = EXPORT_ROOT / BUNDLE_NAME

if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)

BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

print("Bundle directory:")
print(BUNDLE_DIR)

Bundle directory:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/exported_benchmark_bundles/ldbc_snb_sf0_1_mongo_benchmark_bundle


In [238]:
# =========================================================
# BLOCK 14B — Copy benchmark artifacts into final bundle
# =========================================================
#
# Goal:
# - Copy all required methodology and benchmark artifacts.
# =========================================================

bundle_subdirs = {
    "mongo_configurations": MONGO_CONFIG_DIR,
    "final_analytical_matrix": FINAL_MATRIX_DIR,
    "activation": ACTIVATION_DIR,
    "volatility": VOLATILITY_DIR,
    "sharedness": SHAREDNESS_DIR,
    "structural_reduction": STRUCTURAL_DIR,
}

copy_report_rows = []

for target_name, source_dir in bundle_subdirs.items():
    target_dir = BUNDLE_DIR / target_name

    if source_dir.exists():
        shutil.copytree(source_dir, target_dir, dirs_exist_ok=True)

        copied_files = sorted([
            str(p.relative_to(target_dir))
            for p in target_dir.rglob("*")
            if p.is_file()
        ])

        copy_report_rows.append({
            "bundle_section": target_name,
            "source_dir": str(source_dir),
            "target_dir": str(target_dir),
            "n_files": len(copied_files),
            "status": "OK",
        })
    else:
        copy_report_rows.append({
            "bundle_section": target_name,
            "source_dir": str(source_dir),
            "target_dir": str(target_dir),
            "n_files": 0,
            "status": "MISSING_SOURCE_DIR",
        })

bundle_copy_report_df = pd.DataFrame(copy_report_rows)
bundle_copy_report_df

,bundle_section,source_dir,target_dir,n_files,status
0,mongo_configurations,/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_mongo_configurations,/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/exported_benchmark_bundles/ldbc_snb_sf0_1_mongo_benchmark_bundle/mongo_configurations,13,OK
1,final_analytical_matrix,/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_final_analytical_matrix_official,/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/exported_benchmark_bundles/ldbc_snb_sf0_1_mongo_benchmark_bundle/final_analytical_matrix,4,OK
2,activation,/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_activation_official,/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/exported_benchmark_bundles/ldbc_snb_sf0_1_mongo_benchmark_bundle/activation,7,OK
3,volatility,/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_update_volatility_official,/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/exported_benchmark_bundles/ldbc_snb_sf0_1_mongo_benchmark_bundle/volatility,11,OK
4,sharedness,/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_observed_sharedness_official,/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/exported_benchmark_bundles/ldbc_snb_sf0_1_mongo_benchmark_bundle/sharedness,7,OK
5,structural_reduction,/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/benchmark_artifacts_dir/ldbc_snb_structural_reduction_official,/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/exported_benchmark_bundles/ldbc_snb_sf0_1_mongo_benchmark_bundle/structural_reduction,4,OK


In [242]:
# =========================================================
# BLOCK 14C — Create final benchmark bundle manifest
# =========================================================
#
# Goal:
# - Create one high-level manifest for the exported bundle.
# =========================================================

bundle_manifest = {
    "bundle_name": BUNDLE_NAME,
    "dataset": "ldbc_snb",
    "scale_label": "sf0.1",
    "workload": "LDBC SNB Interactive v1 official-based subset",
    "created_at": datetime.utcnow().isoformat() + "Z",

    "methodology_stage": "after_candidate_generation_and_benchmark_plan_selection",

    "summary": {
        "n_final_queries": int(snb_final_analytical_matrix_df["query_name"].nunique())
        if "snb_final_analytical_matrix_df" in globals() else None,

        "n_mongodb_candidates": int(len(snb_mongodb_candidate_specs_df))
        if "snb_mongodb_candidate_specs_df" in globals() else None,

        "n_full_plan_rows": int(len(snb_benchmark_execution_plan_df))
        if "snb_benchmark_execution_plan_df" in globals() else None,

        "n_smoke_plan_rows": int(len(snb_smoke_benchmark_execution_plan_df))
        if "snb_smoke_benchmark_execution_plan_df" in globals() else None,
    },

    "key_files": {
        "candidate_specs_json": "mongo_configurations/mongodb_candidate_specs_by_candidate_id.json",
        "candidate_specs_overview": "mongo_configurations/mongodb_candidate_specs_overview.csv",
        "full_benchmark_plan": "mongo_configurations/benchmark_execution_plan.csv",
        "smoke_benchmark_plan": "mongo_configurations/benchmark_execution_plan_smoke.csv",
        "benchmark_manifest": "mongo_configurations/benchmark_manifest.json",

        "final_analytical_matrix": "final_analytical_matrix/snb_final_analytical_matrix.csv",
        "activation_matrix": "activation/snb_activation_matrix.csv",
        "activation_summary": "activation/snb_activation_summary.csv",
        "volatility_matrix": "volatility/matrix_with_update_volatility.csv",
        "sharedness_matrix": "sharedness/matrix_with_sharedness.csv",
    },

    "benchmark_groups": [
        "primary",
        "secondary_affected",
        "control",
    ],

    "notes": [
        "This bundle contains methodology outputs and benchmark execution plans.",
        "It does not contain raw LDBC SNB CSV data.",
        "The benchmark runner should receive this bundle plus the extracted SF0.1 CSV dataset.",
        "Use benchmark_execution_plan_smoke.csv first before running benchmark_execution_plan.csv.",
    ],
}

bundle_manifest_path = BUNDLE_DIR / "bundle_manifest.json"

with open(bundle_manifest_path, "w", encoding="utf-8") as f:
    json.dump(bundle_manifest, f, indent=2, ensure_ascii=False)

bundle_manifest

{'bundle_name': 'ldbc_snb_sf0_1_mongo_benchmark_bundle',
 'dataset': 'ldbc_snb',
 'scale_label': 'sf0.1',
 'workload': 'LDBC SNB Interactive v1 official-based subset',
 'created_at': '2026-05-08T18:17:58.126412Z',
 'methodology_stage': 'after_candidate_generation_and_benchmark_plan_selection',
 'summary': {'n_final_queries': 22,
  'n_mongodb_candidates': 64,
  'n_full_plan_rows': 64,
  'n_smoke_plan_rows': 24},
 'key_files': {'candidate_specs_json': 'mongo_configurations/mongodb_candidate_specs_by_candidate_id.json',
  'candidate_specs_overview': 'mongo_configurations/mongodb_candidate_specs_overview.csv',
  'full_benchmark_plan': 'mongo_configurations/benchmark_execution_plan.csv',
  'smoke_benchmark_plan': 'mongo_configurations/benchmark_execution_plan_smoke.csv',
  'benchmark_manifest': 'mongo_configurations/benchmark_manifest.json',
  'final_analytical_matrix': 'final_analytical_matrix/snb_final_analytical_matrix.csv',
  'activation_matrix': 'activation/snb_activation_matrix.csv',


In [243]:
# =========================================================
# BLOCK 14D — Create README for exported benchmark bundle
# =========================================================

bundle_readme = f"""# LDBC SNB SF0.1 MongoDB Benchmark Bundle

## 1. Purpose

This bundle contains the artifacts generated by the methodology pipeline for the LDBC SNB dataset.

It includes:

- final analytical matrix;
- G0–G9 activation matrix;
- MongoDB candidate specifications;
- full benchmark execution plan;
- smoke benchmark execution plan;
- validation files.

This bundle is intended to be copied to the benchmark server.

---

## 2. Dataset

Dataset: LDBC SNB Interactive v1  
Scale factor: SF0.1  
CSV serializer used in the notebook: CsvMergeForeign StringDateFormatter

The raw CSV files are not included in this bundle.

Expected dataset folder on the benchmark server:

```bash
/path/to/ldbc_snb/data/sf0.1

SyntaxError: incomplete input (580069515.py, line 5)

In [244]:
# =========================================================
# BLOCK 14F — Compress final benchmark bundle
# =========================================================
#
# Goal:
# - Create a .tar.gz file that can be copied to the benchmark server.
# =========================================================

archive_base = EXPORT_ROOT / BUNDLE_NAME

archive_path = shutil.make_archive(
    base_name=str(archive_base),
    format="gztar",
    root_dir=EXPORT_ROOT,
    base_dir=BUNDLE_NAME,
)

print("Compressed bundle created:")
print(archive_path)

print("\nSize:")
print(Path(archive_path).stat().st_size, "bytes")

Compressed bundle created:
/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/exported_benchmark_bundles/ldbc_snb_sf0_1_mongo_benchmark_bundle.tar.gz

Size:
11305190 bytes


In [245]:
# =========================================================
# BLOCK 14G — Save bundle validation and checkpoint
# =========================================================

# Save validation inside bundle
bundle_copy_report_df.to_csv(
    BUNDLE_DIR / "bundle_copy_report.csv",
    index=False
)

bundle_validation_df.to_csv(
    BUNDLE_DIR / "bundle_validation.csv",
    index=False
)

# Save checkpoint
CHECKPOINT_DIR = PROJECT_DIR / "benchmark_artifacts_dir" / "ldbc_snb_checkpoints"
CHECKPOINT_DATAFRAMES_DIR = CHECKPOINT_DIR / "dataframes"
CHECKPOINT_METADATA_DIR = CHECKPOINT_DIR / "metadata"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_METADATA_DIR.mkdir(parents=True, exist_ok=True)

manifest_path_checkpoint = CHECKPOINT_DIR / "checkpoint_manifest.json"
metadata_path = CHECKPOINT_METADATA_DIR / "checkpoint_metadata.json"

if manifest_path_checkpoint.exists():
    with open(manifest_path_checkpoint, "r", encoding="utf-8") as f:
        checkpoint_manifest = json.load(f)
else:
    checkpoint_manifest = {
        "dataframes": {},
        "metadata": str(metadata_path.relative_to(CHECKPOINT_DIR)),
    }

if metadata_path.exists():
    with open(metadata_path, "r", encoding="utf-8") as f:
        checkpoint_metadata = json.load(f)
else:
    checkpoint_metadata = {}

BLOCK_14_CHECKPOINT_DATAFRAMES = [
    "bundle_copy_report_df",
    "bundle_validation_df",
]

saved_block_14_dataframes = []

for var_name in BLOCK_14_CHECKPOINT_DATAFRAMES:
    obj = globals().get(var_name)

    if isinstance(obj, pd.DataFrame):
        out_path = CHECKPOINT_DATAFRAMES_DIR / f"{var_name}.json"

        obj.to_json(
            out_path,
            orient="records",
            indent=2,
            force_ascii=False
        )

        checkpoint_manifest["dataframes"][var_name] = str(
            out_path.relative_to(CHECKPOINT_DIR)
        )

        saved_block_14_dataframes.append(var_name)

checkpoint_metadata.update({
    "BASE_DIR": str(PROJECT_DIR / "data" / "sf0.1"),
    "BUNDLE_DIR": str(BUNDLE_DIR),
    "BUNDLE_ARCHIVE_PATH": str(archive_path),
    "bundle_manifest": bundle_manifest,
})

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(checkpoint_metadata, f, indent=2, ensure_ascii=False)

checkpoint_manifest["metadata"] = str(metadata_path.relative_to(CHECKPOINT_DIR))

with open(manifest_path_checkpoint, "w", encoding="utf-8") as f:
    json.dump(checkpoint_manifest, f, indent=2, ensure_ascii=False)

print("Block 14 checkpoint saved.")
print("Bundle directory:", BUNDLE_DIR)
print("Bundle archive:", archive_path)

print("\nSaved Block 14 DataFrames:")
for name in saved_block_14_dataframes:
    print(" -", name)

NameError: name 'bundle_validation_df' is not defined